In [ ]:
%pip install numpy pandas scikit-learn scipy lightgbm jupyter

## STA 9890 Prediction Competition
### To see all the models and related information in a scoreboard, scroll to the very end of the notebook

This notebook rebuilds the prediction workflow in a clearer linear order:

1. Data loading
2. Data preprocessing and inspection
3. Data engineering
4. Data modelling
5. Evaluation and final submission

The goal is to reproduce the original workflow and results while keeping each stage organized and readable.



# Data loading

This section loads the four instructor-provided data files.

No preprocessing, merging, feature engineering, or modelling is done here.

In [2]:
import pandas as pd
import numpy as np

In [ ]:
DATA_BASE_URL = "https://michael-weylandt.com/STA9890/competition_data"

scores_training = pd.read_csv(f"{DATA_BASE_URL}/scores_training.csv")
scores_test = pd.read_csv(f"{DATA_BASE_URL}/scores_test.csv")
school_covariates = pd.read_csv(f"{DATA_BASE_URL}/school_covariates.csv")
district_covariates = pd.read_csv(f"{DATA_BASE_URL}/district_covariates.csv")

In [4]:
raw_shapes = pd.DataFrame({
    "dataset": [
        "scores_training",
        "scores_test",
        "school_covariates",
        "district_covariates"
    ],
    "rows": [
        scores_training.shape[0],
        scores_test.shape[0],
        school_covariates.shape[0],
        district_covariates.shape[0]
    ],
    "columns": [
        scores_training.shape[1],
        scores_test.shape[1],
        school_covariates.shape[1],
        district_covariates.shape[1]
    ]
})

raw_shapes

,dataset,rows,columns
0,scores_training,144921,6
1,scores_test,48307,5
2,school_covariates,4754,52
3,district_covariates,674,6


In [5]:
print("scores_training:", scores_training.shape)
print("scores_test:", scores_test.shape)
print("school_covariates:", school_covariates.shape)
print("district_covariates:", district_covariates.shape)

scores_training: (144921, 6)
scores_test: (48307, 5)
school_covariates: (4754, 52)
district_covariates: (674, 6)


In [6]:
scores_training.head()

,ASSESSMENT_ID,SCHOOL,SUBGROUP_NAME,ASSESSMENT_NAME,N_STUDENTS,PERCENT_PROFICIENT
0,3b6deef53665,e037d064,Economically Disadvantaged,ELA4,130,38
1,962a3bfbfe84,5f633522,Male,ELA6,23,65
2,ffe086287b6e,2b76539e,All Students,Regents Algebra I,94,65
3,e6f80847409d,31289ced,All Students,MATH6,56,52
4,676cc6d81961,219db6e5,All Students,ELA8,134,46


In [7]:
scores_test.head()

,ASSESSMENT_ID,SCHOOL,SUBGROUP_NAME,ASSESSMENT_NAME,N_STUDENTS
0,8af5e0382a81,a49eed66,Not Economically Disadvantaged,Regents Phy Set/Earth Sci,5
1,e1591bf8db41,022f98d2,Economically Disadvantaged,MATH6,21
2,547ec44dcea6,255d51d5,All Students,MATH3,84
3,0e200399fc40,9442d8c4,Not Economically Disadvantaged,ELA7,7
4,c2c40438dac7,3f8e3d7a,Male,Regents US History&Gov't (Framework),44


In [8]:
school_covariates.head()

,SCHOOL,DISTRICT,COUNTY,DISTRICT_TYPE,REGION,ATTENDANCE_RATE,LANGUAGE_ARTS_AVERAGE_CLASS_SIZE,MATHEMATICS_AVERAGE_CLASS_SIZE,SCIENCE_AVERAGE_CLASS_SIZE,HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE,...,PERCENT_ASIAN,PERCENT_HISPANIC,PERCENT_WHITE,PERCENT_MULTIRACIAL,PERCENT_WITH_DISABILITIES,PERCENT_ECONOMICALLY_DISADVANTAGED,PERCENT_MIGRANT,PERCENT_HOMELESS,PERCENT_IN_FOSTER_CARE,PERCENT_PARENT_ARMED_FORCES
0,200cb9f4,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",95.0,23.000000,23.000000,21.0,NaN,...,13.0,11.0,43.0,11.0,10.0,36.0,0.0,1.0,0.0,0.0
1,fe6c45ae,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",92.0,12.000000,12.333333,12.0,NaN,...,15.0,22.0,19.0,8.0,15.0,76.0,0.0,4.0,2.0,0.0
2,edd71301,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",91.0,11.666667,11.666667,11.0,NaN,...,17.0,35.0,9.0,7.0,5.0,84.0,0.0,3.0,0.0,0.0
3,f3fc11ad,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",94.0,13.000000,13.000000,15.0,NaN,...,12.0,15.0,38.0,11.0,14.0,48.0,0.0,2.0,0.0,0.0
4,6f19e0a6,f5a6cc88,ec3f85d5,High-Need Urban/Suburban,"Capital District, New York",93.0,19.666667,19.666667,23.0,NaN,...,10.0,17.0,18.0,11.0,13.0,60.0,0.0,6.0,0.0,0.0


In [9]:
district_covariates.head()

,DISTRICT,PERCENT_DIPLOMA,PERCENT_NON_DIPLOMA,PERCENT_STILL_ENROLLED,PERCENT_GED,PERCENT_DROPOUT
0,f5a6cc88,69,2,18,0,11
1,0a4984df,90,1,4,0,4
2,56f2c546,96,1,1,0,2
3,e93bd0b0,84,2,7,0,7
4,b469f4b9,79,1,7,0,14


In [10]:
print("scores_training columns:")
print(scores_training.columns.tolist())

scores_training columns:
['ASSESSMENT_ID', 'SCHOOL', 'SUBGROUP_NAME', 'ASSESSMENT_NAME', 'N_STUDENTS', 'PERCENT_PROFICIENT']


In [11]:
print("scores_test columns:")
print(scores_test.columns.tolist())

scores_test columns:
['ASSESSMENT_ID', 'SCHOOL', 'SUBGROUP_NAME', 'ASSESSMENT_NAME', 'N_STUDENTS']


In [12]:
print("school_covariates columns:")
print(school_covariates.columns.tolist())

school_covariates columns:
['SCHOOL', 'DISTRICT', 'COUNTY', 'DISTRICT_TYPE', 'REGION', 'ATTENDANCE_RATE', 'LANGUAGE_ARTS_AVERAGE_CLASS_SIZE', 'MATHEMATICS_AVERAGE_CLASS_SIZE', 'SCIENCE_AVERAGE_CLASS_SIZE', 'HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE', 'GRADE_1_AVERAGE_CLASS_SIZE', 'GRADE_2_AVERAGE_CLASS_SIZE', 'KINDERGARTEN_AVERAGE_CLASS_SIZE', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'NUMBER_OF_TEACHERS', 'NUMBER_OF_COUNSELORS', 'NUMBER_OF_SOCIAL_WORKERS', 'TEACHER_TURNOVER_RATE', 'PERCENT_OF_STUDENTS_SUSPENDED', 'N_PUPILS', 'FEDERAL_FUNDING_PER_PUPIL', 'LOCAL_FUNDING_PER_PUPIL', 'PRE_K', 'K', 'GRADE_01', 'GRADE_02', 'GRADE_03', 'GRADE_04', 'GRADE_05', 'GRADE_06', 'GRADE_07', 'GRADE_08', 'GRADE_09', 'GRADE_10', 'GRADE_11', 'GRADE_12', 'PERCENT_MALE', 'PERCENT_FEMALE', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_AMERICAN_INDIAN', 'PERCENT_BLACK', 'PERCENT_ASIAN', 'PERCENT_HISPANIC', 'PERCENT_WHITE', 'PERCENT_MULTIRACIAL', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ECONOMICAL

In [13]:
print("district_covariates columns:")
print(district_covariates.columns.tolist())

district_covariates columns:
['DISTRICT', 'PERCENT_DIPLOMA', 'PERCENT_NON_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_GED', 'PERCENT_DROPOUT']


In [14]:
scores_training["PERCENT_PROFICIENT"].describe()

count    144921.000000
mean         54.183521
std          26.429925
min           0.000000
25%          33.000000
50%          53.000000
75%          76.000000
max         100.000000
Name: PERCENT_PROFICIENT, dtype: float64

In [15]:
"PERCENT_PROFICIENT" in scores_test.columns

False

The four raw files have been loaded successfully.

The training score file contains the target variable, `PERCENT_PROFICIENT`, while the test score file does not. The next section will merge the score files with school and district covariates, then inspect missingness and other preprocessing issues.

# Data preprocessing and inspection

This section prepares the raw files for feature construction.

The main steps are:

1. Standardize ID and categorical columns as text.
2. Merge score records with school and district covariates.
3. Inspect missing values.
4. Separate the target, IDs, and raw feature tables.

No feature engineering or modelling is done in this section.

In [16]:
text_cols = [
    "ASSESSMENT_ID",
    "SCHOOL",
    "DISTRICT",
    "COUNTY",
    "SUBGROUP_NAME",
    "ASSESSMENT_NAME",
    "DISTRICT_TYPE",
    "REGION"
]

for df in [scores_training, scores_test, school_covariates, district_covariates]:
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

In [17]:
print("SCHOOL unique in school_covariates:")
print(school_covariates["SCHOOL"].is_unique)

print("\nDISTRICT unique in district_covariates:")
print(district_covariates["DISTRICT"].is_unique)

SCHOOL unique in school_covariates:
True

DISTRICT unique in district_covariates:
True


Every row in both score files has a matching `SCHOOL` in the school covariate table. This means the school-level merge should not create missing school information.

In [18]:
train_full = scores_training.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

test_full = scores_test.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

train_full = train_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

test_full = test_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

print("train_full shape:", train_full.shape)
print("test_full shape:", test_full.shape)

train_full shape: (144921, 62)
test_full shape: (48307, 61)


In [19]:
train_missing = train_full.isna().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)

train_missing = pd.DataFrame({
    "missing_count": train_missing,
    "missing_pct": 100 * train_missing / len(train_full)
})

test_missing = test_full.isna().sum()
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)

test_missing = pd.DataFrame({
    "missing_count": test_missing,
    "missing_pct": 100 * test_missing / len(test_full)
})

print("Train missing columns:", train_missing.shape[0])
print("Test missing columns:", test_missing.shape[0])

Train missing columns: 52
Test missing columns: 52


In [20]:
train_missing.head(15)

,missing_count,missing_pct
TEACHER_TURNOVER_RATE,134136,92.558014
KINDERGARTEN_AVERAGE_CLASS_SIZE,103467,71.395450
GRADE_1_AVERAGE_CLASS_SIZE,102899,71.003512
GRADE_2_AVERAGE_CLASS_SIZE,102779,70.920709
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE,89314,61.629439
PERCENT_DROPOUT,16061,11.082590
PERCENT_GED,16061,11.082590
PERCENT_STILL_ENROLLED,16061,11.082590
PERCENT_NON_DIPLOMA,16061,11.082590
PERCENT_DIPLOMA,16061,11.082590


## Highest missingness rate:

TEACHER_TURNOVER_RATE<br>
KINDERGARTEN_AVERAGE_CLASS_SIZE<br>
GRADE_1_AVERAGE_CLASS_SIZE<br>
GRADE_2_AVERAGE_CLASS_SIZE<br>
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE<br><br>


The same broad missingness pattern appears in both training and test data.

Several class-size and teacher-turnover variables have high missingness. District-level graduation variables also have missing values for some observations.

These variables are not dropped here. Missing-value handling belongs in the data engineering section, where missingness indicators and imputation will be applied consistently to the training and test feature tables.

In [21]:
TARGET = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

train_ids = train_full[ID_COL].copy()
test_ids = test_full[ID_COL].copy()

y_train = train_full[TARGET].copy()

X_train = train_full.drop(columns=[TARGET])
X_test = test_full.copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (144921, 61)
X_test shape: (48307, 61)
y_train shape: (144921,)


In [22]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

if ID_COL in categorical_cols:
    categorical_cols.remove(ID_COL)

high_cardinality_cols = [
    col for col in categorical_cols
    if X_train[col].nunique() > 50
]

low_cardinality_cols = [
    col for col in categorical_cols
    if col not in high_cardinality_cols
]

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

print("\nHigh-cardinality categorical columns:")
print(high_cardinality_cols)

print("\nLow-cardinality categorical columns:")
print(low_cardinality_cols)

Numeric columns: 53
Categorical columns: 7

High-cardinality categorical columns:
['SCHOOL', 'DISTRICT', 'COUNTY']

Low-cardinality categorical columns:
['SUBGROUP_NAME', 'ASSESSMENT_NAME', 'DISTRICT_TYPE', 'REGION']


In [23]:
pd.DataFrame({
    "column": categorical_cols,
    "unique_values": [X_train[col].nunique() for col in categorical_cols],
    "missing_train": [X_train[col].isna().sum() for col in categorical_cols],
    "missing_test": [X_test[col].isna().sum() for col in categorical_cols]
})

,column,unique_values,missing_train,missing_test
0,SCHOOL,4469,0,0
1,SUBGROUP_NAME,5,0,0
2,ASSESSMENT_NAME,32,0,0
3,DISTRICT,710,0,0
4,COUNTY,62,0,0
5,DISTRICT_TYPE,7,0,0
6,REGION,10,0,0


The raw feature tables are now ready for data engineering.

The high-cardinality categorical variables are `SCHOOL`, `DISTRICT`, and `COUNTY`.

The low-cardinality categorical variables are `SUBGROUP_NAME`, `ASSESSMENT_NAME`, `DISTRICT_TYPE`, and `REGION`.

The next section will convert these raw feature tables into model-ready numeric feature matrices.

# Data engineering, Part 1: Base feature matrix

This section converts the raw merged feature tables into a numeric feature matrix.

The base feature matrix uses three simple transformations:

1. Frequency encoding for high-cardinality categorical variables:
   - `SCHOOL`
   - `DISTRICT`
   - `COUNTY`

2. Missing-value handling for numeric variables:
   - add one missingness indicator per numeric column with missing values
   - fill missing numeric values with the training median

3. One-hot encoding for low-cardinality categorical variables:
   - `SUBGROUP_NAME`
   - `ASSESSMENT_NAME`
   - `DISTRICT_TYPE`
   - `REGION`

The identifier column `ASSESSMENT_ID` is kept during feature construction and removed only at the end.

In [24]:
X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

print("Starting training shape:", X_train_proc.shape)
print("Starting test shape:", X_test_proc.shape)

Starting training shape: (144921, 61)
Starting test shape: (48307, 61)


In [25]:
freq_encoding_cols = []

for col in high_cardinality_cols:
    # Frequencies are learned from the training data only.
    # Unseen test categories receive frequency 0.
    freq_map = X_train_proc[col].value_counts(dropna=False)

    new_col = col + "_freq"

    X_train_proc[new_col] = X_train_proc[col].map(freq_map).astype(float)
    X_test_proc[new_col] = X_test_proc[col].map(freq_map).fillna(0).astype(float)

    freq_encoding_cols.append(new_col)

print("Frequency-encoded columns:")
print(freq_encoding_cols)

print("\nShape after frequency encoding:")
print("X_train_proc:", X_train_proc.shape)
print("X_test_proc:", X_test_proc.shape)

Frequency-encoded columns:
['SCHOOL_freq', 'DISTRICT_freq', 'COUNTY_freq']

Shape after frequency encoding:
X_train_proc: (144921, 64)
X_test_proc: (48307, 64)


In [26]:
numeric_cols_extended = numeric_cols + freq_encoding_cols

missing_value_cols = [
    col for col in numeric_cols_extended
    if X_train_proc[col].isna().sum() > 0
]

print("Numeric columns checked:", len(numeric_cols_extended))
print("Numeric columns with missing values:", len(missing_value_cols))

Numeric columns checked: 56
Numeric columns with missing values: 52


In [27]:
train_missing_indicators = pd.DataFrame(
    {
        col + "_missing": X_train_proc[col].isna().astype(int)
        for col in missing_value_cols
    },
    index=X_train_proc.index
)

test_missing_indicators = pd.DataFrame(
    {
        col + "_missing": X_test_proc[col].isna().astype(int)
        for col in missing_value_cols
    },
    index=X_test_proc.index
)

missing_indicator_cols = train_missing_indicators.columns.tolist()

print("Missing indicator columns created:", len(missing_indicator_cols))
print("First five missing indicator columns:")
print(missing_indicator_cols[:5])

Missing indicator columns created: 52
First five missing indicator columns:
['ATTENDANCE_RATE_missing', 'LANGUAGE_ARTS_AVERAGE_CLASS_SIZE_missing', 'MATHEMATICS_AVERAGE_CLASS_SIZE_missing', 'SCIENCE_AVERAGE_CLASS_SIZE_missing', 'HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE_missing']


In [28]:
for col in missing_value_cols:
    # Median is learned from training data only, then applied to both train and test.
    median_value = X_train_proc[col].median()

    X_train_proc[col] = X_train_proc[col].fillna(median_value)
    X_test_proc[col] = X_test_proc[col].fillna(median_value)

X_train_proc = pd.concat(
    [X_train_proc, train_missing_indicators],
    axis=1
)

X_test_proc = pd.concat(
    [X_test_proc, test_missing_indicators],
    axis=1
)

print("Shape after missing indicators and median imputation:")
print("X_train_proc:", X_train_proc.shape)
print("X_test_proc:", X_test_proc.shape)

Shape after missing indicators and median imputation:
X_train_proc: (144921, 116)
X_test_proc: (48307, 116)


In [29]:
X_train_proc = X_train_proc.drop(columns=high_cardinality_cols)
X_test_proc = X_test_proc.drop(columns=high_cardinality_cols)

print("Dropped raw high-cardinality columns:")
print(high_cardinality_cols)

print("\nShape after dropping raw high-cardinality columns:")
print("X_train_proc:", X_train_proc.shape)
print("X_test_proc:", X_test_proc.shape)

Dropped raw high-cardinality columns:
['SCHOOL', 'DISTRICT', 'COUNTY']

Shape after dropping raw high-cardinality columns:
X_train_proc: (144921, 113)
X_test_proc: (48307, 113)


In [30]:
X_train_proc = pd.get_dummies(
    X_train_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

X_test_proc = pd.get_dummies(
    X_test_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

# Align columns so that train and test have exactly the same feature set.
X_train_proc, X_test_proc = X_train_proc.align(
    X_test_proc,
    join="left",
    axis=1,
    fill_value=0
)

print("Shape after one-hot encoding:")
print("X_train_proc:", X_train_proc.shape)
print("X_test_proc:", X_test_proc.shape)

Shape after one-hot encoding:
X_train_proc: (144921, 163)
X_test_proc: (48307, 163)


In [31]:
X_train_proc_model = X_train_proc.drop(columns=[ID_COL]).copy()
X_test_proc_model = X_test_proc.drop(columns=[ID_COL]).copy()

print("Final base modeling matrix:")
print("X_train_proc_model:", X_train_proc_model.shape)
print("X_test_proc_model:", X_test_proc_model.shape)

Final base modeling matrix:
X_train_proc_model: (144921, 162)
X_test_proc_model: (48307, 162)


In [32]:
print("Remaining missing values:")
print("X_train_proc_model:", X_train_proc_model.isna().sum().sum())
print("X_test_proc_model:", X_test_proc_model.isna().sum().sum())

print("\nTrain/test columns aligned:")
print(list(X_train_proc_model.columns) == list(X_test_proc_model.columns))

print("\nAll training columns numeric or boolean:")
print(all(pd.api.types.is_numeric_dtype(X_train_proc_model[col]) for col in X_train_proc_model.columns))

Remaining missing values:
X_train_proc_model: 0
X_test_proc_model: 0

Train/test columns aligned:
True

All training columns numeric or boolean:
True


The base feature matrix is now complete.

The final base modeling tables are:

- `X_train_proc_model`
- `X_test_proc_model`

These contain frequency encodings, missingness indicators, median-imputed numeric variables, and one-hot encoded low-cardinality categorical variables.

The raw identifier `ASSESSMENT_ID` has been removed from the modeling matrix, but the saved `train_ids` and `test_ids` objects are still available for diagnostics and final submission files.

# Data engineering, Part 2: Target/statistical encoding setup

The original notebook introduced target/statistical encoding later in the modelling workflow.

In this cleaned notebook, the target-encoding setup is moved into data engineering because it creates features. However, the modelling motivation is preserved: this branch is treated as an advanced feature branch, not part of the base preprocessing.

This first subsection does not create target encodings yet. It only checks whether the data structure supports them.

### Part 2A: Feasibility audit for target/statistical encoding

Target/statistical encoding uses historical target behavior within groups such as assessment, subgroup, school, district, county, and combinations of these variables.

Before creating those features, we check:

- whether the raw grouping columns are available,
- whether the target is bounded between 0 and 100,
- whether `N_STUDENTS` can be used as a reliability proxy,
- which candidate grouping keys have useful train/test overlap,
- and which keys are too sparse to be reliable.

In [33]:
raw_train_te = train_full.reset_index(drop=True).copy()
raw_test_te = test_full.reset_index(drop=True).copy()

y_arr_te = np.asarray(y_train, dtype=np.float32).reshape(-1)

print("raw_train_te shape:", raw_train_te.shape)
print("raw_test_te shape:", raw_test_te.shape)
print("y_arr_te shape:", y_arr_te.shape)

raw_train_te shape: (144921, 62)
raw_test_te shape: (48307, 61)
y_arr_te shape: (144921,)


In [34]:
print("Alignment checks:")
print("train_full rows match y_train:", len(raw_train_te) == len(y_arr_te))
print("X_train_proc_model rows match y_train:", X_train_proc_model.shape[0] == len(y_arr_te))
print("X_test_proc_model rows match raw_test_te:", X_test_proc_model.shape[0] == len(raw_test_te))
print("test_ids rows match raw_test_te:", len(test_ids) == len(raw_test_te))

Alignment checks:
train_full rows match y_train: True
X_train_proc_model rows match y_train: True
X_test_proc_model rows match raw_test_te: True
test_ids rows match raw_test_te: True


In [35]:
important_te_cols = [
    ID_COL,
    "SCHOOL",
    "DISTRICT",
    "COUNTY",
    "REGION",
    "DISTRICT_TYPE",
    "SUBGROUP_NAME",
    "ASSESSMENT_NAME",
    "N_STUDENTS",
    TARGET,
]

availability_rows = []

for col in important_te_cols:
    availability_rows.append({
        "column": col,
        "in_train": col in raw_train_te.columns,
        "in_test": col in raw_test_te.columns,
        "train_missing_pct": (
            100 * raw_train_te[col].isna().mean()
            if col in raw_train_te.columns
            else np.nan
        ),
        "test_missing_pct": (
            100 * raw_test_te[col].isna().mean()
            if col in raw_test_te.columns
            else np.nan
        ),
        "train_unique": (
            raw_train_te[col].nunique(dropna=False)
            if col in raw_train_te.columns
            else np.nan
        ),
        "test_unique": (
            raw_test_te[col].nunique(dropna=False)
            if col in raw_test_te.columns
            else np.nan
        ),
    })

te_column_availability = pd.DataFrame(availability_rows)

te_column_availability

,column,in_train,in_test,train_missing_pct,test_missing_pct,train_unique,test_unique
0,ASSESSMENT_ID,True,True,0.0,0.0,144921,48307.0
1,SCHOOL,True,True,0.0,0.0,4469,4448.0
2,DISTRICT,True,True,0.0,0.0,710,707.0
3,COUNTY,True,True,0.0,0.0,62,62.0
4,REGION,True,True,0.0,0.0,10,10.0
5,DISTRICT_TYPE,True,True,0.0,0.0,7,7.0
6,SUBGROUP_NAME,True,True,0.0,0.0,5,5.0
7,ASSESSMENT_NAME,True,True,0.0,0.0,32,32.0
8,N_STUDENTS,True,True,0.0,0.0,736,594.0
9,PERCENT_PROFICIENT,True,False,0.0,NaN,101,NaN


In [36]:
pd.Series(y_arr_te).describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    144921.000000
mean         54.183521
std          26.429926
min           0.000000
1%            0.000000
5%           13.000000
25%          33.000000
50%          53.000000
75%          76.000000
95%          98.000000
99%         100.000000
max         100.000000
dtype: float64

In [37]:
print("Target boundary checks:")
print("Share target == 0:", float((y_arr_te == 0).mean()))
print("Share target == 100:", float((y_arr_te == 100).mean()))
print("Share target < 0:", float((y_arr_te < 0).mean()))
print("Share target > 100:", float((y_arr_te > 100).mean()))
print("Share integer-like:", float(np.isclose(y_arr_te, np.round(y_arr_te), atol=1e-9).mean()))
print("Number of unique target values:", int(pd.Series(y_arr_te).nunique()))

Target boundary checks:
Share target == 0: 0.01107499948247666
Share target == 100: 0.04582496670599844
Share target < 0: 0.0
Share target > 100: 0.0
Share integer-like: 1.0
Number of unique target values: 101


This confirms the target is bounded. The exact shares at 0 and 100 can remain as output in the notebook.

In [38]:
n_students_train = raw_train_te["N_STUDENTS"].astype(float)
n_students_test = raw_test_te["N_STUDENTS"].astype(float)

n_students_train.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    144921.000000
mean         57.664748
std          67.211682
min           5.000000
1%            5.000000
5%            8.000000
25%          21.000000
50%          38.000000
75%          69.000000
95%         173.000000
99%         336.000000
max        1683.000000
Name: N_STUDENTS, dtype: float64

In [39]:
valid_count_rows = (
    n_students_train.notna()
    & (n_students_train > 0)
    & pd.Series(y_arr_te).notna()
)

n_valid = n_students_train[valid_count_rows].to_numpy(dtype=float)
y_valid = y_arr_te[valid_count_rows]

# If the target is a reported percent, then y / 100 * N_STUDENTS
# should often be close to an integer count of proficient students.
nearest_proficient_count = np.rint((y_valid / 100.0) * n_valid)
nearest_count_percent = 100.0 * nearest_proficient_count / n_valid
nearest_count_percent_rounded = np.rint(nearest_count_percent)

exact_count_diff = np.abs(y_valid - nearest_count_percent)
rounded_count_diff = np.abs(y_valid - nearest_count_percent_rounded)

print("Rounded-count plausibility:")
print("Valid rows checked:", int(valid_count_rows.sum()))
print("Median absolute difference from nearest exact count percent:", float(np.median(exact_count_diff)))
print("Share compatible with rounded whole-percent count:", float((rounded_count_diff <= 1e-9).mean()))
print("Share within 0.5 percentage points of nearest count percent:", float((exact_count_diff <= 0.5).mean()))
print("Share within 1.0 percentage point of nearest count percent:", float((exact_count_diff <= 1.0).mean()))

Rounded-count plausibility:
Valid rows checked: 144921
Median absolute difference from nearest exact count percent: 0.21739130434782794
Share compatible with rounded whole-percent count: 0.9889319008287274
Share within 0.5 percentage points of nearest count percent: 1.0
Share within 1.0 percentage point of nearest count percent: 1.0


In [40]:
n_students_test.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    48307.000000
mean        58.095576
std         68.502434
min          5.000000
1%           5.000000
5%           8.000000
25%         21.000000
50%         38.000000
75%         69.000000
95%        174.000000
99%        342.000000
max       1514.000000
Name: N_STUDENTS, dtype: float64

In [41]:
raw_train_te["_N_STUDENTS_BIN_TE"] = pd.cut(
    raw_train_te["N_STUDENTS"].astype(float),
    bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
    labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
).astype("string").fillna("<NA>")

raw_test_te["_N_STUDENTS_BIN_TE"] = pd.cut(
    raw_test_te["N_STUDENTS"].astype(float),
    bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
    labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
).astype("string").fillna("<NA>")

print("N_STUDENTS bins in training:")
print(raw_train_te["_N_STUDENTS_BIN_TE"].value_counts().sort_index())

print("\nN_STUDENTS bins in test:")
print(raw_test_te["_N_STUDENTS_BIN_TE"].value_counts().sort_index())

N_STUDENTS bins in training:
_N_STUDENTS_BIN_TE
11-20     23188
21-50     55170
51-100    34123
6-10      10522
<=5        2029
>100      19889
Name: count, dtype: Int64

N_STUDENTS bins in test:
_N_STUDENTS_BIN_TE
11-20      7687
21-50     18529
51-100    11442
6-10       3350
<=5         639
>100       6660
Name: count, dtype: Int64


A count table for the six bins

In [42]:
def make_group_key(df, cols):
    """
    Create a single grouping key from one or more categorical columns.

    This helper is used in the key-feasibility audit and reused later
    when building leakage-safe target/statistical encodings.
    """
    cols = tuple(cols)

    if len(cols) == 1:
        return (
            df[cols[0]]
            .astype("string")
            .fillna("<NA>")
            .astype(str)
            .reset_index(drop=True)
        )

    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
        .reset_index(drop=True)
    )

In [43]:
candidate_te_keys = [
    # Single-column keys
    ("SCHOOL",),
    ("DISTRICT",),
    ("COUNTY",),
    ("REGION",),
    ("DISTRICT_TYPE",),
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),

    # Two-column keys
    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "SUBGROUP_NAME"),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME"),
    ("COUNTY", "SUBGROUP_NAME"),
    ("REGION", "ASSESSMENT_NAME"),
    ("DISTRICT_TYPE", "ASSESSMENT_NAME"),

    # Three-column hierarchical keys
    ("SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"),

    # Student-count interaction keys
    ("ASSESSMENT_NAME", "_N_STUDENTS_BIN_TE"),
    ("SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"),
]

print("Candidate target/statistical encoding keys:", len(candidate_te_keys))

Candidate target/statistical encoding keys: 22


In [44]:
te_key_audit_rows = []

for cols in candidate_te_keys:
    if not all(col in raw_train_te.columns for col in cols):
        continue

    if not all(col in raw_test_te.columns for col in cols):
        continue

    train_key = make_group_key(raw_train_te, cols)
    test_key = make_group_key(raw_test_te, cols)

    train_counts = train_key.value_counts(dropna=False)
    train_groups = set(train_counts.index)

    test_seen = test_key.isin(train_groups)

    te_key_audit_rows.append({
        "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
        "columns": cols,
        "train_groups": int(train_counts.shape[0]),
        "test_groups": int(test_key.nunique(dropna=False)),
        "test_row_coverage": float(test_seen.mean()),
        "median_train_count": float(train_counts.median()),
        "mean_train_count": float(train_counts.mean()),
        "p90_train_count": float(train_counts.quantile(0.90)),
        "p99_train_count": float(train_counts.quantile(0.99)),
        "singleton_group_share": float((train_counts == 1).mean()),
        "max_train_count": int(train_counts.max()),
    })

te_key_audit = pd.DataFrame(te_key_audit_rows)

te_key_audit = te_key_audit.sort_values(
    ["test_row_coverage", "median_train_count", "train_groups"],
    ascending=[False, False, True]
).reset_index(drop=True)

te_key_audit

,key,columns,train_groups,test_groups,test_row_coverage,median_train_count,mean_train_count,p90_train_count,p99_train_count,singleton_group_share,max_train_count
0,SUBGROUP_NAME,"(SUBGROUP_NAME,)",5,5,1.000000,29110.0,28984.200000,33771.8,36417.08,0.000000,36711
1,DISTRICT_TYPE,"(DISTRICT_TYPE,)",7,7,1.000000,13986.0,20703.000000,42308.8,43986.58,0.000000,44173
2,REGION,"(REGION,)",10,10,1.000000,9424.0,14492.100000,21815.9,51955.19,0.000000,55304
3,ASSESSMENT_NAME,"(ASSESSMENT_NAME,)",32,32,1.000000,4532.0,4528.781250,8296.4,8389.21,0.000000,8392
4,SUBGROUP_NAME x N_STUDENTS_BIN,"(SUBGROUP_NAME, _N_STUDENTS_BIN_TE)",30,30,1.000000,3874.5,4830.700000,10516.9,13959.36,0.000000,13964
5,ASSESSMENT_NAME x SUBGROUP_NAME,"(ASSESSMENT_NAME, SUBGROUP_NAME)",132,132,1.000000,1004.5,1097.886364,1771.3,1837.76,0.000000,1883
6,COUNTY,"(COUNTY,)",62,62,1.000000,958.0,2337.435484,7086.5,14432.88,0.000000,17112
7,ASSESSMENT_NAME x N_STUDENTS_BIN,"(ASSESSMENT_NAME, _N_STUDENTS_BIN_TE)",191,188,1.000000,470.0,758.748691,1786.0,3972.70,0.010471,3985
8,DISTRICT_TYPE x ASSESSMENT_NAME,"(DISTRICT_TYPE, ASSESSMENT_NAME)",222,221,1.000000,438.5,652.797297,1523.2,2507.06,0.009009,2545
9,REGION x ASSESSMENT_NAME,"(REGION, ASSESSMENT_NAME)",315,314,1.000000,295.0,460.066667,960.8,3118.04,0.000000,3140


In [45]:
initially_usable_te_keys = te_key_audit[
    (te_key_audit["test_row_coverage"] >= 0.80)
    & (te_key_audit["median_train_count"] >= 2)
].copy()

initially_usable_te_keys[[
    "key",
    "test_row_coverage",
    "train_groups",
    "median_train_count",
    "singleton_group_share",
    "p90_train_count",
    "p99_train_count"
]]

,key,test_row_coverage,train_groups,median_train_count,singleton_group_share,p90_train_count,p99_train_count
0,SUBGROUP_NAME,1.000000,5,29110.0,0.000000,33771.8,36417.08
1,DISTRICT_TYPE,1.000000,7,13986.0,0.000000,42308.8,43986.58
2,REGION,1.000000,10,9424.0,0.000000,21815.9,51955.19
3,ASSESSMENT_NAME,1.000000,32,4532.0,0.000000,8296.4,8389.21
4,SUBGROUP_NAME x N_STUDENTS_BIN,1.000000,30,3874.5,0.000000,10516.9,13959.36
5,ASSESSMENT_NAME x SUBGROUP_NAME,1.000000,132,1004.5,0.000000,1771.3,1837.76
6,COUNTY,1.000000,62,958.0,0.000000,7086.5,14432.88
7,ASSESSMENT_NAME x N_STUDENTS_BIN,1.000000,191,470.0,0.010471,1786.0,3972.70
8,DISTRICT_TYPE x ASSESSMENT_NAME,1.000000,222,438.5,0.009009,1523.2,2507.06
9,REGION x ASSESSMENT_NAME,1.000000,315,295.0,0.000000,960.8,3118.04


The feasibility audit shows which grouping keys are plausible for statistical encoding.

Broad keys, such as assessment, subgroup, region, district type, county, and district, have strong train/test overlap and repeated observations.

More specific keys, such as school-by-assessment and school-by-subgroup, are sparser but may still contain useful local signal when heavily smoothed.

The sparsest three-way keys are diagnostic only unless they have enough coverage and repetition. They should not automatically be used just because they can be constructed.

### Part 2B: Original target-encoding specification

The previous audit motivates which grouping structures are usable.

The following smoothing values are carried over from the original target/statistical encoding experiment. They are pseudo-count shrinkage constants, not fitted model parameters and not values derived by a closed-form rule.

They are kept unchanged because this refactor aims to reproduce the original notebook’s feature branch and results.

Target encoding creates numeric features from categorical groups.

For example, instead of using the raw school name directly, we can summarize how students from that school performed in the training data. Similar summaries can be made for assessment type, subgroup, district, county, and combinations such as `SCHOOL x ASSESSMENT_NAME`.

However, small groups can be noisy. If a school-assessment group has only a few rows, its average score may be unusually high or low just by chance. To reduce this problem, the group average is smoothed toward the overall average.

The smoothed mean uses the following idea:

`smoothed mean = (group total + alpha × overall mean) / (group count + alpha)`

Here, `alpha` controls how strongly the group average is pulled toward the overall average.

A small alpha means the group’s own average matters more.
A large alpha means the group is pulled more strongly toward the overall average.

The smoothing values are larger for more detailed groups because detailed groups usually have fewer observations.

The same idea is also used for a student-count-weighted average. That version uses `alpha_n`, which acts like a pseudo-student count. These values are larger because they work on the student-count scale rather than the row-count scale.

The exact smoothing values used below are carried over from the original target-encoding experiment so that this cleaned notebook can reproduce the same feature branch. They should be understood as practical shrinkage settings, not values derived from a closed-form formula.

In [46]:

# Original target/statistical encoding specification.
#
# These grouping keys were selected after the feasibility audit above.
#
# alpha:
#   smoothing strength for the ordinary group average
#
# alpha_n:
#   smoothing strength for the student-count-weighted group average
#
# Larger values apply stronger shrinkage toward the overall average.
# The exact values are preserved from the original target-encoding branch.

te_key_specs = [
    (("ASSESSMENT_NAME",), 20.0, 200.0),
    (("SUBGROUP_NAME",), 20.0, 200.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 20.0, 200.0),
    (("ASSESSMENT_NAME", "_N_STUDENTS_BIN_TE"), 30.0, 250.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"), 50.0, 300.0),

    (("COUNTY",), 30.0, 250.0),
    (("COUNTY", "SUBGROUP_NAME"), 40.0, 300.0),
    (("COUNTY", "ASSESSMENT_NAME"), 60.0, 400.0),
    (("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"), 100.0, 600.0),

    (("REGION", "ASSESSMENT_NAME"), 40.0, 300.0),
    (("DISTRICT_TYPE", "ASSESSMENT_NAME"), 40.0, 300.0),

    (("DISTRICT",), 60.0, 400.0),
    (("DISTRICT", "SUBGROUP_NAME"), 90.0, 500.0),
    (("DISTRICT", "ASSESSMENT_NAME"), 140.0, 700.0),

    (("SCHOOL",), 100.0, 600.0),
    (("SCHOOL", "SUBGROUP_NAME"), 140.0, 800.0),
    (("SCHOOL", "ASSESSMENT_NAME"), 220.0, 1000.0),
]

te_key_names = [
    " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN")
    for cols, _, _ in te_key_specs
]

te_spec_review = pd.DataFrame({
    "key": te_key_names,
    "alpha": [alpha for _, alpha, _ in te_key_specs],
    "alpha_n": [alpha_n for _, _, alpha_n in te_key_specs],
})

te_spec_review = te_spec_review.merge(
    te_key_audit.drop(columns=["columns"]),
    on="key",
    how="left"
)

te_spec_review[[
    "key",
    "alpha",
    "alpha_n",
    "test_row_coverage",
    "median_train_count",
    "singleton_group_share"
]]

,key,alpha,alpha_n,test_row_coverage,median_train_count,singleton_group_share
0,ASSESSMENT_NAME,20.0,200.0,1.000000,4532.0,0.000000
1,SUBGROUP_NAME,20.0,200.0,1.000000,29110.0,0.000000
2,ASSESSMENT_NAME x SUBGROUP_NAME,20.0,200.0,1.000000,1004.5,0.000000
3,ASSESSMENT_NAME x N_STUDENTS_BIN,30.0,250.0,1.000000,470.0,0.010471
4,ASSESSMENT_NAME x SUBGROUP_NAME x N_STUDENTS_BIN,50.0,300.0,0.999959,133.0,0.010139
5,COUNTY,30.0,250.0,1.000000,958.0,0.000000
6,COUNTY x SUBGROUP_NAME,40.0,300.0,1.000000,197.5,0.000000
7,COUNTY x ASSESSMENT_NAME,60.0,400.0,0.999834,31.0,0.010384
8,COUNTY x ASSESSMENT_NAME x SUBGROUP_NAME,100.0,600.0,0.997619,7.0,0.034279
9,REGION x ASSESSMENT_NAME,40.0,300.0,1.000000,295.0,0.000000


The selected grouping keys are supported by the feasibility audit.

The exact smoothing constants are preserved from the original target/statistical encoding experiment. They act as pseudo-count shrinkage values: larger values pull sparse group estimates more strongly toward the global target mean.

Because these constants were not re-tuned in this cleaned notebook, they should be interpreted as part of the original experimental configuration rather than newly derived quantities.

In [47]:
print("Selected target/statistical encoding keys:", len(te_key_specs))
print("Features per selected key:", 4)
print("Total target/statistical encoding features:", len(te_key_specs) * 4)

Selected target/statistical encoding keys: 17
Features per selected key: 4
Total target/statistical encoding features: 68


### Part 2C: Build leakage-safe target/statistical encoding features

Target encoding must be built carefully.

If a row is encoded using a group average that includes its own target value, the model can indirectly see the answer. That would make validation performance look better than it really is.

To avoid this, the training rows are encoded out-of-fold:

1. Split the training rows into folds.
2. For each fold, compute group statistics using the other folds only.
3. Apply those statistics to the held-out fold.
4. Repeat until every training row has an encoding that did not use its own target value.

For validation or test rows, the encoding is learned from the available training rows and then applied to the new rows.

This cell builds the reusable functions for that process, then runs a holdout construction diagnostic.

In [48]:
from sklearn.model_selection import train_test_split, KFold
import re

In [49]:
RANDOM_STATE = 9890
N_SPLITS = 5

np.random.seed(RANDOM_STATE)

In [50]:
def safe_key_name(cols):
    """
    Convert grouping column names into a safe feature-name fragment.
    """
    name = "__".join(cols)
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

In [51]:
def fit_group_stats(keys, y, n_students, alpha, alpha_n):
    """
    Learn smoothed group statistics from training rows only.

    The ordinary smoothed mean uses row counts.

    The weighted smoothed mean uses N_STUDENTS so that larger tested groups
    contribute more information than very small tested groups.
    """
    y = np.asarray(y, dtype=np.float64)
    n_students = np.asarray(n_students, dtype=np.float64)

    valid_n = np.isfinite(n_students) & (n_students > 0)

    if not valid_n.all():
        fill_value = np.nanmedian(n_students[valid_n]) if valid_n.any() else 1.0
        n_students = np.where(valid_n, n_students, fill_value)

    y_clipped = np.clip(y, 0.0, 100.0)

    proficient_counts = np.rint((y_clipped / 100.0) * n_students)
    proficient_counts = np.clip(proficient_counts, 0.0, n_students)

    global_mean = float(np.mean(y))
    global_std = float(np.std(y, ddof=0))
    global_weighted_mean = float(
        100.0 * proficient_counts.sum() / max(n_students.sum(), 1.0)
    )

    group_data = pd.DataFrame({
        "key": pd.Series(keys).astype(str).to_numpy(),
        "target": y,
        "n_students": n_students,
        "proficient_count": proficient_counts,
    })

    group_stats = group_data.groupby("key", sort=False).agg(
        count=("target", "size"),
        target_sum=("target", "sum"),
        target_std=("target", "std"),
        n_students_sum=("n_students", "sum"),
        proficient_sum=("proficient_count", "sum"),
    )

    group_stats["mean_smooth"] = (
        group_stats["target_sum"] + alpha * global_mean
    ) / (
        group_stats["count"] + alpha
    )

    group_stats["weighted_mean_smooth"] = 100.0 * (
        group_stats["proficient_sum"] + alpha_n * (global_weighted_mean / 100.0)
    ) / (
        group_stats["n_students_sum"] + alpha_n
    )

    group_stats["target_std"] = group_stats["target_std"].fillna(global_std)
    group_stats["log_count"] = np.log1p(group_stats["count"].astype(float))

    defaults = {
        "global_mean": global_mean,
        "global_weighted_mean": global_weighted_mean,
        "global_std": global_std,
    }

    return group_stats, defaults

In [52]:
def apply_group_stats(keys, group_stats, defaults, prefix):
    """
    Apply fitted group statistics to another set of rows.

    Rows with unseen groups receive global fallback values.
    """
    keys = pd.Series(keys).astype(str).reset_index(drop=True)

    encoded = pd.DataFrame(index=np.arange(len(keys)))

    encoded[f"{prefix}_mean"] = (
        keys.map(group_stats["mean_smooth"])
        .fillna(defaults["global_mean"])
        .astype(np.float32)
    )

    encoded[f"{prefix}_wmean"] = (
        keys.map(group_stats["weighted_mean_smooth"])
        .fillna(defaults["global_weighted_mean"])
        .astype(np.float32)
    )

    encoded[f"{prefix}_log_count"] = (
        keys.map(group_stats["log_count"])
        .fillna(0.0)
        .astype(np.float32)
    )

    encoded[f"{prefix}_std"] = (
        keys.map(group_stats["target_std"])
        .fillna(defaults["global_std"])
        .astype(np.float32)
    )

    return encoded

In [53]:
def build_te_oof_and_apply(raw_fit, y_fit, raw_apply, key_specs, n_splits=5, random_state=9890):
    """
    Build target/statistical encoding features safely.

    For raw_fit:
    - features are built out-of-fold
    - each row is encoded using statistics learned from other rows only

    For raw_apply:
    - statistics are learned from all raw_fit rows
    - those statistics are then applied to raw_apply
    """
    raw_fit = raw_fit.reset_index(drop=True)
    raw_apply = raw_apply.reset_index(drop=True)

    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)
    n_fit = raw_fit["N_STUDENTS"].astype(float).to_numpy()

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    oof_feature_blocks = []
    apply_feature_blocks = []
    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        fit_keys = make_group_key(raw_fit, cols)
        apply_keys = make_group_key(raw_apply, cols)

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_values = np.zeros(
            (len(raw_fit), len(feature_cols)),
            dtype=np.float32
        )

        for inner_train_idx, inner_valid_idx in kf.split(np.arange(len(raw_fit))):
            fold_stats, fold_defaults = fit_group_stats(
                fit_keys.iloc[inner_train_idx],
                y_fit[inner_train_idx],
                n_fit[inner_train_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            encoded_valid = apply_group_stats(
                fit_keys.iloc[inner_valid_idx],
                fold_stats,
                fold_defaults,
                prefix,
            )

            oof_values[inner_valid_idx, :] = encoded_valid[feature_cols].to_numpy(
                dtype=np.float32
            )

        full_stats, full_defaults = fit_group_stats(
            fit_keys,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        encoded_apply = apply_group_stats(
            apply_keys,
            full_stats,
            full_defaults,
            prefix,
        )

        oof_feature_blocks.append(
            pd.DataFrame(oof_values, columns=feature_cols)
        )

        apply_feature_blocks.append(
            encoded_apply[feature_cols]
        )

        fit_counts = fit_keys.value_counts(dropna=False)
        apply_seen = apply_keys.isin(set(fit_counts.index))

        summary_rows.append({
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "apply_groups": int(apply_keys.nunique(dropna=False)),
            "apply_row_coverage": float(apply_seen.mean()),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        })

    oof_features = pd.concat(oof_feature_blocks, axis=1)
    apply_features = pd.concat(apply_feature_blocks, axis=1)
    key_summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, key_summary

In [54]:
def take_rows(X, idx):
    """
    Select rows from either a DataFrame or NumPy array.
    """
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx]

    return X[idx]


def to_float32_matrix(X):
    """
    Convert a DataFrame or array to a float32 NumPy matrix.
    """
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)

    return np.asarray(X, dtype=np.float32)


def append_features(X_base, add_df):
    """
    Append engineered feature columns to an existing numeric matrix.
    """
    X_base = to_float32_matrix(X_base)
    X_add = add_df.to_numpy(dtype=np.float32)

    return np.hstack([X_base, X_add])

In [55]:
all_idx_te = np.arange(len(y_arr_te))

tr_idx_te_screen, val_idx_te_screen = train_test_split(
    all_idx_te,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
)

raw_fit_screen = raw_train_te.iloc[tr_idx_te_screen].reset_index(drop=True)
raw_val_screen = raw_train_te.iloc[val_idx_te_screen].reset_index(drop=True)

y_fit_screen = y_arr_te[tr_idx_te_screen]
y_val_screen = y_arr_te[val_idx_te_screen]

print("Screen training rows:", len(tr_idx_te_screen))
print("Screen validation rows:", len(val_idx_te_screen))

Screen training rows: 115936
Screen validation rows: 28985


In [56]:
te_tr_screen, te_val_screen, te_screen_key_summary = build_te_oof_and_apply(
    raw_fit=raw_fit_screen,
    y_fit=y_fit_screen,
    raw_apply=raw_val_screen,
    key_specs=te_key_specs,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

print("Target-encoding training feature shape:", te_tr_screen.shape)
print("Target-encoding validation feature shape:", te_val_screen.shape)

Target-encoding training feature shape: (115936, 68)
Target-encoding validation feature shape: (28985, 68)


In [57]:
X_tr_base_screen = to_float32_matrix(
    take_rows(X_train_proc_model, tr_idx_te_screen)
)

X_val_base_screen = to_float32_matrix(
    take_rows(X_train_proc_model, val_idx_te_screen)
)

X_tr_te_screen = append_features(X_tr_base_screen, te_tr_screen)
X_val_te_screen = append_features(X_val_base_screen, te_val_screen)

print("Base screen shapes:")
print("X_tr_base_screen:", X_tr_base_screen.shape)
print("X_val_base_screen:", X_val_base_screen.shape)

print("\nBase + target-encoding screen shapes:")
print("X_tr_te_screen:", X_tr_te_screen.shape)
print("X_val_te_screen:", X_val_te_screen.shape)

Base screen shapes:
X_tr_base_screen: (115936, 162)
X_val_base_screen: (28985, 162)

Base + target-encoding screen shapes:
X_tr_te_screen: (115936, 230)
X_val_te_screen: (28985, 230)


In [58]:
finite_te_features = (
    np.isfinite(te_tr_screen.to_numpy(dtype=np.float32)).all()
    and np.isfinite(te_val_screen.to_numpy(dtype=np.float32)).all()
)

shape_te_ok = (
    te_tr_screen.shape[0] == len(tr_idx_te_screen)
    and te_val_screen.shape[0] == len(val_idx_te_screen)
    and X_tr_te_screen.shape[0] == len(tr_idx_te_screen)
    and X_val_te_screen.shape[0] == len(val_idx_te_screen)
)

feature_count_te_ok = te_tr_screen.shape[1] == len(te_key_specs) * 4
column_te_ok = te_tr_screen.columns.is_unique and te_val_screen.columns.is_unique

TE_DIAGNOSTIC_PASS = bool(
    finite_te_features
    and shape_te_ok
    and feature_count_te_ok
    and column_te_ok
)

print("TE_DIAGNOSTIC_PASS:", TE_DIAGNOSTIC_PASS)

print("\nFeature checks:")
print("Finite target-encoding features:", finite_te_features)
print("Expected TE feature count:", len(te_key_specs) * 4)
print("Actual TE feature count:", te_tr_screen.shape[1])
print("Unique TE columns:", column_te_ok)

TE_DIAGNOSTIC_PASS: True

Feature checks:
Finite target-encoding features: True
Expected TE feature count: 68
Actual TE feature count: 68
Unique TE columns: True


In [59]:
te_screen_key_summary.sort_values(
    ["apply_row_coverage", "median_fit_count"],
    ascending=[False, False]
).reset_index(drop=True)

,key,alpha,alpha_n,fit_groups,apply_groups,apply_row_coverage,median_fit_count,singleton_share,features_added
0,SUBGROUP_NAME,20.0,200.0,5,5,1.000000,23233.0,0.000000,4
1,ASSESSMENT_NAME,20.0,200.0,32,32,1.000000,3601.0,0.000000,4
2,ASSESSMENT_NAME x SUBGROUP_NAME,20.0,200.0,132,132,1.000000,801.0,0.000000,4
3,COUNTY,30.0,250.0,62,62,1.000000,756.5,0.000000,4
4,DISTRICT_TYPE x ASSESSMENT_NAME,40.0,300.0,222,220,1.000000,354.5,0.009009,4
5,REGION x ASSESSMENT_NAME,40.0,300.0,315,314,1.000000,235.0,0.003175,4
6,COUNTY x SUBGROUP_NAME,40.0,300.0,310,309,1.000000,155.5,0.000000,4
7,DISTRICT,60.0,400.0,710,709,1.000000,78.0,0.000000,4
8,ASSESSMENT_NAME x N_STUDENTS_BIN,30.0,250.0,190,190,0.999965,372.5,0.005263,4
9,ASSESSMENT_NAME x SUBGROUP_NAME x N_STUDENTS_BIN,50.0,300.0,788,761,0.999965,109.0,0.013959,4


In [60]:
te_tr_screen.iloc[:, :10].describe().T

,count,mean,std,min,25%,50%,75%,max
te_ASSESSMENT_NAME_a20_n200_mean,115936.0,54.177391,12.351819,34.510574,43.745338,52.818825,65.268974,86.532150
te_ASSESSMENT_NAME_a20_n200_wmean,115936.0,54.782864,12.578279,31.266104,44.283440,53.665527,62.845219,84.903481
te_ASSESSMENT_NAME_a20_n200_log_count,115936.0,8.170941,0.494378,2.833213,7.962764,8.215007,8.561975,8.594524
te_ASSESSMENT_NAME_a20_n200_std,115936.0,23.093555,2.983797,2.500000,20.966339,22.582537,24.001675,30.780113
te_SUBGROUP_NAME_a20_n200_mean,115936.0,54.198841,4.719690,47.752342,52.528522,53.656490,54.151058,63.541710
te_SUBGROUP_NAME_a20_n200_wmean,115936.0,58.028111,7.377084,47.665009,55.763378,57.394348,57.541130,72.618271
te_SUBGROUP_NAME_a20_n200_log_count,115936.0,9.839026,0.148617,9.657715,9.686512,9.831508,10.059722,10.066923
te_SUBGROUP_NAME_a20_n200_std,115936.0,25.977516,0.553061,24.855640,25.627665,26.247187,26.384045,26.495983
te_ASSESSMENT_NAME_SUBGROUP_NAME_a20_n200_mean,115936.0,54.151497,13.043442,28.668512,43.414429,53.396889,64.705765,86.532150
te_ASSESSMENT_NAME_SUBGROUP_NAME_a20_n200_wmean,115936.0,55.334736,14.385706,26.434465,44.359715,53.848438,64.508759,90.540169


The target/statistical encoding construction diagnostic passes.

The notebook now has two engineered feature representations:

1. `X_train_proc_model` and `X_test_proc_model`  
   Base numeric features used by lower-complexity models.

2. `X_tr_te_screen` and `X_val_te_screen`  
   A holdout-screen version of the base features plus target/statistical encoding features.

The full target-encoding feature matrix for cross-validation will be rebuilt inside the later modelling folds, so validation rows are not encoded using their own target values.

# Data modelling

The modelling section is organized from lower-complexity models to higher-complexity models.

The goal is not to preserve every exploratory experiment from the original notebook. Instead, the cleaned notebook keeps detailed code for model families that were either inexpensive, informative, or important to the final modelling path.

For computationally expensive experiments, detailed code is kept only when the experiment produced a meaningful validation improvement or directly motivated the final model strategy. Expensive dead ends are summarized rather than fully reproduced.

The LightGBM diversity branch from the exploratory notebook is not continued in the main workflow because it only fed an older residual-stack path. It does not feed the later target-encoded LightGBM branch or the accounting / solver branch.

<br><br><br><br><br>

Contents:

Data modelling, Part 1: Linear baselines


Data modelling, Part 2: Regularized and selected linear extensions


Data modelling, Part 3: Step-function linear models


Data modelling, Part 4: Tree and bagging models


Data modelling, Part 5: Base-feature LightGBM boosting and saved-prediction blends


Data modelling, Part 6: Target/statistical-encoded LightGBM models and blends


Data modelling, Part 7: Accounting / solver models


Evaluation and final submission

## Data modelling, Part 1: Linear baselines

We begin modelling with simple linear baselines.

These models are useful because they establish how much signal can be captured by basic additive relationships before moving to more flexible models.

This section fits:

1. one-feature-at-a-time linear regressions,
2. a full multiple linear regression,
3. a degree-2 polynomial model on the strongest individual predictors,
4. a linear interaction model on the strongest individual predictors.

All models in this section use the same fixed 80/20 development split.

In [61]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from itertools import combinations

In [79]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training split:", X_tr.shape)
print("Validation split:", X_val.shape)
print("Training target:", y_tr.shape)
print("Validation target:", y_val.shape)

Training split: (115936, 162)
Validation split: (28985, 162)
Training target: (115936,)
Validation target: (28985,)


In [80]:
model_score_rows = []

In [81]:
simple_lr_results = []

for col in X_train_proc_model.columns:
    linear_model = LinearRegression()
    linear_model.fit(X_tr[[col]], y_tr)

    train_pred = linear_model.predict(X_tr[[col]])
    val_pred = linear_model.predict(X_val[[col]])

    simple_lr_results.append({
        "feature": col,
        "train_mse": mean_squared_error(y_tr, train_pred),
        "val_mse": mean_squared_error(y_val, val_pred),
    })

simple_lr_results = (
    pd.DataFrame(simple_lr_results)
    .sort_values("val_mse")
    .reset_index(drop=True)
)

simple_lr_results.head(30)

,feature,train_mse,val_mse
0,PERCENT_ECONOMICALLY_DISADVANTAGED,570.735244,581.066288
1,PERCENT_FREE_LUNCH,575.082361,584.868424
2,PERCENT_DIPLOMA,621.742550,626.029451
3,PERCENT_STILL_ENROLLED,637.471784,641.000299
4,PERCENT_HOMELESS,640.796399,644.070716
5,PERCENT_BLACK,648.963234,652.648495
6,PERCENT_WITH_DISABILITIES,650.316873,653.054139
7,PERCENT_ENGLISH_LANGUAGE_LEANERS,648.460937,655.348885
8,ATTENDANCE_RATE,650.513825,657.793054
9,PERCENT_WHITE,653.127789,658.601548


In [ ]:
best_simple_lr = simple_lr_results.iloc[0]

model_score_rows.append({
    "model": "simple_linear_best_single_feature",
    "feature_space": "best_single_feature",
    "train_mse": best_simple_lr["train_mse"],
    "val_mse": best_simple_lr["val_mse"],
    "notes": best_simple_lr["feature"],
})

best_simple_lr

The best single-feature linear model uses `PERCENT_ECONOMICALLY_DISADVANTAGED`.

This is useful as an interpretation check, but it is not competitive as a prediction model. Its validation error is much higher than models that combine many features.

In [82]:
linear_full = LinearRegression()
linear_full.fit(X_tr, y_tr)

linear_full_train_pred = linear_full.predict(X_tr)
linear_full_val_pred = linear_full.predict(X_val)

linear_full_train_mse = mean_squared_error(y_tr, linear_full_train_pred)
linear_full_val_mse = mean_squared_error(y_val, linear_full_val_pred)

model_score_rows.append({
    "model": "linear_regression",
    "feature_space": "base",
    "train_mse": linear_full_train_mse,
    "val_mse": linear_full_val_mse,
    "notes": "all base engineered features",
})

print("Train MSE:", linear_full_train_mse)
print("Validation MSE:", linear_full_val_mse)

Train MSE: 305.12409327017724
Validation MSE: 312.6016702920729


The full multiple linear regression model is much stronger than the best single-feature model.

This means the prediction problem is not explained by one dominant variable. Many features contribute useful signal together.

In [83]:
top10_features = simple_lr_results.head(10)["feature"].tolist()
top5_features = simple_lr_results.head(5)["feature"].tolist()

print("Top 10 features for polynomial terms:")
print(top10_features)

print("\nTop 5 features for interaction terms:")
print(top5_features)

Top 10 features for polynomial terms:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 5 features for interaction terms:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']


In [84]:
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_tr_poly_top10 = poly.fit_transform(X_tr[top10_features])
X_val_poly_top10 = poly.transform(X_val[top10_features])

print("Polynomial training shape:", X_tr_poly_top10.shape)
print("Polynomial validation shape:", X_val_poly_top10.shape)

Polynomial training shape: (115936, 65)
Polynomial validation shape: (28985, 65)


This uses degree-2 polynomial features on only the top 10 single-feature predictors. It is intentionally small and simple

In [85]:
linear_poly = LinearRegression()
linear_poly.fit(X_tr_poly_top10, y_tr)

linear_poly_train_pred = linear_poly.predict(X_tr_poly_top10)
linear_poly_val_pred = linear_poly.predict(X_val_poly_top10)

linear_poly_train_mse = mean_squared_error(y_tr, linear_poly_train_pred)
linear_poly_val_mse = mean_squared_error(y_val, linear_poly_val_pred)

model_score_rows.append({
    "model": "linear_regression",
    "feature_space": "top10_degree2_polynomial",
    "train_mse": linear_poly_train_mse,
    "val_mse": linear_poly_val_mse,
    "notes": "degree-2 polynomial expansion of top 10 simple-LR features",
})

print("Train MSE:", linear_poly_train_mse)
print("Validation MSE:", linear_poly_val_mse)

Train MSE: 507.61481933233904
Validation MSE: 515.1911286593601


This is worse than the full multiple linear regression because it uses only a small subset of predictors.

In [86]:
X_tr_interactions = X_tr.copy()
X_val_interactions = X_val.copy()

interaction_cols = []

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"

    X_tr_interactions[new_col] = X_tr[f1] * X_tr[f2]
    X_val_interactions[new_col] = X_val[f1] * X_val[f2]

    interaction_cols.append(new_col)

print("Interaction features added:", len(interaction_cols))
print("Interaction training shape:", X_tr_interactions.shape)
print("Interaction validation shape:", X_val_interactions.shape)

Interaction features added: 10
Interaction training shape: (115936, 172)
Interaction validation shape: (28985, 172)


In [87]:
linear_interactions = LinearRegression()
linear_interactions.fit(X_tr_interactions, y_tr)

linear_interactions_train_pred = linear_interactions.predict(X_tr_interactions)
linear_interactions_val_pred = linear_interactions.predict(X_val_interactions)

linear_interactions_train_mse = mean_squared_error(y_tr, linear_interactions_train_pred)
linear_interactions_val_mse = mean_squared_error(y_val, linear_interactions_val_pred)

model_score_rows.append({
    "model": "linear_regression",
    "feature_space": "base_plus_top5_interactions",
    "train_mse": linear_interactions_train_mse,
    "val_mse": linear_interactions_val_mse,
    "notes": "base features plus pairwise interactions among top 5 simple-LR features",
})

print("Train MSE:", linear_interactions_train_mse)
print("Validation MSE:", linear_interactions_val_mse)

Train MSE: 304.3250676749579
Validation MSE: 311.955526746157


In [88]:
linear_baseline_summary = (
    pd.DataFrame(model_score_rows)
    .sort_values("val_mse")
    .reset_index(drop=True)
)

linear_baseline_summary

,model,feature_space,train_mse,val_mse,notes
0,linear_regression,base_plus_top5_interactions,304.325068,311.955527,base features plus pairwise interactions among...
1,linear_regression,base,305.124093,312.601670,all base engineered features
2,linear_regression,top10_degree2_polynomial,507.614819,515.191129,degree-2 polynomial expansion of top 10 simple...


The linear baseline results show that the full base feature matrix is much stronger than any single feature.

Polynomial terms on only the top 10 predictors perform worse because they discard too much of the full feature set.

Pairwise interactions among the top 5 predictors give a small improvement over plain multiple linear regression, but the gain is modest. This suggests that the early signal is mostly additive, with some limited interaction benefit.

## Data modelling, Part 2: Controlled linear feature spaces

The first linear-baseline section showed that the full base feature matrix performs much better than any single predictor.

The original notebook then tested controlled linear feature spaces:

1. base features
2. base features plus selected squared terms
3. base features plus selected interactions
4. base features plus both squared terms and interactions

This section rebuilds those feature spaces, reruns the forward stepwise search because the original forward run was interrupted, and records the expensive completed searches that should not be rerun.

In [99]:
from itertools import combinations
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

Path("model_results").mkdir(exist_ok=True)

In [100]:
top10_features = simple_lr_results.head(10)["feature"].tolist()
top5_features = simple_lr_results.head(5)["feature"].tolist()

print("Top 10 features used for squared terms:")
print(top10_features)

print("\nTop 5 features used for interactions:")
print(top5_features)

Top 10 features used for squared terms:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 5 features used for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']


In [101]:
X_tr_base = X_tr.copy()
X_val_base = X_val.copy()

X_tr_polyspace = X_tr.copy()
X_val_polyspace = X_val.copy()

poly_cols = []

for col in top10_features:
    new_col = col + "_squared"

    X_tr_polyspace[new_col] = X_tr[col] ** 2
    X_val_polyspace[new_col] = X_val[col] ** 2

    poly_cols.append(new_col)

X_tr_intspace = X_tr.copy()
X_val_intspace = X_val.copy()

interaction_cols_controlled = []

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"

    X_tr_intspace[new_col] = X_tr[f1] * X_tr[f2]
    X_val_intspace[new_col] = X_val[f1] * X_val[f2]

    interaction_cols_controlled.append(new_col)

X_tr_combined = X_tr_polyspace.copy()
X_val_combined = X_val_polyspace.copy()

for col in interaction_cols_controlled:
    X_tr_combined[col] = X_tr_intspace[col]
    X_val_combined[col] = X_val_intspace[col]

feature_spaces = {
    "base": (X_tr_base, X_val_base),
    "base_plus_poly": (X_tr_polyspace, X_val_polyspace),
    "base_plus_interactions": (X_tr_intspace, X_val_intspace),
    "base_plus_poly_interactions": (X_tr_combined, X_val_combined),
}

print("Squared-term columns added:", len(poly_cols))
print("Interaction columns added:", len(interaction_cols_controlled))

print("\nFeature space shapes:")
for name, (X_train_space, X_val_space) in feature_spaces.items():
    print(name, X_train_space.shape, X_val_space.shape)

Squared-term columns added: 10
Interaction columns added: 10

Feature space shapes:
base (115936, 162) (28985, 162)
base_plus_poly (115936, 172) (28985, 172)
base_plus_interactions (115936, 172) (28985, 172)
base_plus_poly_interactions (115936, 182) (28985, 182)


#### Forward stepwise selection

The original forward-stepwise cell was interrupted, so its results should not be treated as completed.

Forward stepwise is rerun here with the same 30-feature cap used in the original notebook. This cap keeps the search focused on whether a small subset of features can approach the full linear model.

The results are saved after each feature space so the run does not have to be repeated if the notebook is restarted.

In [102]:
def forward_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    """
    Add one feature at a time.

    At each step, the feature that gives the lowest validation MSE is added.
    The search stops when the validation MSE stops improving or the feature cap is reached.
    """
    remaining = list(X_train_space.columns)
    selected = []
    rows = []
    best_mse = np.inf

    while remaining and len(selected) < max_features:
        candidates = []

        for feature in remaining:
            trial_features = selected + [feature]

            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)

            val_pred = model.predict(X_val_space[trial_features])
            val_mse = mean_squared_error(y_val, val_pred)

            candidates.append((feature, val_mse))

        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])

        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse

            rows.append({
                "num_features": len(selected),
                "feature_added": best_feature,
                "val_mse": best_mse,
            })
        else:
            break

    return pd.DataFrame(rows), selected

In [103]:
forward_summary_rows = []
forward_paths = {}
forward_selected_features = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "=" * 80)
    print("Forward stepwise:", name)
    print("=" * 80)

    path, selected = forward_stepwise(
        X_train_space,
        X_val_space,
        y_tr,
        y_val,
        max_features=30,
    )

    forward_paths[name] = path
    forward_selected_features[name] = selected

    path.to_csv(
        f"model_results/forward_stepwise_{name}.csv",
        index=False
    )

    if len(path) > 0:
        best_row = path.loc[path["val_mse"].idxmin()]

        forward_summary_rows.append({
            "model_family": "Linear",
            "model": "forward_stepwise",
            "feature_space": name,
            "selected_features": int(best_row["num_features"]),
            "train_mse": np.nan,
            "val_mse": best_row["val_mse"],
            "notes": "rerun in cleaned notebook; max_features=30",
        })

        print(path.tail(10).to_string(index=False))
        print("Best validation MSE:", best_row["val_mse"])
    else:
        print("No features selected.")

forward_stepwise_results = (
    pd.DataFrame(forward_summary_rows)
    .sort_values("val_mse")
    .reset_index(drop=True)
)

forward_stepwise_results


Forward stepwise: base
 num_features                                       feature_added    val_mse
           21                                   DISTRICT_TYPE_NYC 356.639984
           22                     ASSESSMENT_NAME_RegentsScience8 352.619203
           23                               ASSESSMENT_NAME_MATH4 348.915113
           24                               PERCENT_REDUCED_LUNCH 346.330326
           25           ASSESSMENT_NAME_Regents Phy Set/Chemistry 344.123928
           26        ASSESSMENT_NAME_Regents Common Core Geometry 341.579203
           27             ASSESSMENT_NAME_Regents Phy Set/Physics 338.762894
           28 HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE 336.466988
           29                               ASSESSMENT_NAME_MATH7 334.174799
           30                               ASSESSMENT_NAME_MATH3 332.314849
Best validation MSE: 332.3148493948422

Forward stepwise: base_plus_poly
 num_features                                       feat

,model_family,model,feature_space,selected_features,train_mse,val_mse,notes
0,Linear,forward_stepwise,base_plus_poly,30,NaN,329.660042,rerun in cleaned notebook; max_features=30
1,Linear,forward_stepwise,base_plus_poly_interactions,30,NaN,329.660042,rerun in cleaned notebook; max_features=30
2,Linear,forward_stepwise,base,30,NaN,332.314849,rerun in cleaned notebook; max_features=30
3,Linear,forward_stepwise,base_plus_interactions,30,NaN,332.314849,rerun in cleaned notebook; max_features=30


Forward stepwise tests whether a small selected subset of predictors can match the full linear model.

If its validation MSE remains much higher than the full linear model, that indicates the linear signal is spread across many variables rather than concentrated in a small subset.

#### Recorded backward stepwise results

Backward stepwise was completed in the original notebook, but it should not be rerun here.

Unlike forward stepwise, backward stepwise starts with the full feature set and repeatedly tests many possible one-feature removals. This made it much more expensive.

The values below are recorded from the original notebook. They are hardcoded here because the cleaned notebook should preserve the result without repeating the long search.

In [104]:
backward_stepwise_recorded_results = pd.DataFrame([
    {
        "model_family": "Linear",
        "model": "backward_stepwise",
        "feature_space": "base",
        "selected_features": 145,
        "train_mse": np.nan,
        "val_mse": 312.3771461462088,
        "notes": "original backward stepwise result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "backward_stepwise",
        "feature_space": "base_plus_poly",
        "selected_features": 155,
        "train_mse": np.nan,
        "val_mse": 308.7939934983059,
        "notes": "original backward stepwise result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "backward_stepwise",
        "feature_space": "base_plus_interactions",
        "selected_features": 155,
        "train_mse": np.nan,
        "val_mse": 311.70177215183503,
        "notes": "original backward stepwise result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "backward_stepwise",
        "feature_space": "base_plus_poly_interactions",
        "selected_features": 166,
        "train_mse": np.nan,
        "val_mse": 308.13298870014154,
        "notes": "original backward stepwise result; not rerun"
    },
])

backward_stepwise_recorded_results.sort_values("val_mse")

,model_family,model,feature_space,selected_features,train_mse,val_mse,notes
3,Linear,backward_stepwise,base_plus_poly_interactions,166,NaN,308.132989,original backward stepwise result; not rerun
1,Linear,backward_stepwise,base_plus_poly,155,NaN,308.793993,original backward stepwise result; not rerun
2,Linear,backward_stepwise,base_plus_interactions,155,NaN,311.701772,original backward stepwise result; not rerun
0,Linear,backward_stepwise,base,145,NaN,312.377146,original backward stepwise result; not rerun


Backward stepwise was the strongest linear-family search.

It improved validation MSE from roughly 312.6 for the full ordinary linear model to 308.13 on the combined squared-term and interaction feature space.

This improvement is useful, but the run was too expensive to repeat in the cleaned notebook.

#### Recorded hybrid stepwise results

The original notebook also completed a hybrid stepwise search.

Hybrid stepwise alternates between adding a feature and then checking whether any selected feature should be removed. The values below are from that completed original hybrid run.

These are recorded separately from forward stepwise. They should not be described as forward-stepwise results.

In [105]:
hybrid_stepwise_recorded_results = pd.DataFrame([
    {
        "model_family": "Linear",
        "model": "hybrid_stepwise",
        "feature_space": "base",
        "selected_features": 30,
        "train_mse": np.nan,
        "val_mse": 332.3148493948422,
        "notes": "original hybrid stepwise result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "hybrid_stepwise",
        "feature_space": "base_plus_poly",
        "selected_features": 30,
        "train_mse": np.nan,
        "val_mse": 329.6600419738662,
        "notes": "original hybrid stepwise result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "hybrid_stepwise",
        "feature_space": "base_plus_interactions",
        "selected_features": 30,
        "train_mse": np.nan,
        "val_mse": 332.3148493948422,
        "notes": "original hybrid stepwise result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "hybrid_stepwise",
        "feature_space": "base_plus_poly_interactions",
        "selected_features": 30,
        "train_mse": np.nan,
        "val_mse": 329.6600419738662,
        "notes": "original hybrid stepwise result; not rerun"
    },
])

hybrid_stepwise_recorded_results.sort_values("val_mse")

,model_family,model,feature_space,selected_features,train_mse,val_mse,notes
1,Linear,hybrid_stepwise,base_plus_poly,30,NaN,329.660042,original hybrid stepwise result; not rerun
3,Linear,hybrid_stepwise,base_plus_poly_interactions,30,NaN,329.660042,original hybrid stepwise result; not rerun
0,Linear,hybrid_stepwise,base,30,NaN,332.314849,original hybrid stepwise result; not rerun
2,Linear,hybrid_stepwise,base_plus_interactions,30,NaN,332.314849,original hybrid stepwise result; not rerun


Hybrid stepwise did not approach the full linear model or backward stepwise.

This supports the same conclusion as the forward-stepwise screen: limiting the linear model to only 30 selected features loses too much information.

### Recorded regularized-linear results

The original notebook tested Ridge, Lasso, and Elastic Net on the controlled feature spaces.

These results are recorded rather than rerun because they did not beat the best backward-stepwise linear result, and the longer Lasso/Elastic Net searches were not part of the final winning path.

In [106]:
regularized_linear_recorded_results = pd.DataFrame([
    {
        "model_family": "Linear",
        "model": "ridge",
        "feature_space": "base",
        "scaler": "standard",
        "alpha": 1.584893,
        "l1_ratio": np.nan,
        "nonzero_coef": np.nan,
        "train_mse": 305.124257,
        "val_mse": 312.601531,
        "notes": "original Ridge search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "ridge",
        "feature_space": "base_plus_poly",
        "scaler": "standard",
        "alpha": 0.000100,
        "l1_ratio": np.nan,
        "nonzero_coef": np.nan,
        "train_mse": 301.622160,
        "val_mse": 308.925244,
        "notes": "original Ridge search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "ridge",
        "feature_space": "base_plus_interactions",
        "scaler": "standard",
        "alpha": 0.000100,
        "l1_ratio": np.nan,
        "nonzero_coef": np.nan,
        "train_mse": 304.325068,
        "val_mse": 311.955527,
        "notes": "original Ridge search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "ridge",
        "feature_space": "base_plus_poly_interactions",
        "scaler": "standard",
        "alpha": 1.412538,
        "l1_ratio": np.nan,
        "nonzero_coef": np.nan,
        "train_mse": 300.899518,
        "val_mse": 308.356609,
        "notes": "original Ridge search result; not rerun"
    },

    {
        "model_family": "Linear",
        "model": "lasso",
        "feature_space": "base",
        "scaler": "standard",
        "alpha": 0.001126,
        "l1_ratio": np.nan,
        "nonzero_coef": 119,
        "train_mse": 305.153150,
        "val_mse": 312.618107,
        "notes": "original Lasso search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "lasso",
        "feature_space": "base_plus_poly",
        "scaler": "standard",
        "alpha": 0.001126,
        "l1_ratio": np.nan,
        "nonzero_coef": 129,
        "train_mse": 301.662315,
        "val_mse": 308.988249,
        "notes": "original Lasso search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "lasso",
        "feature_space": "base_plus_interactions",
        "scaler": "standard",
        "alpha": 0.001126,
        "l1_ratio": np.nan,
        "nonzero_coef": 128,
        "train_mse": 304.373621,
        "val_mse": 311.977913,
        "notes": "original Lasso search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "lasso",
        "feature_space": "base_plus_poly_interactions",
        "scaler": "standard",
        "alpha": 0.001126,
        "l1_ratio": np.nan,
        "nonzero_coef": 134,
        "train_mse": 301.032358,
        "val_mse": 308.416433,
        "notes": "original Lasso search result; not rerun"
    },

    {
        "model_family": "Linear",
        "model": "elastic_net",
        "feature_space": "base",
        "scaler": "standard",
        "alpha": 0.001137,
        "l1_ratio": 0.99,
        "nonzero_coef": 132,
        "train_mse": 305.154413,
        "val_mse": 312.619191,
        "notes": "original Elastic Net search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "elastic_net",
        "feature_space": "base_plus_poly",
        "scaler": "standard",
        "alpha": 0.001137,
        "l1_ratio": 0.99,
        "nonzero_coef": 141,
        "train_mse": 301.665375,
        "val_mse": 308.993600,
        "notes": "original Elastic Net search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "elastic_net",
        "feature_space": "base_plus_interactions",
        "scaler": "standard",
        "alpha": 0.001137,
        "l1_ratio": 0.99,
        "nonzero_coef": 147,
        "train_mse": 304.377432,
        "val_mse": 311.983215,
        "notes": "original Elastic Net search result; not rerun"
    },
    {
        "model_family": "Linear",
        "model": "elastic_net",
        "feature_space": "base_plus_poly_interactions",
        "scaler": "standard",
        "alpha": 0.001137,
        "l1_ratio": 0.99,
        "nonzero_coef": 152,
        "train_mse": 301.053454,
        "val_mse": 308.431821,
        "notes": "original Elastic Net search result; not rerun"
    },
])

regularized_linear_recorded_results.sort_values("val_mse")

,model_family,model,feature_space,scaler,alpha,l1_ratio,nonzero_coef,train_mse,val_mse,notes
3,Linear,ridge,base_plus_poly_interactions,standard,1.412538,NaN,NaN,300.899518,308.356609,original Ridge search result; not rerun
7,Linear,lasso,base_plus_poly_interactions,standard,0.001126,NaN,134.0,301.032358,308.416433,original Lasso search result; not rerun
11,Linear,elastic_net,base_plus_poly_interactions,standard,0.001137,0.99,152.0,301.053454,308.431821,original Elastic Net search result; not rerun
1,Linear,ridge,base_plus_poly,standard,0.000100,NaN,NaN,301.622160,308.925244,original Ridge search result; not rerun
5,Linear,lasso,base_plus_poly,standard,0.001126,NaN,129.0,301.662315,308.988249,original Lasso search result; not rerun
9,Linear,elastic_net,base_plus_poly,standard,0.001137,0.99,141.0,301.665375,308.993600,original Elastic Net search result; not rerun
2,Linear,ridge,base_plus_interactions,standard,0.000100,NaN,NaN,304.325068,311.955527,original Ridge search result; not rerun
6,Linear,lasso,base_plus_interactions,standard,0.001126,NaN,128.0,304.373621,311.977913,original Lasso search result; not rerun
10,Linear,elastic_net,base_plus_interactions,standard,0.001137,0.99,147.0,304.377432,311.983215,original Elastic Net search result; not rerun
0,Linear,ridge,base,standard,1.584893,NaN,NaN,305.124257,312.601531,original Ridge search result; not rerun


In [107]:
linear_part2_score_rows = pd.concat(
    [
        forward_stepwise_results[
            [
                "model_family",
                "model",
                "feature_space",
                "train_mse",
                "val_mse",
                "notes",
            ]
        ],
        backward_stepwise_recorded_results[
            [
                "model_family",
                "model",
                "feature_space",
                "train_mse",
                "val_mse",
                "notes",
            ]
        ],
        hybrid_stepwise_recorded_results[
            [
                "model_family",
                "model",
                "feature_space",
                "train_mse",
                "val_mse",
                "notes",
            ]
        ],
        regularized_linear_recorded_results[
            [
                "model_family",
                "model",
                "feature_space",
                "train_mse",
                "val_mse",
                "notes",
            ]
        ],
    ],
    axis=0,
    ignore_index=True
)

linear_part2_score_rows.sort_values("val_mse").head(15)

,model_family,model,feature_space,train_mse,val_mse,notes
7,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun
15,Linear,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun
19,Linear,lasso,base_plus_poly_interactions,301.032358,308.416433,original Lasso search result; not rerun
23,Linear,elastic_net,base_plus_poly_interactions,301.053454,308.431821,original Elastic Net search result; not rerun
5,Linear,backward_stepwise,base_plus_poly,NaN,308.793993,original backward stepwise result; not rerun
13,Linear,ridge,base_plus_poly,301.622160,308.925244,original Ridge search result; not rerun
17,Linear,lasso,base_plus_poly,301.662315,308.988249,original Lasso search result; not rerun
21,Linear,elastic_net,base_plus_poly,301.665375,308.993600,original Elastic Net search result; not rerun
6,Linear,backward_stepwise,base_plus_interactions,NaN,311.701772,original backward stepwise result; not rerun
14,Linear,ridge,base_plus_interactions,304.325068,311.955527,original Ridge search result; not rerun


In [108]:
model_scoreboard = pd.concat(
    [
        linear_baseline_summary,
        linear_part2_score_rows,
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard

,model,feature_space,train_mse,val_mse,notes,model_family
0,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun,Linear
1,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun,Linear
2,lasso,base_plus_poly_interactions,301.032358,308.416433,original Lasso search result; not rerun,Linear
3,elastic_net,base_plus_poly_interactions,301.053454,308.431821,original Elastic Net search result; not rerun,Linear
4,backward_stepwise,base_plus_poly,NaN,308.793993,original backward stepwise result; not rerun,Linear
5,ridge,base_plus_poly,301.622160,308.925244,original Ridge search result; not rerun,Linear
6,lasso,base_plus_poly,301.662315,308.988249,original Lasso search result; not rerun,Linear
7,elastic_net,base_plus_poly,301.665375,308.993600,original Elastic Net search result; not rerun,Linear
8,backward_stepwise,base_plus_interactions,NaN,311.701772,original backward stepwise result; not rerun,Linear
9,linear_regression,base_plus_top5_interactions,304.325068,311.955527,base features plus pairwise interactions among...,NaN


The controlled linear experiments show that the best linear-family improvement came from backward stepwise selection on the combined squared-term and interaction feature space.

Forward stepwise was rerun because the original forward cell did not complete. If it remains much worse than the full linear model, this confirms that small selected subsets do not capture enough of the signal.

Backward stepwise and regularized linear models show that selected nonlinear terms help, but the best linear-family result remains far above the later high-performing model families.

In [110]:
model_scoreboard = model_scoreboard.copy()

if "model_family" not in model_scoreboard.columns:
    model_scoreboard["model_family"] = "Linear"

model_scoreboard["model_family"] = model_scoreboard["model_family"].fillna("Linear")

scoreboard_cols = [
    "model_family",
    "model",
    "feature_space",
    "train_mse",
    "val_mse",
    "notes",
]

model_scoreboard = model_scoreboard[scoreboard_cols]

model_scoreboard.head(10)

,model_family,model,feature_space,train_mse,val_mse,notes
0,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun
1,Linear,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun
2,Linear,lasso,base_plus_poly_interactions,301.032358,308.416433,original Lasso search result; not rerun
3,Linear,elastic_net,base_plus_poly_interactions,301.053454,308.431821,original Elastic Net search result; not rerun
4,Linear,backward_stepwise,base_plus_poly,NaN,308.793993,original backward stepwise result; not rerun
5,Linear,ridge,base_plus_poly,301.622160,308.925244,original Ridge search result; not rerun
6,Linear,lasso,base_plus_poly,301.662315,308.988249,original Lasso search result; not rerun
7,Linear,elastic_net,base_plus_poly,301.665375,308.993600,original Elastic Net search result; not rerun
8,Linear,backward_stepwise,base_plus_interactions,NaN,311.701772,original backward stepwise result; not rerun
9,Linear,linear_regression,base_plus_top5_interactions,304.325068,311.955527,base features plus pairwise interactions among...


## Data modelling, Part 3: Step-function linear models

The controlled linear models improved validation MSE to about 308, but they still assume mostly smooth additive relationships.

Step-function models allow selected continuous predictors to have threshold effects. For example, the effect of economic disadvantage or attendance may change across ranges rather than following one straight-line relationship.

This section keeps the step-function searches because they produced a meaningful improvement and were not prohibitively expensive.

The expensive Ridge/Lasso refinements on the final step-function matrix are recorded later instead of rerun, because their gains were very small.

In [111]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

In [112]:
def get_top_continuous_features(k, min_unique=20):
    """
    Select the top k continuous predictors from the simple linear ranking.

    Binary indicators and missingness flags are excluded because step functions
    are only useful for variables with many distinct values.
    """
    features = []

    for col in simple_lr_results["feature"].tolist():
        if col not in X_tr.columns:
            continue

        unique_count = X_tr[col].nunique(dropna=False)

        if unique_count >= min_unique and not col.endswith("_missing"):
            features.append(col)

        if len(features) >= k:
            break

    return features

In [113]:
def make_step_features(X_train_source, X_val_source, columns, n_bins, method):
    """
    Build step-function dummy variables.

    Cutpoints are learned from the training split only and then applied to
    the validation split. This avoids using validation information when
    deciding the bin boundaries.
    """
    train_parts = []
    val_parts = []
    step_cols = []

    for col in columns:
        x_train = X_train_source[col].astype(float)
        x_val = X_val_source[col].astype(float)

        if x_train.nunique(dropna=False) < 2:
            continue

        if method == "quantile":
            try:
                _, edges = pd.qcut(
                    x_train,
                    q=n_bins,
                    retbins=True,
                    duplicates="drop"
                )
            except ValueError:
                continue

        elif method == "equal_width":
            min_val = x_train.min()
            max_val = x_train.max()

            if not np.isfinite(min_val) or not np.isfinite(max_val) or min_val == max_val:
                continue

            edges = np.linspace(min_val, max_val, n_bins + 1)

        else:
            raise ValueError("method must be 'quantile' or 'equal_width'")

        edges = np.unique(edges)

        if len(edges) <= 2:
            continue

        edges = edges.astype(float)
        edges[0] = -np.inf
        edges[-1] = np.inf

        n_actual_bins = len(edges) - 1

        train_codes = pd.cut(
            x_train,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        val_codes = pd.cut(
            x_val,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        train_codes = pd.Categorical(
            train_codes,
            categories=list(range(n_actual_bins))
        )

        val_codes = pd.Categorical(
            val_codes,
            categories=list(range(n_actual_bins))
        )

        prefix = f"{col}_step_{method}_{n_bins}"

        train_dummies = pd.get_dummies(
            train_codes,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )

        val_dummies = pd.get_dummies(
            val_codes,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )

        train_dummies.index = X_train_source.index
        val_dummies.index = X_val_source.index

        train_dummies, val_dummies = train_dummies.align(
            val_dummies,
            join="left",
            axis=1,
            fill_value=0
        )

        train_parts.append(train_dummies)
        val_parts.append(val_dummies)
        step_cols.extend(train_dummies.columns.tolist())

    if len(train_parts) == 0:
        empty_train = pd.DataFrame(index=X_train_source.index)
        empty_val = pd.DataFrame(index=X_val_source.index)

        return empty_train, empty_val, []

    X_train_steps = pd.concat(train_parts, axis=1)
    X_val_steps = pd.concat(val_parts, axis=1)

    return X_train_steps, X_val_steps, step_cols

In [114]:
step_candidate_features = get_top_continuous_features(10)

print("Step-function candidate features:")
for i, col in enumerate(step_candidate_features, start=1):
    print(f"{i:2d}. {col}")

Step-function candidate features:
 1. PERCENT_ECONOMICALLY_DISADVANTAGED
 2. PERCENT_FREE_LUNCH
 3. PERCENT_DIPLOMA
 4. PERCENT_STILL_ENROLLED
 5. PERCENT_HOMELESS
 6. PERCENT_BLACK
 7. PERCENT_WITH_DISABILITIES
 8. PERCENT_ENGLISH_LANGUAGE_LEANERS
 9. ATTENDANCE_RATE
10. PERCENT_WHITE


In [115]:
start_time = time.perf_counter()

step_methods = ["quantile", "equal_width"]
step_bins_grid = [3, 5, 10]

step_results_rows = []

for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for method in step_methods:
        for n_bins in step_bins_grid:
            X_train_steps, X_val_steps, step_cols = make_step_features(
                X_train_source=X_tr,
                X_val_source=X_val,
                columns=step_candidate_features,
                n_bins=n_bins,
                method=method
            )

            X_train_aug = pd.concat(
                [X_train_space, X_train_steps],
                axis=1
            )

            X_val_aug = pd.concat(
                [X_val_space, X_val_steps],
                axis=1
            )

            model = LinearRegression()
            model.fit(X_train_aug, y_tr)

            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)

            step_results_rows.append({
                "model_family": "Step functions",
                "model": "linear_regression_with_steps",
                "feature_space": base_space_name,
                "step_method": method,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
            })

step_results = pd.DataFrame(step_results_rows)

best_step_by_space = (
    step_results
    .sort_values("val_mse")
    .groupby("feature_space", as_index=False)
    .first()
    .sort_values("val_mse")
    .reset_index(drop=True)
)

best_step_model = step_results.sort_values("val_mse").iloc[0]

print("Best step-function model by base feature space:")
display(best_step_by_space)

print("\nOverall best step-function model:")
display(best_step_model)

print("\nElapsed time:", round(time.perf_counter() - start_time, 2), "seconds")

Best step-function model by base feature space:


,feature_space,model_family,model,step_method,n_bins,n_step_features_added,total_features,train_mse,val_mse
0,base_plus_poly_interactions,Step functions,linear_regression_with_steps,quantile,10,83,265,296.544041,303.570375
1,base_plus_poly,Step functions,linear_regression_with_steps,quantile,10,83,255,297.288261,304.119794
2,base_plus_interactions,Step functions,linear_regression_with_steps,quantile,10,83,255,297.540321,304.612830
3,base,Step functions,linear_regression_with_steps,quantile,10,83,245,298.302277,305.261413



Overall best step-function model:


model_family                           Step functions
model                    linear_regression_with_steps
feature_space             base_plus_poly_interactions
step_method                                  quantile
n_bins                                             10
n_step_features_added                              83
total_features                                    265
train_mse                                  296.544041
val_mse                                    303.570375
Name: 20, dtype: object


Elapsed time: 16.88 seconds


The first step-function search improves on the best linear-family result.

Quantile bins perform best because they create balanced groups across skewed predictors. The best model uses the combined squared-term and interaction feature space plus 10-bin step functions.

In [116]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Step functions",
                "model": "linear_regression_with_steps",
                "feature_space": best_step_model["feature_space"],
                "train_mse": best_step_model["train_mse"],
                "val_mse": best_step_model["val_mse"],
                "notes": (
                    f"basic step search; method={best_step_model['step_method']}; "
                    f"bins={int(best_step_model['n_bins'])}; "
                    f"step_features={int(best_step_model['n_step_features_added'])}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(10)

,model_family,model,feature_space,train_mse,val_mse,notes
0,Step functions,linear_regression_with_steps,base_plus_poly_interactions,296.544041,303.570375,basic step search; method=quantile; bins=10; s...
1,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun
2,Linear,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun
3,Linear,lasso,base_plus_poly_interactions,301.032358,308.416433,original Lasso search result; not rerun
4,Linear,elastic_net,base_plus_poly_interactions,301.053454,308.431821,original Elastic Net search result; not rerun
5,Linear,backward_stepwise,base_plus_poly,NaN,308.793993,original backward stepwise result; not rerun
6,Linear,ridge,base_plus_poly,301.622160,308.925244,original Ridge search result; not rerun
7,Linear,lasso,base_plus_poly,301.662315,308.988249,original Lasso search result; not rerun
8,Linear,elastic_net,base_plus_poly,301.665375,308.993600,original Elastic Net search result; not rerun
9,Linear,backward_stepwise,base_plus_interactions,NaN,311.701772,original backward stepwise result; not rerun


#### Expanded step-function search

The first step-function search used 10 continuous predictors and at most 10 bins.

Because the first search improved validation MSE substantially, the original notebook expanded the search to more candidate predictors and more bins.

This expanded search is still reasonable to rerun because it is not a long model-training branch and it produced another meaningful improvement.

In [117]:
start_time = time.perf_counter()

candidate_feature_counts = [10, 15, 20]
expanded_step_bins_grid = [5, 10, 15, 20]
expanded_step_method = "quantile"

expanded_step_rows = []

for k in candidate_feature_counts:
    candidate_features = get_top_continuous_features(k)

    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)

    for n_bins in expanded_step_bins_grid:
        X_train_steps, X_val_steps, step_cols = make_step_features(
            X_train_source=X_tr,
            X_val_source=X_val,
            columns=candidate_features,
            n_bins=n_bins,
            method=expanded_step_method
        )

        for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
            X_train_aug = pd.concat(
                [X_train_space, X_train_steps],
                axis=1
            )

            X_val_aug = pd.concat(
                [X_val_space, X_val_steps],
                axis=1
            )

            model = LinearRegression()
            model.fit(X_train_aug, y_tr)

            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)

            expanded_step_rows.append({
                "model_family": "Step functions",
                "model": "expanded_linear_regression_with_steps",
                "feature_space": base_space_name,
                "step_method": expanded_step_method,
                "candidate_feature_count": k,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
            })

expanded_step_results = pd.DataFrame(expanded_step_rows)

expanded_best_by_space = (
    expanded_step_results
    .sort_values("val_mse")
    .groupby("feature_space", as_index=False)
    .first()
    .sort_values("val_mse")
    .reset_index(drop=True)
)

best_expanded_step_model = expanded_step_results.sort_values("val_mse").iloc[0]

print("\nBest expanded step-function model by base feature space:")
display(expanded_best_by_space)

print("\nOverall best expanded step-function model:")
display(best_expanded_step_model)

print("\nElapsed time:", round(time.perf_counter() - start_time, 2), "seconds")


Top 10 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 15 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11']

Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN

,feature_space,model_family,model,step_method,candidate_feature_count,n_bins,n_step_features_added,total_features,train_mse,val_mse
0,base_plus_poly_interactions,Step functions,expanded_linear_regression_with_steps,quantile,20,20,252,434,289.144890,296.137338
1,base_plus_interactions,Step functions,expanded_linear_regression_with_steps,quantile,20,20,252,424,289.690511,296.561235
2,base_plus_poly,Step functions,expanded_linear_regression_with_steps,quantile,20,20,252,424,289.860260,296.592385
3,base,Step functions,expanded_linear_regression_with_steps,quantile,20,20,252,414,290.309896,297.044336



Overall best expanded step-function model:


model_family                                      Step functions
model                      expanded_linear_regression_with_steps
feature_space                        base_plus_poly_interactions
step_method                                             quantile
candidate_feature_count                                       20
n_bins                                                        20
n_step_features_added                                        252
total_features                                               434
train_mse                                              289.14489
val_mse                                               296.137338
Name: 47, dtype: object


Elapsed time: 49.47 seconds


In [118]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Step functions",
                "model": "expanded_linear_regression_with_steps",
                "feature_space": best_expanded_step_model["feature_space"],
                "train_mse": best_expanded_step_model["train_mse"],
                "val_mse": best_expanded_step_model["val_mse"],
                "notes": (
                    f"expanded step search; candidates={int(best_expanded_step_model['candidate_feature_count'])}; "
                    f"bins={int(best_expanded_step_model['n_bins'])}; "
                    f"step_features={int(best_expanded_step_model['n_step_features_added'])}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(10)

,model_family,model,feature_space,train_mse,val_mse,notes
0,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...
1,Step functions,linear_regression_with_steps,base_plus_poly_interactions,296.544041,303.570375,basic step search; method=quantile; bins=10; s...
2,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun
3,Linear,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun
4,Linear,lasso,base_plus_poly_interactions,301.032358,308.416433,original Lasso search result; not rerun
5,Linear,elastic_net,base_plus_poly_interactions,301.053454,308.431821,original Elastic Net search result; not rerun
6,Linear,backward_stepwise,base_plus_poly,NaN,308.793993,original backward stepwise result; not rerun
7,Linear,ridge,base_plus_poly,301.622160,308.925244,original Ridge search result; not rerun
8,Linear,lasso,base_plus_poly,301.662315,308.988249,original Lasso search result; not rerun
9,Linear,elastic_net,base_plus_poly,301.665375,308.993600,original Elastic Net search result; not rerun


The expanded step-function search gives another clear improvement.

The best model uses 20 continuous predictors and 20 quantile bins on the combined squared-term and interaction base feature space.

### Focused expanded step-function search

The expanded search showed that more continuous predictors and more quantile bins improved performance.

The next search focuses only on the best base feature space, `base_plus_poly_interactions`, and tests larger step-function designs.

This is kept because it produced the strongest step-function result and another meaningful validation improvement.

In [119]:
start_time = time.perf_counter()

focused_base_space_name = "base_plus_poly_interactions"
X_train_focused_base, X_val_focused_base = feature_spaces[focused_base_space_name]

focused_candidate_counts = [20, 25, 30]
focused_bins_grid = [20, 25, 30]
focused_step_method = "quantile"

focused_step_rows = []

for k in focused_candidate_counts:
    candidate_features = get_top_continuous_features(k)

    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)

    for n_bins in focused_bins_grid:
        X_train_steps, X_val_steps, step_cols = make_step_features(
            X_train_source=X_tr,
            X_val_source=X_val,
            columns=candidate_features,
            n_bins=n_bins,
            method=focused_step_method
        )

        X_train_aug = pd.concat(
            [X_train_focused_base, X_train_steps],
            axis=1
        )

        X_val_aug = pd.concat(
            [X_val_focused_base, X_val_steps],
            axis=1
        )

        model = LinearRegression()
        model.fit(X_train_aug, y_tr)

        train_pred = model.predict(X_train_aug)
        val_pred = model.predict(X_val_aug)

        focused_step_rows.append({
            "model_family": "Step functions",
            "model": "focused_expanded_linear_regression_with_steps",
            "feature_space": focused_base_space_name,
            "step_method": focused_step_method,
            "candidate_feature_count": k,
            "n_bins": n_bins,
            "n_step_features_added": len(step_cols),
            "total_features": X_train_aug.shape[1],
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
        })

focused_step_results = (
    pd.DataFrame(focused_step_rows)
    .sort_values("val_mse")
    .reset_index(drop=True)
)

best_focused_step_model = focused_step_results.iloc[0]

print("\nFocused expanded step-function results:")
display(focused_step_results)

print("\nBest focused expanded step-function model:")
display(best_focused_step_model)

print("\nElapsed time:", round(time.perf_counter() - start_time, 2), "seconds")


Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS']

Top 25 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS', 'N_PUPILS', 'FEDERAL_FUNDING_PER_PUPIL', 'GRADE_04', 'GRADE_03', 'SCHOOL_freq']

Top 30 continuous step-

,model_family,model,feature_space,step_method,candidate_feature_count,n_bins,n_step_features_added,total_features,train_mse,val_mse
0,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,30,30,526,708,281.634289,290.109374
1,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,30,25,448,630,282.706508,290.473023
2,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,25,25,399,581,283.926249,291.319274
3,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,25,30,467,649,283.636449,291.788912
4,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,30,20,368,550,284.615841,292.258181
5,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,25,20,329,511,286.070626,293.647148
6,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,20,25,303,485,287.180460,294.125353
7,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,20,30,350,532,287.711205,295.376534
8,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,quantile,20,20,252,434,289.144890,296.137338



Best focused expanded step-function model:


model_family                                              Step functions
model                      focused_expanded_linear_regression_with_steps
feature_space                                base_plus_poly_interactions
step_method                                                     quantile
candidate_feature_count                                               30
n_bins                                                                30
n_step_features_added                                                526
total_features                                                       708
train_mse                                                     281.634289
val_mse                                                       290.109374
Name: 0, dtype: object


Elapsed time: 28.39 seconds


In [120]:
best_step_candidate_count = int(best_focused_step_model["candidate_feature_count"])
best_step_bins = int(best_focused_step_model["n_bins"])
best_step_method = best_focused_step_model["step_method"]

best_step_candidates = get_top_continuous_features(best_step_candidate_count)

X_train_steps_best, X_val_steps_best, best_step_cols = make_step_features(
    X_train_source=X_tr,
    X_val_source=X_val,
    columns=best_step_candidates,
    n_bins=best_step_bins,
    method=best_step_method
)

X_tr_step_best = pd.concat(
    [X_train_focused_base, X_train_steps_best],
    axis=1
)

X_val_step_best = pd.concat(
    [X_val_focused_base, X_val_steps_best],
    axis=1
)

print("Best step-function design rebuilt:")
print("Base space:", focused_base_space_name)
print("Candidate feature count:", best_step_candidate_count)
print("Bins:", best_step_bins)
print("Step features added:", len(best_step_cols))
print("Training shape:", X_tr_step_best.shape)
print("Validation shape:", X_val_step_best.shape)

Best step-function design rebuilt:
Base space: base_plus_poly_interactions
Candidate feature count: 30
Bins: 30
Step features added: 526
Training shape: (115936, 708)
Validation shape: (28985, 708)


In [121]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Step functions",
                "model": "focused_expanded_linear_regression_with_steps",
                "feature_space": best_focused_step_model["feature_space"],
                "train_mse": best_focused_step_model["train_mse"],
                "val_mse": best_focused_step_model["val_mse"],
                "notes": (
                    f"focused step search; candidates={best_step_candidate_count}; "
                    f"bins={best_step_bins}; "
                    f"step_features={int(best_focused_step_model['n_step_features_added'])}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(10)

,model_family,model,feature_space,train_mse,val_mse,notes
0,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,281.634289,290.109374,focused step search; candidates=30; bins=30; s...
1,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...
2,Step functions,linear_regression_with_steps,base_plus_poly_interactions,296.544041,303.570375,basic step search; method=quantile; bins=10; s...
3,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun
4,Linear,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun
5,Linear,lasso,base_plus_poly_interactions,301.032358,308.416433,original Lasso search result; not rerun
6,Linear,elastic_net,base_plus_poly_interactions,301.053454,308.431821,original Elastic Net search result; not rerun
7,Linear,backward_stepwise,base_plus_poly,NaN,308.793993,original backward stepwise result; not rerun
8,Linear,ridge,base_plus_poly,301.622160,308.925244,original Ridge search result; not rerun
9,Linear,lasso,base_plus_poly,301.662315,308.988249,original Lasso search result; not rerun


The focused step-function model is the strongest engineered linear model so far.

It improves validation MSE from about 308 for the best controlled linear model to about 290. This is a large enough improvement to justify keeping the runnable step-function code in the cleaned notebook.

#### Recorded Ridge and Lasso refinements on the step-function matrix

The original notebook also tested Ridge and Lasso on the final 708-feature step-function matrix.

These refinements are not rerun here.

Ridge took substantially longer than the ordinary step-function fit and improved validation MSE by only about 0.03.

Lasso took much longer and improved validation MSE by only about 0.02 compared with Ridge. It also kept nearly all coefficients nonzero, so it did not meaningfully simplify the model.

The recorded results are kept below for completeness, but the expensive searches are not repeated in the cleaned notebook.

In [122]:
step_regularized_recorded_results = pd.DataFrame([
    {
        "model_family": "Step functions",
        "model": "ridge_with_steps",
        "feature_space": "base_plus_poly_interactions_plus_steps",
        "scaler": "robust",
        "alpha": 0.473151,
        "nonzero_coef": np.nan,
        "train_mse": 281.664053,
        "val_mse": 290.079273,
        "notes": "original Ridge-step search result; not rerun"
    },
    {
        "model_family": "Step functions",
        "model": "lasso_with_steps",
        "feature_space": "base_plus_poly_interactions_plus_steps",
        "scaler": "standard",
        "alpha": 0.000310,
        "nonzero_coef": 657,
        "train_mse": 281.688365,
        "val_mse": 290.058976,
        "notes": "original Lasso-step search result; not rerun"
    },
])

step_regularized_recorded_results.sort_values("val_mse")

,model_family,model,feature_space,scaler,alpha,nonzero_coef,train_mse,val_mse,notes
1,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,standard,0.000310,657.0,281.688365,290.058976,original Lasso-step search result; not rerun
0,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,robust,0.473151,NaN,281.664053,290.079273,original Ridge-step search result; not rerun


In [141]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        step_regularized_recorded_results[
            [
                "model_family",
                "model",
                "feature_space",
                "train_mse",
                "val_mse",
                "notes",
            ]
        ],
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(15)

,model_family,model,feature_space,train_mse,val_mse,notes
0,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun
1,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun
2,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun
3,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun
4,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,281.634289,290.109374,focused step search; candidates=30; bins=30; s...
5,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...
6,Step functions,linear_regression_with_steps,base_plus_poly_interactions,296.544041,303.570375,basic step search; method=quantile; bins=10; s...
7,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun
8,Linear,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun
9,Linear,lasso,base_plus_poly_interactions,301.032358,308.416433,original Lasso search result; not rerun


The step-function family substantially improves over the earlier linear models.

The important gain came from adding step-function features. Ridge and Lasso produced only tiny additional improvements after long searches, so those refinements are recorded rather than rerun.

The next model family is regression trees. Trees are a natural next step because they can learn threshold behavior and interactions directly, without manually creating step-function features.

## Data modelling, Part 4: Tree and bagging models

The strongest step-function linear model reached validation MSE around 290.

The next modelling stage uses tree-based methods. These models can learn threshold effects and interactions directly, without manually creating step-function dummy variables.

This section begins with:

1. regression trees,
2. random forests,
3. random forest out-of-fold predictions.

These models are more computationally expensive than linear models, but they are kept as runnable code because they produced very large MSE improvements in the original workflow.

In [142]:
from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

#### Shared setup for tree-family models

The regression tree holdout screen uses the controlled feature spaces already built earlier:

- `base`
- `base_plus_poly`
- `base_plus_interactions`
- `base_plus_poly_interactions`

For cross-validation, squared terms and interactions are rebuilt inside each fold. This avoids choosing top polynomial/interaction features using validation-fold information.

In [143]:
tree_feature_spaces = {
    "base": feature_spaces["base"],
    "base_plus_poly": feature_spaces["base_plus_poly"],
    "base_plus_interactions": feature_spaces["base_plus_interactions"],
    "base_plus_poly_interactions": feature_spaces["base_plus_poly_interactions"],
}

print("Tree feature spaces:")

for name, (X_train_space, X_val_space) in tree_feature_spaces.items():
    print(name, X_train_space.shape, X_val_space.shape)

Tree feature spaces:
base (115936, 162) (28985, 162)
base_plus_poly (115936, 172) (28985, 172)
base_plus_interactions (115936, 172) (28985, 172)
base_plus_poly_interactions (115936, 182) (28985, 182)


In [144]:
def select_top_by_abs_corr(X_train_df, y_train_array, top_n):
    """
    Select predictors by absolute correlation with the target.

    This is used inside cross-validation folds so that fold-specific
    squared/interacted features are chosen using training-fold data only.
    """
    y_arr = np.asarray(y_train_array, dtype=float)
    y_centered = y_arr - y_arr.mean()
    y_norm = np.sqrt(np.sum(y_centered ** 2))

    scores = []

    for col in X_train_df.columns:
        x_arr = X_train_df[col].to_numpy(dtype=float)
        x_centered = x_arr - x_arr.mean()
        x_norm = np.sqrt(np.sum(x_centered ** 2))

        denom = x_norm * y_norm

        if denom == 0 or not np.isfinite(denom):
            score = 0.0
        else:
            score = abs(float(np.sum(x_centered * y_centered) / denom))

        scores.append((col, score))

    scores = sorted(scores, key=lambda item: item[1], reverse=True)

    return [col for col, _ in scores[:top_n]]

In [145]:
def build_fold_tree_feature_spaces(X_train_raw, X_eval_raw, y_train_array):
    """
    Build fold-safe versions of the controlled feature spaces.

    The base feature space is used as-is.

    The polynomial and interaction feature spaces are rebuilt using top
    predictors selected only from the training part of the current fold.
    """
    top10_features_fold = select_top_by_abs_corr(
        X_train_raw,
        y_train_array,
        top_n=10
    )

    top5_features_fold = select_top_by_abs_corr(
        X_train_raw,
        y_train_array,
        top_n=5
    )

    X_train_base = X_train_raw.copy()
    X_eval_base = X_eval_raw.copy()

    X_train_poly = X_train_raw.copy()
    X_eval_poly = X_eval_raw.copy()

    for col in top10_features_fold:
        new_col = col + "_squared"
        X_train_poly[new_col] = X_train_raw[col] ** 2
        X_eval_poly[new_col] = X_eval_raw[col] ** 2

    X_train_interactions = X_train_raw.copy()
    X_eval_interactions = X_eval_raw.copy()

    for f1, f2 in combinations(top5_features_fold, 2):
        new_col = f"{f1}_x_{f2}"
        X_train_interactions[new_col] = X_train_raw[f1] * X_train_raw[f2]
        X_eval_interactions[new_col] = X_eval_raw[f1] * X_eval_raw[f2]

    X_train_combined = X_train_poly.copy()
    X_eval_combined = X_eval_poly.copy()

    for col in X_train_interactions.columns:
        if col not in X_train_combined.columns:
            X_train_combined[col] = X_train_interactions[col]
            X_eval_combined[col] = X_eval_interactions[col]

    fold_feature_spaces = {
        "base": (X_train_base, X_eval_base),
        "base_plus_poly": (X_train_poly, X_eval_poly),
        "base_plus_interactions": (X_train_interactions, X_eval_interactions),
        "base_plus_poly_interactions": (X_train_combined, X_eval_combined),
    }

    return fold_feature_spaces, top10_features_fold, top5_features_fold

In [146]:
def append_checkpoint(row_df, path):
    """
    Append one or more result rows to a CSV checkpoint.
    """
    write_header = not path.exists()
    row_df.to_csv(path, mode="a", header=write_header, index=False)

#### 4A. Regression tree holdout screen

A regression tree is the simplest tree-based model here.

The model recursively splits the data into regions and predicts the average target value inside each region. This lets the model learn threshold behavior directly.

The holdout screen tests four controlled feature spaces and a small grid of tree complexity settings:

- `max_depth`: maximum tree depth,
- `min_samples_leaf`: minimum number of rows allowed in each terminal leaf,
- `min_samples_split`: set to twice `min_samples_leaf`.

Larger leaves reduce overfitting. Deeper trees allow more complex interactions.

In [147]:
start_time = time.perf_counter()

TREE_HOLDOUT_RESULTS_PATH = RESULTS_DIR / "regression_tree_results_holdout.csv"
TREE_HOLDOUT_BEST_BY_SPACE_PATH = RESULTS_DIR / "regression_tree_best_by_space_holdout.csv"
TREE_HOLDOUT_BEST_OVERALL_PATH = RESULTS_DIR / "regression_tree_best_overall_holdout.csv"

for path in [
    TREE_HOLDOUT_RESULTS_PATH,
    TREE_HOLDOUT_BEST_BY_SPACE_PATH,
    TREE_HOLDOUT_BEST_OVERALL_PATH,
]:
    if path.exists():
        path.unlink()

max_depth_grid = [3, 5, 7, 9, 12, 15, None]
min_samples_leaf_grid = [25, 50, 100, 250, 500, 1000, 2000]

tree_holdout_rows = []

for feature_space_name, (X_train_space, X_val_space) in tree_feature_spaces.items():
    print("\nFeature space:", feature_space_name)

    for max_depth in max_depth_grid:
        for min_samples_leaf in min_samples_leaf_grid:
            min_samples_split = max(2, 2 * min_samples_leaf)

            model = DecisionTreeRegressor(
                criterion="squared_error",
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                min_samples_split=min_samples_split,
                random_state=RANDOM_STATE
            )

            model.fit(X_train_space, y_tr)

            train_pred = model.predict(X_train_space)
            val_pred = model.predict(X_val_space)

            row = {
                "model_class": "Regression Tree",
                "feature_space": feature_space_name,
                "max_depth": "None" if max_depth is None else max_depth,
                "min_samples_leaf": min_samples_leaf,
                "min_samples_split": min_samples_split,
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
                "n_leaves": model.get_n_leaves(),
                "tree_depth": model.get_depth(),
            }

            tree_holdout_rows.append(row)
            append_checkpoint(pd.DataFrame([row]), TREE_HOLDOUT_RESULTS_PATH)

tree_holdout_results = (
    pd.DataFrame(tree_holdout_rows)
    .sort_values("val_mse")
    .reset_index(drop=True)
)

tree_holdout_best_by_space = (
    tree_holdout_results
    .sort_values("val_mse")
    .groupby("feature_space", as_index=False)
    .first()
    .sort_values("val_mse")
    .reset_index(drop=True)
)

best_tree_holdout = tree_holdout_results.iloc[0]

tree_holdout_best_by_space.to_csv(TREE_HOLDOUT_BEST_BY_SPACE_PATH, index=False)
pd.DataFrame([best_tree_holdout]).to_csv(TREE_HOLDOUT_BEST_OVERALL_PATH, index=False)

print("Best regression tree by feature space:")
display(tree_holdout_best_by_space)

print("\nOverall best regression tree holdout result:")
display(best_tree_holdout)

print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))


Feature space: base

Feature space: base_plus_poly

Feature space: base_plus_interactions

Feature space: base_plus_poly_interactions
Best regression tree by feature space:


,feature_space,model_class,max_depth,min_samples_leaf,min_samples_split,train_mse,val_mse,n_leaves,tree_depth
0,base_plus_poly,Regression Tree,None,25,50,190.544257,240.566256,3522,45
1,base,Regression Tree,None,25,50,190.544257,240.589540,3522,45
2,base_plus_interactions,Regression Tree,None,25,50,189.192813,241.312566,3526,46
3,base_plus_poly_interactions,Regression Tree,None,25,50,189.192804,241.352112,3525,46



Overall best regression tree holdout result:


model_class          Regression Tree
feature_space         base_plus_poly
max_depth                       None
min_samples_leaf                  25
min_samples_split                 50
train_mse                 190.544257
val_mse                   240.566256
n_leaves                        3522
tree_depth                        45
Name: 0, dtype: object


Elapsed seconds: 291.39


In [148]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Tree",
                "model": "regression_tree_holdout_screen",
                "feature_space": best_tree_holdout["feature_space"],
                "metric_source": "fixed holdout",
                "train_mse": best_tree_holdout["train_mse"],
                "val_mse": best_tree_holdout["val_mse"],
                "notes": (
                    f"max_depth={best_tree_holdout['max_depth']}; "
                    f"min_samples_leaf={int(best_tree_holdout['min_samples_leaf'])}; "
                    f"leaves={int(best_tree_holdout['n_leaves'])}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(12)

,model_family,model,feature_space,train_mse,val_mse,notes,metric_source
0,Tree,regression_tree_holdout_screen,base_plus_poly,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522,fixed holdout
1,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
2,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
3,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
4,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
5,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,281.634289,290.109374,focused step search; candidates=30; bins=30; s...,NaN
6,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...,NaN
7,Step functions,linear_regression_with_steps,base_plus_poly_interactions,296.544041,303.570375,basic step search; method=quantile; bins=10; s...,NaN
8,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun,NaN
9,Linear,ridge,base_plus_poly_interactions,300.899518,308.356609,original Ridge search result; not rerun,NaN


The regression tree holdout screen gives a large improvement over the step-function linear models.

This is the first sign that tree-based models can capture nonlinear structure more effectively than manually engineered step functions.

#### 4B. Regression tree 5-fold cross-validation

The holdout screen uses one validation split. Cross-validation gives a more stable estimate.

For each fold, the controlled polynomial and interaction feature spaces are rebuilt using only that fold’s training rows. This avoids using validation-fold information when choosing squared and interaction features.

The same tree grid is evaluated using 5-fold cross-validation.

In [149]:
start_time = time.perf_counter()

TREE_CV_FOLD_RESULTS_PATH = RESULTS_DIR / "regression_tree_cv_fold_results.csv"
TREE_CV_SUMMARY_PATH = RESULTS_DIR / "regression_tree_cv_summary.csv"
TREE_CV_BEST_OVERALL_PATH = RESULTS_DIR / "regression_tree_cv_best_overall.csv"

for path in [
    TREE_CV_FOLD_RESULTS_PATH,
    TREE_CV_SUMMARY_PATH,
    TREE_CV_BEST_OVERALL_PATH,
]:
    if path.exists():
        path.unlink()

N_SPLITS_TREE_CV = 5

kf_tree = KFold(
    n_splits=N_SPLITS_TREE_CV,
    shuffle=True,
    random_state=RANDOM_STATE
)

tree_cv_rows = []
y_tr_array = np.asarray(y_tr, dtype=float)

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf_tree.split(X_tr), start=1):
    print("\nRegression tree CV fold:", fold_id)

    X_cv_train_raw = X_tr.iloc[cv_train_idx]
    X_cv_valid_raw = X_tr.iloc[cv_valid_idx]

    y_cv_train = y_tr_array[cv_train_idx]
    y_cv_valid = y_tr_array[cv_valid_idx]

    fold_feature_spaces, fold_top10, fold_top5 = build_fold_tree_feature_spaces(
        X_cv_train_raw,
        X_cv_valid_raw,
        y_cv_train
    )

    for feature_space_name, (X_cv_train_space, X_cv_valid_space) in fold_feature_spaces.items():
        for max_depth in max_depth_grid:
            for min_samples_leaf in min_samples_leaf_grid:
                min_samples_split = max(2, 2 * min_samples_leaf)

                model = DecisionTreeRegressor(
                    criterion="squared_error",
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    min_samples_split=min_samples_split,
                    random_state=RANDOM_STATE + fold_id
                )

                model.fit(X_cv_train_space, y_cv_train)

                train_pred = model.predict(X_cv_train_space)
                valid_pred = model.predict(X_cv_valid_space)

                row = {
                    "model_class": "Regression Tree",
                    "fold": fold_id,
                    "feature_space": feature_space_name,
                    "max_depth": "None" if max_depth is None else max_depth,
                    "min_samples_leaf": min_samples_leaf,
                    "min_samples_split": min_samples_split,
                    "fold_train_mse": mean_squared_error(y_cv_train, train_pred),
                    "fold_valid_mse": mean_squared_error(y_cv_valid, valid_pred),
                    "n_leaves": model.get_n_leaves(),
                    "tree_depth": model.get_depth(),
                }

                tree_cv_rows.append(row)
                append_checkpoint(pd.DataFrame([row]), TREE_CV_FOLD_RESULTS_PATH)

    del fold_feature_spaces
    gc.collect()

tree_cv_fold_results = pd.DataFrame(tree_cv_rows)

tree_cv_group_cols = [
    "model_class",
    "feature_space",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split",
]

tree_cv_summary = (
    tree_cv_fold_results
    .groupby(tree_cv_group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std"),
        mean_leaves=("n_leaves", "mean"),
        mean_depth=("tree_depth", "mean"),
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
    .reset_index(drop=True)
)

best_tree_cv = tree_cv_summary.iloc[0]

tree_cv_summary.to_csv(TREE_CV_SUMMARY_PATH, index=False)
pd.DataFrame([best_tree_cv]).to_csv(TREE_CV_BEST_OVERALL_PATH, index=False)

print("Best regression tree CV results:")
display(tree_cv_summary.head(15))

print("\nOverall best regression tree CV result:")
display(best_tree_cv)

print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))


Regression tree CV fold: 1

Regression tree CV fold: 2

Regression tree CV fold: 3

Regression tree CV fold: 4

Regression tree CV fold: 5
Best regression tree CV results:


,model_class,feature_space,max_depth,min_samples_leaf,min_samples_split,cv_train_mse_mean,cv_train_mse_std,cv_valid_mse_mean,cv_valid_mse_std,mean_leaves,mean_depth
0,Regression Tree,base_plus_poly_interactions,None,25,50,193.942697,2.041149,250.689631,4.651331,2828.2,40.0
1,Regression Tree,base_plus_interactions,None,25,50,193.945488,2.044750,250.719660,4.654088,2828.0,40.0
2,Regression Tree,base_plus_poly,None,25,50,195.208861,1.474547,251.137172,4.297448,2825.8,41.4
3,Regression Tree,base,None,25,50,195.209028,1.474847,251.142112,4.291054,2825.8,41.4
4,Regression Tree,base_plus_poly_interactions,None,50,100,235.111359,2.596096,271.587271,6.111003,1411.8,36.0
5,Regression Tree,base_plus_interactions,None,50,100,235.111359,2.596096,271.587928,6.110520,1411.8,36.0
6,Regression Tree,base,None,50,100,236.318847,2.026537,271.687981,5.025428,1408.4,36.6
7,Regression Tree,base_plus_poly,None,50,100,236.318847,2.026537,271.704979,4.990109,1408.4,36.6
8,Regression Tree,base,15,25,50,255.192159,2.655166,284.590183,6.193787,1176.2,15.0
9,Regression Tree,base_plus_poly,15,25,50,255.192159,2.655166,284.613878,6.198155,1176.2,15.0



Overall best regression tree CV result:


model_class                      Regression Tree
feature_space        base_plus_poly_interactions
max_depth                                   None
min_samples_leaf                              25
min_samples_split                             50
cv_train_mse_mean                     193.942697
cv_train_mse_std                        2.041149
cv_valid_mse_mean                     250.689631
cv_valid_mse_std                        4.651331
mean_leaves                               2828.2
mean_depth                                  40.0
Name: 0, dtype: object


Elapsed seconds: 893.81


In [150]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Tree",
                "model": "regression_tree_cv",
                "feature_space": best_tree_cv["feature_space"],
                "metric_source": "5-fold CV mean",
                "train_mse": best_tree_cv["cv_train_mse_mean"],
                "val_mse": best_tree_cv["cv_valid_mse_mean"],
                "notes": (
                    f"max_depth={best_tree_cv['max_depth']}; "
                    f"min_samples_leaf={int(best_tree_cv['min_samples_leaf'])}; "
                    f"CV std={best_tree_cv['cv_valid_mse_std']:.4f}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(12)

,model_family,model,feature_space,train_mse,val_mse,notes,metric_source
0,Tree,regression_tree_holdout_screen,base_plus_poly,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522,fixed holdout
1,Tree,regression_tree_cv,base_plus_poly_interactions,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....,5-fold CV mean
2,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
3,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
4,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
5,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
6,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,281.634289,290.109374,focused step search; candidates=30; bins=30; s...,NaN
7,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...,NaN
8,Step functions,linear_regression_with_steps,base_plus_poly_interactions,296.544041,303.570375,basic step search; method=quantile; bins=10; s...,NaN
9,Linear,backward_stepwise,base_plus_poly_interactions,NaN,308.132989,original backward stepwise result; not rerun,NaN


The regression tree cross-validation result is slightly worse than the best single holdout tree, but it is more stable because it averages across five validation folds.

The next step is random forests. A random forest averages many trees, which usually improves over a single tree by reducing instability.

#### 4C. Random forest screen and refinement

A random forest fits many trees and averages their predictions.

Each tree is trained on a randomized sample of rows and considers only a subset of predictors at each split. This reduces the instability of a single tree.

The original workflow used three random forest stages:

1. a controlled 3-fold screen using 120 trees,
2. a stronger 5-fold refinement using 300 trees,
3. a final 500-tree out-of-fold artifact using the selected base-feature configuration.

The refined search is computationally heavier than the earlier cells, but it is kept because the MSE improvement is very large.

In [151]:
start_time = time.perf_counter()

RF_SCREEN_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_cv_fold_results_screen.csv"
RF_SCREEN_SUMMARY_PATH = RESULTS_DIR / "random_forest_cv_summary_screen.csv"
RF_SCREEN_BEST_OVERALL_PATH = RESULTS_DIR / "random_forest_cv_best_overall_screen.csv"

for path in [
    RF_SCREEN_FOLD_RESULTS_PATH,
    RF_SCREEN_SUMMARY_PATH,
    RF_SCREEN_BEST_OVERALL_PATH,
]:
    if path.exists():
        path.unlink()

N_SPLITS_RF_SCREEN = 3
RF_SCREEN_N_ESTIMATORS = 120
RF_SCREEN_MAX_SAMPLES = 0.80

rf_screen_param_grid = list(ParameterGrid({
    "max_features": ["sqrt", 0.50],
    "min_samples_leaf": [1, 5, 25],
}))

kf_rf_screen = KFold(
    n_splits=N_SPLITS_RF_SCREEN,
    shuffle=True,
    random_state=RANDOM_STATE
)

rf_screen_rows = []

X_rf_screen = X_tr
y_rf_screen = np.asarray(y_tr, dtype=float)

print("RF screen total fits:", N_SPLITS_RF_SCREEN * len(rf_screen_param_grid))

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf_rf_screen.split(X_rf_screen), start=1):
    print("\nRF screen fold:", fold_id)

    X_cv_train = X_rf_screen.iloc[cv_train_idx]
    X_cv_valid = X_rf_screen.iloc[cv_valid_idx]

    y_cv_train = y_rf_screen[cv_train_idx]
    y_cv_valid = y_rf_screen[cv_valid_idx]

    for combo_id, params in enumerate(rf_screen_param_grid, start=1):
        min_samples_leaf = int(params["min_samples_leaf"])
        min_samples_split = max(2, 2 * min_samples_leaf)

        model = RandomForestRegressor(
            n_estimators=RF_SCREEN_N_ESTIMATORS,
            criterion="squared_error",
            max_depth=None,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=params["max_features"],
            bootstrap=True,
            max_samples=RF_SCREEN_MAX_SAMPLES,
            oob_score=True,
            n_jobs=-1,
            random_state=RANDOM_STATE + 1000 * fold_id + combo_id
        )

        model.fit(X_cv_train, y_cv_train)

        train_pred = model.predict(X_cv_train)
        valid_pred = model.predict(X_cv_valid)

        oob_pred = model.oob_prediction_
        oob_mask = np.isfinite(oob_pred)

        if oob_mask.sum() > 0:
            oob_mse = mean_squared_error(y_cv_train[oob_mask], oob_pred[oob_mask])
        else:
            oob_mse = np.nan

        row = {
            "model_class": "Random Forest",
            "screen": "initial_3fold",
            "fold": fold_id,
            "feature_space": "base",
            "n_estimators": RF_SCREEN_N_ESTIMATORS,
            "max_samples": RF_SCREEN_MAX_SAMPLES,
            "max_features": str(params["max_features"]),
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split,
            "fold_train_mse": mean_squared_error(y_cv_train, train_pred),
            "fold_oob_mse": oob_mse,
            "fold_valid_mse": mean_squared_error(y_cv_valid, valid_pred),
        }

        rf_screen_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), RF_SCREEN_FOLD_RESULTS_PATH)

rf_screen_fold_results = pd.DataFrame(rf_screen_rows)

rf_screen_group_cols = [
    "model_class",
    "screen",
    "feature_space",
    "n_estimators",
    "max_samples",
    "max_features",
    "min_samples_leaf",
    "min_samples_split",
]

rf_screen_summary = (
    rf_screen_fold_results
    .groupby(rf_screen_group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_oob_mse_mean=("fold_oob_mse", "mean"),
        cv_oob_mse_std=("fold_oob_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std"),
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
    .reset_index(drop=True)
)

best_rf_screen = rf_screen_summary.iloc[0]

rf_screen_summary.to_csv(RF_SCREEN_SUMMARY_PATH, index=False)
pd.DataFrame([best_rf_screen]).to_csv(RF_SCREEN_BEST_OVERALL_PATH, index=False)

print("Random forest screen summary:")
display(rf_screen_summary)

print("\nBest random forest screen result:")
display(best_rf_screen)

print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))

RF screen total fits: 18

RF screen fold: 1

RF screen fold: 2

RF screen fold: 3
Random forest screen summary:


,model_class,screen,feature_space,n_estimators,max_samples,max_features,min_samples_leaf,min_samples_split,cv_train_mse_mean,cv_train_mse_std,cv_oob_mse_mean,cv_oob_mse_std,cv_valid_mse_mean,cv_valid_mse_std
0,Random Forest,initial_3fold,base,120,0.8,0.5,1,2,30.847553,0.250012,151.327034,0.944001,149.549571,0.282329
1,Random Forest,initial_3fold,base,120,0.8,0.5,5,10,108.977327,0.651871,181.536103,1.038970,180.342356,0.464139
2,Random Forest,initial_3fold,base,120,0.8,sqrt,1,2,37.703749,0.313783,185.144290,1.537173,182.424852,0.956699
3,Random Forest,initial_3fold,base,120,0.8,0.5,25,50,198.955523,0.911707,229.386657,0.901637,228.486152,1.117831
4,Random Forest,initial_3fold,base,120,0.8,sqrt,5,10,185.342204,1.705048,237.617356,1.862579,236.461401,1.955710
5,Random Forest,initial_3fold,base,120,0.8,sqrt,25,50,269.962266,1.889142,293.180623,1.911917,292.328583,2.571598



Best random forest screen result:


model_class          Random Forest
screen               initial_3fold
feature_space                 base
n_estimators                   120
max_samples                    0.8
max_features                   0.5
min_samples_leaf                 1
min_samples_split                2
cv_train_mse_mean        30.847553
cv_train_mse_std          0.250012
cv_oob_mse_mean         151.327034
cv_oob_mse_std            0.944001
cv_valid_mse_mean       149.549571
cv_valid_mse_std          0.282329
Name: 0, dtype: object


Elapsed seconds: 178.72


In [152]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Bagging",
                "model": "random_forest_screen",
                "feature_space": "base",
                "metric_source": "3-fold CV mean",
                "train_mse": best_rf_screen["cv_train_mse_mean"],
                "val_mse": best_rf_screen["cv_valid_mse_mean"],
                "notes": (
                    f"n_estimators={int(best_rf_screen['n_estimators'])}; "
                    f"max_samples={best_rf_screen['max_samples']}; "
                    f"max_features={best_rf_screen['max_features']}; "
                    f"min_samples_leaf={int(best_rf_screen['min_samples_leaf'])}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(12)

,model_family,model,feature_space,train_mse,val_mse,notes,metric_source
0,Bagging,random_forest_screen,base,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...,3-fold CV mean
1,Tree,regression_tree_holdout_screen,base_plus_poly,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522,fixed holdout
2,Tree,regression_tree_cv,base_plus_poly_interactions,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....,5-fold CV mean
3,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
4,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
5,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
6,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
7,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,281.634289,290.109374,focused step search; candidates=30; bins=30; s...,NaN
8,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...,NaN
9,Step functions,linear_regression_with_steps,base_plus_poly_interactions,296.544041,303.570375,basic step search; method=quantile; bins=10; s...,NaN


The first random forest screen gives a major improvement over the single regression tree.

The strongest screen configuration uses the base feature space. This suggests that the forest can learn useful nonlinear structure without the manually added polynomial and interaction terms.

#### Random forest 5-fold refinement

The refined random forest search keeps the same modelling idea but evaluates a more serious grid:

- 5-fold cross-validation,
- 300 trees per model,
- base and fold-safe combined feature spaces,
- several `max_features`, `min_samples_leaf`, and `max_samples` settings.

This is computationally heavier than the initial screen, but it remains part of the cleaned notebook because it produced a large MSE gain.

In [153]:
start_time = time.perf_counter()

RF_REFINED_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_cv_fold_results_refined.csv"
RF_REFINED_SUMMARY_PATH = RESULTS_DIR / "random_forest_cv_summary_refined.csv"
RF_REFINED_BEST_OVERALL_PATH = RESULTS_DIR / "random_forest_cv_best_overall_refined.csv"

for path in [
    RF_REFINED_FOLD_RESULTS_PATH,
    RF_REFINED_SUMMARY_PATH,
    RF_REFINED_BEST_OVERALL_PATH,
]:
    if path.exists():
        path.unlink()

N_SPLITS_RF_REFINED = 5
RF_REFINED_N_ESTIMATORS = 300

rf_refined_param_grid = list(ParameterGrid({
    "feature_space": ["base", "base_plus_poly_interactions"],
    "max_features": [0.40, 0.50, 0.70],
    "min_samples_leaf": [1, 2, 5],
    "max_samples": [0.80, None],
}))

kf_rf_refined = KFold(
    n_splits=N_SPLITS_RF_REFINED,
    shuffle=True,
    random_state=RANDOM_STATE
)

rf_refined_rows = []
y_rf_refined = np.asarray(y_tr, dtype=float)

print("RF refined total fits:", N_SPLITS_RF_REFINED * len(rf_refined_param_grid))

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf_rf_refined.split(X_tr), start=1):
    print("\nRF refined fold:", fold_id)

    X_cv_train_raw = X_tr.iloc[cv_train_idx]
    X_cv_valid_raw = X_tr.iloc[cv_valid_idx]

    y_cv_train = y_rf_refined[cv_train_idx]
    y_cv_valid = y_rf_refined[cv_valid_idx]

    fold_feature_spaces, fold_top10, fold_top5 = build_fold_tree_feature_spaces(
        X_cv_train_raw,
        X_cv_valid_raw,
        y_cv_train
    )

    for combo_id, params in enumerate(rf_refined_param_grid, start=1):
        feature_space_name = params["feature_space"]
        X_cv_train, X_cv_valid = fold_feature_spaces[feature_space_name]

        min_samples_leaf = int(params["min_samples_leaf"])
        min_samples_split = max(2, 2 * min_samples_leaf)

        model = RandomForestRegressor(
            n_estimators=RF_REFINED_N_ESTIMATORS,
            criterion="squared_error",
            max_depth=None,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=params["max_features"],
            bootstrap=True,
            max_samples=params["max_samples"],
            oob_score=True,
            n_jobs=-1,
            random_state=RANDOM_STATE + 1000 * fold_id + combo_id
        )

        model.fit(X_cv_train, y_cv_train)

        train_pred = model.predict(X_cv_train)
        valid_pred = model.predict(X_cv_valid)

        oob_pred = model.oob_prediction_
        oob_mask = np.isfinite(oob_pred)

        if oob_mask.sum() > 0:
            oob_mse = mean_squared_error(y_cv_train[oob_mask], oob_pred[oob_mask])
        else:
            oob_mse = np.nan

        row = {
            "model_class": "Random Forest",
            "screen": "refined_5fold",
            "fold": fold_id,
            "feature_space": feature_space_name,
            "n_estimators": RF_REFINED_N_ESTIMATORS,
            "max_samples": "None" if params["max_samples"] is None else str(params["max_samples"]),
            "max_features": str(params["max_features"]),
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split,
            "fold_train_mse": mean_squared_error(y_cv_train, train_pred),
            "fold_oob_mse": oob_mse,
            "fold_valid_mse": mean_squared_error(y_cv_valid, valid_pred),
        }

        rf_refined_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), RF_REFINED_FOLD_RESULTS_PATH)

    del fold_feature_spaces
    gc.collect()

rf_refined_fold_results = pd.DataFrame(rf_refined_rows)

rf_refined_group_cols = [
    "model_class",
    "screen",
    "feature_space",
    "n_estimators",
    "max_samples",
    "max_features",
    "min_samples_leaf",
    "min_samples_split",
]

rf_refined_summary = (
    rf_refined_fold_results
    .groupby(rf_refined_group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_oob_mse_mean=("fold_oob_mse", "mean"),
        cv_oob_mse_std=("fold_oob_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std"),
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
    .reset_index(drop=True)
)

best_rf_refined = rf_refined_summary.iloc[0]

rf_refined_summary.to_csv(RF_REFINED_SUMMARY_PATH, index=False)
pd.DataFrame([best_rf_refined]).to_csv(RF_REFINED_BEST_OVERALL_PATH, index=False)

print("Random forest refined summary:")
display(rf_refined_summary.head(15))

print("\nBest refined random forest result:")
display(best_rf_refined)

print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))

RF refined total fits: 180

RF refined fold: 1

RF refined fold: 2

RF refined fold: 3

RF refined fold: 4

RF refined fold: 5
Random forest refined summary:


,model_class,screen,feature_space,n_estimators,max_samples,max_features,min_samples_leaf,min_samples_split,cv_train_mse_mean,cv_train_mse_std,cv_oob_mse_mean,cv_oob_mse_std,cv_valid_mse_mean,cv_valid_mse_std
0,Random Forest,refined_5fold,base,300,None,0.5,1,2,18.459212,0.131071,135.578589,0.831340,134.708097,2.847304
1,Random Forest,refined_5fold,base,300,None,0.4,1,2,18.497945,0.095325,135.817888,0.625915,134.828198,2.969135
2,Random Forest,refined_5fold,base,300,None,0.7,1,2,18.614073,0.101051,136.762601,0.662909,136.042749,3.167066
3,Random Forest,refined_5fold,base_plus_poly_interactions,300,None,0.5,1,2,18.682136,0.099630,137.169209,0.669220,136.338753,3.181541
4,Random Forest,refined_5fold,base_plus_poly_interactions,300,None,0.4,1,2,18.706106,0.148673,137.438430,0.972772,136.527062,3.013754
5,Random Forest,refined_5fold,base_plus_poly_interactions,300,None,0.7,1,2,18.785101,0.103550,138.039564,0.692605,137.278115,3.347808
6,Random Forest,refined_5fold,base,300,0.8,0.5,1,2,28.245653,0.138608,139.282025,0.726741,138.835202,2.987776
7,Random Forest,refined_5fold,base,300,0.8,0.4,1,2,28.271050,0.178547,139.494729,0.811035,138.897775,2.828281
8,Random Forest,refined_5fold,base,300,0.8,0.7,1,2,28.369595,0.166544,139.870036,0.821909,139.479075,2.940695
9,Random Forest,refined_5fold,base_plus_poly_interactions,300,0.8,0.5,1,2,28.549979,0.135143,140.758187,0.627951,140.169313,3.208860



Best refined random forest result:


model_class          Random Forest
screen               refined_5fold
feature_space                 base
n_estimators                   300
max_samples                   None
max_features                   0.5
min_samples_leaf                 1
min_samples_split                2
cv_train_mse_mean        18.459212
cv_train_mse_std          0.131071
cv_oob_mse_mean         135.578589
cv_oob_mse_std             0.83134
cv_valid_mse_mean       134.708097
cv_valid_mse_std          2.847304
Name: 0, dtype: object


Elapsed seconds: 12100.69


In [154]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Bagging",
                "model": "random_forest_refined",
                "feature_space": best_rf_refined["feature_space"],
                "metric_source": "5-fold CV mean",
                "train_mse": best_rf_refined["cv_train_mse_mean"],
                "val_mse": best_rf_refined["cv_valid_mse_mean"],
                "notes": (
                    f"n_estimators={int(best_rf_refined['n_estimators'])}; "
                    f"max_samples={best_rf_refined['max_samples']}; "
                    f"max_features={best_rf_refined['max_features']}; "
                    f"min_samples_leaf={int(best_rf_refined['min_samples_leaf'])}; "
                    f"CV std={best_rf_refined['cv_valid_mse_std']:.4f}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(15)

,model_family,model,feature_space,train_mse,val_mse,notes,metric_source
0,Bagging,random_forest_refined,base,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...,5-fold CV mean
1,Bagging,random_forest_screen,base,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...,3-fold CV mean
2,Tree,regression_tree_holdout_screen,base_plus_poly,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522,fixed holdout
3,Tree,regression_tree_cv,base_plus_poly_interactions,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....,5-fold CV mean
4,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
5,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
6,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
7,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
8,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,281.634289,290.109374,focused step search; candidates=30; bins=30; s...,NaN
9,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...,NaN


The refined random forest improves substantially over the initial random forest screen.

The best refined configuration again uses the base feature space. This reinforces that bagged trees can learn the useful nonlinearities directly from the base engineered features.

#### Random forest 500-tree OOF artifact

The refined random forest selected a base-feature configuration.

The next cell trains a 5-fold, 500-tree random forest artifact on the full training data:

- each training row receives an out-of-fold prediction,
- each test row receives one prediction from each fold model,
- the test prediction is averaged across folds.

These OOF predictions are needed later for blending and stacking.

In [155]:
start_time = time.perf_counter()

RF_OOF_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_oof_500_fold_results.csv"
RF_OOF_PATH = RESULTS_DIR / "oof_rf_500_base.csv"
RF_TEST_FOLD_PATH = RESULTS_DIR / "testpred_rf_500_base_folds.csv"
RF_SUBMISSION_PATH = Path("submission_rf_500_base_oof_foldavg.csv")

for path in [
    RF_OOF_FOLD_RESULTS_PATH,
    RF_OOF_PATH,
    RF_TEST_FOLD_PATH,
    RF_SUBMISSION_PATH,
]:
    if path.exists():
        path.unlink()

RF_OOF_N_SPLITS = 5
RF_OOF_N_ESTIMATORS = 500

rf_oof_params = {
    "n_estimators": RF_OOF_N_ESTIMATORS,
    "criterion": "squared_error",
    "max_depth": None,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "max_features": 0.5,
    "bootstrap": True,
    "max_samples": None,
    "oob_score": True,
    "n_jobs": -1,
}

print("RF OOF settings:")
print(rf_oof_params)

kf_rf_oof = KFold(
    n_splits=RF_OOF_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

X_rf_all = X_train_proc_model
X_rf_test = X_test_proc_model
y_rf_all = np.asarray(y_train, dtype=float)

rf_oof_pred = np.zeros(len(X_rf_all), dtype=float)
rf_test_pred_folds = np.zeros((len(X_rf_test), RF_OOF_N_SPLITS), dtype=float)

rf_oof_fold_rows = []

for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf_rf_oof.split(X_rf_all), start=1):
    fold_start = time.perf_counter()

    print("\nRF OOF fold:", fold_id)

    X_fold_train = X_rf_all.iloc[fold_train_idx]
    X_fold_valid = X_rf_all.iloc[fold_valid_idx]

    y_fold_train = y_rf_all[fold_train_idx]
    y_fold_valid = y_rf_all[fold_valid_idx]

    model = RandomForestRegressor(
        **rf_oof_params,
        random_state=RANDOM_STATE + fold_id
    )

    model.fit(X_fold_train, y_fold_train)

    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    test_pred = model.predict(X_rf_test)

    rf_oof_pred[fold_valid_idx] = valid_pred
    rf_test_pred_folds[:, fold_id - 1] = test_pred

    oob_pred = model.oob_prediction_
    oob_mask = np.isfinite(oob_pred)

    if oob_mask.sum() > 0:
        oob_mse = mean_squared_error(y_fold_train[oob_mask], oob_pred[oob_mask])
    else:
        oob_mse = np.nan

    row = {
        "model_class": "Random Forest",
        "artifact": "oof_fold_ensemble",
        "fold": fold_id,
        "n_estimators": RF_OOF_N_ESTIMATORS,
        "max_features": 0.5,
        "max_samples": "None",
        "min_samples_leaf": 1,
        "min_samples_split": 2,
        "fold_train_mse": mean_squared_error(y_fold_train, train_pred),
        "fold_oob_mse": oob_mse,
        "fold_valid_mse": mean_squared_error(y_fold_valid, valid_pred),
        "elapsed_sec": time.perf_counter() - fold_start,
    }

    rf_oof_fold_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), RF_OOF_FOLD_RESULTS_PATH)

    print("fold_valid_mse:", row["fold_valid_mse"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))

    del model
    gc.collect()

rf_oof_fold_results = pd.DataFrame(rf_oof_fold_rows)

rf_oof_mse = mean_squared_error(y_rf_all, rf_oof_pred)

rf_test_pred_mean_raw = rf_test_pred_folds.mean(axis=1)
rf_test_pred_mean = np.clip(rf_test_pred_mean_raw, 0, 100)

rf_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": train_ids.astype(str),
    "y_true": y_rf_all,
    "rf_500_base_oof_pred": rf_oof_pred,
})

rf_test_fold_df = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
})

for fold_id in range(RF_OOF_N_SPLITS):
    rf_test_fold_df[f"rf_500_base_fold{fold_id + 1}_pred"] = rf_test_pred_folds[:, fold_id]

rf_test_fold_df["rf_500_base_foldavg_pred"] = rf_test_pred_mean

rf_submission_oof = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
    "PERCENT_PROFICIENT": rf_test_pred_mean,
})

rf_oof_df.to_csv(RF_OOF_PATH, index=False)
rf_test_fold_df.to_csv(RF_TEST_FOLD_PATH, index=False)
rf_submission_oof.to_csv(RF_SUBMISSION_PATH, index=False)

print("\nRF OOF fold results:")
display(rf_oof_fold_results)

print("\nOverall RF OOF MSE:", rf_oof_mse)

print("\nSaved files:")
print(RF_OOF_PATH)
print(RF_TEST_FOLD_PATH)
print(RF_SUBMISSION_PATH)

print("\nSubmission shape:", rf_submission_oof.shape)
print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))

RF OOF settings:
{'n_estimators': 500, 'criterion': 'squared_error', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'max_features': 0.5, 'bootstrap': True, 'max_samples': None, 'oob_score': True, 'n_jobs': -1}

RF OOF fold: 1
fold_valid_mse: 122.88128050280964
elapsed_sec: 217.01

RF OOF fold: 2
fold_valid_mse: 120.14224548830677
elapsed_sec: 178.22

RF OOF fold: 3
fold_valid_mse: 123.54697677953702
elapsed_sec: 165.49

RF OOF fold: 4
fold_valid_mse: 119.76183630285226
elapsed_sec: 206.04

RF OOF fold: 5
fold_valid_mse: 125.30776421592398
elapsed_sec: 192.4

RF OOF fold results:


,model_class,artifact,fold,n_estimators,max_features,max_samples,min_samples_leaf,min_samples_split,fold_train_mse,fold_oob_mse,fold_valid_mse,elapsed_sec
0,Random Forest,oof_fold_ensemble,1,500,0.5,None,1,2,16.708704,122.960428,122.881281,217.009852
1,Random Forest,oof_fold_ensemble,2,500,0.5,None,1,2,16.725163,123.335287,120.142245,178.216108
2,Random Forest,oof_fold_ensemble,3,500,0.5,None,1,2,16.707048,122.797429,123.546977,165.487264
3,Random Forest,oof_fold_ensemble,4,500,0.5,None,1,2,16.806578,123.720044,119.761836,206.039390
4,Random Forest,oof_fold_ensemble,5,500,0.5,None,1,2,16.672325,122.655667,125.307764,192.400543



Overall RF OOF MSE: 122.32802447555102

Saved files:
model_results/oof_rf_500_base.csv
model_results/testpred_rf_500_base_folds.csv
submission_rf_500_base_oof_foldavg.csv

Submission shape: (48307, 2)

Elapsed seconds: 961.81


In [156]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Bagging",
                "model": "random_forest_oof_500",
                "feature_space": "base",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": rf_oof_mse,
                "notes": "500-tree RF OOF artifact; fold-averaged test predictions saved",
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(15)

,model_family,model,feature_space,train_mse,val_mse,notes,metric_source
0,Bagging,random_forest_oof_500,base,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...,5-fold OOF
1,Bagging,random_forest_refined,base,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...,5-fold CV mean
2,Bagging,random_forest_screen,base,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...,3-fold CV mean
3,Tree,regression_tree_holdout_screen,base_plus_poly,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522,fixed holdout
4,Tree,regression_tree_cv,base_plus_poly_interactions,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....,5-fold CV mean
5,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
6,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,281.688365,290.058976,original Lasso-step search result; not rerun,NaN
7,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
8,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,281.664053,290.079273,original Ridge-step search result; not rerun,NaN
9,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,281.634289,290.109374,focused step search; candidates=30; bins=30; s...,NaN


The 500-tree random forest OOF artifact is the strongest result so far.

It improves substantially over the refined random forest CV screen and provides saved OOF and test predictions for later blending.

The next subsection will move to ExtraTrees, which uses a similar tree-averaging idea but adds more randomness to split selection.

In [158]:
# removes 2 duplicates
model_scoreboard = model_scoreboard.copy()

if "metric_source" not in model_scoreboard.columns:
    model_scoreboard["metric_source"] = np.nan

model_scoreboard["metric_source"] = model_scoreboard["metric_source"].fillna("fixed holdout / recorded")

scoreboard_cols = [
    "model_family",
    "model",
    "feature_space",
    "metric_source",
    "train_mse",
    "val_mse",
    "notes",
]

model_scoreboard = model_scoreboard[scoreboard_cols]

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(15)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
1,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
2,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...
3,Tree,regression_tree_holdout_screen,base_plus_poly,fixed holdout,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522
4,Tree,regression_tree_cv,base_plus_poly_interactions,5-fold CV mean,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....
5,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.688365,290.058976,original Lasso-step search result; not rerun
6,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.664053,290.079273,original Ridge-step search result; not rerun
7,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,fixed holdout / recorded,281.634289,290.109374,focused step search; candidates=30; bins=30; s...
8,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,fixed holdout / recorded,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...
9,Step functions,linear_regression_with_steps,base_plus_poly_interactions,fixed holdout / recorded,296.544041,303.570375,basic step search; method=quantile; bins=10; s...


#### 4D. ExtraTrees

ExtraTrees is a tree-ensemble model, like random forest.

Both models average many decision trees. The main difference is that ExtraTrees adds more randomness when choosing split thresholds. This can reduce variance and often gives predictions that are different enough from random forest predictions to help later blending.

This section uses only the base engineered feature matrix. The random forest searches already showed that the base feature matrix was strongest for tree ensembles, so the ExtraTrees search does not repeat the controlled polynomial and interaction feature spaces.

The ExtraTrees workflow has two stages:

1. a small holdout screen to choose a reasonable configuration,
2. a 5-fold out-of-fold run using the selected configuration.

The out-of-fold predictions are saved because they will be used later for model blending.

In [167]:
from sklearn.ensemble import ExtraTreesRegressor

The ExtraTrees configuration is intentionally focused.

The random forest section already showed that tree ensembles produce a major gain, so this section does not use a large tuning grid. It tests a small set of reasonable configurations and then builds the OOF artifact from the best holdout configuration.

In [168]:
EXTRATREES_N_JOBS = 1
EXTRATREES_SCREEN_N_ESTIMATORS = 180
EXTRATREES_OOF_N_ESTIMATORS = 250
EXTRATREES_OOF_N_SPLITS = 5

EXTRATREES_SCREEN_RESULTS_PATH = RESULTS_DIR / "extratrees_holdout_screen.csv"
EXTRATREES_SCREEN_BEST_PATH = RESULTS_DIR / "extratrees_holdout_best.csv"

EXTRATREES_OOF_FOLD_RESULTS_PATH = RESULTS_DIR / "extratrees_oof_fold_results.csv"
EXTRATREES_OOF_SUMMARY_PATH = RESULTS_DIR / "extratrees_oof_summary.csv"
EXTRATREES_OOF_PRED_PATH = RESULTS_DIR / "oof_extratrees_base.csv"
EXTRATREES_TEST_PRED_PATH = RESULTS_DIR / "testpred_extratrees_base_foldavg.csv"

EXTRATREES_SUBMISSION_PATH = Path("submission_extratrees_base_oof_foldavg.csv")

for path in [
    EXTRATREES_SCREEN_RESULTS_PATH,
    EXTRATREES_SCREEN_BEST_PATH,
    EXTRATREES_OOF_FOLD_RESULTS_PATH,
    EXTRATREES_OOF_SUMMARY_PATH,
    EXTRATREES_OOF_PRED_PATH,
    EXTRATREES_TEST_PRED_PATH,
    EXTRATREES_SUBMISSION_PATH,
]:
    if path.exists():
        path.unlink()

print("ExtraTrees setup:")
print("Development split:", X_tr.shape, X_val.shape)
print("Full training matrix:", X_train_proc_model.shape)
print("Test matrix:", X_test_proc_model.shape)
print("n_jobs:", EXTRATREES_N_JOBS)
print("Screen n_estimators:", EXTRATREES_SCREEN_N_ESTIMATORS)
print("OOF n_estimators:", EXTRATREES_OOF_N_ESTIMATORS)

ExtraTrees setup:
Development split: (115936, 162) (28985, 162)
Full training matrix: (144921, 162)
Test matrix: (48307, 162)
n_jobs: 1
Screen n_estimators: 180
OOF n_estimators: 250


In [169]:
assert list(X_tr.columns) == list(X_val.columns)
assert list(X_train_proc_model.columns) == list(X_test_proc_model.columns)
assert len(y_train) == len(X_train_proc_model)
assert len(test_ids) == len(X_test_proc_model)

print("ExtraTrees input checks passed.")

ExtraTrees input checks passed.


In [170]:
def make_extratrees_model(config, random_state):
    """
    Build one ExtraTrees model from a configuration dictionary.
    """
    return ExtraTreesRegressor(
        n_estimators=config["n_estimators"],
        criterion="squared_error",
        max_depth=None,
        min_samples_leaf=config["min_samples_leaf"],
        min_samples_split=2,
        max_features=config["max_features"],
        bootstrap=config["bootstrap"],
        max_samples=config.get("max_samples", None),
        n_jobs=EXTRATREES_N_JOBS,
        random_state=random_state,
    )


def id_array(ids):
    """
    Convert saved ID objects to a one-dimensional string array.
    """
    if isinstance(ids, pd.DataFrame):
        ids = ids.iloc[:, 0]

    return pd.Series(ids).astype(str).to_numpy()

##### ExtraTrees holdout screen

The holdout screen tests four ExtraTrees configurations.

The settings vary:

- `max_features`: how many predictors each split can consider,
- `min_samples_leaf`: how small terminal leaves are allowed to be,
- `bootstrap`: whether trees are trained on bootstrap samples.

The expected strongest configuration uses `max_features = 0.5`, `min_samples_leaf = 1`, and no bootstrap sampling.

In [171]:
extratrees_screen_configs = [
    {
        "config_name": "et_base_180_mf0.5_leaf1_no_bootstrap",
        "n_estimators": EXTRATREES_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 1,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.7_leaf1_no_bootstrap",
        "n_estimators": EXTRATREES_SCREEN_N_ESTIMATORS,
        "max_features": 0.70,
        "min_samples_leaf": 1,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.5_leaf2_no_bootstrap",
        "n_estimators": EXTRATREES_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 2,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.5_leaf1_bootstrap0.8",
        "n_estimators": EXTRATREES_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 1,
        "bootstrap": True,
        "max_samples": 0.80,
    },
]

pd.DataFrame(extratrees_screen_configs)

,config_name,n_estimators,max_features,min_samples_leaf,bootstrap,max_samples
0,et_base_180_mf0.5_leaf1_no_bootstrap,180,0.5,1,False,NaN
1,et_base_180_mf0.7_leaf1_no_bootstrap,180,0.7,1,False,NaN
2,et_base_180_mf0.5_leaf2_no_bootstrap,180,0.5,2,False,NaN
3,et_base_180_mf0.5_leaf1_bootstrap0.8,180,0.5,1,True,0.8


In [172]:
start_time = time.perf_counter()

X_et_screen_train = X_tr
X_et_screen_val = X_val

y_et_screen_train = np.asarray(y_tr, dtype=float)
y_et_screen_val = np.asarray(y_val, dtype=float)

extratrees_screen_rows = []

for config_id, config in enumerate(extratrees_screen_configs, start=1):
    config_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(f"ExtraTrees holdout config {config_id} / {len(extratrees_screen_configs)}")
    print(config)
    print("=" * 80)

    model = make_extratrees_model(
        config=config,
        random_state=RANDOM_STATE + 3000 + config_id
    )

    model.fit(X_et_screen_train, y_et_screen_train)

    train_pred = model.predict(X_et_screen_train)
    val_pred = model.predict(X_et_screen_val)

    row = {
        "model_class": "ExtraTrees",
        "stage": "holdout_screen",
        "feature_space": "base",
        "config_name": config["config_name"],
        "n_estimators": config["n_estimators"],
        "max_features": config["max_features"],
        "min_samples_leaf": config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": config["bootstrap"],
        "max_samples": config.get("max_samples", np.nan),
        "train_mse": mean_squared_error(y_et_screen_train, train_pred),
        "holdout_val_mse": mean_squared_error(y_et_screen_val, val_pred),
        "elapsed_sec": time.perf_counter() - config_start,
    }

    extratrees_screen_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), EXTRATREES_SCREEN_RESULTS_PATH)

    print("train_mse:", row["train_mse"])
    print("holdout_val_mse:", row["holdout_val_mse"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))

    del model, train_pred, val_pred
    gc.collect()

extratrees_screen_results = (
    pd.DataFrame(extratrees_screen_rows)
    .sort_values("holdout_val_mse")
    .reset_index(drop=True)
)

best_extratrees_screen = extratrees_screen_results.iloc[0].to_dict()

extratrees_screen_results.to_csv(EXTRATREES_SCREEN_RESULTS_PATH, index=False)
pd.DataFrame([best_extratrees_screen]).to_csv(EXTRATREES_SCREEN_BEST_PATH, index=False)

print("ExtraTrees holdout screen results:")
display(extratrees_screen_results)

print("\nBest ExtraTrees holdout configuration:")
display(pd.DataFrame([best_extratrees_screen]))

print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))


ExtraTrees holdout config 1 / 4
{'config_name': 'et_base_180_mf0.5_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse: 0.02512247710955044
holdout_val_mse: 111.62730960485095
elapsed_sec: 199.33

ExtraTrees holdout config 2 / 4
{'config_name': 'et_base_180_mf0.7_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.7, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse: 0.02512161882417886
holdout_val_mse: 111.87404767073008
elapsed_sec: 228.15

ExtraTrees holdout config 3 / 4
{'config_name': 'et_base_180_mf0.5_leaf2_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 2, 'bootstrap': False}
train_mse: 19.159447351854915
holdout_val_mse: 119.02691779822973
elapsed_sec: 139.52

ExtraTrees holdout config 4 / 4
{'config_name': 'et_base_180_mf0.5_leaf1_bootstrap0.8', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': True, 'max_samples': 0.8}
train_mse: 25.15842815592

,model_class,stage,feature_space,config_name,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,train_mse,holdout_val_mse,elapsed_sec
0,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf1_no_bootstrap,180,0.5,1,2,False,NaN,0.025122,111.627310,199.334307
1,ExtraTrees,holdout_screen,base,et_base_180_mf0.7_leaf1_no_bootstrap,180,0.7,1,2,False,NaN,0.025122,111.874048,228.153361
2,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf2_no_bootstrap,180,0.5,2,2,False,NaN,19.159447,119.026918,139.524718
3,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf1_bootstrap0.8,180,0.5,1,2,True,0.8,25.158428,123.513316,96.425045



Best ExtraTrees holdout configuration:


,model_class,stage,feature_space,config_name,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,train_mse,holdout_val_mse,elapsed_sec
0,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf1_no_bootstrap,180,0.5,1,2,False,NaN,0.025122,111.62731,199.334307



Elapsed seconds: 664.61


In [173]:
start_time = time.perf_counter()

X_et_screen_train = X_tr
X_et_screen_val = X_val

y_et_screen_train = np.asarray(y_tr, dtype=float)
y_et_screen_val = np.asarray(y_val, dtype=float)

extratrees_screen_rows = []

for config_id, config in enumerate(extratrees_screen_configs, start=1):
    config_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(f"ExtraTrees holdout config {config_id} / {len(extratrees_screen_configs)}")
    print(config)
    print("=" * 80)

    model = make_extratrees_model(
        config=config,
        random_state=RANDOM_STATE + 3000 + config_id
    )

    model.fit(X_et_screen_train, y_et_screen_train)

    train_pred = model.predict(X_et_screen_train)
    val_pred = model.predict(X_et_screen_val)

    row = {
        "model_class": "ExtraTrees",
        "stage": "holdout_screen",
        "feature_space": "base",
        "config_name": config["config_name"],
        "n_estimators": config["n_estimators"],
        "max_features": config["max_features"],
        "min_samples_leaf": config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": config["bootstrap"],
        "max_samples": config.get("max_samples", np.nan),
        "train_mse": mean_squared_error(y_et_screen_train, train_pred),
        "holdout_val_mse": mean_squared_error(y_et_screen_val, val_pred),
        "elapsed_sec": time.perf_counter() - config_start,
    }

    extratrees_screen_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), EXTRATREES_SCREEN_RESULTS_PATH)

    print("train_mse:", row["train_mse"])
    print("holdout_val_mse:", row["holdout_val_mse"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))

    del model, train_pred, val_pred
    gc.collect()

extratrees_screen_results = (
    pd.DataFrame(extratrees_screen_rows)
    .sort_values("holdout_val_mse")
    .reset_index(drop=True)
)

best_extratrees_screen = extratrees_screen_results.iloc[0].to_dict()

extratrees_screen_results.to_csv(EXTRATREES_SCREEN_RESULTS_PATH, index=False)
pd.DataFrame([best_extratrees_screen]).to_csv(EXTRATREES_SCREEN_BEST_PATH, index=False)

print("ExtraTrees holdout screen results:")
display(extratrees_screen_results)

print("\nBest ExtraTrees holdout configuration:")
display(pd.DataFrame([best_extratrees_screen]))

print("\nElapsed seconds:", round(time.perf_counter() - start_time, 2))


ExtraTrees holdout config 1 / 4
{'config_name': 'et_base_180_mf0.5_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse: 0.02512247710955044
holdout_val_mse: 111.62730960485095
elapsed_sec: 176.67

ExtraTrees holdout config 2 / 4
{'config_name': 'et_base_180_mf0.7_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.7, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse: 0.02512161882417886
holdout_val_mse: 111.87404767073008
elapsed_sec: 240.69

ExtraTrees holdout config 3 / 4
{'config_name': 'et_base_180_mf0.5_leaf2_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 2, 'bootstrap': False}
train_mse: 19.159447351854915
holdout_val_mse: 119.02691779822973
elapsed_sec: 151.97

ExtraTrees holdout config 4 / 4
{'config_name': 'et_base_180_mf0.5_leaf1_bootstrap0.8', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': True, 'max_samples': 0.8}
train_mse: 25.15842815592

,model_class,stage,feature_space,config_name,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,train_mse,holdout_val_mse,elapsed_sec
0,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf1_no_bootstrap,180,0.5,1,2,False,NaN,0.025122,111.627310,176.674511
1,ExtraTrees,holdout_screen,base,et_base_180_mf0.7_leaf1_no_bootstrap,180,0.7,1,2,False,NaN,0.025122,111.874048,240.692870
2,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf2_no_bootstrap,180,0.5,2,2,False,NaN,19.159447,119.026918,151.969575
3,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf1_bootstrap0.8,180,0.5,1,2,True,0.8,25.158428,123.513316,110.879818



Best ExtraTrees holdout configuration:


,model_class,stage,feature_space,config_name,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,train_mse,holdout_val_mse,elapsed_sec
0,ExtraTrees,holdout_screen,base,et_base_180_mf0.5_leaf1_no_bootstrap,180,0.5,1,2,False,NaN,0.025122,111.62731,176.674511



Elapsed seconds: 681.14


In [174]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Bagging",
                "model": "extratrees_holdout_screen",
                "feature_space": "base",
                "metric_source": "fixed holdout",
                "train_mse": best_extratrees_screen["train_mse"],
                "val_mse": best_extratrees_screen["holdout_val_mse"],
                "notes": (
                    f"n_estimators={int(best_extratrees_screen['n_estimators'])}; "
                    f"max_features={best_extratrees_screen['max_features']}; "
                    f"min_samples_leaf={int(best_extratrees_screen['min_samples_leaf'])}; "
                    f"bootstrap={best_extratrees_screen['bootstrap']}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(15)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
1,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
2,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
3,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...
4,Tree,regression_tree_holdout_screen,base_plus_poly,fixed holdout,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522
5,Tree,regression_tree_cv,base_plus_poly_interactions,5-fold CV mean,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....
6,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.688365,290.058976,original Lasso-step search result; not rerun
7,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.664053,290.079273,original Ridge-step search result; not rerun
8,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,fixed holdout / recorded,281.634289,290.109374,focused step search; candidates=30; bins=30; s...
9,Step functions,expanded_linear_regression_with_steps,base_plus_poly_interactions,fixed holdout / recorded,289.144890,296.137338,expanded step search; candidates=20; bins=20; ...


The ExtraTrees holdout screen improves sharply over the 500-tree random forest.

The best holdout configuration uses the base feature matrix, `max_features = 0.5`, `min_samples_leaf = 1`, and no bootstrap sampling.

The next cell trains this configuration as a 5-fold out-of-fold artifact.

##### ExtraTrees out-of-fold artifact

The selected ExtraTrees configuration is trained using 5-fold out-of-fold prediction.

For each fold:

1. fit ExtraTrees on four folds,
2. predict the held-out fold,
3. predict the test set,
4. average the five test predictions.

This gives one OOF prediction for every training row and one averaged prediction for every test row.

In [175]:
selected_extratrees_config = [
    config for config in extratrees_screen_configs
    if config["config_name"] == best_extratrees_screen["config_name"]
][0].copy()

selected_extratrees_config["config_name"] = (
    "et_oof_from_" + selected_extratrees_config["config_name"]
)

selected_extratrees_config["n_estimators"] = EXTRATREES_OOF_N_ESTIMATORS

print("Selected ExtraTrees OOF configuration:")
print(selected_extratrees_config)

Selected ExtraTrees OOF configuration:
{'config_name': 'et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap', 'n_estimators': 250, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': False}


In [176]:
start_time = time.perf_counter()

X_et_all = X_train_proc_model
X_et_test = X_test_proc_model
y_et_all = np.asarray(y_train, dtype=float)

extratrees_oof_pred = np.full(len(X_et_all), np.nan, dtype=float)
extratrees_test_pred_sum = np.zeros(len(X_et_test), dtype=float)

extratrees_oof_fold_rows = []

kf_extratrees = KFold(
    n_splits=EXTRATREES_OOF_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf_extratrees.split(X_et_all), start=1):
    fold_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(f"ExtraTrees OOF fold {fold_id} / {EXTRATREES_OOF_N_SPLITS}")
    print("=" * 80)

    model = make_extratrees_model(
        config=selected_extratrees_config,
        random_state=RANDOM_STATE + 4000 + fold_id
    )

    model.fit(
        X_et_all.iloc[fold_train_idx],
        y_et_all[fold_train_idx]
    )

    valid_pred = model.predict(
        X_et_all.iloc[fold_valid_idx]
    )

    test_pred = model.predict(
        X_et_test
    )

    extratrees_oof_pred[fold_valid_idx] = valid_pred
    extratrees_test_pred_sum += test_pred / EXTRATREES_OOF_N_SPLITS

    row = {
        "model_class": "ExtraTrees",
        "stage": "oof_selected_config",
        "feature_space": "base",
        "fold": fold_id,
        "config_name": selected_extratrees_config["config_name"],
        "n_estimators": selected_extratrees_config["n_estimators"],
        "max_features": selected_extratrees_config["max_features"],
        "min_samples_leaf": selected_extratrees_config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": selected_extratrees_config["bootstrap"],
        "max_samples": selected_extratrees_config.get("max_samples", np.nan),
        "fold_valid_mse": mean_squared_error(
            y_et_all[fold_valid_idx],
            valid_pred
        ),
        "elapsed_sec": time.perf_counter() - fold_start,
    }

    extratrees_oof_fold_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), EXTRATREES_OOF_FOLD_RESULTS_PATH)

    print("fold_valid_mse:", row["fold_valid_mse"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))
    print(
        "OOF predictions filled:",
        np.isfinite(extratrees_oof_pred).sum(),
        "/",
        len(extratrees_oof_pred)
    )

    del model, valid_pred, test_pred
    gc.collect()

assert np.isfinite(extratrees_oof_pred).all()

extratrees_oof_fold_results = pd.DataFrame(extratrees_oof_fold_rows)
extratrees_oof_fold_results.to_csv(EXTRATREES_OOF_FOLD_RESULTS_PATH, index=False)

extratrees_oof_mse = mean_squared_error(y_et_all, extratrees_oof_pred)

extratrees_oof_pred_clipped = np.clip(extratrees_oof_pred, 0, 100)
extratrees_test_pred_avg = np.clip(extratrees_test_pred_sum, 0, 100)

extratrees_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(train_ids),
    "y_true": y_et_all,
    "extratrees_oof_pred": extratrees_oof_pred,
    "extratrees_oof_pred_clipped": extratrees_oof_pred_clipped,
})

extratrees_test_pred_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "PERCENT_PROFICIENT": extratrees_test_pred_avg,
})

extratrees_oof_df.to_csv(EXTRATREES_OOF_PRED_PATH, index=False)
extratrees_test_pred_df.to_csv(EXTRATREES_TEST_PRED_PATH, index=False)
extratrees_test_pred_df.to_csv(EXTRATREES_SUBMISSION_PATH, index=False)

extratrees_oof_summary = pd.DataFrame([
    {
        "model_key": "ExtraTrees OOF base",
        "model_class": "ExtraTrees",
        "feature_space": "base",
        "screen_holdout_val_mse": float(best_extratrees_screen["holdout_val_mse"]),
        "oof_mse": float(extratrees_oof_mse),
        "fold_valid_mse_mean": float(extratrees_oof_fold_results["fold_valid_mse"].mean()),
        "fold_valid_mse_std": float(extratrees_oof_fold_results["fold_valid_mse"].std()),
        "n_splits": EXTRATREES_OOF_N_SPLITS,
        "n_estimators": EXTRATREES_OOF_N_ESTIMATORS,
        "max_features": selected_extratrees_config["max_features"],
        "min_samples_leaf": selected_extratrees_config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": selected_extratrees_config["bootstrap"],
        "max_samples": selected_extratrees_config.get("max_samples", np.nan),
        "n_jobs": EXTRATREES_N_JOBS,
        "submission_path": str(EXTRATREES_SUBMISSION_PATH),
        "elapsed_sec": time.perf_counter() - start_time,
    }
])

extratrees_oof_summary.to_csv(EXTRATREES_OOF_SUMMARY_PATH, index=False)

print("ExtraTrees OOF fold results:")
display(extratrees_oof_fold_results)

print("\nExtraTrees OOF summary:")
display(extratrees_oof_summary)

print("\nSaved files:")
print(EXTRATREES_OOF_FOLD_RESULTS_PATH)
print(EXTRATREES_OOF_SUMMARY_PATH)
print(EXTRATREES_OOF_PRED_PATH)
print(EXTRATREES_TEST_PRED_PATH)
print(EXTRATREES_SUBMISSION_PATH)


ExtraTrees OOF fold 1 / 5
fold_valid_mse: 111.63933039144383
elapsed_sec: 241.6
OOF predictions filled: 28985 / 144921

ExtraTrees OOF fold 2 / 5
fold_valid_mse: 109.15533429271322
elapsed_sec: 251.11
OOF predictions filled: 57969 / 144921

ExtraTrees OOF fold 3 / 5
fold_valid_mse: 111.87028989470052
elapsed_sec: 227.83
OOF predictions filled: 86953 / 144921

ExtraTrees OOF fold 4 / 5
fold_valid_mse: 108.40540941995584
elapsed_sec: 226.7
OOF predictions filled: 115937 / 144921

ExtraTrees OOF fold 5 / 5
fold_valid_mse: 112.6760763288711
elapsed_sec: 274.73
OOF predictions filled: 144921 / 144921
ExtraTrees OOF fold results:


,model_class,stage,feature_space,fold,config_name,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,fold_valid_mse,elapsed_sec
0,ExtraTrees,oof_selected_config,base,1,et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap,250,0.5,1,2,False,NaN,111.639330,241.595767
1,ExtraTrees,oof_selected_config,base,2,et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap,250,0.5,1,2,False,NaN,109.155334,251.110072
2,ExtraTrees,oof_selected_config,base,3,et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap,250,0.5,1,2,False,NaN,111.870290,227.831579
3,ExtraTrees,oof_selected_config,base,4,et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap,250,0.5,1,2,False,NaN,108.405409,226.703315
4,ExtraTrees,oof_selected_config,base,5,et_oof_from_et_base_180_mf0.5_leaf1_no_bootstrap,250,0.5,1,2,False,NaN,112.676076,274.730880



ExtraTrees OOF summary:


,model_key,model_class,feature_space,screen_holdout_val_mse,oof_mse,fold_valid_mse_mean,fold_valid_mse_std,n_splits,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,n_jobs,submission_path,elapsed_sec
0,ExtraTrees OOF base,ExtraTrees,base,111.62731,110.749294,110.749288,1.857134,5,250,0.5,1,2,False,NaN,1,submission_extratrees_base_oof_foldavg.csv,1225.590434



Saved files:
model_results/extratrees_oof_fold_results.csv
model_results/extratrees_oof_summary.csv
model_results/oof_extratrees_base.csv
model_results/testpred_extratrees_base_foldavg.csv
submission_extratrees_base_oof_foldavg.csv


In [177]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Bagging",
                "model": "extratrees_oof_base",
                "feature_space": "base",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": extratrees_oof_mse,
                "notes": (
                    f"n_estimators={EXTRATREES_OOF_N_ESTIMATORS}; "
                    f"max_features={selected_extratrees_config['max_features']}; "
                    f"min_samples_leaf={selected_extratrees_config['min_samples_leaf']}; "
                    f"bootstrap={selected_extratrees_config['bootstrap']}; "
                    f"n_jobs={EXTRATREES_N_JOBS}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(15)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
1,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
2,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
3,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
4,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...
5,Tree,regression_tree_holdout_screen,base_plus_poly,fixed holdout,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522
6,Tree,regression_tree_cv,base_plus_poly_interactions,5-fold CV mean,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....
7,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.688365,290.058976,original Lasso-step search result; not rerun
8,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.664053,290.079273,original Ridge-step search result; not rerun
9,Step functions,focused_expanded_linear_regression_with_steps,base_plus_poly_interactions,fixed holdout / recorded,281.634289,290.109374,focused step search; candidates=30; bins=30; s...


ExtraTrees improves over the 500-tree random forest.

The random forest OOF MSE was about 122.33. ExtraTrees OOF MSE is about 110.75.

This is a large gain, so ExtraTrees is kept as a fully runnable part of the cleaned notebook.

#### 4E. Random forest + ExtraTrees blend

The random forest and ExtraTrees models are both tree ensembles, but they use different randomization strategies.

Random forest reached OOF MSE around 122.33.

ExtraTrees reached OOF MSE around 110.75.

Since the two models may make different errors, we test a simple weighted average of their out-of-fold predictions.

The blend has the form:

`blend prediction = w_rf × random forest prediction + w_et × ExtraTrees prediction`

where: w_rf + w_et = 1

In [179]:
rf_oof_for_blend = pd.read_csv(RF_OOF_PATH)
rf_test_for_blend = pd.read_csv(RF_TEST_FOLD_PATH)

extratrees_oof_for_blend = pd.read_csv(EXTRATREES_OOF_PRED_PATH)
extratrees_test_for_blend = pd.read_csv(EXTRATREES_TEST_PRED_PATH)

print("RF OOF shape:", rf_oof_for_blend.shape)
print("ExtraTrees OOF shape:", extratrees_oof_for_blend.shape)
print("RF test shape:", rf_test_for_blend.shape)
print("ExtraTrees test shape:", extratrees_test_for_blend.shape)

RF OOF shape: (144921, 3)
ExtraTrees OOF shape: (144921, 4)
RF test shape: (48307, 7)
ExtraTrees test shape: (48307, 2)


In [182]:
rf_blend_oof = rf_oof_for_blend[
    ["ASSESSMENT_ID", "y_true", "rf_500_base_oof_pred"]
].copy()

rf_blend_oof["ASSESSMENT_ID"] = rf_blend_oof["ASSESSMENT_ID"].astype(str)

extratrees_blend_oof = extratrees_oof_for_blend[
    ["ASSESSMENT_ID", "y_true", "extratrees_oof_pred_clipped"]
].copy()

extratrees_blend_oof["ASSESSMENT_ID"] = extratrees_blend_oof["ASSESSMENT_ID"].astype(str)

blend_oof_data = rf_blend_oof.merge(
    extratrees_blend_oof,
    on="ASSESSMENT_ID",
    how="inner",
    suffixes=("_rf", "_extratrees")
)

print("Blended OOF rows:", blend_oof_data.shape[0])
print("Expected training rows:", len(train_ids))

print("\nTargets aligned:")
print(np.allclose(
    blend_oof_data["y_true_rf"],
    blend_oof_data["y_true_extratrees"]
))

Blended OOF rows: 144921
Expected training rows: 144921

Targets aligned:
True


In [183]:
if blend_oof_data.shape[0] != len(train_ids):
    raise ValueError("OOF blend data does not contain the expected number of training rows.")

if not np.allclose(
    blend_oof_data["y_true_rf"],
    blend_oof_data["y_true_extratrees"]
):
    raise ValueError("RF and ExtraTrees OOF target columns are not aligned.")

blend_oof_data["y_true"] = blend_oof_data["y_true_rf"]

y_blend = blend_oof_data["y_true"].to_numpy(dtype=float)

rf_blend_pred = blend_oof_data["rf_500_base_oof_pred"].to_numpy(dtype=float)
extratrees_blend_pred = blend_oof_data["extratrees_oof_pred_clipped"].to_numpy(dtype=float)

print("RF OOF MSE:")
print(mean_squared_error(y_blend, np.clip(rf_blend_pred, 0, 100)))

print("\nExtraTrees OOF MSE:")
print(mean_squared_error(y_blend, np.clip(extratrees_blend_pred, 0, 100)))

RF OOF MSE:
122.32802447555102

ExtraTrees OOF MSE:
110.74929420710593


In [184]:
blend_weight_rows = []

for w_extratrees in np.linspace(0, 1, 1001):
    w_rf = 1.0 - w_extratrees

    blend_pred_raw = (
        w_rf * rf_blend_pred
        + w_extratrees * extratrees_blend_pred
    )

    blend_pred_clipped = np.clip(
        blend_pred_raw,
        0,
        100
    )

    blend_weight_rows.append({
        "w_rf": w_rf,
        "w_extratrees": w_extratrees,
        "oof_mse_raw": mean_squared_error(y_blend, blend_pred_raw),
        "oof_mse_clipped": mean_squared_error(y_blend, blend_pred_clipped),
    })

rf_extratrees_blend_screen = (
    pd.DataFrame(blend_weight_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

best_rf_extratrees_blend = rf_extratrees_blend_screen.iloc[0]

best_rf_extratrees_blend

w_rf                 0.152000
w_extratrees         0.848000
oof_mse_raw        110.363092
oof_mse_clipped    110.363092
Name: 0, dtype: float64

In [185]:
rf_extratrees_blend_screen.head(10)

,w_rf,w_extratrees,oof_mse_raw,oof_mse_clipped
0,0.152,0.848,110.363092,110.363092
1,0.153,0.847,110.363099,110.363099
2,0.151,0.849,110.363118,110.363118
3,0.154,0.846,110.363139,110.363139
4,0.150,0.850,110.363178,110.363178
5,0.155,0.845,110.363212,110.363212
6,0.149,0.851,110.363272,110.363272
7,0.156,0.844,110.363318,110.363318
8,0.148,0.852,110.363398,110.363398
9,0.157,0.843,110.363458,110.363458


The best blend puts most of the weight on ExtraTrees and a smaller weight on random forest.

This makes sense because ExtraTrees is the stronger individual model. The random forest still receives some weight because it contributes a slightly different prediction pattern.

In [186]:
best_w_rf = float(best_rf_extratrees_blend["w_rf"])
best_w_extratrees = float(best_rf_extratrees_blend["w_extratrees"])

blend_oof_data["blend_pred_raw"] = (
    best_w_rf * rf_blend_pred
    + best_w_extratrees * extratrees_blend_pred
)

blend_oof_data["blend_pred_clipped"] = np.clip(
    blend_oof_data["blend_pred_raw"],
    0,
    100
)

rf_extratrees_blend_oof_mse = mean_squared_error(
    blend_oof_data["y_true"],
    blend_oof_data["blend_pred_clipped"]
)

RF_EXTRATREES_BLEND_SCREEN_PATH = RESULTS_DIR / "blend_rf500_extratrees_weight_screen.csv"
RF_EXTRATREES_BLEND_OOF_PATH = RESULTS_DIR / "oof_blend_rf500_extratrees.csv"

rf_extratrees_blend_screen.to_csv(
    RF_EXTRATREES_BLEND_SCREEN_PATH,
    index=False
)

blend_oof_data.to_csv(
    RF_EXTRATREES_BLEND_OOF_PATH,
    index=False
)

print("Best RF weight:", best_w_rf)
print("Best ExtraTrees weight:", best_w_extratrees)
print("RF + ExtraTrees blend OOF MSE:", rf_extratrees_blend_oof_mse)

Best RF weight: 0.15200000000000002
Best ExtraTrees weight: 0.848
RF + ExtraTrees blend OOF MSE: 110.36309185851032


The RF + ExtraTrees blend gives a small OOF improvement over ExtraTrees alone.

The improvement is modest, but the blend is inexpensive because both OOF prediction files already exist.

In [187]:
rf_test_pred_col = "rf_500_base_foldavg_pred"

rf_test_blend = rf_test_for_blend[
    ["ASSESSMENT_ID", rf_test_pred_col]
].copy()

rf_test_blend["ASSESSMENT_ID"] = rf_test_blend["ASSESSMENT_ID"].astype(str)

rf_test_blend = rf_test_blend.rename(
    columns={rf_test_pred_col: "rf_test_pred"}
)

extratrees_test_blend = extratrees_test_for_blend[
    ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
].copy()

extratrees_test_blend["ASSESSMENT_ID"] = extratrees_test_blend["ASSESSMENT_ID"].astype(str)

extratrees_test_blend = extratrees_test_blend.rename(
    columns={"PERCENT_PROFICIENT": "extratrees_test_pred"}
)

blend_test_data = rf_test_blend.merge(
    extratrees_test_blend,
    on="ASSESSMENT_ID",
    how="inner"
)

print("Blend test rows:", blend_test_data.shape[0])
print("Expected test rows:", len(test_ids))

Blend test rows: 48307
Expected test rows: 48307


In [188]:
if blend_test_data.shape[0] != len(test_ids):
    raise ValueError("Blend test data does not contain the expected number of test rows.")

test_order = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "_test_order": np.arange(len(test_ids))
})

blend_test_data = test_order.merge(
    blend_test_data,
    on="ASSESSMENT_ID",
    how="left"
)

if blend_test_data["rf_test_pred"].isna().sum() > 0:
    raise ValueError("RF test predictions are missing after alignment.")

if blend_test_data["extratrees_test_pred"].isna().sum() > 0:
    raise ValueError("ExtraTrees test predictions are missing after alignment.")

blend_test_data = (
    blend_test_data
    .sort_values("_test_order")
    .drop(columns=["_test_order"])
    .reset_index(drop=True)
)

blend_test_data["PERCENT_PROFICIENT"] = np.clip(
    best_w_rf * blend_test_data["rf_test_pred"]
    + best_w_extratrees * blend_test_data["extratrees_test_pred"],
    0,
    100
)

RF_EXTRATREES_BLEND_TEST_PATH = RESULTS_DIR / "testpred_blend_rf500_extratrees.csv"
RF_EXTRATREES_BLEND_SUBMISSION_PATH = Path("submission_blend_rf500_extratrees_oof_weighted.csv")

blend_test_data.to_csv(
    RF_EXTRATREES_BLEND_TEST_PATH,
    index=False
)

submission_rf_extratrees_blend = blend_test_data[
    ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
].copy()

submission_rf_extratrees_blend.to_csv(
    RF_EXTRATREES_BLEND_SUBMISSION_PATH,
    index=False
)

print("Blend test prediction shape:", blend_test_data.shape)
print("Blend submission shape:", submission_rf_extratrees_blend.shape)

print("\nSaved:")
print(RF_EXTRATREES_BLEND_TEST_PATH)
print(RF_EXTRATREES_BLEND_SUBMISSION_PATH)

Blend test prediction shape: (48307, 4)
Blend submission shape: (48307, 2)

Saved:
model_results/testpred_blend_rf500_extratrees.csv
submission_blend_rf500_extratrees_oof_weighted.csv


In [189]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Bagging",
                "model": "rf_extratrees_blend",
                "feature_space": "OOF prediction blend",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": rf_extratrees_blend_oof_mse,
                "notes": (
                    f"w_rf={best_w_rf:.3f}; "
                    f"w_extratrees={best_w_extratrees:.3f}; "
                    "RF 500 base + ExtraTrees base"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
1,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
2,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
3,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
4,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
5,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...
6,Tree,regression_tree_holdout_screen,base_plus_poly,fixed holdout,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522
7,Tree,regression_tree_cv,base_plus_poly_interactions,5-fold CV mean,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....
8,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.688365,290.058976,original Lasso-step search result; not rerun
9,Step functions,ridge_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.664053,290.079273,original Ridge-step search result; not rerun


The RF + ExtraTrees blend is the strongest bagging-family result.

The next modelling stage moves from bagging to boosting. Boosting fits trees sequentially, with each new tree focusing on errors from earlier trees.

## Data modelling, Part 5: Base-feature LightGBM boosting and saved-prediction blends

The best bagging-family result so far is the Random Forest + ExtraTrees blend, with OOF MSE around 110.36.

The next model family is gradient boosting, starting with LightGBM.

Bagging methods, such as Random Forest and ExtraTrees, average many independently trained trees. Boosting works differently: it builds trees sequentially, where later trees focus on mistakes made by earlier trees.

This section uses the base engineered feature matrix. The earlier tree-ensemble experiments showed that the base feature matrix worked best for tree models, so LightGBM also starts from the base feature matrix.

The LightGBM workflow has two stages:

1. a broad holdout screen,
2. a targeted holdout refinement around the strongest configuration.

The selected configuration will then be promoted to a 5-fold OOF artifact in the next subsection.

In [191]:
from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error

try:
    import lightgbm as lgb
except ImportError as err:
    raise ImportError(
        "LightGBM is required for this section. Install it before running the LightGBM cells."
    ) from err

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


#### LightGBM input setup

LightGBM is run on NumPy `float32` arrays rather than pandas DataFrames.

This keeps memory usage lower and avoids repeated conversion inside model fitting.

In [192]:
X_train_lgb_holdout = np.ascontiguousarray(
    np.asarray(X_tr, dtype=np.float32)
)

X_val_lgb_holdout = np.ascontiguousarray(
    np.asarray(X_val, dtype=np.float32)
)

y_train_lgb_holdout = np.asarray(
    y_tr,
    dtype=np.float32
).ravel()

y_val_lgb_holdout = np.asarray(
    y_val,
    dtype=np.float32
).ravel()

print("LightGBM holdout train shape:", X_train_lgb_holdout.shape)
print("LightGBM holdout validation shape:", X_val_lgb_holdout.shape)

LightGBM holdout train shape: (115936, 162)
LightGBM holdout validation shape: (28985, 162)


#### LightGBM holdout-screen helper

The same holdout-screen logic is used twice:

1. first for a broad screen,
2. then for a targeted refinement.

The helper below fits each configuration, evaluates train and validation MSE, saves each result immediately, and returns a sorted result table.

The function is kept because it is used more than once in this LightGBM section.

In [193]:
def run_lgbm_holdout_screen(
    configs,
    output_path,
    screen_name,
    early_stopping_rounds,
    log_period,
):
    """
    Run a checkpointed LightGBM holdout screen.

    Each configuration is fit on X_train_lgb_holdout and evaluated on X_val_lgb_holdout.
    Results are saved after every configuration.
    """
    output_path = Path(output_path)

    if output_path.exists():
        completed = pd.read_csv(output_path)
        completed_configs = set(completed["config_name"].astype(str))
        result_rows = completed.to_dict("records")
    else:
        completed_configs = set()
        result_rows = []

    print("Screen:", screen_name)
    print("Checkpoint:", output_path)
    print("Already completed configs:", sorted(completed_configs))

    common_params = {
        "objective": "regression",
        "metric": "l2",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
        "verbosity": -1,
        "force_col_wise": True,
    }

    for config in configs:
        config_name = config["config_name"]

        if config_name in completed_configs:
            print("\nSkipping completed config:", config_name)
            continue

        print("\n" + "=" * 80)
        print("Running LightGBM config:", config_name)
        print("=" * 80)

        start_time = time.perf_counter()

        model_params = common_params.copy()
        model_params.update({
            key: value
            for key, value in config.items()
            if key != "config_name"
        })

        model = lgb.LGBMRegressor(**model_params)

        model.fit(
            X_train_lgb_holdout,
            y_train_lgb_holdout,
            eval_set=[(X_val_lgb_holdout, y_val_lgb_holdout)],
            eval_metric="l2",
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=early_stopping_rounds,
                    verbose=True
                ),
                lgb.log_evaluation(period=log_period),
            ],
        )

        best_iteration = model.best_iteration_

        train_pred_raw = model.predict(
            X_train_lgb_holdout,
            num_iteration=best_iteration
        )

        val_pred_raw = model.predict(
            X_val_lgb_holdout,
            num_iteration=best_iteration
        )

        train_pred_clipped = np.clip(train_pred_raw, 0, 100)
        val_pred_clipped = np.clip(val_pred_raw, 0, 100)

        row = {
            "screen_name": screen_name,
            "config_name": config_name,
            "best_iteration": best_iteration,
            "train_mse_raw": mean_squared_error(y_train_lgb_holdout, train_pred_raw),
            "train_mse_clipped": mean_squared_error(y_train_lgb_holdout, train_pred_clipped),
            "holdout_val_mse_raw": mean_squared_error(y_val_lgb_holdout, val_pred_raw),
            "holdout_val_mse_clipped": mean_squared_error(y_val_lgb_holdout, val_pred_clipped),
            "elapsed_sec": time.perf_counter() - start_time,
        }

        row.update(model_params)

        result_rows.append(row)

        pd.DataFrame(result_rows).to_csv(output_path, index=False)

        print("Completed:", config_name)
        print("Best iteration:", best_iteration)
        print("Train MSE clipped:", row["train_mse_clipped"])
        print("Holdout MSE clipped:", row["holdout_val_mse_clipped"])
        print("Elapsed seconds:", round(row["elapsed_sec"], 2))

        del model, train_pred_raw, val_pred_raw, train_pred_clipped, val_pred_clipped
        gc.collect()

    results = (
        pd.DataFrame(result_rows)
        .sort_values("holdout_val_mse_clipped")
        .reset_index(drop=True)
    )

    results.to_csv(output_path, index=False)

    return results

#### 5A. Broad LightGBM holdout screen

The first LightGBM screen tests a small set of broad configurations.

The configurations vary:

- number of leaves,
- minimum child samples,
- learning rate,
- row and column subsampling,
- L2 regularization.

These settings control how flexible the boosted trees are. The goal is not exhaustive tuning; the goal is to identify a strong region of the LightGBM parameter space.

child samples refers to the data points assigned to a resulting leaf or node after a split.

In [194]:
lgbm_broad_configs = [
    {
        "config_name": "lgbm_broad_l63_child80_l2_5",
        "n_estimators": 2500,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_child_samples": 80,
        "subsample": 0.80,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "max_depth": -1,
    },
    {
        "config_name": "lgbm_broad_l63_child120_l2_10",
        "n_estimators": 2500,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_child_samples": 120,
        "subsample": 0.80,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.0,
        "reg_lambda": 10.0,
        "max_depth": -1,
    },
    {
        "config_name": "lgbm_broad_l63_child80_l2_10_lr02",
        "n_estimators": 3500,
        "learning_rate": 0.02,
        "num_leaves": 63,
        "min_child_samples": 80,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.0,
        "reg_lambda": 10.0,
        "max_depth": -1,
    },
    {
        "config_name": "lgbm_broad_l127_child120_l2_20",
        "n_estimators": 2500,
        "learning_rate": 0.03,
        "num_leaves": 127,
        "min_child_samples": 120,
        "subsample": 0.80,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.0,
        "reg_lambda": 20.0,
        "max_depth": -1,
    },
]

pd.DataFrame(lgbm_broad_configs)

,config_name,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,max_depth
0,lgbm_broad_l63_child80_l2_5,2500,0.03,63,80,0.80,1,0.80,0.0,5.0,-1
1,lgbm_broad_l63_child120_l2_10,2500,0.03,63,120,0.80,1,0.80,0.0,10.0,-1
2,lgbm_broad_l63_child80_l2_10_lr02,3500,0.02,63,80,0.85,1,0.85,0.0,10.0,-1
3,lgbm_broad_l127_child120_l2_20,2500,0.03,127,120,0.80,1,0.80,0.0,20.0,-1


In [195]:
LGBM_BROAD_SCREEN_PATH = RESULTS_DIR / "lgbm_broad_base_holdout_screen.csv"

lgbm_broad_results = run_lgbm_holdout_screen(
    configs=lgbm_broad_configs,
    output_path=LGBM_BROAD_SCREEN_PATH,
    screen_name="broad_base_holdout_screen",
    early_stopping_rounds=100,
    log_period=250,
)

lgbm_broad_results[
    [
        "config_name",
        "train_mse_clipped",
        "holdout_val_mse_clipped",
        "best_iteration",
        "elapsed_sec",
        "n_estimators",
        "learning_rate",
        "num_leaves",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
    ]
]

Screen: broad_base_holdout_screen
Checkpoint: model_results/lgbm_broad_base_holdout_screen.csv
Already completed configs: []

Running LightGBM config: lgbm_broad_l63_child80_l2_5
Training until validation scores don't improve for 100 rounds
[250]	valid_0's l2: 199.162
[500]	valid_0's l2: 174.406
[750]	valid_0's l2: 163.192
[1000]	valid_0's l2: 156.177
[1250]	valid_0's l2: 151.088
[1500]	valid_0's l2: 147.206
[1750]	valid_0's l2: 144.006
[2000]	valid_0's l2: 141.114
[2250]	valid_0's l2: 138.747
[2500]	valid_0's l2: 136.724
Did not meet early stopping. Best iteration is:
[2500]	valid_0's l2: 136.724
Completed: lgbm_broad_l63_child80_l2_5
Best iteration: 2500
Train MSE clipped: 92.78294057631074
Holdout MSE clipped: 136.40299287236442
Elapsed seconds: 91.59

Running LightGBM config: lgbm_broad_l63_child120_l2_10
Training until validation scores don't improve for 100 rounds
[250]	valid_0's l2: 200.153
[500]	valid_0's l2: 175.401
[750]	valid_0's l2: 164.531
[1000]	valid_0's l2: 157.828
[125

,config_name,train_mse_clipped,holdout_val_mse_clipped,best_iteration,elapsed_sec,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda
0,lgbm_broad_l127_child120_l2_20,70.014712,126.393035,2500,150.242053,2500,0.03,127,120,0.80,0.80,20.0
1,lgbm_broad_l63_child80_l2_5,92.782941,136.402993,2500,91.589604,2500,0.03,63,80,0.80,0.80,5.0
2,lgbm_broad_l63_child80_l2_10_lr02,95.873512,137.583865,3500,141.868279,3500,0.02,63,80,0.85,0.85,10.0
3,lgbm_broad_l63_child120_l2_10,97.946833,139.495044,2500,100.050993,2500,0.03,63,120,0.80,0.80,10.0


In [196]:
best_lgbm_broad = lgbm_broad_results.iloc[0]

model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Boosting",
                "model": "lightgbm_broad_holdout_screen",
                "feature_space": "base",
                "metric_source": "fixed holdout",
                "train_mse": best_lgbm_broad["train_mse_clipped"],
                "val_mse": best_lgbm_broad["holdout_val_mse_clipped"],
                "notes": (
                    f"{best_lgbm_broad['config_name']}; "
                    f"best_iteration={int(best_lgbm_broad['best_iteration'])}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = model_scoreboard.sort_values("val_mse").reset_index(drop=True)

model_scoreboard.head(15)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
1,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
2,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
3,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
4,Boosting,lightgbm_broad_holdout_screen,base,fixed holdout,70.014712,126.393035,lgbm_broad_l127_child120_l2_20; best_iteration...
5,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
6,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...
7,Tree,regression_tree_holdout_screen,base_plus_poly,fixed holdout,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522
8,Tree,regression_tree_cv,base_plus_poly_interactions,5-fold CV mean,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....
9,Step functions,lasso_with_steps,base_plus_poly_interactions_plus_steps,fixed holdout / recorded,281.688365,290.058976,original Lasso-step search result; not rerun


The broad LightGBM screen shows that boosting is competitive, but the best broad configuration is not yet better than ExtraTrees.

The strongest broad result uses more leaves and stronger L2 regularization. The next screen refines around that region by testing deeper boosting runs with more trees.

#### 5B. Selected LightGBM OOF artifact

The targeted holdout refinement selected the following LightGBM configuration:

- `learning_rate = 0.03`
- `num_leaves = 95`
- `min_child_samples = 60`
- `subsample = 0.85`
- `colsample_bytree = 0.90`
- `reg_lambda = 5.0`

The original workflow then promoted this configuration to a full 5-fold OOF artifact with a larger tree cap:

- `n_estimators = 12000`
- `early_stopping_rounds = 500`

This section recreates that selected LightGBM artifact.

For each fold:

1. train LightGBM on four folds,
2. predict the held-out fold,
3. predict the test set,
4. average the five test predictions.

The OOF predictions are saved for later blending.

In [207]:
LGBM_SELECTED_ARTIFACT = "lgbm_t03_base_5fold_oof"

LGBM_SELECTED_METRICS_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_fold_metrics.csv"
LGBM_SELECTED_OOF_NPY_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_oof.npy"
LGBM_SELECTED_TEST_SUM_NPY_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_test_pred_sum.npy"

LGBM_SELECTED_OOF_PATH = RESULTS_DIR / f"oof_{LGBM_SELECTED_ARTIFACT}.csv"
LGBM_SELECTED_TEST_PRED_PATH = RESULTS_DIR / f"testpred_{LGBM_SELECTED_ARTIFACT}_foldavg.csv"
LGBM_SELECTED_SUBMISSION_PATH = Path(f"submission_{LGBM_SELECTED_ARTIFACT}_foldavg.csv")

print("Selected LightGBM artifact:", LGBM_SELECTED_ARTIFACT)
print("Metrics path:", LGBM_SELECTED_METRICS_PATH)
print("OOF path:", LGBM_SELECTED_OOF_PATH)
print("Test prediction path:", LGBM_SELECTED_TEST_PRED_PATH)
print("Submission path:", LGBM_SELECTED_SUBMISSION_PATH)

Selected LightGBM artifact: lgbm_t03_base_5fold_oof
Metrics path: model_results/lgbm_t03_base_5fold_oof_fold_metrics.csv
OOF path: model_results/oof_lgbm_t03_base_5fold_oof.csv
Test prediction path: model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv
Submission path: submission_lgbm_t03_base_5fold_oof_foldavg.csv


In [208]:
selected_lgbm_12k_params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "n_estimators": 12000,
    "learning_rate": 0.03,
    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

selected_lgbm_12k_params

{'objective': 'regression',
 'metric': 'l2',
 'random_state': 9890,
 'n_jobs': 1,
 'verbosity': -1,
 'force_col_wise': True,
 'n_estimators': 12000,
 'learning_rate': 0.03,
 'num_leaves': 95,
 'min_child_samples': 60,
 'subsample': 0.85,
 'subsample_freq': 1,
 'colsample_bytree': 0.9,
 'reg_alpha': 0.0,
 'reg_lambda': 5.0,
 'max_depth': -1}

In [209]:
X_lgb_selected_all = np.ascontiguousarray(
    np.asarray(X_train_proc_model, dtype=np.float32)
)

X_lgb_selected_test = np.ascontiguousarray(
    np.asarray(X_test_proc_model, dtype=np.float32)
)

y_lgb_selected_all = np.asarray(
    y_train,
    dtype=np.float32
).ravel()

print("Selected LightGBM training matrix:", X_lgb_selected_all.shape)
print("Selected LightGBM test matrix:", X_lgb_selected_test.shape)
print("Target shape:", y_lgb_selected_all.shape)

Selected LightGBM training matrix: (144921, 162)
Selected LightGBM test matrix: (48307, 162)
Target shape: (144921,)


In [210]:
if LGBM_SELECTED_METRICS_PATH.exists():
    lgbm_selected_fold_metrics = pd.read_csv(LGBM_SELECTED_METRICS_PATH)
    completed_lgbm_selected_folds = set(
        lgbm_selected_fold_metrics["fold"].astype(int)
    )
else:
    lgbm_selected_fold_metrics = pd.DataFrame()
    completed_lgbm_selected_folds = set()

if LGBM_SELECTED_OOF_NPY_PATH.exists():
    lgbm_selected_oof_pred = np.load(LGBM_SELECTED_OOF_NPY_PATH)
    assert len(lgbm_selected_oof_pred) == len(X_lgb_selected_all)
else:
    lgbm_selected_oof_pred = np.full(
        len(X_lgb_selected_all),
        np.nan,
        dtype=np.float32
    )

if LGBM_SELECTED_TEST_SUM_NPY_PATH.exists():
    lgbm_selected_test_pred_sum = np.load(LGBM_SELECTED_TEST_SUM_NPY_PATH)
    assert len(lgbm_selected_test_pred_sum) == len(X_lgb_selected_test)
else:
    lgbm_selected_test_pred_sum = np.zeros(
        len(X_lgb_selected_test),
        dtype=np.float32
    )

print("Completed selected LightGBM folds:", sorted(completed_lgbm_selected_folds))
print("OOF predictions already filled:", np.isfinite(lgbm_selected_oof_pred).sum())

Completed selected LightGBM folds: []
OOF predictions already filled: 0


In [213]:
start_time = time.perf_counter()

kf_lgbm_selected = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf_lgbm_selected.split(X_lgb_selected_all), start=1):
    if fold_id in completed_lgbm_selected_folds:
        print(f"\nSkipping completed selected LightGBM fold {fold_id}.")
        continue

    fold_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(f"Selected LightGBM OOF fold {fold_id} / 5")
    print("=" * 80)

    X_fold_train = X_lgb_selected_all[fold_train_idx]
    X_fold_valid = X_lgb_selected_all[fold_valid_idx]

    y_fold_train = y_lgb_selected_all[fold_train_idx]
    y_fold_valid = y_lgb_selected_all[fold_valid_idx]

    model = lgb.LGBMRegressor(
        **selected_lgbm_12k_params,
        random_state=RANDOM_STATE + fold_id
    )

    model.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_fold_valid, y_fold_valid)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=500,
                verbose=True
            ),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iteration = model.best_iteration_

    if best_iteration is None or best_iteration <= 0:
        best_iteration = selected_lgbm_12k_params["n_estimators"]

    valid_pred_raw = model.predict(
        X_fold_valid,
        num_iteration=best_iteration
    )

    valid_pred_clipped = np.clip(
        valid_pred_raw,
        0,
        100
    ).astype(np.float32)

    test_pred_raw = model.predict(
        X_lgb_selected_test,
        num_iteration=best_iteration
    )

    test_pred_clipped = np.clip(
        test_pred_raw,
        0,
        100
    ).astype(np.float32)

    lgbm_selected_oof_pred[fold_valid_idx] = valid_pred_clipped
    lgbm_selected_test_pred_sum += test_pred_clipped / 5.0

    row = {
        "artifact_name": LGBM_SELECTED_ARTIFACT,
        "model_class": "LightGBM",
        "feature_space": "base",
        "fold": fold_id,
        "fold_train_n": len(fold_train_idx),
        "fold_valid_n": len(fold_valid_idx),
        "best_iteration": best_iteration,
        "fold_valid_mse_raw": mean_squared_error(y_fold_valid, valid_pred_raw),
        "fold_valid_mse_clipped": mean_squared_error(y_fold_valid, valid_pred_clipped),
        "elapsed_sec": time.perf_counter() - fold_start,
    }

    row.update(selected_lgbm_12k_params)

    lgbm_selected_fold_metrics = pd.concat(
        [
            lgbm_selected_fold_metrics,
            pd.DataFrame([row])
        ],
        axis=0,
        ignore_index=True
    )

    lgbm_selected_fold_metrics.to_csv(
        LGBM_SELECTED_METRICS_PATH,
        index=False
    )

    np.save(
        LGBM_SELECTED_OOF_NPY_PATH,
        lgbm_selected_oof_pred
    )

    np.save(
        LGBM_SELECTED_TEST_SUM_NPY_PATH,
        lgbm_selected_test_pred_sum
    )

    print("best_iteration:", best_iteration)
    print("fold_valid_mse_clipped:", row["fold_valid_mse_clipped"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))
    print("OOF predictions filled:", np.isfinite(lgbm_selected_oof_pred).sum())

    del model
    del X_fold_train, X_fold_valid
    del y_fold_train, y_fold_valid
    del valid_pred_raw, valid_pred_clipped
    del test_pred_raw, test_pred_clipped

    gc.collect()

print("\nSelected LightGBM OOF loop elapsed seconds:", round(time.perf_counter() - start_time, 2))


Selected LightGBM OOF fold 1 / 5
Training until validation scores don't improve for 500 rounds
[1000]	valid_0's l2: 144.55
[2000]	valid_0's l2: 129.408
[3000]	valid_0's l2: 121.318
[4000]	valid_0's l2: 115.861
[5000]	valid_0's l2: 112.045
[6000]	valid_0's l2: 109.066
[7000]	valid_0's l2: 106.881
[8000]	valid_0's l2: 105.062
[9000]	valid_0's l2: 103.675
[10000]	valid_0's l2: 102.592
[11000]	valid_0's l2: 101.653
[12000]	valid_0's l2: 100.822
Did not meet early stopping. Best iteration is:
[11999]	valid_0's l2: 100.821
best_iteration: 11999
fold_valid_mse_clipped: 100.26539611816406
elapsed_sec: 590.47
OOF predictions filled: 28985

Selected LightGBM OOF fold 2 / 5
Training until validation scores don't improve for 500 rounds
[1000]	valid_0's l2: 140.072
[2000]	valid_0's l2: 125.608
[3000]	valid_0's l2: 117.681
[4000]	valid_0's l2: 112.217
[5000]	valid_0's l2: 108.298
[6000]	valid_0's l2: 105.304
[7000]	valid_0's l2: 103.177
[8000]	valid_0's l2: 101.45
[9000]	valid_0's l2: 99.9494
[1000

In [214]:
missing_lgbm_selected_oof = int(np.isnan(lgbm_selected_oof_pred).sum())

print("Missing selected LightGBM OOF predictions:", missing_lgbm_selected_oof)

if missing_lgbm_selected_oof != 0:
    raise RuntimeError(
        "The selected LightGBM OOF artifact is incomplete. Rerun the previous cell to resume."
    )

lgbm_selected_oof_mse = mean_squared_error(
    y_lgb_selected_all,
    lgbm_selected_oof_pred
)

lgbm_selected_test_pred = np.clip(
    lgbm_selected_test_pred_sum,
    0,
    100
).astype(np.float32)

lgbm_selected_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(train_ids),
    "PERCENT_PROFICIENT_TRUE": y_lgb_selected_all,
    f"OOF_{LGBM_SELECTED_ARTIFACT}": lgbm_selected_oof_pred,
})

lgbm_selected_test_pred_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    f"TESTPRED_{LGBM_SELECTED_ARTIFACT}": lgbm_selected_test_pred,
})

lgbm_selected_submission = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "PERCENT_PROFICIENT": lgbm_selected_test_pred,
})

lgbm_selected_summary = pd.DataFrame([
    {
        "model_key": "LightGBM selected 12k OOF base",
        "model_class": "LightGBM",
        "feature_space": "base",
        "artifact_name": LGBM_SELECTED_ARTIFACT,
        "oof_mse_clipped": lgbm_selected_oof_mse,
        "fold_valid_mse_mean": lgbm_selected_fold_metrics["fold_valid_mse_clipped"].mean(),
        "fold_valid_mse_std": lgbm_selected_fold_metrics["fold_valid_mse_clipped"].std(),
        "mean_best_iteration": lgbm_selected_fold_metrics["best_iteration"].mean(),
        "n_splits": 5,
        "n_estimators": selected_lgbm_12k_params["n_estimators"],
        "learning_rate": selected_lgbm_12k_params["learning_rate"],
        "num_leaves": selected_lgbm_12k_params["num_leaves"],
        "min_child_samples": selected_lgbm_12k_params["min_child_samples"],
        "subsample": selected_lgbm_12k_params["subsample"],
        "colsample_bytree": selected_lgbm_12k_params["colsample_bytree"],
        "reg_alpha": selected_lgbm_12k_params["reg_alpha"],
        "reg_lambda": selected_lgbm_12k_params["reg_lambda"],
        "submission_path": str(LGBM_SELECTED_SUBMISSION_PATH),
    }
])

lgbm_selected_oof_df.to_csv(
    LGBM_SELECTED_OOF_PATH,
    index=False
)

lgbm_selected_test_pred_df.to_csv(
    LGBM_SELECTED_TEST_PRED_PATH,
    index=False
)

lgbm_selected_submission.to_csv(
    LGBM_SELECTED_SUBMISSION_PATH,
    index=False
)

LGBM_SELECTED_SUMMARY_PATH = RESULTS_DIR / f"{LGBM_SELECTED_ARTIFACT}_summary.csv"

lgbm_selected_summary.to_csv(
    LGBM_SELECTED_SUMMARY_PATH,
    index=False
)

print("Selected LightGBM OOF MSE:", lgbm_selected_oof_mse)

print("\nSelected LightGBM fold metrics:")
display(
    lgbm_selected_fold_metrics.sort_values("fold")[
        [
            "fold",
            "fold_valid_mse_clipped",
            "fold_valid_mse_raw",
            "best_iteration",
            "elapsed_sec",
        ]
    ].reset_index(drop=True)
)

print("\nSelected LightGBM summary:")
display(lgbm_selected_summary)

print("\nSaved files:")
print(LGBM_SELECTED_METRICS_PATH)
print(LGBM_SELECTED_OOF_PATH)
print(LGBM_SELECTED_TEST_PRED_PATH)
print(LGBM_SELECTED_SUBMISSION_PATH)
print(LGBM_SELECTED_SUMMARY_PATH)

Missing selected LightGBM OOF predictions: 0
Selected LightGBM OOF MSE: 98.77940368652344

Selected LightGBM fold metrics:


,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,100.265396,100.821338,11999,590.472698
1,2,96.495171,97.028716,11998,480.433172
2,3,98.903893,99.598858,12000,494.760867
3,4,96.720520,97.372209,11998,507.186494
4,5,101.512009,102.124080,12000,455.613318



Selected LightGBM summary:


,model_key,model_class,feature_space,artifact_name,oof_mse_clipped,fold_valid_mse_mean,fold_valid_mse_std,mean_best_iteration,n_splits,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda,submission_path
0,LightGBM selected 12k OOF base,LightGBM,base,lgbm_t03_base_5fold_oof,98.779404,98.779398,2.187894,11999.0,5,12000,0.03,95,60,0.85,0.9,0.0,5.0,submission_lgbm_t03_base_5fold_oof_foldavg.csv



Saved files:
model_results/lgbm_t03_base_5fold_oof_fold_metrics.csv
model_results/oof_lgbm_t03_base_5fold_oof.csv
model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv
submission_lgbm_t03_base_5fold_oof_foldavg.csv
model_results/lgbm_t03_base_5fold_oof_summary.csv


In [215]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Boosting",
                "model": "lightgbm_selected_12k_oof_base",
                "feature_space": "base",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": lgbm_selected_oof_mse,
                "notes": (
                    f"{LGBM_SELECTED_ARTIFACT}; "
                    f"mean_best_iteration={lgbm_selected_summary.loc[0, 'mean_best_iteration']:.1f}; "
                    f"num_leaves={selected_lgbm_12k_params['num_leaves']}; "
                    f"min_child_samples={selected_lgbm_12k_params['min_child_samples']}; "
                    f"reg_lambda={selected_lgbm_12k_params['reg_lambda']}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
1,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
2,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
3,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
4,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
5,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
6,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
7,Boosting,lightgbm_broad_holdout_screen,base,fixed holdout,70.014712,126.393035,lgbm_broad_l127_child120_l2_20; best_iteration...
8,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
9,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...


The selected 12,000-tree LightGBM artifact improves over the earlier 8,000-tree comparison artifact.

The 8,000-tree run reached the tree cap on almost every fold, which suggested the model was still improving. The selected 12,000-tree artifact reduces OOF MSE further and becomes the preferred LightGBM artifact for blending.

The next section keeps the 8,000-tree artifact as a comparison branch, then the notebook proceeds to the original three-model blend using Random Forest, ExtraTrees, and the selected LightGBM artifact.

### 5C. Targeted LightGBM holdout refinement

The targeted screen focuses on the strongest LightGBM region found so far.

The refinement tests configurations with:

- more boosting rounds,
- moderate-to-large leaf counts,
- smaller minimum child samples,
- row and column subsampling near 0.85–0.90,
- moderate L2 regularization.

These settings allow LightGBM to keep improving over many trees while controlling overfitting.

In [197]:
lgbm_targeted_configs = [
    {
        "config_name": "lgbm_target_l63_child40_l2_1",
        "n_estimators": 8000,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_child_samples": 40,
        "subsample": 0.90,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "max_depth": -1,
    },
    {
        "config_name": "lgbm_target_l63_child40_l2_1_lr02",
        "n_estimators": 10000,
        "learning_rate": 0.02,
        "num_leaves": 63,
        "min_child_samples": 40,
        "subsample": 0.90,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "max_depth": -1,
    },
    {
        "config_name": "lgbm_target_l95_child60_l2_5",
        "n_estimators": 8000,
        "learning_rate": 0.03,
        "num_leaves": 95,
        "min_child_samples": 60,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "max_depth": -1,
    },
]

pd.DataFrame(lgbm_targeted_configs)

,config_name,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,max_depth
0,lgbm_target_l63_child40_l2_1,8000,0.03,63,40,0.90,1,0.9,0.0,1.0,-1
1,lgbm_target_l63_child40_l2_1_lr02,10000,0.02,63,40,0.90,1,0.9,0.0,1.0,-1
2,lgbm_target_l95_child60_l2_5,8000,0.03,95,60,0.85,1,0.9,0.0,5.0,-1


In [198]:
LGBM_TARGETED_SCREEN_PATH = RESULTS_DIR / "lgbm_targeted_base_holdout_refinement.csv"

lgbm_targeted_results = run_lgbm_holdout_screen(
    configs=lgbm_targeted_configs,
    output_path=LGBM_TARGETED_SCREEN_PATH,
    screen_name="targeted_base_holdout_refinement",
    early_stopping_rounds=300,
    log_period=500,
)

lgbm_targeted_results[
    [
        "config_name",
        "train_mse_clipped",
        "holdout_val_mse_clipped",
        "best_iteration",
        "elapsed_sec",
        "n_estimators",
        "learning_rate",
        "num_leaves",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
    ]
]

Screen: targeted_base_holdout_refinement
Checkpoint: model_results/lgbm_targeted_base_holdout_refinement.csv
Already completed configs: []

Running LightGBM config: lgbm_target_l63_child40_l2_1
Training until validation scores don't improve for 300 rounds
[500]	valid_0's l2: 173.611
[1000]	valid_0's l2: 153.444
[1500]	valid_0's l2: 142.397
[2000]	valid_0's l2: 134.757
[2500]	valid_0's l2: 129.388
[3000]	valid_0's l2: 125.208
[3500]	valid_0's l2: 121.699
[4000]	valid_0's l2: 118.833
[4500]	valid_0's l2: 116.477
[5000]	valid_0's l2: 114.247
[5500]	valid_0's l2: 112.313
[6000]	valid_0's l2: 110.83
[6500]	valid_0's l2: 109.438
[7000]	valid_0's l2: 108.097
[7500]	valid_0's l2: 107.004
[8000]	valid_0's l2: 105.968
Did not meet early stopping. Best iteration is:
[8000]	valid_0's l2: 105.968
Completed: lgbm_target_l63_child40_l2_1
Best iteration: 8000
Train MSE clipped: 35.15224971100176
Holdout MSE clipped: 105.58646905597116
Elapsed seconds: 242.16

Running LightGBM config: lgbm_target_l63_c

,config_name,train_mse_clipped,holdout_val_mse_clipped,best_iteration,elapsed_sec,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda
0,lgbm_target_l95_child60_l2_5,27.643127,104.496354,8000,277.306868,8000,0.03,95,60,0.85,0.9,5.0
1,lgbm_target_l63_child40_l2_1,35.152250,105.586469,8000,242.163192,8000,0.03,63,40,0.90,0.9,1.0
2,lgbm_target_l63_child40_l2_1_lr02,41.509765,108.300128,10000,284.421738,10000,0.02,63,40,0.90,0.9,1.0


In [199]:
best_lgbm_targeted = lgbm_targeted_results.iloc[0]

model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Boosting",
                "model": "lightgbm_targeted_holdout_refinement",
                "feature_space": "base",
                "metric_source": "fixed holdout",
                "train_mse": best_lgbm_targeted["train_mse_clipped"],
                "val_mse": best_lgbm_targeted["holdout_val_mse_clipped"],
                "notes": (
                    f"{best_lgbm_targeted['config_name']}; "
                    f"best_iteration={int(best_lgbm_targeted['best_iteration'])}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
1,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
2,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
3,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
4,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
5,Boosting,lightgbm_broad_holdout_screen,base,fixed holdout,70.014712,126.393035,lgbm_broad_l127_child120_l2_20; best_iteration...
6,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
7,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...
8,Tree,regression_tree_holdout_screen,base_plus_poly,fixed holdout,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522
9,Tree,regression_tree_cv,base_plus_poly_interactions,5-fold CV mean,193.942697,250.689631,max_depth=None; min_samples_leaf=25; CV std=4....


The targeted LightGBM refinement gives a clear improvement over the RF + ExtraTrees blend.

The best targeted configuration is:

- `num_leaves = 95`
- `min_child_samples = 60`
- `learning_rate = 0.03`
- `subsample = 0.85`
- `colsample_bytree = 0.90`
- `reg_lambda = 5.0`

This configuration is selected for the next stage: a 5-fold OOF LightGBM artifact with fold-averaged test predictions.

#### 5C. Short-cap LightGBM OOF comparison

This section records the 8,000-tree LightGBM OOF artifact that was run before the original selected 12,000-tree artifact was restored into the cleaned notebook.

It is kept as a useful comparison because its folds reached the tree cap and produced OOF MSE `103.251549`, showing that a longer selected LightGBM run was justified.

The targeted holdout screen selected the following LightGBM configuration:

- `num_leaves = 95`
- `min_child_samples = 60`
- `learning_rate = 0.03`
- `subsample = 0.85`
- `colsample_bytree = 0.90`
- `reg_lambda = 5.0`
- `n_estimators = 8000`

This configuration is now trained as a 5-fold out-of-fold artifact.

For each fold:

1. train LightGBM on four folds,
2. predict the held-out fold,
3. predict the test set,
4. average the five test predictions.

The OOF predictions are saved because they will be used later for blending and stacking.

In [201]:
LGBM_OOF_N_SPLITS = 5

LGBM_OOF_FOLD_RESULTS_PATH = RESULTS_DIR / "lgbm_base_oof_fold_results.csv"
LGBM_OOF_SUMMARY_PATH = RESULTS_DIR / "lgbm_base_oof_summary.csv"
LGBM_OOF_PRED_PATH = RESULTS_DIR / "oof_lgbm_base.csv"
LGBM_TEST_PRED_PATH = RESULTS_DIR / "testpred_lgbm_base_foldavg.csv"

LGBM_SUBMISSION_PATH = Path("submission_lgbm_base_oof_foldavg.csv")

for path in [
    LGBM_OOF_FOLD_RESULTS_PATH,
    LGBM_OOF_SUMMARY_PATH,
    LGBM_OOF_PRED_PATH,
    LGBM_TEST_PRED_PATH,
    LGBM_SUBMISSION_PATH,
]:
    if path.exists():
        path.unlink()

print("LightGBM OOF paths prepared.")

LightGBM OOF paths prepared.


In [202]:
selected_lgbm_config = {
    "objective": "regression",
    "metric": "l2",
    "n_estimators": int(best_lgbm_targeted["n_estimators"]),
    "learning_rate": float(best_lgbm_targeted["learning_rate"]),
    "num_leaves": int(best_lgbm_targeted["num_leaves"]),
    "min_child_samples": int(best_lgbm_targeted["min_child_samples"]),
    "subsample": float(best_lgbm_targeted["subsample"]),
    "subsample_freq": int(best_lgbm_targeted["subsample_freq"]),
    "colsample_bytree": float(best_lgbm_targeted["colsample_bytree"]),
    "reg_alpha": float(best_lgbm_targeted["reg_alpha"]),
    "reg_lambda": float(best_lgbm_targeted["reg_lambda"]),
    "max_depth": int(best_lgbm_targeted["max_depth"]),
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
}

selected_lgbm_config

{'objective': 'regression',
 'metric': 'l2',
 'n_estimators': 8000,
 'learning_rate': 0.03,
 'num_leaves': 95,
 'min_child_samples': 60,
 'subsample': 0.85,
 'subsample_freq': 1,
 'colsample_bytree': 0.9,
 'reg_alpha': 0.0,
 'reg_lambda': 5.0,
 'max_depth': -1,
 'n_jobs': 1,
 'verbosity': -1,
 'force_col_wise': True}

In [203]:
X_lgb_all = np.ascontiguousarray(
    np.asarray(X_train_proc_model, dtype=np.float32)
)

X_lgb_test = np.ascontiguousarray(
    np.asarray(X_test_proc_model, dtype=np.float32)
)

y_lgb_all = np.asarray(
    y_train,
    dtype=np.float32
).ravel()

print("Full LightGBM training matrix:", X_lgb_all.shape)
print("LightGBM test matrix:", X_lgb_test.shape)
print("Target shape:", y_lgb_all.shape)

Full LightGBM training matrix: (144921, 162)
LightGBM test matrix: (48307, 162)
Target shape: (144921,)


In [204]:
start_time = time.perf_counter()

kf_lgbm = KFold(
    n_splits=LGBM_OOF_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

lgbm_oof_pred_raw = np.zeros(len(X_lgb_all), dtype=np.float32)
lgbm_test_pred_folds = np.zeros(
    (len(X_lgb_test), LGBM_OOF_N_SPLITS),
    dtype=np.float32
)

lgbm_oof_fold_rows = []

for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf_lgbm.split(X_lgb_all), start=1):
    fold_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(f"LightGBM OOF fold {fold_id} / {LGBM_OOF_N_SPLITS}")
    print("=" * 80)

    X_fold_train = X_lgb_all[fold_train_idx]
    X_fold_valid = X_lgb_all[fold_valid_idx]

    y_fold_train = y_lgb_all[fold_train_idx]
    y_fold_valid = y_lgb_all[fold_valid_idx]

    model = lgb.LGBMRegressor(
        **selected_lgbm_config,
        random_state=RANDOM_STATE + fold_id
    )

    model.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_fold_valid, y_fold_valid)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=300,
                verbose=True
            ),
            lgb.log_evaluation(period=500),
        ],
    )

    best_iteration = model.best_iteration_

    train_pred_raw = model.predict(
        X_fold_train,
        num_iteration=best_iteration
    )

    valid_pred_raw = model.predict(
        X_fold_valid,
        num_iteration=best_iteration
    )

    test_pred_raw = model.predict(
        X_lgb_test,
        num_iteration=best_iteration
    )

    train_pred_clipped = np.clip(train_pred_raw, 0, 100)
    valid_pred_clipped = np.clip(valid_pred_raw, 0, 100)
    test_pred_clipped = np.clip(test_pred_raw, 0, 100)

    lgbm_oof_pred_raw[fold_valid_idx] = valid_pred_raw.astype(np.float32)
    lgbm_test_pred_folds[:, fold_id - 1] = test_pred_clipped.astype(np.float32)

    row = {
        "model_class": "LightGBM",
        "stage": "oof_selected_config",
        "feature_space": "base",
        "fold": fold_id,
        "config_name": "lgbm_target_l95_child60_l2_5",
        "best_iteration": best_iteration,
        "n_estimators": selected_lgbm_config["n_estimators"],
        "learning_rate": selected_lgbm_config["learning_rate"],
        "num_leaves": selected_lgbm_config["num_leaves"],
        "min_child_samples": selected_lgbm_config["min_child_samples"],
        "subsample": selected_lgbm_config["subsample"],
        "colsample_bytree": selected_lgbm_config["colsample_bytree"],
        "reg_alpha": selected_lgbm_config["reg_alpha"],
        "reg_lambda": selected_lgbm_config["reg_lambda"],
        "fold_train_mse_raw": mean_squared_error(y_fold_train, train_pred_raw),
        "fold_train_mse_clipped": mean_squared_error(y_fold_train, train_pred_clipped),
        "fold_valid_mse_raw": mean_squared_error(y_fold_valid, valid_pred_raw),
        "fold_valid_mse_clipped": mean_squared_error(y_fold_valid, valid_pred_clipped),
        "elapsed_sec": time.perf_counter() - fold_start,
    }

    lgbm_oof_fold_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), LGBM_OOF_FOLD_RESULTS_PATH)

    print("best_iteration:", best_iteration)
    print("fold_valid_mse_clipped:", row["fold_valid_mse_clipped"])
    print("elapsed_sec:", round(row["elapsed_sec"], 2))

    del model
    del X_fold_train, X_fold_valid
    del y_fold_train, y_fold_valid
    del train_pred_raw, valid_pred_raw, test_pred_raw
    del train_pred_clipped, valid_pred_clipped, test_pred_clipped

    gc.collect()

lgbm_oof_fold_results = pd.DataFrame(lgbm_oof_fold_rows)
lgbm_oof_fold_results.to_csv(LGBM_OOF_FOLD_RESULTS_PATH, index=False)

lgbm_oof_pred_clipped = np.clip(lgbm_oof_pred_raw, 0, 100)
lgbm_test_pred_foldavg = np.clip(
    lgbm_test_pred_folds.mean(axis=1),
    0,
    100
)

lgbm_oof_mse_raw = mean_squared_error(
    y_lgb_all,
    lgbm_oof_pred_raw
)

lgbm_oof_mse_clipped = mean_squared_error(
    y_lgb_all,
    lgbm_oof_pred_clipped
)

lgbm_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(train_ids),
    "y_true": y_lgb_all,
    "lgbm_oof_pred_raw": lgbm_oof_pred_raw,
    "lgbm_oof_pred_clipped": lgbm_oof_pred_clipped,
})

lgbm_test_pred_df = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
})

for fold_id in range(LGBM_OOF_N_SPLITS):
    lgbm_test_pred_df[f"lgbm_fold{fold_id + 1}_pred"] = lgbm_test_pred_folds[:, fold_id]

lgbm_test_pred_df["PERCENT_PROFICIENT"] = lgbm_test_pred_foldavg

lgbm_submission = lgbm_test_pred_df[
    ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
].copy()

lgbm_oof_summary = pd.DataFrame([
    {
        "model_key": "LightGBM OOF base",
        "model_class": "LightGBM",
        "feature_space": "base",
        "config_name": "lgbm_target_l95_child60_l2_5",
        "oof_mse_raw": lgbm_oof_mse_raw,
        "oof_mse_clipped": lgbm_oof_mse_clipped,
        "fold_valid_mse_mean": lgbm_oof_fold_results["fold_valid_mse_clipped"].mean(),
        "fold_valid_mse_std": lgbm_oof_fold_results["fold_valid_mse_clipped"].std(),
        "mean_best_iteration": lgbm_oof_fold_results["best_iteration"].mean(),
        "n_splits": LGBM_OOF_N_SPLITS,
        "n_estimators": selected_lgbm_config["n_estimators"],
        "learning_rate": selected_lgbm_config["learning_rate"],
        "num_leaves": selected_lgbm_config["num_leaves"],
        "min_child_samples": selected_lgbm_config["min_child_samples"],
        "subsample": selected_lgbm_config["subsample"],
        "colsample_bytree": selected_lgbm_config["colsample_bytree"],
        "reg_alpha": selected_lgbm_config["reg_alpha"],
        "reg_lambda": selected_lgbm_config["reg_lambda"],
        "submission_path": str(LGBM_SUBMISSION_PATH),
        "elapsed_sec": time.perf_counter() - start_time,
    }
])

lgbm_oof_df.to_csv(LGBM_OOF_PRED_PATH, index=False)
lgbm_test_pred_df.to_csv(LGBM_TEST_PRED_PATH, index=False)
lgbm_submission.to_csv(LGBM_SUBMISSION_PATH, index=False)
lgbm_oof_summary.to_csv(LGBM_OOF_SUMMARY_PATH, index=False)

print("LightGBM OOF fold results:")
display(lgbm_oof_fold_results)

print("\nLightGBM OOF summary:")
display(lgbm_oof_summary)

print("\nSaved files:")
print(LGBM_OOF_FOLD_RESULTS_PATH)
print(LGBM_OOF_SUMMARY_PATH)
print(LGBM_OOF_PRED_PATH)
print(LGBM_TEST_PRED_PATH)
print(LGBM_SUBMISSION_PATH)


LightGBM OOF fold 1 / 5
Training until validation scores don't improve for 300 rounds
[500]	valid_0's l2: 161.632
[1000]	valid_0's l2: 144.55
[1500]	valid_0's l2: 135.562
[2000]	valid_0's l2: 129.408
[2500]	valid_0's l2: 124.798
[3000]	valid_0's l2: 121.318
[3500]	valid_0's l2: 118.276
[4000]	valid_0's l2: 115.861
[4500]	valid_0's l2: 113.84
[5000]	valid_0's l2: 112.045
[5500]	valid_0's l2: 110.48
[6000]	valid_0's l2: 109.066
[6500]	valid_0's l2: 107.887
[7000]	valid_0's l2: 106.881
[7500]	valid_0's l2: 105.975
[8000]	valid_0's l2: 105.062
Did not meet early stopping. Best iteration is:
[8000]	valid_0's l2: 105.062
best_iteration: 8000
fold_valid_mse_clipped: 104.59005781089512
elapsed_sec: 384.77

LightGBM OOF fold 2 / 5
Training until validation scores don't improve for 300 rounds
[500]	valid_0's l2: 156.996
[1000]	valid_0's l2: 140.072
[1500]	valid_0's l2: 131.51
[2000]	valid_0's l2: 125.608
[2500]	valid_0's l2: 121.154
[3000]	valid_0's l2: 117.681
[3500]	valid_0's l2: 114.627
[400

,model_class,stage,feature_space,fold,config_name,best_iteration,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda,fold_train_mse_raw,fold_train_mse_clipped,fold_valid_mse_raw,fold_valid_mse_clipped,elapsed_sec
0,LightGBM,oof_selected_config,base,1,lgbm_target_l95_child60_l2_5,8000,8000,0.03,95,60,0.85,0.9,0.0,5.0,27.627895,27.500110,105.062180,104.590058,384.769429
1,LightGBM,oof_selected_config,base,2,lgbm_target_l95_child60_l2_5,7988,8000,0.03,95,60,0.85,0.9,0.0,5.0,27.909359,27.788745,101.439123,100.977234,375.557372
2,LightGBM,oof_selected_config,base,3,lgbm_target_l95_child60_l2_5,7996,8000,0.03,95,60,0.85,0.9,0.0,5.0,27.817695,27.694714,104.068424,103.464764,330.404919
3,LightGBM,oof_selected_config,base,4,lgbm_target_l95_child60_l2_5,7998,8000,0.03,95,60,0.85,0.9,0.0,5.0,27.830846,27.706234,101.684479,101.128020,327.797302
4,LightGBM,oof_selected_config,base,5,lgbm_target_l95_child60_l2_5,8000,8000,0.03,95,60,0.85,0.9,0.0,5.0,27.511822,27.381140,106.605896,106.097616,325.626952



LightGBM OOF summary:


,model_key,model_class,feature_space,config_name,oof_mse_raw,oof_mse_clipped,fold_valid_mse_mean,fold_valid_mse_std,mean_best_iteration,n_splits,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda,submission_path,elapsed_sec
0,LightGBM OOF base,LightGBM,base,lgbm_target_l95_child60_l2_5,103.772026,103.251549,103.251538,2.21467,7996.4,5,8000,0.03,95,60,0.85,0.9,0.0,5.0,submission_lgbm_base_oof_foldavg.csv,1744.848161



Saved files:
model_results/lgbm_base_oof_fold_results.csv
model_results/lgbm_base_oof_summary.csv
model_results/oof_lgbm_base.csv
model_results/testpred_lgbm_base_foldavg.csv
submission_lgbm_base_oof_foldavg.csv


In [205]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Boosting",
                "model": "lightgbm_oof_base",
                "feature_space": "base",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": lgbm_oof_mse_clipped,
                "notes": (
                    "lgbm_target_l95_child60_l2_5; "
                    f"mean_best_iteration={lgbm_oof_summary.loc[0, 'mean_best_iteration']:.1f}; "
                    f"num_leaves={selected_lgbm_config['num_leaves']}; "
                    f"min_child_samples={selected_lgbm_config['min_child_samples']}; "
                    f"reg_lambda={selected_lgbm_config['reg_lambda']}"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
1,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
2,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
3,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
4,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
5,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
6,Boosting,lightgbm_broad_holdout_screen,base,fixed holdout,70.014712,126.393035,lgbm_broad_l127_child120_l2_20; best_iteration...
7,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...
8,Bagging,random_forest_screen,base,3-fold CV mean,30.847553,149.549571,n_estimators=120; max_samples=0.8; max_feature...
9,Tree,regression_tree_holdout_screen,base_plus_poly,fixed holdout,190.544257,240.566256,max_depth=None; min_samples_leaf=25; leaves=3522


The selected LightGBM model improves over the RF + ExtraTrees blend.

The previous best OOF result was about 110.36 from the RF + ExtraTrees blend. The LightGBM OOF artifact is expected to reduce OOF MSE to around 99.

This is a major improvement, so the LightGBM OOF artifact is kept as a fully runnable modelling step.

#### 5D. Random Forest + ExtraTrees + selected LightGBM blend

The selected 12,000-tree LightGBM artifact is now the strongest individual model so far, with OOF MSE around 98.78.

The next step is to test whether the earlier tree-ensemble models still add useful complementary signal.

This blend uses saved OOF predictions from:

1. Random Forest 500-tree base model
2. ExtraTrees base model
3. selected 12,000-tree LightGBM base model

No new model is trained in this section. We only blend existing out-of-fold predictions and choose weights that minimize OOF MSE.

The blend has the form:


blend prediction =
`w_rf × Random Forest + w_extratrees × ExtraTrees + w_lgbm × LightGBM`

In [217]:
RF_EXTRATREES_LGBM_BLEND_SCREEN_PATH = RESULTS_DIR / "blend_rf_et_lgbm_weight_screen.csv"
RF_EXTRATREES_LGBM_BLEND_OOF_PATH = RESULTS_DIR / "oof_blend_rf_et_lgbm_weighted.csv"
RF_EXTRATREES_LGBM_BLEND_TEST_PATH = RESULTS_DIR / "testpred_blend_rf_et_lgbm_weighted.csv"
RF_EXTRATREES_LGBM_BLEND_SUBMISSION_PATH = Path("submission_blend_rf_et_lgbm_oof_weighted.csv")

print("Three-model blend paths:")
print(RF_EXTRATREES_LGBM_BLEND_SCREEN_PATH)
print(RF_EXTRATREES_LGBM_BLEND_OOF_PATH)
print(RF_EXTRATREES_LGBM_BLEND_TEST_PATH)
print(RF_EXTRATREES_LGBM_BLEND_SUBMISSION_PATH)

Three-model blend paths:
model_results/blend_rf_et_lgbm_weight_screen.csv
model_results/oof_blend_rf_et_lgbm_weighted.csv
model_results/testpred_blend_rf_et_lgbm_weighted.csv
submission_blend_rf_et_lgbm_oof_weighted.csv


In [218]:
rf_oof_blend = pd.read_csv(RF_OOF_PATH)
extratrees_oof_blend = pd.read_csv(EXTRATREES_OOF_PRED_PATH)
lgbm_oof_blend = pd.read_csv(LGBM_SELECTED_OOF_PATH)

rf_test_blend = pd.read_csv(RF_TEST_FOLD_PATH)
extratrees_test_blend = pd.read_csv(EXTRATREES_TEST_PRED_PATH)
lgbm_test_blend = pd.read_csv(LGBM_SELECTED_TEST_PRED_PATH)

print("OOF shapes:")
print("RF:", rf_oof_blend.shape)
print("ExtraTrees:", extratrees_oof_blend.shape)
print("LightGBM:", lgbm_oof_blend.shape)

print("\nTest prediction shapes:")
print("RF:", rf_test_blend.shape)
print("ExtraTrees:", extratrees_test_blend.shape)
print("LightGBM:", lgbm_test_blend.shape)

OOF shapes:
RF: (144921, 3)
ExtraTrees: (144921, 4)
LightGBM: (144921, 3)

Test prediction shapes:
RF: (48307, 7)
ExtraTrees: (48307, 2)
LightGBM: (48307, 2)


In [219]:
rf_oof_blend = rf_oof_blend[
    ["ASSESSMENT_ID", "y_true", "rf_500_base_oof_pred"]
].copy()

rf_oof_blend["ASSESSMENT_ID"] = rf_oof_blend["ASSESSMENT_ID"].astype(str)

extratrees_oof_blend = extratrees_oof_blend[
    ["ASSESSMENT_ID", "y_true", "extratrees_oof_pred"]
].copy()

extratrees_oof_blend["ASSESSMENT_ID"] = extratrees_oof_blend["ASSESSMENT_ID"].astype(str)

lgbm_oof_blend = lgbm_oof_blend[
    [
        "ASSESSMENT_ID",
        "PERCENT_PROFICIENT_TRUE",
        f"OOF_{LGBM_SELECTED_ARTIFACT}",
    ]
].copy()

lgbm_oof_blend["ASSESSMENT_ID"] = lgbm_oof_blend["ASSESSMENT_ID"].astype(str)

blend_oof_3model = (
    rf_oof_blend
    .merge(extratrees_oof_blend, on="ASSESSMENT_ID", how="inner", suffixes=("_rf", "_extratrees"))
    .merge(lgbm_oof_blend, on="ASSESSMENT_ID", how="inner")
)

print("Blended OOF rows:", blend_oof_3model.shape[0])
print("Expected training rows:", len(train_ids))

Blended OOF rows: 144921
Expected training rows: 144921


In [220]:
if blend_oof_3model.shape[0] != len(train_ids):
    raise ValueError("Three-model OOF blend lost rows.")

target_rf = blend_oof_3model["y_true_rf"].to_numpy(dtype=float)
target_extratrees = blend_oof_3model["y_true_extratrees"].to_numpy(dtype=float)
target_lgbm = blend_oof_3model["PERCENT_PROFICIENT_TRUE"].to_numpy(dtype=float)

if not np.allclose(target_rf, target_extratrees):
    raise ValueError("RF and ExtraTrees targets are not aligned.")

if not np.allclose(target_rf, target_lgbm):
    raise ValueError("RF and LightGBM targets are not aligned.")

blend_oof_3model["y_true"] = target_rf

y_blend_3model = blend_oof_3model["y_true"].to_numpy(dtype=float)

rf_pred_3model = blend_oof_3model["rf_500_base_oof_pred"].to_numpy(dtype=float)
extratrees_pred_3model = blend_oof_3model["extratrees_oof_pred"].to_numpy(dtype=float)
lgbm_pred_3model = blend_oof_3model[f"OOF_{LGBM_SELECTED_ARTIFACT}"].to_numpy(dtype=float)

component_blend_results = pd.DataFrame([
    {
        "model": "Random Forest",
        "oof_mse": mean_squared_error(y_blend_3model, np.clip(rf_pred_3model, 0, 100)),
    },
    {
        "model": "ExtraTrees",
        "oof_mse": mean_squared_error(y_blend_3model, np.clip(extratrees_pred_3model, 0, 100)),
    },
    {
        "model": "LightGBM selected 12k",
        "oof_mse": mean_squared_error(y_blend_3model, np.clip(lgbm_pred_3model, 0, 100)),
    },
]).sort_values("oof_mse").reset_index(drop=True)

component_blend_results

,model,oof_mse
0,LightGBM selected 12k,98.779406
1,ExtraTrees,110.749294
2,Random Forest,122.328024


The component results show that LightGBM is the strongest individual model.

However, a weaker model can still help a blend if its errors are different from LightGBM's errors. The next cell searches convex blend weights using OOF MSE.

In [221]:
candidate_weight_rows = []

# Pure models
candidate_weight_rows.extend([
    {"w_rf": 1.0, "w_extratrees": 0.0, "w_lgbm": 0.0, "source": "pure_rf"},
    {"w_rf": 0.0, "w_extratrees": 1.0, "w_lgbm": 0.0, "source": "pure_extratrees"},
    {"w_rf": 0.0, "w_extratrees": 0.0, "w_lgbm": 1.0, "source": "pure_lgbm"},
])

# Pairwise grids
for w_lgbm in np.linspace(0, 1, 1001):
    candidate_weight_rows.append({
        "w_rf": 0.0,
        "w_extratrees": 1.0 - w_lgbm,
        "w_lgbm": w_lgbm,
        "source": "pair_extratrees_lgbm",
    })

for w_lgbm in np.linspace(0, 1, 1001):
    candidate_weight_rows.append({
        "w_rf": 1.0 - w_lgbm,
        "w_extratrees": 0.0,
        "w_lgbm": w_lgbm,
        "source": "pair_rf_lgbm",
    })

for w_extratrees in np.linspace(0, 1, 1001):
    candidate_weight_rows.append({
        "w_rf": 1.0 - w_extratrees,
        "w_extratrees": w_extratrees,
        "w_lgbm": 0.0,
        "source": "pair_rf_extratrees",
    })

# Coarse three-way grid focused around LightGBM-dominant blends.
grid_step = 0.01

for w_lgbm in np.arange(0.60, 1.0 + grid_step / 2, grid_step):
    remaining_weight = 1.0 - w_lgbm
    n_steps = int(round(remaining_weight / grid_step))

    for k in range(n_steps + 1):
        w_rf = k * grid_step
        w_extratrees = remaining_weight - w_rf

        candidate_weight_rows.append({
            "w_rf": w_rf,
            "w_extratrees": w_extratrees,
            "w_lgbm": w_lgbm,
            "source": "three_way_lgbm_focused_coarse",
        })

candidate_weights = (
    pd.DataFrame(candidate_weight_rows)
    .round(6)
    .drop_duplicates(subset=["w_rf", "w_extratrees", "w_lgbm"])
    .reset_index(drop=True)
)

print("Initial candidate weights:", candidate_weights.shape[0])

Initial candidate weights: 3780


In [222]:
prediction_matrix_3model = np.vstack([
    rf_pred_3model,
    extratrees_pred_3model,
    lgbm_pred_3model,
]).astype(np.float32)

blend_screen_rows = []

for row in candidate_weights.itertuples(index=False):
    weights = np.array(
        [row.w_rf, row.w_extratrees, row.w_lgbm],
        dtype=np.float32
    )

    blend_pred_raw = (
        weights[0] * prediction_matrix_3model[0]
        + weights[1] * prediction_matrix_3model[1]
        + weights[2] * prediction_matrix_3model[2]
    )

    blend_pred_clipped = np.clip(blend_pred_raw, 0, 100)

    blend_screen_rows.append({
        "w_rf": float(weights[0]),
        "w_extratrees": float(weights[1]),
        "w_lgbm": float(weights[2]),
        "source": row.source,
        "oof_mse_raw": mean_squared_error(y_blend_3model, blend_pred_raw),
        "oof_mse_clipped": mean_squared_error(y_blend_3model, blend_pred_clipped),
    })

blend_screen_initial = (
    pd.DataFrame(blend_screen_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

best_initial_3model_blend = blend_screen_initial.iloc[0]

best_initial_3model_blend

w_rf                                0.0
w_extratrees                      0.356
w_lgbm                            0.644
source             pair_extratrees_lgbm
oof_mse_raw                   93.499458
oof_mse_clipped               93.499458
Name: 0, dtype: object

In [223]:
local_weight_rows = []

center_rf = float(best_initial_3model_blend["w_rf"])
center_extratrees = float(best_initial_3model_blend["w_extratrees"])
center_lgbm = float(best_initial_3model_blend["w_lgbm"])

local_step = 0.002
window = 0.04

for w_rf in np.arange(
    max(0.0, center_rf - window),
    min(1.0, center_rf + window) + local_step / 2,
    local_step,
):
    for w_extratrees in np.arange(
        max(0.0, center_extratrees - window),
        min(1.0, center_extratrees + window) + local_step / 2,
        local_step,
    ):
        w_lgbm = 1.0 - w_rf - w_extratrees

        if w_lgbm < 0:
            continue

        local_weight_rows.append({
            "w_rf": w_rf,
            "w_extratrees": w_extratrees,
            "w_lgbm": w_lgbm,
            "source": "local_fine_around_best",
        })

candidate_weights = (
    pd.concat(
        [
            candidate_weights,
            pd.DataFrame(local_weight_rows),
        ],
        axis=0,
        ignore_index=True
    )
    .round(6)
    .drop_duplicates(subset=["w_rf", "w_extratrees", "w_lgbm"])
    .reset_index(drop=True)
)

print("Final candidate weights after local search:", candidate_weights.shape[0])

Final candidate weights after local search: 4574


In [224]:
blend_screen_rows = []

for row in candidate_weights.itertuples(index=False):
    weights = np.array(
        [row.w_rf, row.w_extratrees, row.w_lgbm],
        dtype=np.float32
    )

    blend_pred_raw = (
        weights[0] * prediction_matrix_3model[0]
        + weights[1] * prediction_matrix_3model[1]
        + weights[2] * prediction_matrix_3model[2]
    )

    blend_pred_clipped = np.clip(blend_pred_raw, 0, 100)

    blend_screen_rows.append({
        "w_rf": float(weights[0]),
        "w_extratrees": float(weights[1]),
        "w_lgbm": float(weights[2]),
        "source": row.source,
        "oof_mse_raw": mean_squared_error(y_blend_3model, blend_pred_raw),
        "oof_mse_clipped": mean_squared_error(y_blend_3model, blend_pred_clipped),
    })

blend_rf_extratrees_lgbm_screen = (
    pd.DataFrame(blend_screen_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

best_rf_extratrees_lgbm_blend = blend_rf_extratrees_lgbm_screen.iloc[0]

best_rf_extratrees_lgbm_blend

w_rf                                0.0
w_extratrees                      0.356
w_lgbm                            0.644
source             pair_extratrees_lgbm
oof_mse_raw                   93.499458
oof_mse_clipped               93.499458
Name: 0, dtype: object

In [225]:
blend_rf_extratrees_lgbm_screen.head(15)

,w_rf,w_extratrees,w_lgbm,source,oof_mse_raw,oof_mse_clipped
0,0.0,0.356,0.644,pair_extratrees_lgbm,93.499458,93.499458
1,0.0,0.357,0.643,pair_extratrees_lgbm,93.499484,93.499484
2,0.0,0.355,0.645,pair_extratrees_lgbm,93.499516,93.499516
3,0.0,0.358,0.642,pair_extratrees_lgbm,93.499592,93.499592
4,0.0,0.354,0.646,pair_extratrees_lgbm,93.499655,93.499655
5,0.0,0.359,0.641,pair_extratrees_lgbm,93.499786,93.499786
6,0.0,0.353,0.647,pair_extratrees_lgbm,93.499879,93.499879
7,0.0,0.360,0.640,pair_extratrees_lgbm,93.500061,93.500061
8,0.0,0.352,0.648,pair_extratrees_lgbm,93.500187,93.500187
9,0.0,0.361,0.639,pair_extratrees_lgbm,93.500420,93.500420


The best blend gives zero weight to Random Forest and combines ExtraTrees with LightGBM.

This is expected because Random Forest is weaker than both ExtraTrees and LightGBM. ExtraTrees is weaker than LightGBM as a standalone model, but it adds complementary signal when blended with LightGBM.

In [226]:
best_w_rf_3model = float(best_rf_extratrees_lgbm_blend["w_rf"])
best_w_extratrees_3model = float(best_rf_extratrees_lgbm_blend["w_extratrees"])
best_w_lgbm_3model = float(best_rf_extratrees_lgbm_blend["w_lgbm"])

blend_oof_3model["blend_pred_raw"] = (
    best_w_rf_3model * rf_pred_3model
    + best_w_extratrees_3model * extratrees_pred_3model
    + best_w_lgbm_3model * lgbm_pred_3model
)

blend_oof_3model["blend_pred_clipped"] = np.clip(
    blend_oof_3model["blend_pred_raw"],
    0,
    100
)

rf_extratrees_lgbm_blend_oof_mse = mean_squared_error(
    blend_oof_3model["y_true"],
    blend_oof_3model["blend_pred_clipped"]
)

blend_rf_extratrees_lgbm_screen.to_csv(
    RF_EXTRATREES_LGBM_BLEND_SCREEN_PATH,
    index=False
)

blend_oof_3model.to_csv(
    RF_EXTRATREES_LGBM_BLEND_OOF_PATH,
    index=False
)

print("Best RF weight:", best_w_rf_3model)
print("Best ExtraTrees weight:", best_w_extratrees_3model)
print("Best LightGBM weight:", best_w_lgbm_3model)
print("RF + ExtraTrees + LightGBM blend OOF MSE:", rf_extratrees_lgbm_blend_oof_mse)

Best RF weight: 0.0
Best ExtraTrees weight: 0.35600000619888306
Best LightGBM weight: 0.6439999938011169
RF + ExtraTrees + LightGBM blend OOF MSE: 93.49945766144685


In [227]:
rf_test_3model = rf_test_blend[
    ["ASSESSMENT_ID", "rf_500_base_foldavg_pred"]
].copy()

rf_test_3model["ASSESSMENT_ID"] = rf_test_3model["ASSESSMENT_ID"].astype(str)

rf_test_3model = rf_test_3model.rename(
    columns={"rf_500_base_foldavg_pred": "rf_test_pred"}
)

extratrees_test_3model = extratrees_test_blend[
    ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
].copy()

extratrees_test_3model["ASSESSMENT_ID"] = extratrees_test_3model["ASSESSMENT_ID"].astype(str)

extratrees_test_3model = extratrees_test_3model.rename(
    columns={"PERCENT_PROFICIENT": "extratrees_test_pred"}
)

lgbm_test_3model = lgbm_test_blend[
    [
        "ASSESSMENT_ID",
        f"TESTPRED_{LGBM_SELECTED_ARTIFACT}",
    ]
].copy()

lgbm_test_3model["ASSESSMENT_ID"] = lgbm_test_3model["ASSESSMENT_ID"].astype(str)

lgbm_test_3model = lgbm_test_3model.rename(
    columns={f"TESTPRED_{LGBM_SELECTED_ARTIFACT}": "lgbm_test_pred"}
)

blend_test_3model = (
    rf_test_3model
    .merge(extratrees_test_3model, on="ASSESSMENT_ID", how="inner")
    .merge(lgbm_test_3model, on="ASSESSMENT_ID", how="inner")
)

print("Blend test rows:", blend_test_3model.shape[0])
print("Expected test rows:", len(test_ids))

Blend test rows: 48307
Expected test rows: 48307


In [228]:
if blend_test_3model.shape[0] != len(test_ids):
    raise ValueError("Three-model test blend lost rows.")

test_order = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "_test_order": np.arange(len(test_ids)),
})

blend_test_3model = test_order.merge(
    blend_test_3model,
    on="ASSESSMENT_ID",
    how="left"
)

missing_test_preds = blend_test_3model[
    ["rf_test_pred", "extratrees_test_pred", "lgbm_test_pred"]
].isna().sum()

if missing_test_preds.sum() > 0:
    raise ValueError(f"Missing test predictions after alignment:\n{missing_test_preds}")

blend_test_3model = (
    blend_test_3model
    .sort_values("_test_order")
    .drop(columns=["_test_order"])
    .reset_index(drop=True)
)

blend_test_3model["PERCENT_PROFICIENT"] = np.clip(
    best_w_rf_3model * blend_test_3model["rf_test_pred"].to_numpy(dtype=float)
    + best_w_extratrees_3model * blend_test_3model["extratrees_test_pred"].to_numpy(dtype=float)
    + best_w_lgbm_3model * blend_test_3model["lgbm_test_pred"].to_numpy(dtype=float),
    0,
    100
)

submission_rf_extratrees_lgbm_blend = blend_test_3model[
    ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
].copy()

blend_test_3model.to_csv(
    RF_EXTRATREES_LGBM_BLEND_TEST_PATH,
    index=False
)

submission_rf_extratrees_lgbm_blend.to_csv(
    RF_EXTRATREES_LGBM_BLEND_SUBMISSION_PATH,
    index=False
)

print("Blend test prediction shape:", blend_test_3model.shape)
print("Blend submission shape:", submission_rf_extratrees_lgbm_blend.shape)

print("\nSaved:")
print(RF_EXTRATREES_LGBM_BLEND_TEST_PATH)
print(RF_EXTRATREES_LGBM_BLEND_SUBMISSION_PATH)

Blend test prediction shape: (48307, 5)
Blend submission shape: (48307, 2)

Saved:
model_results/testpred_blend_rf_et_lgbm_weighted.csv
submission_blend_rf_et_lgbm_oof_weighted.csv


In [229]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Blend",
                "model": "rf_extratrees_lightgbm_blend",
                "feature_space": "OOF prediction blend",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": rf_extratrees_lgbm_blend_oof_mse,
                "notes": (
                    f"w_rf={best_w_rf_3model:.3f}; "
                    f"w_extratrees={best_w_extratrees_3model:.3f}; "
                    f"w_lgbm={best_w_lgbm_3model:.3f}; "
                    "RF 500 base + ExtraTrees base + selected LightGBM 12k"
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
1,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
2,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
3,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
4,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
5,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
6,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
7,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...
8,Boosting,lightgbm_broad_holdout_screen,base,fixed holdout,70.014712,126.393035,lgbm_broad_l127_child120_l2_20; best_iteration...
9,Bagging,random_forest_refined,base,5-fold CV mean,18.459212,134.708097,n_estimators=300; max_samples=None; max_featur...


The three-model blend improves over pure selected LightGBM.

Random Forest receives zero weight, while ExtraTrees and LightGBM are blended. This means Random Forest does not add useful signal once ExtraTrees and LightGBM are both included.

The blend is kept because it improves OOF MSE by more than 5 points relative to selected LightGBM alone.

#### 5E. Long selected LightGBM suite and automatic re-blend

The selected 12,000-tree LightGBM artifact reached the estimator cap on nearly every fold.

This suggests that the selected LightGBM configuration was still improving when training stopped. The original workflow therefore trained two longer versions of the same selected structure:

1. `long80k_lr03`  
   - learning rate `0.03`
   - maximum `80,000` trees
   - early stopping after `2,500` rounds

2. `long100k_lr02`  
   - learning rate `0.02`
   - maximum `100,000` trees
   - early stopping after `3,000` rounds

After both long artifacts are trained, the notebook automatically re-blends:

- Random Forest
- ExtraTrees
- selected 12k LightGBM
- long80k LightGBM
- long100k LightGBM

This section is kept because the original workflow found a meaningful OOF gain from the long LightGBM artifacts and the automatic re-blend.

In [231]:
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [233]:
LONG_LGBM_N_SPLITS = 5

X_long_lgbm = np.ascontiguousarray(
    np.asarray(X_train_proc_model, dtype=np.float32)
)

X_long_lgbm_test = np.ascontiguousarray(
    np.asarray(X_test_proc_model, dtype=np.float32)
)

y_long_lgbm = np.asarray(
    y_train,
    dtype=np.float32
).ravel()

print("Long LightGBM training matrix:", X_long_lgbm.shape)
print("Long LightGBM test matrix:", X_long_lgbm_test.shape)
print("Target shape:", y_long_lgbm.shape)

Long LightGBM training matrix: (144921, 162)
Long LightGBM test matrix: (48307, 162)
Target shape: (144921,)


The long LightGBM models use the same selected structure as the 12k artifact.

The only changes are the learning rate and maximum number of trees.

In [234]:
long_lgbm_base_params = {
    "objective": "regression",
    "metric": "l2",
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

long_lgbm_jobs = [
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long80k_lr03",
        "learning_rate": 0.03,
        "n_estimators": 80000,
        "early_stopping_rounds": 2500,
        "log_period": 2000,
    },
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long100k_lr02",
        "learning_rate": 0.02,
        "n_estimators": 100000,
        "early_stopping_rounds": 3000,
        "log_period": 2500,
    },
]

pd.DataFrame(long_lgbm_jobs)

,artifact_name,learning_rate,n_estimators,early_stopping_rounds,log_period
0,lgbm_t03_base_5fold_oof_long80k_lr03,0.03,80000,2500,2000
1,lgbm_t03_base_5fold_oof_long100k_lr02,0.02,100000,3000,2500


The long LightGBM helper below is checkpointed by fold.

For each artifact it saves:

- fold metrics,
- partial OOF predictions,
- one test-prediction file per fold,
- final OOF CSV,
- final fold-averaged test prediction CSV,
- final submission CSV.

If a long run is interrupted, rerunning the same cell resumes from completed folds.

In [235]:
def run_long_lgbm_artifact(job):
    artifact_name = job["artifact_name"]

    metrics_path = RESULTS_DIR / f"{artifact_name}_fold_metrics.csv"
    oof_npy_path = RESULTS_DIR / f"{artifact_name}_oof.npy"

    oof_csv_path = RESULTS_DIR / f"oof_{artifact_name}.csv"
    testpred_csv_path = RESULTS_DIR / f"testpred_{artifact_name}_foldavg.csv"
    submission_path = Path(f"submission_{artifact_name}_foldavg.csv")

    params = long_lgbm_base_params.copy()
    params.update({
        "n_estimators": job["n_estimators"],
        "learning_rate": job["learning_rate"],
    })

    if metrics_path.exists():
        fold_metrics = pd.read_csv(metrics_path)
        completed_folds = set(fold_metrics["fold"].astype(int))
    else:
        fold_metrics = pd.DataFrame()
        completed_folds = set()

    if oof_npy_path.exists():
        oof_pred = np.load(oof_npy_path)
        assert len(oof_pred) == len(X_long_lgbm)
    else:
        oof_pred = np.full(
            len(X_long_lgbm),
            np.nan,
            dtype=np.float32
        )

    print("\n" + "#" * 90)
    print("Long LightGBM artifact:", artifact_name)
    print("#" * 90)

    print("\nParameters:")
    for key, value in params.items():
        print(f"{key}: {value}")

    print("\nCompleted folds:", sorted(completed_folds))
    print("OOF predictions filled:", int(np.isfinite(oof_pred).sum()))

    kf = KFold(
        n_splits=LONG_LGBM_N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    artifact_start = time.perf_counter()

    for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(kf.split(X_long_lgbm), start=1):
        fold_test_path = RESULTS_DIR / f"{artifact_name}_fold{fold_id}_test_pred.npy"

        fold_oof_done = np.isfinite(oof_pred[fold_valid_idx]).all()
        fold_test_done = fold_test_path.exists()
        fold_metric_done = fold_id in completed_folds

        if fold_oof_done and fold_test_done and fold_metric_done:
            print(f"\nSkipping completed fold {fold_id}.")
            continue

        if fold_oof_done or fold_test_done or fold_metric_done:
            print(f"\nFold {fold_id} has partial checkpoint state. Rerunning it cleanly.")

            if len(fold_metrics) > 0:
                fold_metrics = fold_metrics[
                    fold_metrics["fold"].astype(int) != fold_id
                ].copy()

                fold_metrics.to_csv(
                    metrics_path,
                    index=False
                )

            if fold_test_path.exists():
                fold_test_path.unlink()

            oof_pred[fold_valid_idx] = np.nan
            np.save(oof_npy_path, oof_pred)

        print("\n" + "=" * 80)
        print(f"{artifact_name}: fold {fold_id} / {LONG_LGBM_N_SPLITS}")
        print("=" * 80)

        fold_start = time.perf_counter()

        X_fold_train = X_long_lgbm[fold_train_idx]
        X_fold_valid = X_long_lgbm[fold_valid_idx]

        y_fold_train = y_long_lgbm[fold_train_idx]
        y_fold_valid = y_long_lgbm[fold_valid_idx]

        model = lgb.LGBMRegressor(
            **params,
            random_state=RANDOM_STATE + fold_id
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            eval_set=[(X_fold_valid, y_fold_valid)],
            eval_metric="l2",
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=job["early_stopping_rounds"],
                    verbose=True
                ),
                lgb.log_evaluation(
                    period=job["log_period"]
                ),
            ],
        )

        best_iteration = model.best_iteration_

        if best_iteration is None or best_iteration <= 0:
            best_iteration = job["n_estimators"]

        valid_pred_raw = model.predict(
            X_fold_valid,
            num_iteration=best_iteration
        )

        valid_pred_clipped = np.clip(
            valid_pred_raw,
            0,
            100
        ).astype(np.float32)

        test_pred_raw = model.predict(
            X_long_lgbm_test,
            num_iteration=best_iteration
        )

        test_pred_clipped = np.clip(
            test_pred_raw,
            0,
            100
        ).astype(np.float32)

        oof_pred[fold_valid_idx] = valid_pred_clipped

        np.save(
            oof_npy_path,
            oof_pred
        )

        np.save(
            fold_test_path,
            test_pred_clipped
        )

        row = {
            "artifact_name": artifact_name,
            "model_class": "LightGBM",
            "feature_space": "base",
            "fold": fold_id,
            "fold_train_n": len(fold_train_idx),
            "fold_valid_n": len(fold_valid_idx),
            "best_iteration": best_iteration,
            "fold_valid_mse_raw": mean_squared_error(y_fold_valid, valid_pred_raw),
            "fold_valid_mse_clipped": mean_squared_error(y_fold_valid, valid_pred_clipped),
            "elapsed_sec": time.perf_counter() - fold_start,
        }

        row.update(params)

        fold_metrics = pd.concat(
            [
                fold_metrics,
                pd.DataFrame([row])
            ],
            axis=0,
            ignore_index=True
        )

        fold_metrics = (
            fold_metrics
            .sort_values("fold")
            .reset_index(drop=True)
        )

        fold_metrics.to_csv(
            metrics_path,
            index=False
        )

        print("best_iteration:", best_iteration)
        print("fold_valid_mse_clipped:", row["fold_valid_mse_clipped"])
        print("elapsed_sec:", round(row["elapsed_sec"], 2))
        print("OOF predictions filled:", int(np.isfinite(oof_pred).sum()))

        del model
        del X_fold_train, X_fold_valid
        del y_fold_train, y_fold_valid
        del valid_pred_raw, valid_pred_clipped
        del test_pred_raw, test_pred_clipped

        gc.collect()

    missing_oof = int(np.isnan(oof_pred).sum())

    fold_test_arrays = []

    for fold_id in range(1, LONG_LGBM_N_SPLITS + 1):
        fold_test_path = RESULTS_DIR / f"{artifact_name}_fold{fold_id}_test_pred.npy"

        if not fold_test_path.exists():
            print("Missing fold test prediction:", fold_test_path)
            continue

        fold_test_arrays.append(
            np.load(fold_test_path).astype(np.float32)
        )

    if missing_oof != 0 or len(fold_test_arrays) != LONG_LGBM_N_SPLITS:
        print("\nArtifact is incomplete. Rerun this same cell to resume.")
        print("Missing OOF predictions:", missing_oof)
        print("Fold test files found:", len(fold_test_arrays))

        return {
            "artifact_name": artifact_name,
            "completed": False,
            "oof_mse_clipped": np.nan,
            "metrics_path": metrics_path,
            "oof_csv_path": oof_csv_path,
            "testpred_csv_path": testpred_csv_path,
            "submission_path": submission_path,
        }

    oof_mse_clipped = mean_squared_error(
        y_long_lgbm,
        oof_pred
    )

    test_pred_foldavg = np.clip(
        np.mean(np.vstack(fold_test_arrays), axis=0),
        0,
        100
    ).astype(np.float32)

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": id_array(train_ids),
        "PERCENT_PROFICIENT_TRUE": y_long_lgbm,
        f"OOF_{artifact_name}": oof_pred,
    })

    testpred_df = pd.DataFrame({
        "ASSESSMENT_ID": id_array(test_ids),
        f"TESTPRED_{artifact_name}": test_pred_foldavg,
    })

    submission_df = pd.DataFrame({
        "ASSESSMENT_ID": id_array(test_ids),
        "PERCENT_PROFICIENT": test_pred_foldavg,
    })

    oof_df.to_csv(
        oof_csv_path,
        index=False
    )

    testpred_df.to_csv(
        testpred_csv_path,
        index=False
    )

    submission_df.to_csv(
        submission_path,
        index=False
    )

    artifact_elapsed = time.perf_counter() - artifact_start

    print("\nCompleted artifact:", artifact_name)
    print("OOF MSE clipped:", oof_mse_clipped)

    print("\nFold metrics:")
    display(
        fold_metrics[
            [
                "fold",
                "fold_valid_mse_clipped",
                "fold_valid_mse_raw",
                "best_iteration",
                "elapsed_sec",
            ]
        ].reset_index(drop=True)
    )

    print("\nSaved:")
    print(metrics_path)
    print(oof_csv_path)
    print(testpred_csv_path)
    print(submission_path)

    print("\nArtifact elapsed seconds:", round(artifact_elapsed, 2))

    return {
        "artifact_name": artifact_name,
        "completed": True,
        "oof_mse_clipped": float(oof_mse_clipped),
        "metrics_path": metrics_path,
        "oof_csv_path": oof_csv_path,
        "testpred_csv_path": testpred_csv_path,
        "submission_path": submission_path,
    }

##### Long LightGBM artifact 1: long80k, learning rate 0.03

This artifact keeps the selected LightGBM structure but allows up to `80,000` trees at learning rate `0.03`.

Expected OOF MSE from the original workflow is about `94.16`.

In [236]:
long80k_result = run_long_lgbm_artifact(
    long_lgbm_jobs[0]
)

long80k_result


##########################################################################################
Long LightGBM artifact: lgbm_t03_base_5fold_oof_long80k_lr03
##########################################################################################

Parameters:
objective: regression
metric: l2
n_jobs: 1
verbosity: -1
force_col_wise: True
num_leaves: 95
min_child_samples: 60
subsample: 0.85
subsample_freq: 1
colsample_bytree: 0.9
reg_alpha: 0.0
reg_lambda: 5.0
max_depth: -1
n_estimators: 80000
learning_rate: 0.03

Completed folds: []
OOF predictions filled: 0

lgbm_t03_base_5fold_oof_long80k_lr03: fold 1 / 5
Training until validation scores don't improve for 2500 rounds
[2000]	valid_0's l2: 129.408
[4000]	valid_0's l2: 115.861
[6000]	valid_0's l2: 109.066
[8000]	valid_0's l2: 105.062
[10000]	valid_0's l2: 102.592
[12000]	valid_0's l2: 100.822
[14000]	valid_0's l2: 99.6255
[16000]	valid_0's l2: 98.7821
[18000]	valid_0's l2: 98.2385
[20000]	valid_0's l2: 97.7689
[22000]	valid_0's l2: 97.4802
[

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,95.764534,96.739767,39024,2664.440927
1,2,91.844963,92.593493,39385,2508.991979
2,3,94.334686,95.260975,34065,1947.446119
3,4,92.318062,93.309677,36884,2002.379572
4,5,96.555138,97.543099,40463,2046.100898



Saved:
model_results/lgbm_t03_base_5fold_oof_long80k_lr03_fold_metrics.csv
model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv
model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv
submission_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv

Artifact elapsed seconds: 11171.97


{'artifact_name': 'lgbm_t03_base_5fold_oof_long80k_lr03',
 'completed': True,
 'oof_mse_clipped': 94.16348266601562,
 'metrics_path': PosixPath('model_results/lgbm_t03_base_5fold_oof_long80k_lr03_fold_metrics.csv'),
 'oof_csv_path': PosixPath('model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv'),
 'testpred_csv_path': PosixPath('model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv'),
 'submission_path': PosixPath('submission_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv')}

##### Long LightGBM artifact 2: long100k, learning rate 0.02

This artifact lowers the learning rate to `0.02` and allows up to `100,000` trees.

Expected OOF MSE from the original workflow is about `93.73`.

##### Long LightGBM suite summary

The two long LightGBM artifacts are summarized below.

The long100k model is expected to be the stronger of the two individual long LightGBM artifacts.

In [244]:
long100k_result = run_long_lgbm_artifact(
    long_lgbm_jobs[1]
)

long100k_result


##########################################################################################
Long LightGBM artifact: lgbm_t03_base_5fold_oof_long100k_lr02
##########################################################################################

Parameters:
objective: regression
metric: l2
n_jobs: 1
verbosity: -1
force_col_wise: True
num_leaves: 95
min_child_samples: 60
subsample: 0.85
subsample_freq: 1
colsample_bytree: 0.9
reg_alpha: 0.0
reg_lambda: 5.0
max_depth: -1
n_estimators: 100000
learning_rate: 0.02

Completed folds: []
OOF predictions filled: 0

lgbm_t03_base_5fold_oof_long100k_lr02: fold 1 / 5
Training until validation scores don't improve for 3000 rounds
[2500]	valid_0's l2: 132.896
[5000]	valid_0's l2: 119.103
[7500]	valid_0's l2: 111.575
[10000]	valid_0's l2: 107.136
[12500]	valid_0's l2: 104.191
[15000]	valid_0's l2: 102.111
[17500]	valid_0's l2: 100.632
[20000]	valid_0's l2: 99.4575
[22500]	valid_0's l2: 98.6736
[25000]	valid_0's l2: 98.0836
[27500]	valid_0's l2: 97.63

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,95.241920,96.194398,54700,2768.080933
1,2,91.291603,92.035591,52940,2410.600852
2,3,93.777443,94.710169,59506,2978.432545
3,4,92.410759,93.340667,53392,2427.601405
4,5,95.909294,96.860579,64138,3072.133163



Saved:
model_results/lgbm_t03_base_5fold_oof_long100k_lr02_fold_metrics.csv
model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv
model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv
submission_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv

Artifact elapsed seconds: 13658.4


{'artifact_name': 'lgbm_t03_base_5fold_oof_long100k_lr02',
 'completed': True,
 'oof_mse_clipped': 93.7262191772461,
 'metrics_path': PosixPath('model_results/lgbm_t03_base_5fold_oof_long100k_lr02_fold_metrics.csv'),
 'oof_csv_path': PosixPath('model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv'),
 'testpred_csv_path': PosixPath('model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv'),
 'submission_path': PosixPath('submission_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv')}

In [245]:
for result in [long80k_result, long100k_result]:
    if result["completed"]:
        model_scoreboard = pd.concat(
            [
                model_scoreboard,
                pd.DataFrame([
                    {
                        "model_family": "Boosting",
                        "model": result["artifact_name"],
                        "feature_space": "base",
                        "metric_source": "5-fold OOF",
                        "train_mse": np.nan,
                        "val_mse": result["oof_mse_clipped"],
                        "notes": "long selected LightGBM artifact",
                    }
                ])
            ],
            axis=0,
            ignore_index=True
        )

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
1,Boosting,lgbm_t03_base_5fold_oof_long100k_lr02,base,5-fold OOF,NaN,93.726219,long selected LightGBM artifact
2,Boosting,lgbm_t03_base_5fold_oof_long80k_lr03,base,5-fold OOF,NaN,94.163485,long selected LightGBM artifact
3,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
4,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
5,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
6,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
7,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
8,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...
9,Bagging,random_forest_oof_500,base,5-fold OOF,NaN,122.328024,500-tree RF OOF artifact; fold-averaged test p...


##### Blend after long LightGBM models

Both long LightGBM artifacts are now complete.

The next step is to blend the saved OOF predictions from:

- Random Forest
- ExtraTrees
- selected 12k LightGBM
- long 80k LightGBM
- long 100k LightGBM

A blend is just a weighted average of model predictions.

For example:

`blended prediction = 0.30 × ExtraTrees prediction + 0.70 × long LightGBM prediction`

The weights are required to be nonnegative and add up to 1. This keeps the final prediction as an average of existing model predictions.

The purpose is to check whether the long 100k LightGBM model and ExtraTrees make different enough errors that averaging them improves OOF MSE.

In [246]:
blend_components = [
    {
        "name": "random_forest",
        "display_name": "Random Forest",
        "oof_path": RF_OOF_PATH,
        "test_path": RF_TEST_FOLD_PATH,
        "oof_pred_col": "rf_500_base_oof_pred",
        "test_pred_col": "rf_500_base_foldavg_pred",
        "target_col": "y_true",
    },
    {
        "name": "extratrees",
        "display_name": "ExtraTrees",
        "oof_path": EXTRATREES_OOF_PRED_PATH,
        "test_path": EXTRATREES_TEST_PRED_PATH,
        "oof_pred_col": "extratrees_oof_pred",
        "test_pred_col": "PERCENT_PROFICIENT",
        "target_col": "y_true",
    },
    {
        "name": "lgbm_12k",
        "display_name": "LightGBM selected 12k",
        "oof_path": LGBM_SELECTED_OOF_PATH,
        "test_path": LGBM_SELECTED_TEST_PRED_PATH,
        "oof_pred_col": f"OOF_{LGBM_SELECTED_ARTIFACT}",
        "test_pred_col": f"TESTPRED_{LGBM_SELECTED_ARTIFACT}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_80k_lr03",
        "display_name": "LightGBM long 80k",
        "oof_path": long80k_result["oof_csv_path"],
        "test_path": long80k_result["testpred_csv_path"],
        "oof_pred_col": f"OOF_{long80k_result['artifact_name']}",
        "test_pred_col": f"TESTPRED_{long80k_result['artifact_name']}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_100k_lr02",
        "display_name": "LightGBM long 100k",
        "oof_path": long100k_result["oof_csv_path"],
        "test_path": long100k_result["testpred_csv_path"],
        "oof_pred_col": f"OOF_{long100k_result['artifact_name']}",
        "test_pred_col": f"TESTPRED_{long100k_result['artifact_name']}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
]

pd.DataFrame([
    {
        "name": component["name"],
        "display_name": component["display_name"],
        "oof_path": str(component["oof_path"]),
        "test_path": str(component["test_path"]),
    }
    for component in blend_components
])

,name,display_name,oof_path,test_path
0,random_forest,Random Forest,model_results/oof_rf_500_base.csv,model_results/testpred_rf_500_base_folds.csv
1,extratrees,ExtraTrees,model_results/oof_extratrees_base.csv,model_results/testpred_extratrees_base_foldavg...
2,lgbm_12k,LightGBM selected 12k,model_results/oof_lgbm_t03_base_5fold_oof.csv,model_results/testpred_lgbm_t03_base_5fold_oof...
3,lgbm_80k_lr03,LightGBM long 80k,model_results/oof_lgbm_t03_base_5fold_oof_long...,model_results/testpred_lgbm_t03_base_5fold_oof...
4,lgbm_100k_lr02,LightGBM long 100k,model_results/oof_lgbm_t03_base_5fold_oof_long...,model_results/testpred_lgbm_t03_base_5fold_oof...


In [247]:
blend_oof_data = None
component_names = []
component_display_names = {}

for component in blend_components:
    oof_df = pd.read_csv(component["oof_path"])

    current = oof_df[
        [
            "ASSESSMENT_ID",
            component["target_col"],
            component["oof_pred_col"],
        ]
    ].copy()

    current["ASSESSMENT_ID"] = current["ASSESSMENT_ID"].astype(str)

    current = current.rename(
        columns={
            component["target_col"]: "y_true",
            component["oof_pred_col"]: component["name"],
        }
    )

    if blend_oof_data is None:
        blend_oof_data = current.copy()
    else:
        blend_oof_data = blend_oof_data.merge(
            current,
            on="ASSESSMENT_ID",
            how="inner",
            suffixes=("", "_new")
        )

        if not np.allclose(blend_oof_data["y_true"], blend_oof_data["y_true_new"]):
            raise ValueError(f"Target mismatch after merging {component['name']}.")

        blend_oof_data = blend_oof_data.drop(columns=["y_true_new"])

    component_names.append(component["name"])
    component_display_names[component["name"]] = component["display_name"]

print("Merged OOF rows:", blend_oof_data.shape[0])
print("Expected training rows:", len(train_ids))
print("Components:", component_names)

Merged OOF rows: 144921
Expected training rows: 144921
Components: ['random_forest', 'extratrees', 'lgbm_12k', 'lgbm_80k_lr03', 'lgbm_100k_lr02']


In [248]:
if blend_oof_data.shape[0] != len(train_ids):
    raise ValueError("Blend OOF data lost rows.")

y_blend = blend_oof_data["y_true"].to_numpy(dtype=float)

component_oof_results = []

for name in component_names:
    pred = np.clip(blend_oof_data[name].to_numpy(dtype=float), 0, 100)

    component_oof_results.append({
        "component": name,
        "display_name": component_display_names[name],
        "oof_mse_clipped": mean_squared_error(y_blend, pred),
        "pred_mean": float(np.mean(pred)),
        "pred_std": float(np.std(pred)),
    })

component_oof_results = (
    pd.DataFrame(component_oof_results)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

component_oof_results

,component,display_name,oof_mse_clipped,pred_mean,pred_std
0,lgbm_100k_lr02,LightGBM long 100k,93.726214,54.153144,24.734667
1,lgbm_80k_lr03,LightGBM long 80k,94.163485,54.147019,24.743266
2,lgbm_12k,LightGBM selected 12k,98.779406,54.152808,24.283770
3,extratrees,ExtraTrees,110.749294,54.127367,23.857745
4,random_forest,Random Forest,122.328024,54.134143,22.839348


The long 100k LightGBM model is the strongest individual component.

The blend search now checks whether ExtraTrees or any of the earlier models can improve on it through weighted averaging.

In [249]:
prediction_matrix = np.vstack([
    blend_oof_data[name].to_numpy(dtype=float)
    for name in component_names
]).astype(np.float32)

candidate_weights = []
candidate_sources = []
seen_weight_keys = set()

def add_candidate_weights(weights, source):
    weights = np.asarray(weights, dtype=float)

    if np.any(weights < -1e-12):
        return

    total = weights.sum()

    if total <= 0:
        return

    weights = weights / total
    weights[np.abs(weights) < 1e-12] = 0.0

    key = tuple(np.round(weights, 6))

    if key in seen_weight_keys:
        return

    seen_weight_keys.add(key)
    candidate_weights.append(weights)
    candidate_sources.append(source)


n_components = len(component_names)

# Single-model candidates
for i in range(n_components):
    weights = np.zeros(n_components)
    weights[i] = 1.0
    add_candidate_weights(weights, "single_model")

# Two-model weighted averages
for i in range(n_components):
    for j in range(i + 1, n_components):
        for weight_i in np.linspace(0, 1, 1001):
            weights = np.zeros(n_components)
            weights[i] = weight_i
            weights[j] = 1.0 - weight_i
            add_candidate_weights(weights, "two_model_grid")

# Random weighted averages focused toward stronger models
rng = np.random.default_rng(RANDOM_STATE)

best_single_mse = float(component_oof_results["oof_mse_clipped"].min())

dirichlet_alpha = []

for name in component_names:
    mse_value = float(
        component_oof_results.loc[
            component_oof_results["component"] == name,
            "oof_mse_clipped"
        ].iloc[0]
    )

    if mse_value <= best_single_mse + 2:
        dirichlet_alpha.append(8.0)
    elif "lgbm" in name:
        dirichlet_alpha.append(5.0)
    elif "extratrees" in name:
        dirichlet_alpha.append(3.0)
    else:
        dirichlet_alpha.append(1.0)

dirichlet_alpha = np.asarray(dirichlet_alpha, dtype=float)

for _ in range(30000):
    add_candidate_weights(
        rng.dirichlet(dirichlet_alpha),
        "random_weight_search"
    )

candidate_weights = np.vstack(candidate_weights).astype(np.float32)

print("Blend components:", component_names)
print("Candidate weights:", candidate_weights.shape[0])

Blend components: ['random_forest', 'extratrees', 'lgbm_12k', 'lgbm_80k_lr03', 'lgbm_100k_lr02']
Candidate weights: 39995


In [250]:
blend_rows = []

for row_idx, weights in enumerate(candidate_weights):
    pred_raw = np.dot(weights, prediction_matrix)
    pred_clipped = np.clip(pred_raw, 0, 100)

    row = {
        "source": candidate_sources[row_idx],
        "oof_mse_raw": mean_squared_error(y_blend, pred_raw),
        "oof_mse_clipped": mean_squared_error(y_blend, pred_clipped),
    }

    for name, weight in zip(component_names, weights):
        row[f"w_{name}"] = float(weight)

    blend_rows.append(row)

long_lgbm_blend_screen = (
    pd.DataFrame(blend_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

best_long_lgbm_blend = long_lgbm_blend_screen.iloc[0]

best_long_lgbm_blend

source              two_model_grid
oof_mse_raw              88.966368
oof_mse_clipped          88.966368
w_random_forest                0.0
w_extratrees                 0.319
w_lgbm_12k                     0.0
w_lgbm_80k_lr03                0.0
w_lgbm_100k_lr02             0.681
Name: 0, dtype: object

In [251]:
long_lgbm_blend_screen.head(20)

,source,oof_mse_raw,oof_mse_clipped,w_random_forest,w_extratrees,w_lgbm_12k,w_lgbm_80k_lr03,w_lgbm_100k_lr02
0,two_model_grid,88.966368,88.966368,0.0,0.319,0.0,0.0,0.681
1,two_model_grid,88.966373,88.966373,0.0,0.318,0.0,0.0,0.682
2,two_model_grid,88.966458,88.966458,0.0,0.320,0.0,0.0,0.680
3,two_model_grid,88.966470,88.966470,0.0,0.317,0.0,0.0,0.683
4,two_model_grid,88.966640,88.966640,0.0,0.321,0.0,0.0,0.679
5,two_model_grid,88.966662,88.966662,0.0,0.316,0.0,0.0,0.684
6,two_model_grid,88.966919,88.966919,0.0,0.322,0.0,0.0,0.678
7,two_model_grid,88.966949,88.966949,0.0,0.315,0.0,0.0,0.685
8,two_model_grid,88.967288,88.967288,0.0,0.323,0.0,0.0,0.677
9,two_model_grid,88.967329,88.967329,0.0,0.314,0.0,0.0,0.686


In [252]:
best_blend_weights = np.array(
    [
        best_long_lgbm_blend[f"w_{name}"]
        for name in component_names
    ],
    dtype=np.float32
)

long_lgbm_blend_oof_pred = np.clip(
    np.dot(best_blend_weights, prediction_matrix),
    0,
    100
).astype(np.float32)

long_lgbm_blend_oof_mse = mean_squared_error(
    y_blend,
    long_lgbm_blend_oof_pred
)

blend_oof_data["blend_pred"] = long_lgbm_blend_oof_pred

LONG_LGBM_BLEND_SCREEN_PATH = RESULTS_DIR / "blend_long_lgbm_weight_screen.csv"
LONG_LGBM_BLEND_OOF_PATH = RESULTS_DIR / "oof_blend_long_lgbm_weighted.csv"

# Original-compatible aliases.
LONG_LGBM_BLEND_SCREEN_ALIAS_PATH = RESULTS_DIR / "blend_auto_et_lgbm_long_oof_weighted_weight_screen.csv"
LONG_LGBM_BLEND_OOF_ALIAS_PATH = RESULTS_DIR / "oof_blend_auto_et_lgbm_long_oof_weighted.csv"

long_lgbm_blend_screen.to_csv(LONG_LGBM_BLEND_SCREEN_PATH, index=False)
blend_oof_data.to_csv(LONG_LGBM_BLEND_OOF_PATH, index=False)

long_lgbm_blend_screen.to_csv(LONG_LGBM_BLEND_SCREEN_ALIAS_PATH, index=False)
blend_oof_data.to_csv(LONG_LGBM_BLEND_OOF_ALIAS_PATH, index=False)

print("Long LightGBM blend OOF MSE:", long_lgbm_blend_oof_mse)

print("\nBest weights:")
for name, weight in zip(component_names, best_blend_weights):
    print(f"{name}: {weight:.6f}")

Long LightGBM blend OOF MSE: 88.96636830459983

Best weights:
random_forest: 0.000000
extratrees: 0.319000
lgbm_12k: 0.000000
lgbm_80k_lr03: 0.000000
lgbm_100k_lr02: 0.681000


In [253]:
blend_test_data = None

for component in blend_components:
    test_df = pd.read_csv(component["test_path"])

    current = test_df[
        [
            "ASSESSMENT_ID",
            component["test_pred_col"],
        ]
    ].copy()

    current["ASSESSMENT_ID"] = current["ASSESSMENT_ID"].astype(str)

    current = current.rename(
        columns={
            component["test_pred_col"]: component["name"]
        }
    )

    if blend_test_data is None:
        blend_test_data = current.copy()
    else:
        blend_test_data = blend_test_data.merge(
            current,
            on="ASSESSMENT_ID",
            how="inner"
        )

print("Merged test rows:", blend_test_data.shape[0])
print("Expected test rows:", len(test_ids))

Merged test rows: 48307
Expected test rows: 48307


In [254]:
if blend_test_data.shape[0] != len(test_ids):
    raise ValueError("Blend test data lost rows.")

test_order = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "_test_order": np.arange(len(test_ids)),
})

blend_test_data = test_order.merge(
    blend_test_data,
    on="ASSESSMENT_ID",
    how="left"
)

missing_test_values = blend_test_data[component_names].isna().sum()

if missing_test_values.sum() > 0:
    raise ValueError(f"Missing test predictions:\n{missing_test_values}")

blend_test_data = (
    blend_test_data
    .sort_values("_test_order")
    .drop(columns=["_test_order"])
    .reset_index(drop=True)
)

test_prediction_matrix = np.vstack([
    blend_test_data[name].to_numpy(dtype=float)
    for name in component_names
]).astype(np.float32)

long_lgbm_blend_test_pred = np.clip(
    np.dot(best_blend_weights, test_prediction_matrix),
    0,
    100
).astype(np.float32)

blend_test_data["PERCENT_PROFICIENT"] = long_lgbm_blend_test_pred

LONG_LGBM_BLEND_TEST_PATH = RESULTS_DIR / "testpred_blend_long_lgbm_weighted.csv"
LONG_LGBM_BLEND_SUBMISSION_PATH = Path("submission_blend_long_lgbm_weighted.csv")

# Original-compatible aliases.
LONG_LGBM_BLEND_TEST_ALIAS_PATH = RESULTS_DIR / "testpred_blend_auto_et_lgbm_long_oof_weighted.csv"
LONG_LGBM_BLEND_SUBMISSION_ALIAS_PATH = Path("submission_blend_auto_et_lgbm_long_oof_weighted.csv")

blend_test_data.to_csv(LONG_LGBM_BLEND_TEST_PATH, index=False)

submission_long_lgbm_blend = pd.DataFrame({
    "ASSESSMENT_ID": id_array(test_ids),
    "PERCENT_PROFICIENT": long_lgbm_blend_test_pred,
})

submission_long_lgbm_blend.to_csv(LONG_LGBM_BLEND_SUBMISSION_PATH, index=False)

blend_test_data.to_csv(LONG_LGBM_BLEND_TEST_ALIAS_PATH, index=False)
submission_long_lgbm_blend.to_csv(LONG_LGBM_BLEND_SUBMISSION_ALIAS_PATH, index=False)

print("Blend test prediction shape:", blend_test_data.shape)
print("Blend submission shape:", submission_long_lgbm_blend.shape)

print("\nSaved:")
print(LONG_LGBM_BLEND_TEST_PATH)
print(LONG_LGBM_BLEND_SUBMISSION_PATH)
print(LONG_LGBM_BLEND_TEST_ALIAS_PATH)
print(LONG_LGBM_BLEND_SUBMISSION_ALIAS_PATH)

Blend test prediction shape: (48307, 7)
Blend submission shape: (48307, 2)

Saved:
model_results/testpred_blend_long_lgbm_weighted.csv
submission_blend_long_lgbm_weighted.csv
model_results/testpred_blend_auto_et_lgbm_long_oof_weighted.csv
submission_blend_auto_et_lgbm_long_oof_weighted.csv


In [255]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Blend",
                "model": "long_lightgbm_extratrees_blend",
                "feature_space": "OOF prediction blend",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": long_lgbm_blend_oof_mse,
                "notes": "; ".join(
                    [
                        f"w_{name}={weight:.3f}"
                        for name, weight in zip(component_names, best_blend_weights)
                        if weight > 0.0005
                    ]
                ),
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Blend,long_lightgbm_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,88.966368,w_extratrees=0.319; w_lgbm_100k_lr02=0.681
1,Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
2,Boosting,lgbm_t03_base_5fold_oof_long100k_lr02,base,5-fold OOF,NaN,93.726219,long selected LightGBM artifact
3,Boosting,lgbm_t03_base_5fold_oof_long80k_lr03,base,5-fold OOF,NaN,94.163485,long selected LightGBM artifact
4,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
5,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
6,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
7,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
8,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...
9,Bagging,extratrees_holdout_screen,base,fixed holdout,0.025122,111.627310,n_estimators=180; max_features=0.5; min_sample...


The long LightGBM blend improves over the earlier ExtraTrees + selected 12k LightGBM blend.

The earlier blend had OOF MSE around `93.50`. The long LightGBM blend is expected to reduce OOF MSE to about `88.97`.

The best blend mostly combines ExtraTrees with the long 100k LightGBM model. This shows that the long LightGBM run adds meaningful signal, while ExtraTrees still contributes complementary predictions.

#### 5F. Saved-prediction blend refinement after public-score checkpoint

The long LightGBM + ExtraTrees blend was the strongest model so far.

This section records the public-score checkpoint and tests a few fast refinements using only saved OOF and test predictions.

No base model is retrained here.

The methods are:

1. Recreate the submitted ExtraTrees + long 100k LightGBM blend.
2. Slightly refine that two-model blend weight using OOF MSE.
3. Try a weighted average over all saved prediction files.
4. Try a small Ridge stack using saved OOF predictions as inputs.

A weighted average means the model predictions are averaged with nonnegative weights that add to 1.

A stack means a small second-stage model learns how to combine the saved predictions.

In [258]:
from scipy.optimize import minimize, minimize_scalar
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

TARGET_COL = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

public_checkpoint_mse = 80.662
public_checkpoint_file = "submission_blend_auto_et_lgbm_long_oof_weighted.csv"

print("Public checkpoint file:", public_checkpoint_file)
print("Public MSE:", public_checkpoint_mse)

Public checkpoint file: submission_blend_auto_et_lgbm_long_oof_weighted.csv
Public MSE: 80.662


In [259]:
submission_tracker_path = RESULTS_DIR / "submission_tracker.csv"

public_checkpoint = pd.DataFrame([
    {
        "file": public_checkpoint_file,
        "model_family": "Blend",
        "public_mse": public_checkpoint_mse,
        "notes": "ExtraTrees + long 100k LightGBM blend"
    }
])

if submission_tracker_path.exists():
    submission_tracker = pd.read_csv(submission_tracker_path)
    submission_tracker = pd.concat([submission_tracker, public_checkpoint], ignore_index=True)
else:
    submission_tracker = public_checkpoint.copy()

submission_tracker = (
    submission_tracker
    .drop_duplicates(subset=["file"], keep="last")
    .reset_index(drop=True)
)

submission_tracker.to_csv(submission_tracker_path, index=False)

submission_tracker.tail()

,file,model_family,public_mse,notes
0,submission_blend_auto_et_lgbm_long_oof_weighte...,Blend,80.662,ExtraTrees + long 100k LightGBM blend


##### Load saved OOF and test predictions

The next cell loads the saved prediction files used in this refinement stage.

The components are:

- Random Forest
- ExtraTrees
- selected 12k LightGBM
- long 80k LightGBM
- long 100k LightGBM

In [260]:
def read_prediction(path, pred_col):
    df = pd.read_csv(path)

    if pred_col not in df.columns:
        raise ValueError(f"{pred_col} not found in {path}")

    return np.clip(df[pred_col].to_numpy(dtype=float), 0, 100)


saved_components = {
    "random_forest": {
        "oof": (RF_OOF_PATH, "rf_500_base_oof_pred"),
        "test": (RF_TEST_FOLD_PATH, "rf_500_base_foldavg_pred"),
    },
    "extratrees": {
        "oof": (EXTRATREES_OOF_PRED_PATH, "extratrees_oof_pred"),
        "test": (EXTRATREES_TEST_PRED_PATH, "PERCENT_PROFICIENT"),
    },
    "lgbm_12k": {
        "oof": (LGBM_SELECTED_OOF_PATH, f"OOF_{LGBM_SELECTED_ARTIFACT}"),
        "test": (LGBM_SELECTED_TEST_PRED_PATH, f"TESTPRED_{LGBM_SELECTED_ARTIFACT}"),
    },
    "lgbm_80k_lr03": {
        "oof": (long80k_result["oof_csv_path"], f"OOF_{long80k_result['artifact_name']}"),
        "test": (long80k_result["testpred_csv_path"], f"TESTPRED_{long80k_result['artifact_name']}"),
    },
    "lgbm_100k_lr02": {
        "oof": (long100k_result["oof_csv_path"], f"OOF_{long100k_result['artifact_name']}"),
        "test": (long100k_result["testpred_csv_path"], f"TESTPRED_{long100k_result['artifact_name']}"),
    },
}

y_refine = np.asarray(y_train, dtype=float).ravel()

oof_preds = {}
test_preds = {}

for name, files in saved_components.items():
    oof_preds[name] = read_prediction(*files["oof"])
    test_preds[name] = read_prediction(*files["test"])

component_summary = pd.DataFrame([
    {
        "component": name,
        "oof_mse": mean_squared_error(y_refine, pred),
        "oof_mean": pred.mean(),
        "oof_std": pred.std(),
    }
    for name, pred in oof_preds.items()
]).sort_values("oof_mse").reset_index(drop=True)

component_summary

,component,oof_mse,oof_mean,oof_std
0,lgbm_100k_lr02,93.726214,54.153144,24.734667
1,lgbm_80k_lr03,94.163485,54.147019,24.743266
2,lgbm_12k,98.779406,54.152808,24.283770
3,extratrees,110.749294,54.127367,23.857745
4,random_forest,122.328024,54.134143,22.839348


In [261]:
def save_candidate(name, oof_pred, test_pred):
    oof_pred = np.clip(oof_pred, 0, 100)
    test_pred = np.clip(test_pred, 0, 100)

    oof_path = RESULTS_DIR / f"oof_{name}.csv"
    test_path = RESULTS_DIR / f"testpred_{name}.csv"
    submission_path = Path(f"submission_{name}.csv")

    pd.DataFrame({
        ID_COL: id_array(train_ids),
        TARGET_COL: y_refine,
        f"OOF_{name}": oof_pred,
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: id_array(test_ids),
        f"TESTPRED_{name}": test_pred,
    }).to_csv(test_path, index=False)

    pd.DataFrame({
        ID_COL: id_array(test_ids),
        TARGET_COL: test_pred,
    }).to_csv(submission_path, index=False)

    return {
        "candidate": name,
        "oof_path": str(oof_path),
        "testpred_path": str(test_path),
        "submission_path": str(submission_path),
    }


refinement_rows = []
saved_candidates = []

##### Two-model blend refinement

The submitted blend used:

`0.319 × ExtraTrees + 0.681 × long 100k LightGBM`

The next cell checks whether a slightly different ExtraTrees weight improves OOF MSE.

In [262]:
submitted_oof = (
    0.319 * oof_preds["extratrees"]
    + 0.681 * oof_preds["lgbm_100k_lr02"]
)

submitted_test = (
    0.319 * test_preds["extratrees"]
    + 0.681 * test_preds["lgbm_100k_lr02"]
)

refinement_rows.append({
    "candidate": "submitted_public80p662_recreated",
    "method": "recorded submitted blend",
    "oof_mse": mean_squared_error(y_refine, submitted_oof),
    "w_extratrees": 0.319,
    "w_lgbm_100k_lr02": 0.681,
})


def two_model_loss(w_extratrees):
    pred = (
        w_extratrees * oof_preds["extratrees"]
        + (1 - w_extratrees) * oof_preds["lgbm_100k_lr02"]
    )

    return mean_squared_error(y_refine, np.clip(pred, 0, 100))


two_model_result = minimize_scalar(
    two_model_loss,
    bounds=(0, 1),
    method="bounded"
)

w_et = float(two_model_result.x)
w_lgbm100 = 1 - w_et

two_model_oof = (
    w_et * oof_preds["extratrees"]
    + w_lgbm100 * oof_preds["lgbm_100k_lr02"]
)

two_model_test = (
    w_et * test_preds["extratrees"]
    + w_lgbm100 * test_preds["lgbm_100k_lr02"]
)

two_model_name = "blend_extratrees_lgbm100k_refined"

refinement_rows.append({
    "candidate": two_model_name,
    "method": "two-model weight refinement",
    "oof_mse": mean_squared_error(y_refine, np.clip(two_model_oof, 0, 100)),
    "w_extratrees": w_et,
    "w_lgbm_100k_lr02": w_lgbm100,
})

saved_candidates.append(
    save_candidate(two_model_name, two_model_oof, two_model_test)
)

print("ExtraTrees weight:", w_et)
print("Long 100k LightGBM weight:", w_lgbm100)
print("OOF MSE:", mean_squared_error(y_refine, np.clip(two_model_oof, 0, 100)))

ExtraTrees weight: 0.31854753344512543
Long 100k LightGBM weight: 0.6814524665548746
OOF MSE: 88.96635862409047


##### Weighted average over all saved components

The next cell allows all saved prediction files to receive a weight.

The weights are constrained to be nonnegative and add up to 1. This keeps the result as an average of model predictions rather than an unrestricted fitted model.

In [263]:
component_order = [
    "random_forest",
    "extratrees",
    "lgbm_12k",
    "lgbm_80k_lr03",
    "lgbm_100k_lr02",
]

X_oof_stack = np.column_stack([
    oof_preds[name]
    for name in component_order
])

X_test_stack = np.column_stack([
    test_preds[name]
    for name in component_order
])


def weighted_average_loss(weights):
    pred = X_oof_stack @ weights
    return mean_squared_error(y_refine, np.clip(pred, 0, 100))


start_weights = np.zeros(len(component_order))
start_weights[component_order.index("extratrees")] = w_et
start_weights[component_order.index("lgbm_100k_lr02")] = w_lgbm100

weighted_result = minimize(
    weighted_average_loss,
    x0=start_weights,
    method="SLSQP",
    bounds=[(0, 1)] * len(component_order),
    constraints={"type": "eq", "fun": lambda w: w.sum() - 1},
)

weighted_average_weights = np.clip(weighted_result.x, 0, 1)
weighted_average_weights = weighted_average_weights / weighted_average_weights.sum()

weighted_average_oof = X_oof_stack @ weighted_average_weights
weighted_average_test = X_test_stack @ weighted_average_weights

weighted_average_name = "blend_all_saved_predictions_weighted"

weighted_details = {
    f"w_{name}": weight
    for name, weight in zip(component_order, weighted_average_weights)
}

refinement_rows.append({
    "candidate": weighted_average_name,
    "method": "all-component weighted average",
    "oof_mse": mean_squared_error(y_refine, np.clip(weighted_average_oof, 0, 100)),
    **weighted_details,
})

saved_candidates.append(
    save_candidate(weighted_average_name, weighted_average_oof, weighted_average_test)
)

print("OOF MSE:", mean_squared_error(y_refine, np.clip(weighted_average_oof, 0, 100)))

print("\nWeights:")
for name, weight in zip(component_order, weighted_average_weights):
    print(f"{name}: {weight:.6f}")

OOF MSE: 88.86553460597635

Weights:
random_forest: 0.000000
extratrees: 0.315230
lgbm_12k: 0.000000
lgbm_80k_lr03: 0.247714
lgbm_100k_lr02: 0.437056


##### Conservative calibration

This cell makes a small scale-and-shift adjustment to the refined two-model blend:

`adjusted prediction = a × prediction + b`

The adjustment is limited to a narrow range so it cannot make a large unstable correction.

In [264]:
def calibration_loss(params):
    a, b = params
    pred = a * two_model_oof + b

    return mean_squared_error(y_refine, np.clip(pred, 0, 100))


calibration_result = minimize(
    calibration_loss,
    x0=np.array([1.0, 0.0]),
    method="L-BFGS-B",
    bounds=[(0.90, 1.10), (-3, 3)],
)

a_cal, b_cal = calibration_result.x

calibrated_oof = a_cal * two_model_oof + b_cal
calibrated_test = a_cal * two_model_test + b_cal

calibrated_name = "blend_extratrees_lgbm100k_calibrated"

refinement_rows.append({
    "candidate": calibrated_name,
    "method": "small scale-and-shift calibration",
    "oof_mse": mean_squared_error(y_refine, np.clip(calibrated_oof, 0, 100)),
    "a": a_cal,
    "b": b_cal,
    "base_w_extratrees": w_et,
    "base_w_lgbm_100k_lr02": w_lgbm100,
})

saved_candidates.append(
    save_candidate(calibrated_name, calibrated_oof, calibrated_test)
)

print("a:", a_cal)
print("b:", b_cal)
print("OOF MSE:", mean_squared_error(y_refine, np.clip(calibrated_oof, 0, 100)))

a: 1.0197589451630706
b: -1.0175215308303818
OOF MSE: 88.75522676403227


##### Ridge stack over saved predictions

The final refinement fits a small Ridge model using the saved predictions as input features.

This is a second-stage model, so it is evaluated with 5-fold cross-validation over the OOF prediction features.

In [265]:
kf_stack = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

ridge_stack_oof = np.zeros(len(y_refine))

for fold_id, (tr_idx, val_idx) in enumerate(kf_stack.split(X_oof_stack), start=1):
    ridge_stack = RidgeCV(
        alphas=np.logspace(-6, 3, 60),
        fit_intercept=True
    )

    ridge_stack.fit(
        X_oof_stack[tr_idx],
        y_refine[tr_idx]
    )

    ridge_stack_oof[val_idx] = ridge_stack.predict(
        X_oof_stack[val_idx]
    )

ridge_stack_final = RidgeCV(
    alphas=np.logspace(-6, 3, 60),
    fit_intercept=True
)

ridge_stack_final.fit(
    X_oof_stack,
    y_refine
)

ridge_stack_test = ridge_stack_final.predict(
    X_test_stack
)

ridge_stack_name = "stack_ridge_saved_predictions"

ridge_details = {
    f"coef_{name}": coef
    for name, coef in zip(component_order, ridge_stack_final.coef_)
}

ridge_details["intercept"] = float(ridge_stack_final.intercept_)
ridge_details["alpha"] = float(ridge_stack_final.alpha_)

refinement_rows.append({
    "candidate": ridge_stack_name,
    "method": "ridge stack over saved predictions",
    "oof_mse": mean_squared_error(y_refine, np.clip(ridge_stack_oof, 0, 100)),
    **ridge_details,
})

saved_candidates.append(
    save_candidate(ridge_stack_name, ridge_stack_oof, ridge_stack_test)
)

print("Ridge stack OOF MSE:", mean_squared_error(y_refine, np.clip(ridge_stack_oof, 0, 100)))
print("Selected alpha:", ridge_stack_final.alpha_)

Ridge stack OOF MSE: 87.9764793610966
Selected alpha: 1000.0


In [266]:
blend_refinement_screen = (
    pd.DataFrame(refinement_rows)
    .sort_values("oof_mse")
    .reset_index(drop=True)
)

saved_blend_candidates = pd.DataFrame(saved_candidates)

BLEND_REFINEMENT_SCREEN_PATH = RESULTS_DIR / "blend_refinement_after_public80p662_screen.csv"
BLEND_REFINEMENT_SAVED_PATH = RESULTS_DIR / "blend_refinement_after_public80p662_saved_candidates.csv"

blend_refinement_screen.to_csv(BLEND_REFINEMENT_SCREEN_PATH, index=False)
saved_blend_candidates.to_csv(BLEND_REFINEMENT_SAVED_PATH, index=False)

blend_refinement_screen

,candidate,method,oof_mse,w_extratrees,w_lgbm_100k_lr02,w_random_forest,w_lgbm_12k,w_lgbm_80k_lr03,a,b,base_w_extratrees,base_w_lgbm_100k_lr02,coef_random_forest,coef_extratrees,coef_lgbm_12k,coef_lgbm_80k_lr03,coef_lgbm_100k_lr02,intercept,alpha
0,stack_ridge_saved_predictions,ridge stack over saved predictions,87.976479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.171527,0.480278,-0.159616,0.402591,0.455197,-0.333024,1000.0
1,blend_extratrees_lgbm100k_calibrated,small scale-and-shift calibration,88.755227,NaN,NaN,NaN,NaN,NaN,1.019759,-1.017522,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,blend_all_saved_predictions_weighted,all-component weighted average,88.865535,0.315230,0.437056,0.0,0.0,0.247714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,blend_extratrees_lgbm100k_refined,two-model weight refinement,88.966359,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,submitted_public80p662_recreated,recorded submitted blend,88.966368,0.319000,0.681000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [267]:
saved_blend_candidates

,candidate,oof_path,testpred_path,submission_path
0,blend_extratrees_lgbm100k_refined,model_results/oof_blend_extratrees_lgbm100k_re...,model_results/testpred_blend_extratrees_lgbm10...,submission_blend_extratrees_lgbm100k_refined.csv
1,blend_all_saved_predictions_weighted,model_results/oof_blend_all_saved_predictions_...,model_results/testpred_blend_all_saved_predict...,submission_blend_all_saved_predictions_weighte...
2,blend_extratrees_lgbm100k_calibrated,model_results/oof_blend_extratrees_lgbm100k_ca...,model_results/testpred_blend_extratrees_lgbm10...,submission_blend_extratrees_lgbm100k_calibrate...
3,stack_ridge_saved_predictions,model_results/oof_stack_ridge_saved_prediction...,model_results/testpred_stack_ridge_saved_predi...,submission_stack_ridge_saved_predictions.csv


In [268]:
best_refinement = blend_refinement_screen.iloc[0]

model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Blend",
                "model": best_refinement["candidate"],
                "feature_space": "saved prediction stack",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": best_refinement["oof_mse"],
                "notes": best_refinement["method"],
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Blend,stack_ridge_saved_predictions,saved prediction stack,5-fold OOF,NaN,87.976479,ridge stack over saved predictions
1,Blend,long_lightgbm_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,88.966368,w_extratrees=0.319; w_lgbm_100k_lr02=0.681
2,Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
3,Boosting,lgbm_t03_base_5fold_oof_long100k_lr02,base,5-fold OOF,NaN,93.726219,long selected LightGBM artifact
4,Boosting,lgbm_t03_base_5fold_oof_long80k_lr03,base,5-fold OOF,NaN,94.163485,long selected LightGBM artifact
5,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
6,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
7,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
8,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...
9,Bagging,extratrees_oof_base,base,5-fold OOF,NaN,110.749294,n_estimators=250; max_features=0.5; min_sample...


The saved-prediction refinement improves internal OOF performance.

The strongest candidate in this section is expected to be the Ridge stack over saved model predictions. Since this is a second-stage model, it should be reviewed carefully before using a public submission slot.

The saved-prediction refinement is kept as the final checkpoint for the base-feature LightGBM branch.

The exploratory LightGBM diversity screen is not continued in the cleaned main workflow because it only supported an older residual-stack branch. It does not feed the later target/statistical-encoded LightGBM model or the accounting / solver models.

The next section moves to the target/statistical-encoding branch, which is the next material improvement in the final modelling path.

## Data modelling, Part 6: Target/statistical-encoded LightGBM models and safe blend

The base-feature LightGBM branch used only the numeric feature matrix created during preprocessing.

This section trains LightGBM on the augmented feature representation created in data engineering: the base numeric matrix plus leakage-safe target/statistical encoding features.

The target/statistical encoding features are rebuilt inside each cross-validation fold. This is important because validation rows must never be encoded using their own target values.

This section starts with a holdout screen to verify that the target/statistical encoding representation materially improves LightGBM. It then trains the full 5-fold OOF artifact, one fold cell at a time, with checkpoint files saved after every fold.

In [273]:
from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

try:
    import lightgbm as lgb
except ImportError as err:
    raise ImportError(
        "LightGBM is required for Part 6. Install lightgbm before running these cells."
    ) from err

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

TARGET_COL = globals().get("TARGET_COL", globals().get("TARGET", "PERCENT_PROFICIENT"))
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

TE_LGBM_ARTIFACT = "lgbm_te_base_5fold_oof_v1"

TE_HOLDOUT_SCREEN_PATH = RESULTS_DIR / "lgbm_te_holdout_screen_results.csv"

TE_LGBM_METRICS_PATH = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold_metrics.csv"
TE_LGBM_OOF_RAW_NPY_PATH = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_oof_raw.npy"
TE_LGBM_OOF_CLIPPED_NPY_PATH = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_oof_clipped.npy"

TE_LGBM_OOF_PATH = RESULTS_DIR / f"oof_{TE_LGBM_ARTIFACT}.csv"
TE_LGBM_TEST_PATH = RESULTS_DIR / f"testpred_{TE_LGBM_ARTIFACT}_foldavg.csv"
TE_LGBM_SUBMISSION_PATH = Path(f"submission_{TE_LGBM_ARTIFACT}_foldavg.csv")

required_part6_objects = [
    "raw_train_te",
    "raw_test_te",
    "te_key_specs",
    "make_group_key",
    "safe_key_name",
    "fit_group_stats",
    "apply_group_stats",
    "take_rows",
    "to_float32_matrix",
    "append_features",
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
    "X_tr_base_screen",
    "X_val_base_screen",
    "X_tr_te_screen",
    "X_val_te_screen",
    "y_fit_screen",
    "y_val_screen",
    "TE_DIAGNOSTIC_PASS",
]

missing_part6_objects = [
    name for name in required_part6_objects
    if name not in globals()
]

if missing_part6_objects:
    raise ValueError(f"Missing Part 6 prerequisite objects: {missing_part6_objects}")

if not TE_DIAGNOSTIC_PASS:
    raise ValueError("Target/statistical encoding diagnostic did not pass. Re-run the data engineering TE setup first.")

y_te_lgbm_all = np.asarray(y_train, dtype=np.float32).reshape(-1)

print("Part 6 artifact:", TE_LGBM_ARTIFACT)
print("Training rows:", X_train_proc_model.shape[0])
print("Test rows:", X_test_proc_model.shape[0])
print("Base feature count:", X_train_proc_model.shape[1])
print("Target/statistical encoding feature count:", len(te_key_specs) * 4)
print("Expected augmented feature count:", X_train_proc_model.shape[1] + len(te_key_specs) * 4)

print("\nCheckpoint folder:", RESULTS_DIR)

Part 6 artifact: lgbm_te_base_5fold_oof_v1
Training rows: 144921
Test rows: 48307
Base feature count: 162
Target/statistical encoding feature count: 68
Expected augmented feature count: 230

Checkpoint folder: model_results


In [274]:
def fit_te_lgbm_holdout_screen(X_train_matrix, y_train_vector, X_valid_matrix, y_valid_vector, label):
    """
    Fit one LightGBM holdout-screen model and return clipped train/validation metrics.

    This function is used twice in the holdout screen:
    once for the base matrix and once for the target/statistical-encoded matrix.
    """
    params = {
        "objective": "regression",
        "metric": "l2",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
        "verbosity": -1,
        "force_col_wise": True,

        "n_estimators": 20000,
        "learning_rate": 0.03,
        "num_leaves": 95,
        "min_child_samples": 60,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "max_depth": -1,
    }

    model = lgb.LGBMRegressor(**params)

    start_time = time.perf_counter()

    model.fit(
        X_train_matrix,
        y_train_vector,
        eval_set=[(X_valid_matrix, y_valid_vector)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )

    elapsed_seconds = time.perf_counter() - start_time
    best_iteration = int(model.best_iteration_ or params["n_estimators"])

    train_pred = np.clip(
        model.predict(X_train_matrix, num_iteration=best_iteration),
        0,
        100
    )

    valid_pred = np.clip(
        model.predict(X_valid_matrix, num_iteration=best_iteration),
        0,
        100
    )

    result = {
        "label": label,
        "train_mse_clipped": mean_squared_error(y_train_vector, train_pred),
        "valid_mse_clipped": mean_squared_error(y_valid_vector, valid_pred),
        "best_iteration": best_iteration,
        "elapsed_seconds": elapsed_seconds,
        "n_features": X_train_matrix.shape[1],
        "pred_valid_mean": valid_pred.mean(),
        "pred_valid_std": valid_pred.std(),
        "pred_valid_min": valid_pred.min(),
        "pred_valid_max": valid_pred.max(),
    }

    del model
    gc.collect()

    return result


RETRAIN_TE_HOLDOUT_SCREEN = globals().get("RETRAIN_TE_HOLDOUT_SCREEN", False)

if TE_HOLDOUT_SCREEN_PATH.exists() and not RETRAIN_TE_HOLDOUT_SCREEN:
    lgbm_te_holdout_results = pd.read_csv(TE_HOLDOUT_SCREEN_PATH)
    print("Loaded existing TE holdout screen:", TE_HOLDOUT_SCREEN_PATH)

else:
    print("Training base LightGBM holdout screen model...")

    base_result_te_screen = fit_te_lgbm_holdout_screen(
        X_train_matrix=X_tr_base_screen,
        y_train_vector=y_fit_screen,
        X_valid_matrix=X_val_base_screen,
        y_valid_vector=y_val_screen,
        label="base_lgbm_screen",
    )

    print("\nTraining base + target/statistical encoding LightGBM holdout screen model...")

    aug_result_te_screen = fit_te_lgbm_holdout_screen(
        X_train_matrix=X_tr_te_screen,
        y_train_vector=y_fit_screen,
        X_valid_matrix=X_val_te_screen,
        y_valid_vector=y_val_screen,
        label="base_plus_te_lgbm_screen",
    )

    lgbm_te_holdout_results = pd.DataFrame([
        base_result_te_screen,
        aug_result_te_screen,
    ])

    base_mse = float(
        lgbm_te_holdout_results.loc[
            lgbm_te_holdout_results["label"] == "base_lgbm_screen",
            "valid_mse_clipped"
        ].iloc[0]
    )

    lgbm_te_holdout_results["valid_gain_vs_base"] = (
        base_mse - lgbm_te_holdout_results["valid_mse_clipped"]
    )

    lgbm_te_holdout_results.to_csv(
        TE_HOLDOUT_SCREEN_PATH,
        index=False
    )

base_te_holdout_mse = float(
    lgbm_te_holdout_results.loc[
        lgbm_te_holdout_results["label"] == "base_lgbm_screen",
        "valid_mse_clipped"
    ].iloc[0]
)

aug_te_holdout_mse = float(
    lgbm_te_holdout_results.loc[
        lgbm_te_holdout_results["label"] == "base_plus_te_lgbm_screen",
        "valid_mse_clipped"
    ].iloc[0]
)

te_holdout_gain = base_te_holdout_mse - aug_te_holdout_mse
TE_HOLDOUT_PASS = bool(te_holdout_gain >= 1.0)

print("TE_HOLDOUT_PASS:", TE_HOLDOUT_PASS)
print("Base LightGBM holdout MSE:", base_te_holdout_mse)
print("Base + TE LightGBM holdout MSE:", aug_te_holdout_mse)
print("Holdout gain from TE features:", te_holdout_gain)

lgbm_te_holdout_results[
    [
        "label",
        "n_features",
        "best_iteration",
        "train_mse_clipped",
        "valid_mse_clipped",
        "valid_gain_vs_base",
        "pred_valid_mean",
        "pred_valid_std",
        "elapsed_seconds",
    ]
]

Training base LightGBM holdout screen model...
[1000]	valid_0's l2: 144.669
[2000]	valid_0's l2: 129.649
[3000]	valid_0's l2: 121.526
[4000]	valid_0's l2: 115.833
[5000]	valid_0's l2: 112.05
[6000]	valid_0's l2: 109.168
[7000]	valid_0's l2: 106.872
[8000]	valid_0's l2: 105.021
[9000]	valid_0's l2: 103.557
[10000]	valid_0's l2: 102.335
[11000]	valid_0's l2: 101.414
[12000]	valid_0's l2: 100.64
[13000]	valid_0's l2: 99.9944
[14000]	valid_0's l2: 99.4335
[15000]	valid_0's l2: 99.0222
[16000]	valid_0's l2: 98.6226
[17000]	valid_0's l2: 98.3013
[18000]	valid_0's l2: 98.0536
[19000]	valid_0's l2: 97.8065
[20000]	valid_0's l2: 97.6178

Training base + target/statistical encoding LightGBM holdout screen model...
[1000]	valid_0's l2: 90.631
[2000]	valid_0's l2: 89.6559
[3000]	valid_0's l2: 89.2743
[4000]	valid_0's l2: 89.1099
[5000]	valid_0's l2: 89.1189
TE_HOLDOUT_PASS: True
Base LightGBM holdout MSE: 96.84990207455618
Base + TE LightGBM holdout MSE: 88.80805200212511
Holdout gain from TE feat

,label,n_features,best_iteration,train_mse_clipped,valid_mse_clipped,valid_gain_vs_base,pred_valid_mean,pred_valid_std,elapsed_seconds
0,base_lgbm_screen,162,19992,9.283496,96.849902,0.00000,54.109896,24.564745,597.644955
1,base_plus_te_lgbm_screen,230,4868,12.467271,88.808052,8.04185,55.502333,25.017372,247.665155


#### 6B. Full 5-fold target/statistical-encoded LightGBM artifact

The holdout screen confirms that the target/statistical encoding feature representation materially improves LightGBM.

The next step trains the full 5-fold OOF artifact. Each fold is run in a separate cell. After every fold, the notebook saves:

- the fold validation predictions into the OOF checkpoint arrays,
- the fold test predictions,
- and the fold metric row.

This makes the run resumable. If a fold has already completed, rerunning its cell will skip it.

In [275]:
if not TE_HOLDOUT_PASS:
    raise ValueError("TE_HOLDOUT_PASS is False. Do not run the full TE LightGBM artifact.")

TE_LGBM_N_SPLITS = 5

te_lgbm_params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "n_estimators": 20000,
    "learning_rate": 0.03,
    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

n_train_te_lgbm = X_train_proc_model.shape[0]
n_test_te_lgbm = X_test_proc_model.shape[0]

te_outer_kf = KFold(
    n_splits=TE_LGBM_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

te_outer_fold_indices = list(
    te_outer_kf.split(np.arange(n_train_te_lgbm))
)

if TE_LGBM_OOF_RAW_NPY_PATH.exists():
    te_lgbm_oof_raw = np.load(TE_LGBM_OOF_RAW_NPY_PATH)

    if len(te_lgbm_oof_raw) != n_train_te_lgbm:
        raise ValueError("Existing TE raw OOF checkpoint has the wrong row count.")
else:
    te_lgbm_oof_raw = np.full(
        n_train_te_lgbm,
        np.nan,
        dtype=np.float32
    )

if TE_LGBM_OOF_CLIPPED_NPY_PATH.exists():
    te_lgbm_oof_clipped = np.load(TE_LGBM_OOF_CLIPPED_NPY_PATH)

    if len(te_lgbm_oof_clipped) != n_train_te_lgbm:
        raise ValueError("Existing TE clipped OOF checkpoint has the wrong row count.")
else:
    te_lgbm_oof_clipped = np.full(
        n_train_te_lgbm,
        np.nan,
        dtype=np.float32
    )

if TE_LGBM_METRICS_PATH.exists():
    te_lgbm_fold_metrics = pd.read_csv(TE_LGBM_METRICS_PATH)
else:
    te_lgbm_fold_metrics = pd.DataFrame()

completed_te_folds = (
    set(te_lgbm_fold_metrics["fold"].astype(int))
    if "fold" in te_lgbm_fold_metrics.columns
    else set()
)

te_fold_status_rows = []

for fold_id, (_, fold_valid_idx) in enumerate(te_outer_fold_indices, start=1):
    fold_test_path = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy"

    te_fold_status_rows.append({
        "fold": fold_id,
        "metrics_saved": fold_id in completed_te_folds,
        "test_pred_saved": fold_test_path.exists(),
        "oof_values_saved": np.isfinite(te_lgbm_oof_clipped[fold_valid_idx]).all(),
    })

te_lgbm_preflight = pd.DataFrame(te_fold_status_rows)

TE_OOF_PREFLIGHT_PASS = bool(
    len(te_key_specs) > 0
    and X_train_proc_model.shape[0] == len(y_te_lgbm_all)
    and X_test_proc_model.shape[0] == len(raw_test_te)
    and len(test_ids) == len(raw_test_te)
    and X_train_proc_model.shape[1] + len(te_key_specs) * 4 == 230
)

print("TE_OOF_PREFLIGHT_PASS:", TE_OOF_PREFLIGHT_PASS)
print("Artifact:", TE_LGBM_ARTIFACT)
print("Base features:", X_train_proc_model.shape[1])
print("TE features:", len(te_key_specs) * 4)
print("Total features expected:", X_train_proc_model.shape[1] + len(te_key_specs) * 4)
print("Completed folds from checkpoint:", sorted(completed_te_folds))

te_lgbm_preflight

TE_OOF_PREFLIGHT_PASS: True
Artifact: lgbm_te_base_5fold_oof_v1
Base features: 162
TE features: 68
Total features expected: 230
Completed folds from checkpoint: []


,fold,metrics_saved,test_pred_saved,oof_values_saved
0,1,False,False,False
1,2,False,False,False
2,3,False,False,False
3,4,False,False,False
4,5,False,False,False


In [276]:
def build_te_oof_and_apply_many(
    raw_fit,
    y_fit,
    raw_apply_dict,
    key_specs,
    n_splits=5,
    random_state=9890,
):
    """
    Build leakage-safe target/statistical encoding features for one outer fold.

    The function is used once per outer fold.

    For the outer-training rows:
    - target/statistical encoding features are built out-of-fold.

    For the outer-validation and test rows:
    - mappings are learned only from the outer-training rows.
    """
    raw_fit = raw_fit.reset_index(drop=True)

    raw_apply_dict = {
        name: frame.reset_index(drop=True)
        for name, frame in raw_apply_dict.items()
    }

    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)
    n_fit = raw_fit["N_STUDENTS"].astype(float).to_numpy()

    inner_kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    oof_feature_blocks = []
    apply_feature_blocks = {
        name: []
        for name in raw_apply_dict
    }

    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        fit_keys = make_group_key(raw_fit, cols)

        apply_keys = {
            name: make_group_key(frame, cols)
            for name, frame in raw_apply_dict.items()
        }

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_values = np.zeros(
            (len(raw_fit), len(feature_cols)),
            dtype=np.float32,
        )

        for inner_train_idx, inner_valid_idx in inner_kf.split(np.arange(len(raw_fit))):
            fold_stats, fold_defaults = fit_group_stats(
                fit_keys.iloc[inner_train_idx],
                y_fit[inner_train_idx],
                n_fit[inner_train_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            encoded_inner_valid = apply_group_stats(
                fit_keys.iloc[inner_valid_idx],
                fold_stats,
                fold_defaults,
                prefix,
            )

            oof_values[inner_valid_idx, :] = encoded_inner_valid[feature_cols].to_numpy(
                dtype=np.float32
            )

        full_stats, full_defaults = fit_group_stats(
            fit_keys,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        oof_feature_blocks.append(
            pd.DataFrame(oof_values, columns=feature_cols)
        )

        for name, keys in apply_keys.items():
            encoded_apply = apply_group_stats(
                keys,
                full_stats,
                full_defaults,
                prefix,
            )

            apply_feature_blocks[name].append(
                encoded_apply[feature_cols]
            )

        fit_counts = fit_keys.value_counts(dropna=False)

        summary_row = {
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        }

        fit_group_set = set(fit_counts.index)

        for name, keys in apply_keys.items():
            summary_row[f"{name}_groups"] = int(keys.nunique(dropna=False))
            summary_row[f"{name}_row_coverage"] = float(keys.isin(fit_group_set).mean())

        summary_rows.append(summary_row)

    oof_features = pd.concat(oof_feature_blocks, axis=1)

    apply_features = {
        name: pd.concat(blocks, axis=1)
        for name, blocks in apply_feature_blocks.items()
    }

    key_summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, key_summary

In [277]:
def run_te_lgbm_fold(fold_id):
    """
    Train one fold of the target/statistical-encoded LightGBM artifact.

    This function is called separately for folds 1 through 5 so each fold has its own notebook cell.
    """
    if not TE_OOF_PREFLIGHT_PASS:
        raise ValueError("TE_OOF_PREFLIGHT_PASS is False. Re-run the Part 6 preflight cell.")

    if fold_id < 1 or fold_id > TE_LGBM_N_SPLITS:
        raise ValueError(f"fold_id must be between 1 and {TE_LGBM_N_SPLITS}.")

    fold_train_idx, fold_valid_idx = te_outer_fold_indices[fold_id - 1]
    fold_test_path = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy"

    if TE_LGBM_OOF_RAW_NPY_PATH.exists():
        oof_raw = np.load(TE_LGBM_OOF_RAW_NPY_PATH)
    else:
        oof_raw = np.full(n_train_te_lgbm, np.nan, dtype=np.float32)

    if TE_LGBM_OOF_CLIPPED_NPY_PATH.exists():
        oof_clipped = np.load(TE_LGBM_OOF_CLIPPED_NPY_PATH)
    else:
        oof_clipped = np.full(n_train_te_lgbm, np.nan, dtype=np.float32)

    if TE_LGBM_METRICS_PATH.exists():
        fold_metrics = pd.read_csv(TE_LGBM_METRICS_PATH)
    else:
        fold_metrics = pd.DataFrame()

    fold_already_complete = (
        "fold" in fold_metrics.columns
        and fold_id in set(fold_metrics["fold"].astype(int))
        and fold_test_path.exists()
        and np.isfinite(oof_clipped[fold_valid_idx]).all()
    )

    if fold_already_complete:
        print(f"TE LightGBM fold {fold_id} is already complete. Skipping training.")
        return (
            fold_metrics
            .loc[fold_metrics["fold"].astype(int) == fold_id]
            .iloc[0]
            .to_dict()
        )

    print("\n" + "=" * 80)
    print(f"Training TE LightGBM fold {fold_id} / {TE_LGBM_N_SPLITS}")
    print("=" * 80)

    fold_start = time.perf_counter()

    raw_fit_fold = raw_train_te.iloc[fold_train_idx].reset_index(drop=True)
    raw_valid_fold = raw_train_te.iloc[fold_valid_idx].reset_index(drop=True)

    y_fit_fold = y_te_lgbm_all[fold_train_idx]
    y_valid_fold = y_te_lgbm_all[fold_valid_idx]

    print("Building leakage-safe TE features for this fold...")

    te_fit_fold, te_apply_fold, te_key_summary_fold = build_te_oof_and_apply_many(
        raw_fit=raw_fit_fold,
        y_fit=y_fit_fold,
        raw_apply_dict={
            "valid": raw_valid_fold,
            "test": raw_test_te,
        },
        key_specs=te_key_specs,
        n_splits=TE_LGBM_N_SPLITS,
        random_state=RANDOM_STATE + 100 * fold_id,
    )

    finite_te_ok = (
        np.isfinite(te_fit_fold.to_numpy(dtype=np.float32)).all()
        and np.isfinite(te_apply_fold["valid"].to_numpy(dtype=np.float32)).all()
        and np.isfinite(te_apply_fold["test"].to_numpy(dtype=np.float32)).all()
    )

    if not finite_te_ok:
        raise ValueError(f"Non-finite TE features detected in fold {fold_id}.")

    X_fit_base_fold = to_float32_matrix(
        take_rows(X_train_proc_model, fold_train_idx)
    )

    X_valid_base_fold = to_float32_matrix(
        take_rows(X_train_proc_model, fold_valid_idx)
    )

    X_test_base_fold = to_float32_matrix(X_test_proc_model)

    X_fit_fold = append_features(X_fit_base_fold, te_fit_fold)
    X_valid_fold = append_features(X_valid_base_fold, te_apply_fold["valid"])
    X_test_fold = append_features(X_test_base_fold, te_apply_fold["test"])

    print("Fold train shape:", X_fit_fold.shape)
    print("Fold valid shape:", X_valid_fold.shape)
    print("Fold test shape:", X_test_fold.shape)

    model = lgb.LGBMRegressor(**te_lgbm_params)

    print("Training LightGBM...")

    model.fit(
        X_fit_fold,
        y_fit_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iteration = int(model.best_iteration_ or te_lgbm_params["n_estimators"])

    fit_pred_raw = model.predict(
        X_fit_fold,
        num_iteration=best_iteration
    )

    valid_pred_raw = model.predict(
        X_valid_fold,
        num_iteration=best_iteration
    )

    test_pred_raw = model.predict(
        X_test_fold,
        num_iteration=best_iteration
    )

    fit_pred_clipped = np.clip(fit_pred_raw, 0, 100).astype(np.float32)
    valid_pred_clipped = np.clip(valid_pred_raw, 0, 100).astype(np.float32)
    test_pred_clipped = np.clip(test_pred_raw, 0, 100).astype(np.float32)

    oof_raw[fold_valid_idx] = valid_pred_raw.astype(np.float32)
    oof_clipped[fold_valid_idx] = valid_pred_clipped

    np.save(TE_LGBM_OOF_RAW_NPY_PATH, oof_raw)
    np.save(TE_LGBM_OOF_CLIPPED_NPY_PATH, oof_clipped)
    np.save(fold_test_path, test_pred_clipped)

    elapsed_seconds = time.perf_counter() - fold_start

    fold_row = {
        "artifact": TE_LGBM_ARTIFACT,
        "fold": fold_id,
        "train_rows": int(len(fold_train_idx)),
        "valid_rows": int(len(fold_valid_idx)),
        "n_features": int(X_fit_fold.shape[1]),
        "best_iteration": best_iteration,
        "train_mse_raw": mean_squared_error(y_fit_fold, fit_pred_raw),
        "train_mse_clipped": mean_squared_error(y_fit_fold, fit_pred_clipped),
        "valid_mse_raw": mean_squared_error(y_valid_fold, valid_pred_raw),
        "valid_mse_clipped": mean_squared_error(y_valid_fold, valid_pred_clipped),
        "pred_valid_mean": float(valid_pred_clipped.mean()),
        "pred_valid_std": float(valid_pred_clipped.std()),
        "pred_valid_min": float(valid_pred_clipped.min()),
        "pred_valid_max": float(valid_pred_clipped.max()),
        "elapsed_seconds": elapsed_seconds,
    }

    if len(fold_metrics) > 0 and "fold" in fold_metrics.columns:
        fold_metrics = fold_metrics[
            fold_metrics["fold"].astype(int) != fold_id
        ].copy()

    fold_metrics = pd.concat(
        [
            fold_metrics,
            pd.DataFrame([fold_row]),
        ],
        ignore_index=True,
    )

    fold_metrics = (
        fold_metrics
        .sort_values("fold")
        .reset_index(drop=True)
    )

    fold_metrics.to_csv(
        TE_LGBM_METRICS_PATH,
        index=False
    )

    print("\nFold result:")
    display(pd.DataFrame([fold_row]).T)

    del model
    del raw_fit_fold, raw_valid_fold
    del te_fit_fold, te_apply_fold, te_key_summary_fold
    del X_fit_base_fold, X_valid_base_fold, X_test_base_fold
    del X_fit_fold, X_valid_fold, X_test_fold
    del fit_pred_raw, valid_pred_raw, test_pred_raw
    del fit_pred_clipped, valid_pred_clipped, test_pred_clipped

    gc.collect()

    return fold_row

In [278]:
te_lgbm_fold_1 = run_te_lgbm_fold(1)

pd.DataFrame([te_lgbm_fold_1])


Training TE LightGBM fold 1 / 5
Building leakage-safe TE features for this fold...
Fold train shape: (115936, 230)
Fold valid shape: (28985, 230)
Fold test shape: (48307, 230)
Training LightGBM...
[1000]	valid_0's l2: 85.3052
[2000]	valid_0's l2: 84.3444
[3000]	valid_0's l2: 84.0699
[4000]	valid_0's l2: 84.0502
[5000]	valid_0's l2: 83.9653
[6000]	valid_0's l2: 84.0682

Fold result:


,0
artifact,lgbm_te_base_5fold_oof_v1
fold,1
train_rows,115936
valid_rows,28985
n_features,230
best_iteration,5033
train_mse_raw,11.666033
train_mse_clipped,11.650144
valid_mse_raw,83.961103
valid_mse_clipped,83.846176


,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,1,115936,28985,230,5033,11.666033,11.650144,83.961103,83.846176,54.934315,24.891148,0.0,100.0,350.812325


In [279]:
te_lgbm_fold_2 = run_te_lgbm_fold(2)

pd.DataFrame([te_lgbm_fold_2])


Training TE LightGBM fold 2 / 5
Building leakage-safe TE features for this fold...
Fold train shape: (115937, 230)
Fold valid shape: (28984, 230)
Fold test shape: (48307, 230)
Training LightGBM...
[1000]	valid_0's l2: 81.6343
[2000]	valid_0's l2: 80.6412
[3000]	valid_0's l2: 80.2265
[4000]	valid_0's l2: 80.0547
[5000]	valid_0's l2: 79.9343
[6000]	valid_0's l2: 79.9516

Fold result:


,0
artifact,lgbm_te_base_5fold_oof_v1
fold,2
train_rows,115937
valid_rows,28984
n_features,230
best_iteration,5379
train_mse_raw,10.535665
train_mse_clipped,10.520914
valid_mse_raw,79.906313
valid_mse_clipped,79.800461


,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,2,115937,28984,230,5379,10.535665,10.520914,79.906313,79.800461,53.856457,25.086536,0.0,100.0,346.021772


In [280]:
te_lgbm_fold_3 = run_te_lgbm_fold(3)

pd.DataFrame([te_lgbm_fold_3])


Training TE LightGBM fold 3 / 5
Building leakage-safe TE features for this fold...
Fold train shape: (115937, 230)
Fold valid shape: (28984, 230)
Fold test shape: (48307, 230)
Training LightGBM...
[1000]	valid_0's l2: 86.351
[2000]	valid_0's l2: 85.2369
[3000]	valid_0's l2: 84.8399
[4000]	valid_0's l2: 84.6823
[5000]	valid_0's l2: 84.6529
[6000]	valid_0's l2: 84.6459

Fold result:


,0
artifact,lgbm_te_base_5fold_oof_v1
fold,3
train_rows,115937
valid_rows,28984
n_features,230
best_iteration,5629
train_mse_raw,9.836355
train_mse_clipped,9.822016
valid_mse_raw,84.624473
valid_mse_clipped,84.527496


,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,3,115937,28984,230,5629,9.836355,9.822016,84.624473,84.527496,54.604282,25.013144,0.0,100.0,366.167276


In [281]:
te_lgbm_fold_4 = run_te_lgbm_fold(4)

pd.DataFrame([te_lgbm_fold_4])


Training TE LightGBM fold 4 / 5
Building leakage-safe TE features for this fold...
Fold train shape: (115937, 230)
Fold valid shape: (28984, 230)
Fold test shape: (48307, 230)
Training LightGBM...
[1000]	valid_0's l2: 85.2357
[2000]	valid_0's l2: 83.9971
[3000]	valid_0's l2: 83.7619
[4000]	valid_0's l2: 83.7085
[5000]	valid_0's l2: 83.6448

Fold result:


,0
artifact,lgbm_te_base_5fold_oof_v1
fold,4
train_rows,115937
valid_rows,28984
n_features,230
best_iteration,4379
train_mse_raw,14.653784
train_mse_clipped,14.633527
valid_mse_raw,83.62574
valid_mse_clipped,83.476036


,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,4,115937,28984,230,4379,14.653784,14.633527,83.62574,83.476036,54.854794,24.772558,0.0,100.0,298.582994


In [282]:
te_lgbm_fold_5 = run_te_lgbm_fold(5)

pd.DataFrame([te_lgbm_fold_5])


Training TE LightGBM fold 5 / 5
Building leakage-safe TE features for this fold...
Fold train shape: (115937, 230)
Fold valid shape: (28984, 230)
Fold test shape: (48307, 230)
Training LightGBM...
[1000]	valid_0's l2: 84.5481
[2000]	valid_0's l2: 83.5317
[3000]	valid_0's l2: 83.0842
[4000]	valid_0's l2: 82.9912
[5000]	valid_0's l2: 83.0368

Fold result:


,0
artifact,lgbm_te_base_5fold_oof_v1
fold,5
train_rows,115937
valid_rows,28984
n_features,230
best_iteration,4240
train_mse_raw,15.136295
train_mse_clipped,15.119184
valid_mse_raw,82.946
valid_mse_clipped,82.887871


,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,5,115937,28984,230,4240,15.136295,15.119184,82.946,82.887871,54.425667,24.821636,0.0,100.0,279.822828


#### 6C. Finalize the target/statistical-encoded LightGBM artifact

After all five folds are complete, the checkpointed OOF and test predictions are assembled into the final artifact files.

The final model files are:

- `model_results/oof_lgbm_te_base_5fold_oof_v1.csv`
- `model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv`
- `submission_lgbm_te_base_5fold_oof_v1_foldavg.csv`

In [283]:
if not TE_LGBM_OOF_RAW_NPY_PATH.exists():
    raise FileNotFoundError(f"Missing raw OOF checkpoint: {TE_LGBM_OOF_RAW_NPY_PATH}")

if not TE_LGBM_OOF_CLIPPED_NPY_PATH.exists():
    raise FileNotFoundError(f"Missing clipped OOF checkpoint: {TE_LGBM_OOF_CLIPPED_NPY_PATH}")

if not TE_LGBM_METRICS_PATH.exists():
    raise FileNotFoundError(f"Missing fold metrics file: {TE_LGBM_METRICS_PATH}")

te_lgbm_oof_raw = np.load(TE_LGBM_OOF_RAW_NPY_PATH)
te_lgbm_oof_clipped = np.load(TE_LGBM_OOF_CLIPPED_NPY_PATH)
te_lgbm_fold_metrics = pd.read_csv(TE_LGBM_METRICS_PATH)

missing_fold_test_paths = [
    RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy"
    for fold_id in range(1, TE_LGBM_N_SPLITS + 1)
    if not (RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy").exists()
]

TE_OOF_RUN_PASS = bool(
    np.isfinite(te_lgbm_oof_raw).all()
    and np.isfinite(te_lgbm_oof_clipped).all()
    and len(missing_fold_test_paths) == 0
    and te_lgbm_fold_metrics["fold"].nunique() == TE_LGBM_N_SPLITS
)

print("TE_OOF_RUN_PASS:", TE_OOF_RUN_PASS)

if missing_fold_test_paths:
    print("Missing fold test prediction files:")
    for path in missing_fold_test_paths:
        print("-", path)

if not TE_OOF_RUN_PASS:
    raise ValueError("The TE LightGBM OOF artifact is incomplete. Finish all five fold cells before finalizing.")

te_lgbm_oof_mse_raw = mean_squared_error(
    y_te_lgbm_all,
    te_lgbm_oof_raw
)

te_lgbm_oof_mse_clipped = mean_squared_error(
    y_te_lgbm_all,
    te_lgbm_oof_clipped
)

te_lgbm_fold_test_preds = []

for fold_id in range(1, TE_LGBM_N_SPLITS + 1):
    fold_test_path = RESULTS_DIR / f"{TE_LGBM_ARTIFACT}_fold{fold_id}_test_pred.npy"
    te_lgbm_fold_test_preds.append(
        np.load(fold_test_path).astype(np.float32)
    )

te_lgbm_test_pred = np.mean(
    np.vstack(te_lgbm_fold_test_preds),
    axis=0
)

te_lgbm_test_pred = np.clip(
    te_lgbm_test_pred,
    0,
    100
).astype(np.float32)

te_lgbm_oof_df = pd.DataFrame({
    "row_index": np.arange(n_train_te_lgbm),
    TARGET_COL: y_te_lgbm_all,
    "pred_raw": te_lgbm_oof_raw,
    "pred_clipped": te_lgbm_oof_clipped,
})

te_lgbm_test_df = pd.DataFrame({
    ID_COL: id_array(test_ids),
    TARGET_COL: te_lgbm_test_pred,
})

te_lgbm_submission = te_lgbm_test_df[
    [ID_COL, TARGET_COL]
].copy()

te_lgbm_oof_df.to_csv(
    TE_LGBM_OOF_PATH,
    index=False
)

te_lgbm_test_df.to_csv(
    TE_LGBM_TEST_PATH,
    index=False
)

te_lgbm_submission.to_csv(
    TE_LGBM_SUBMISSION_PATH,
    index=False
)

te_lgbm_artifact_summary = pd.DataFrame([
    {
        "artifact": TE_LGBM_ARTIFACT,
        "feature_space": "base + target/statistical encoding",
        "base_features": X_train_proc_model.shape[1],
        "target_encoding_features": len(te_key_specs) * 4,
        "total_features": X_train_proc_model.shape[1] + len(te_key_specs) * 4,
        "folds": TE_LGBM_N_SPLITS,
        "mean_fold_valid_mse": te_lgbm_fold_metrics["valid_mse_clipped"].mean(),
        "overall_oof_mse_raw": te_lgbm_oof_mse_raw,
        "overall_oof_mse_clipped": te_lgbm_oof_mse_clipped,
        "test_pred_mean": te_lgbm_test_pred.mean(),
        "test_pred_std": te_lgbm_test_pred.std(),
    }
])

print("Saved files:")
print("-", TE_LGBM_OOF_PATH)
print("-", TE_LGBM_TEST_PATH)
print("-", TE_LGBM_SUBMISSION_PATH)

print("\nOOF MSE raw:", te_lgbm_oof_mse_raw)
print("OOF MSE clipped:", te_lgbm_oof_mse_clipped)

display(te_lgbm_fold_metrics)
te_lgbm_artifact_summary

TE_OOF_RUN_PASS: True
Saved files:
- model_results/oof_lgbm_te_base_5fold_oof_v1.csv
- model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv
- submission_lgbm_te_base_5fold_oof_v1_foldavg.csv

OOF MSE raw: 83.01273345947266
OOF MSE clipped: 82.90760803222656


,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,1,115936,28985,230,5033,11.666033,11.650144,83.961103,83.846176,54.934315,24.891148,0.0,100.0,350.812325
1,lgbm_te_base_5fold_oof_v1,2,115937,28984,230,5379,10.535665,10.520914,79.906313,79.800461,53.856457,25.086536,0.0,100.0,346.021772
2,lgbm_te_base_5fold_oof_v1,3,115937,28984,230,5629,9.836355,9.822016,84.624473,84.527496,54.604282,25.013144,0.0,100.0,366.167276
3,lgbm_te_base_5fold_oof_v1,4,115937,28984,230,4379,14.653784,14.633527,83.625740,83.476036,54.854794,24.772558,0.0,100.0,298.582994
4,lgbm_te_base_5fold_oof_v1,5,115937,28984,230,4240,15.136295,15.119184,82.946000,82.887871,54.425667,24.821636,0.0,100.0,279.822828


,artifact,feature_space,base_features,target_encoding_features,total_features,folds,mean_fold_valid_mse,overall_oof_mse_raw,overall_oof_mse_clipped,test_pred_mean,test_pred_std
0,lgbm_te_base_5fold_oof_v1,base + target/statistical encoding,162,68,230,5,82.907608,83.012733,82.907608,54.500694,24.778822


In [284]:
model_scoreboard = pd.concat(
    [
        model_scoreboard,
        pd.DataFrame([
            {
                "model_family": "Target/statistical-encoded boosting",
                "model": TE_LGBM_ARTIFACT,
                "feature_space": "base + target/statistical encoding",
                "metric_source": "5-fold OOF",
                "train_mse": np.nan,
                "val_mse": te_lgbm_oof_mse_clipped,
                "notes": "LightGBM with leakage-safe target/statistical encoding features",
            }
        ])
    ],
    axis=0,
    ignore_index=True
)

model_scoreboard = (
    model_scoreboard
    .drop_duplicates(
        subset=["model_family", "model", "feature_space", "val_mse", "notes"],
        keep="last"
    )
    .sort_values("val_mse")
    .reset_index(drop=True)
)

model_scoreboard.head(20)

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Target/statistical-encoded boosting,lgbm_te_base_5fold_oof_v1,base + target/statistical encoding,5-fold OOF,NaN,82.907608,LightGBM with leakage-safe target/statistical ...
1,Blend,stack_ridge_saved_predictions,saved prediction stack,5-fold OOF,NaN,87.976479,ridge stack over saved predictions
2,Blend,long_lightgbm_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,88.966368,w_extratrees=0.319; w_lgbm_100k_lr02=0.681
3,Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
4,Boosting,lgbm_t03_base_5fold_oof_long100k_lr02,base,5-fold OOF,NaN,93.726219,long selected LightGBM artifact
5,Boosting,lgbm_t03_base_5fold_oof_long80k_lr03,base,5-fold OOF,NaN,94.163485,long selected LightGBM artifact
6,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
7,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
8,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000
9,Bagging,rf_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,110.363092,w_rf=0.152; w_extratrees=0.848; RF 500 base + ...


In [285]:
submission_tracker_path = RESULTS_DIR / "submission_tracker.csv"

te_lgbm_public_checkpoint = pd.DataFrame([
    {
        "file": TE_LGBM_SUBMISSION_PATH.name,
        "model_family": "Target/statistical-encoded boosting",
        "public_mse": 70.997,
        "notes": "LightGBM on base features plus leakage-safe target/statistical encodings",
    }
])

if submission_tracker_path.exists():
    submission_tracker = pd.read_csv(submission_tracker_path)
    submission_tracker = pd.concat(
        [submission_tracker, te_lgbm_public_checkpoint],
        ignore_index=True
    )
else:
    submission_tracker = te_lgbm_public_checkpoint.copy()

submission_tracker = (
    submission_tracker
    .drop_duplicates(subset=["file"], keep="last")
    .reset_index(drop=True)
)

submission_tracker.to_csv(
    submission_tracker_path,
    index=False
)

submission_tracker.tail(10)

,file,model_family,public_mse,notes
0,submission_blend_auto_et_lgbm_long_oof_weighte...,Blend,80.662,ExtraTrees + long 100k LightGBM blend
1,submission_lgbm_te_base_5fold_oof_v1_foldavg.csv,Target/statistical-encoded boosting,70.997,LightGBM on base features plus leakage-safe ta...


In [286]:
te_part6_diagnostic = pd.DataFrame([
    {
        "check": "Holdout TE improvement",
        "value": te_holdout_gain,
        "expected": "about 8.04185",
        "pass": bool(TE_HOLDOUT_PASS and te_holdout_gain > 8.0),
    },
    {
        "check": "Full TE OOF clipped MSE",
        "value": te_lgbm_oof_mse_clipped,
        "expected": "about 82.9076",
        "pass": bool(te_lgbm_oof_mse_clipped < 83.0),
    },
    {
        "check": "Total feature count",
        "value": X_train_proc_model.shape[1] + len(te_key_specs) * 4,
        "expected": 230,
        "pass": bool(X_train_proc_model.shape[1] + len(te_key_specs) * 4 == 230),
    },
    {
        "check": "Completed fold count",
        "value": te_lgbm_fold_metrics["fold"].nunique(),
        "expected": 5,
        "pass": bool(te_lgbm_fold_metrics["fold"].nunique() == 5),
    },
    {
        "check": "Test prediction rows",
        "value": len(te_lgbm_test_pred),
        "expected": len(test_ids),
        "pass": bool(len(te_lgbm_test_pred) == len(test_ids)),
    },
])

te_part6_diagnostic

,check,value,expected,pass
0,Holdout TE improvement,8.041850,about 8.04185,True
1,Full TE OOF clipped MSE,82.907608,about 82.9076,True
2,Total feature count,230.000000,230,True
3,Completed fold count,5.000000,5,True
4,Test prediction rows,48307.000000,48307,True


#### 6D. Verified non-residual blend components

The tree and bagging models were already completed in Part 4. This section does not retrain Random Forest or ExtraTrees.

Instead, it verifies the saved OOF/test prediction artifacts from:

- the Part 4 Random Forest 500-tree OOF artifact,
- the Part 4 ExtraTrees OOF artifact,
- the Part 5 selected base LightGBM artifact,
- the Part 5 long LightGBM artifacts,
- the Part 6 target/statistical-encoded LightGBM artifact.

These verified non-residual components are then blended in the next cell.

In [289]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

required_part6d_objects = [
    "y_train",
    "train_ids",
    "test_ids",
]

missing_part6d_objects = [
    name for name in required_part6d_objects
    if name not in globals()
]

if missing_part6d_objects:
    raise ValueError(f"Missing required objects for Part 6D: {missing_part6d_objects}")

y_safe_te_blend = np.asarray(y_train, dtype=float).reshape(-1)
train_id_reference = pd.Series(train_ids).astype(str).to_numpy()
test_id_reference = pd.Series(test_ids).astype(str).to_numpy()

rf_oof_path = Path(globals().get("RF_OOF_PATH", RESULTS_DIR / "oof_rf_500_base.csv"))
rf_test_path = Path(globals().get("RF_TEST_FOLD_PATH", RESULTS_DIR / "testpred_rf_500_base_folds.csv"))

extratrees_oof_path = Path(globals().get("EXTRATREES_OOF_PRED_PATH", RESULTS_DIR / "oof_extratrees_base.csv"))
extratrees_test_path = Path(globals().get("EXTRATREES_TEST_PRED_PATH", RESULTS_DIR / "testpred_extratrees_base_foldavg.csv"))

lgbm_selected_artifact = globals().get("LGBM_SELECTED_ARTIFACT", "lgbm_t03_base_5fold_oof")
lgbm_selected_oof_path = Path(
    globals().get(
        "LGBM_SELECTED_OOF_PATH",
        RESULTS_DIR / f"oof_{lgbm_selected_artifact}.csv",
    )
)
lgbm_selected_test_path = Path(
    globals().get(
        "LGBM_SELECTED_TEST_PRED_PATH",
        RESULTS_DIR / f"testpred_{lgbm_selected_artifact}_foldavg.csv",
    )
)

long80k_artifact = "lgbm_t03_base_5fold_oof_long80k_lr03"
long100k_artifact = "lgbm_t03_base_5fold_oof_long100k_lr02"

long80k_oof_path = Path(
    long80k_result["oof_csv_path"]
    if "long80k_result" in globals()
    else RESULTS_DIR / f"oof_{long80k_artifact}.csv"
)

long80k_test_path = Path(
    long80k_result["testpred_csv_path"]
    if "long80k_result" in globals()
    else RESULTS_DIR / f"testpred_{long80k_artifact}_foldavg.csv"
)

long100k_oof_path = Path(
    long100k_result["oof_csv_path"]
    if "long100k_result" in globals()
    else RESULTS_DIR / f"oof_{long100k_artifact}.csv"
)

long100k_test_path = Path(
    long100k_result["testpred_csv_path"]
    if "long100k_result" in globals()
    else RESULTS_DIR / f"testpred_{long100k_artifact}_foldavg.csv"
)

te_lgbm_oof_path = RESULTS_DIR / "oof_lgbm_te_base_5fold_oof_v1.csv"
te_lgbm_test_path = RESULTS_DIR / "testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv"

safe_te_component_specs = [
    {
        "name": "rf500",
        "source_section": "Part 4",
        "display_name": "Random Forest 500",
        "oof_path": rf_oof_path,
        "test_path": rf_test_path,
        "oof_pred_col": "rf_500_base_oof_pred",
        "test_pred_col": "rf_500_base_foldavg_pred",
        "target_col": "y_true",
    },
    {
        "name": "extratrees",
        "source_section": "Part 4",
        "display_name": "ExtraTrees",
        "oof_path": extratrees_oof_path,
        "test_path": extratrees_test_path,
        "oof_pred_col": "extratrees_oof_pred",
        "test_pred_col": TARGET_COL,
        "target_col": "y_true",
    },
    {
        "name": "lgbm_12k",
        "source_section": "Part 5",
        "display_name": "LightGBM selected 12k",
        "oof_path": lgbm_selected_oof_path,
        "test_path": lgbm_selected_test_path,
        "oof_pred_col": f"OOF_{lgbm_selected_artifact}",
        "test_pred_col": f"TESTPRED_{lgbm_selected_artifact}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_80k_lr03",
        "source_section": "Part 5",
        "display_name": "LightGBM long 80k",
        "oof_path": long80k_oof_path,
        "test_path": long80k_test_path,
        "oof_pred_col": f"OOF_{long80k_artifact}",
        "test_pred_col": f"TESTPRED_{long80k_artifact}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_100k_lr02",
        "source_section": "Part 5",
        "display_name": "LightGBM long 100k",
        "oof_path": long100k_oof_path,
        "test_path": long100k_test_path,
        "oof_pred_col": f"OOF_{long100k_artifact}",
        "test_pred_col": f"TESTPRED_{long100k_artifact}",
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_te_v1",
        "source_section": "Part 6",
        "display_name": "Target/statistical-encoded LightGBM",
        "oof_path": te_lgbm_oof_path,
        "test_path": te_lgbm_test_path,
        "oof_pred_col": "pred_clipped",
        "test_pred_col": TARGET_COL,
        "target_col": TARGET_COL,
    },
]

component_file_check = []

for component in safe_te_component_specs:
    component_file_check.append({
        "component": component["name"],
        "source_section": component["source_section"],
        "display_name": component["display_name"],
        "oof_path": str(component["oof_path"]),
        "oof_exists": component["oof_path"].exists(),
        "test_path": str(component["test_path"]),
        "test_exists": component["test_path"].exists(),
    })

component_file_check = pd.DataFrame(component_file_check)

PART6D_COMPONENT_FILES_READY = bool(
    component_file_check["oof_exists"].all()
    and component_file_check["test_exists"].all()
)

print("PART6D_COMPONENT_FILES_READY:", PART6D_COMPONENT_FILES_READY)

if not PART6D_COMPONENT_FILES_READY:
    display(component_file_check)

    raise FileNotFoundError(
        "One or more required blend component artifacts are missing. "
        "Random Forest and ExtraTrees should come from Part 4; "
        "base and long LightGBM artifacts should come from Part 5; "
        "the TE LightGBM artifact should come from Part 6C."
    )

component_file_check

PART6D_COMPONENT_FILES_READY: True


,component,source_section,display_name,oof_path,oof_exists,test_path,test_exists
0,rf500,Part 4,Random Forest 500,model_results/oof_rf_500_base.csv,True,model_results/testpred_rf_500_base_folds.csv,True
1,extratrees,Part 4,ExtraTrees,model_results/oof_extratrees_base.csv,True,model_results/testpred_extratrees_base_foldavg...,True
2,lgbm_12k,Part 5,LightGBM selected 12k,model_results/oof_lgbm_t03_base_5fold_oof.csv,True,model_results/testpred_lgbm_t03_base_5fold_oof...,True
3,lgbm_80k_lr03,Part 5,LightGBM long 80k,model_results/oof_lgbm_t03_base_5fold_oof_long...,True,model_results/testpred_lgbm_t03_base_5fold_oof...,True
4,lgbm_100k_lr02,Part 5,LightGBM long 100k,model_results/oof_lgbm_t03_base_5fold_oof_long...,True,model_results/testpred_lgbm_t03_base_5fold_oof...,True
5,lgbm_te_v1,Part 6,Target/statistical-encoded LightGBM,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,True,model_results/testpred_lgbm_te_base_5fold_oof_...,True


The required non-residual component files are available.

The blend now loads and aligns the saved OOF/test predictions. Random Forest and ExtraTrees are included as already-completed Part 4 artifacts; they are not retrained here.

The canonical blend component names match the successful original no-residual TE blend:

- `rf500`
- `extratrees_safe`
- `lgbm_12k`
- `lgbm_long80k`
- `lgbm_long100k`
- `lgbm_te_v1`

In [290]:
SAFE_TE_BLEND_NAME = "blend_te_verified_noresid_weighted"

SAFE_TE_BLEND_OOF_PATH = RESULTS_DIR / f"oof_{SAFE_TE_BLEND_NAME}.csv"
SAFE_TE_BLEND_TEST_PATH = RESULTS_DIR / f"testpred_{SAFE_TE_BLEND_NAME}.csv"
SAFE_TE_BLEND_SUBMISSION_PATH = Path(f"submission_{SAFE_TE_BLEND_NAME}.csv")

SAFE_TE_BLEND_SCREEN_PATH = RESULTS_DIR / "blend_te_verified_noresid_weight_screen.csv"
SAFE_TE_BLEND_WEIGHTS_PATH = RESULTS_DIR / "blend_te_verified_noresid_best_weights.csv"

safe_te_blend_component_specs = [
    {
        "name": "rf500",
        "display_name": "Random Forest 500",
        "source_section": "Part 4",
        "oof_path": rf_oof_path,
        "test_path": rf_test_path,
        "oof_pred_cols": ["rf_500_base_oof_pred"],
        "test_pred_cols": ["rf_500_base_foldavg_pred"],
        "target_col": "y_true",
    },
    {
        "name": "extratrees_safe",
        "display_name": "ExtraTrees",
        "source_section": "Part 4",
        "oof_path": extratrees_oof_path,
        "test_path": extratrees_test_path,
        "oof_pred_cols": ["extratrees_oof_pred", "extratrees_oof_pred_clipped", "oof_pred_clipped", "oof_pred"],
        "test_pred_cols": [TARGET_COL, "PERCENT_PROFICIENT", "extratrees_test_pred"],
        "target_col": "y_true",
    },
    {
        "name": "lgbm_12k",
        "display_name": "LightGBM selected 12k",
        "source_section": "Part 5",
        "oof_path": lgbm_selected_oof_path,
        "test_path": lgbm_selected_test_path,
        "oof_pred_cols": [f"OOF_{lgbm_selected_artifact}"],
        "test_pred_cols": [f"TESTPRED_{lgbm_selected_artifact}"],
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_long80k",
        "display_name": "LightGBM long 80k",
        "source_section": "Part 5",
        "oof_path": long80k_oof_path,
        "test_path": long80k_test_path,
        "oof_pred_cols": [f"OOF_{long80k_artifact}"],
        "test_pred_cols": [f"TESTPRED_{long80k_artifact}"],
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_long100k",
        "display_name": "LightGBM long 100k",
        "source_section": "Part 5",
        "oof_path": long100k_oof_path,
        "test_path": long100k_test_path,
        "oof_pred_cols": [f"OOF_{long100k_artifact}"],
        "test_pred_cols": [f"TESTPRED_{long100k_artifact}"],
        "target_col": "PERCENT_PROFICIENT_TRUE",
    },
    {
        "name": "lgbm_te_v1",
        "display_name": "Target/statistical-encoded LightGBM",
        "source_section": "Part 6",
        "oof_path": te_lgbm_oof_path,
        "test_path": te_lgbm_test_path,
        "oof_pred_cols": ["pred_clipped"],
        "test_pred_cols": [TARGET_COL, "PERCENT_PROFICIENT"],
        "target_col": TARGET_COL,
    },
]


def first_present_column(df, candidate_cols, file_path, component_name):
    """
    Return the first expected prediction column present in a saved artifact.

    This is used for every blend component because some artifacts have cleaned
    names while the archive has original-compatible names.
    """
    for col in candidate_cols:
        if col in df.columns:
            return col

    raise ValueError(
        f"No expected prediction column found for {component_name} in {file_path}. "
        f"Expected one of {candidate_cols}. Observed columns: {list(df.columns)}"
    )


def load_aligned_oof_prediction(component):
    """
    Load one OOF prediction artifact and align it to the current training row order.
    """
    path = Path(component["oof_path"])
    df = pd.read_csv(path)

    pred_col = first_present_column(
        df=df,
        candidate_cols=component["oof_pred_cols"],
        file_path=path,
        component_name=component["name"],
    )

    aligned = df.copy()

    if "row_index" in aligned.columns:
        aligned = (
            aligned
            .sort_values("row_index")
            .reset_index(drop=True)
        )

        expected_row_index = np.arange(len(aligned))

        if not np.array_equal(aligned["row_index"].to_numpy(), expected_row_index):
            raise ValueError(f"OOF row_index is not consecutive for {component['name']}.")

    elif ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in OOF file for {component['name']}.")

        train_order = pd.DataFrame({
            ID_COL: train_id_reference,
            "_train_order": np.arange(len(train_id_reference)),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = train_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"OOF predictions could not be aligned for {component['name']}.")

        aligned = (
            aligned
            .sort_values("_train_order")
            .drop(columns=["_train_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

    if len(aligned) != len(y_safe_te_blend):
        raise ValueError(
            f"OOF row count mismatch for {component['name']}: "
            f"{len(aligned)} rows vs {len(y_safe_te_blend)} expected."
        )

    target_col = component["target_col"]

    if target_col in aligned.columns:
        saved_target = aligned[target_col].to_numpy(dtype=float)

        if not np.allclose(saved_target, y_safe_te_blend):
            raise ValueError(f"Saved target values do not align for {component['name']}.")

    pred = aligned[pred_col].to_numpy(dtype=float)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF predictions found for {component['name']}.")

    return np.clip(pred, 0, 100), pred_col


def load_aligned_test_prediction(component):
    """
    Load one test prediction artifact and align it to the current test row order.
    """
    path = Path(component["test_path"])
    df = pd.read_csv(path)

    pred_col = first_present_column(
        df=df,
        candidate_cols=component["test_pred_cols"],
        file_path=path,
        component_name=component["name"],
    )

    aligned = df.copy()

    if ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in test file for {component['name']}.")

        test_order = pd.DataFrame({
            ID_COL: test_id_reference,
            "_test_order": np.arange(len(test_id_reference)),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = test_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"Test predictions could not be aligned for {component['name']}.")

        aligned = (
            aligned
            .sort_values("_test_order")
            .drop(columns=["_test_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

    if len(aligned) != len(test_id_reference):
        raise ValueError(
            f"Test row count mismatch for {component['name']}: "
            f"{len(aligned)} rows vs {len(test_id_reference)} expected."
        )

    pred = aligned[pred_col].to_numpy(dtype=float)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test predictions found for {component['name']}.")

    return np.clip(pred, 0, 100), pred_col

In [291]:
safe_te_component_names = []
safe_te_component_display_names = []
safe_te_oof_predictions = []
safe_te_test_predictions = []
safe_te_component_rows = []

for component in safe_te_blend_component_specs:
    oof_pred, oof_pred_col = load_aligned_oof_prediction(component)
    test_pred, test_pred_col = load_aligned_test_prediction(component)

    component_oof_mse = mean_squared_error(
        y_safe_te_blend,
        oof_pred,
    )

    safe_te_component_names.append(component["name"])
    safe_te_component_display_names.append(component["display_name"])
    safe_te_oof_predictions.append(oof_pred)
    safe_te_test_predictions.append(test_pred)

    safe_te_component_rows.append({
        "component": component["name"],
        "display_name": component["display_name"],
        "source_section": component["source_section"],
        "oof_mse": component_oof_mse,
        "oof_file": str(component["oof_path"]),
        "oof_column": oof_pred_col,
        "test_file": str(component["test_path"]),
        "test_column": test_pred_col,
        "oof_mean": oof_pred.mean(),
        "oof_std": oof_pred.std(),
        "test_mean": test_pred.mean(),
        "test_std": test_pred.std(),
    })

safe_te_oof_matrix = np.column_stack(safe_te_oof_predictions).astype(np.float64)
safe_te_test_matrix = np.column_stack(safe_te_test_predictions).astype(np.float64)

safe_te_component_summary = (
    pd.DataFrame(safe_te_component_rows)
    .sort_values("oof_mse")
    .reset_index(drop=True)
)

SAFE_TE_BLEND_PREFLIGHT_PASS = bool(
    safe_te_component_names == [
        "rf500",
        "extratrees_safe",
        "lgbm_12k",
        "lgbm_long80k",
        "lgbm_long100k",
        "lgbm_te_v1",
    ]
    and safe_te_oof_matrix.shape == (len(y_safe_te_blend), 6)
    and safe_te_test_matrix.shape == (len(test_id_reference), 6)
    and np.isfinite(safe_te_oof_matrix).all()
    and np.isfinite(safe_te_test_matrix).all()
)

print("SAFE_TE_BLEND_PREFLIGHT_PASS:", SAFE_TE_BLEND_PREFLIGHT_PASS)
print("Component order:", safe_te_component_names)
print("OOF matrix shape:", safe_te_oof_matrix.shape)
print("Test matrix shape:", safe_te_test_matrix.shape)

if not SAFE_TE_BLEND_PREFLIGHT_PASS:
    raise ValueError("Safe TE blend component loading failed.")

safe_te_component_summary

SAFE_TE_BLEND_PREFLIGHT_PASS: True
Component order: ['rf500', 'extratrees_safe', 'lgbm_12k', 'lgbm_long80k', 'lgbm_long100k', 'lgbm_te_v1']
OOF matrix shape: (144921, 6)
Test matrix shape: (48307, 6)


,component,display_name,source_section,oof_mse,oof_file,oof_column,test_file,test_column,oof_mean,oof_std,test_mean,test_std
0,lgbm_te_v1,Target/statistical-encoded LightGBM,Part 6,82.907614,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,pred_clipped,model_results/testpred_lgbm_te_base_5fold_oof_...,PERCENT_PROFICIENT,54.535106,24.920245,54.500693,24.778823
1,lgbm_long100k,LightGBM long 100k,Part 5,93.726214,model_results/oof_lgbm_t03_base_5fold_oof_long...,OOF_lgbm_t03_base_5fold_oof_long100k_lr02,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02,54.153144,24.734667,54.074208,24.605725
2,lgbm_long80k,LightGBM long 80k,Part 5,94.163485,model_results/oof_lgbm_t03_base_5fold_oof_long...,OOF_lgbm_t03_base_5fold_oof_long80k_lr03,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03,54.147019,24.743266,54.067601,24.607933
3,lgbm_12k,LightGBM selected 12k,Part 5,98.779406,model_results/oof_lgbm_t03_base_5fold_oof.csv,OOF_lgbm_t03_base_5fold_oof,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof,54.152808,24.283770,54.086671,24.207260
4,extratrees_safe,ExtraTrees,Part 4,110.749294,model_results/oof_extratrees_base.csv,extratrees_oof_pred,model_results/testpred_extratrees_base_foldavg...,PERCENT_PROFICIENT,54.127367,23.857745,54.040173,23.782901
5,rf500,Random Forest 500,Part 4,122.328024,model_results/oof_rf_500_base.csv,rf_500_base_oof_pred,model_results/testpred_rf_500_base_folds.csv,rf_500_base_foldavg_pred,54.134143,22.839348,54.056767,22.817463


#### 6E. Safe convex blend with the target/statistical-encoded LightGBM artifact

The blend is restricted to verified non-residual OOF/test prediction pairs.

The weights are constrained to be nonnegative and sum to one. This keeps the blend as an average of existing predictions rather than an unrestricted second-stage regression.

The search includes pure models, pairwise blends, and random simplex searches focused around the strongest components.

In [292]:
n_safe_te_blend, k_safe_te_blend = safe_te_oof_matrix.shape

safe_te_gram = (safe_te_oof_matrix.T @ safe_te_oof_matrix) / n_safe_te_blend
safe_te_target_cross = (safe_te_oof_matrix.T @ y_safe_te_blend) / n_safe_te_blend
safe_te_target_square = float((y_safe_te_blend @ y_safe_te_blend) / n_safe_te_blend)


def safe_te_mse_for_weights(weights):
    """
    Compute OOF MSE for one convex weight vector.
    """
    weights = np.asarray(weights, dtype=np.float64)

    return float(
        weights @ safe_te_gram @ weights
        - 2.0 * (weights @ safe_te_target_cross)
        + safe_te_target_square
    )


def safe_te_mse_for_weight_matrix(weight_matrix):
    """
    Compute OOF MSE for many convex weight vectors.
    """
    weight_matrix = np.asarray(weight_matrix, dtype=np.float64)

    return (
        np.einsum("ij,jk,ik->i", weight_matrix, safe_te_gram, weight_matrix)
        - 2.0 * (weight_matrix @ safe_te_target_cross)
        + safe_te_target_square
    )


safe_te_blend_candidates = []


def add_safe_te_candidate(label, weights):
    """
    Add one normalized nonnegative blend candidate.
    """
    weights = np.asarray(weights, dtype=np.float64)
    weights = np.maximum(weights, 0)

    if weights.sum() <= 0:
        return

    weights = weights / weights.sum()

    safe_te_blend_candidates.append({
        "label": label,
        "mse": safe_te_mse_for_weights(weights),
        "weights": weights,
    })


for component_idx, component_name in enumerate(safe_te_component_names):
    weights = np.zeros(k_safe_te_blend, dtype=np.float64)
    weights[component_idx] = 1.0

    add_safe_te_candidate(
        label=f"pure_{component_name}",
        weights=weights,
    )


pair_grid = np.linspace(0.0, 1.0, 1001)

for i in range(k_safe_te_blend):
    for j in range(i + 1, k_safe_te_blend):
        weight_matrix = np.zeros(
            (len(pair_grid), k_safe_te_blend),
            dtype=np.float64,
        )

        weight_matrix[:, i] = pair_grid
        weight_matrix[:, j] = 1.0 - pair_grid

        pair_mses = safe_te_mse_for_weight_matrix(weight_matrix)
        best_pair_idx = int(np.argmin(pair_mses))

        add_safe_te_candidate(
            label=f"pair_{safe_te_component_names[i]}__{safe_te_component_names[j]}",
            weights=weight_matrix[best_pair_idx],
        )


safe_te_rng = np.random.default_rng(RANDOM_STATE + 2809)

uniform_weights = safe_te_rng.dirichlet(
    np.ones(k_safe_te_blend),
    size=50000,
)

uniform_mses = safe_te_mse_for_weight_matrix(uniform_weights)

add_safe_te_candidate(
    label="random_dirichlet_uniform_50k",
    weights=uniform_weights[int(np.argmin(uniform_mses))],
)


component_mses = np.array([
    mean_squared_error(y_safe_te_blend, safe_te_oof_matrix[:, i])
    for i in range(k_safe_te_blend)
])

te_component_idx = safe_te_component_names.index("lgbm_te_v1")
component_order_by_mse = np.argsort(component_mses)

te_focused_alpha = np.ones(k_safe_te_blend) * 0.25
te_focused_alpha[te_component_idx] = 10.0

for idx in component_order_by_mse[:min(3, k_safe_te_blend)]:
    te_focused_alpha[idx] = max(te_focused_alpha[idx], 2.5)

te_focused_weights = safe_te_rng.dirichlet(
    te_focused_alpha,
    size=75000,
)

te_focused_mses = safe_te_mse_for_weight_matrix(te_focused_weights)

add_safe_te_candidate(
    label="random_dirichlet_te_focused_75k",
    weights=te_focused_weights[int(np.argmin(te_focused_mses))],
)


top_focused_alpha = np.ones(k_safe_te_blend) * 0.20

for rank, idx in enumerate(component_order_by_mse[:min(4, k_safe_te_blend)]):
    top_focused_alpha[idx] = 6.0 / (rank + 1)

top_focused_weights = safe_te_rng.dirichlet(
    top_focused_alpha,
    size=75000,
)

top_focused_mses = safe_te_mse_for_weight_matrix(top_focused_weights)

add_safe_te_candidate(
    label="random_dirichlet_top_focused_75k",
    weights=top_focused_weights[int(np.argmin(top_focused_mses))],
)


safe_te_blend_screen_rows = []

for candidate in safe_te_blend_candidates:
    row = {
        "label": candidate["label"],
        "mse": candidate["mse"],
    }

    for component_name, weight in zip(safe_te_component_names, candidate["weights"]):
        row[f"w_{component_name}"] = weight

    safe_te_blend_screen_rows.append(row)

safe_te_blend_screen = (
    pd.DataFrame(safe_te_blend_screen_rows)
    .sort_values("mse")
    .reset_index(drop=True)
)

best_safe_te_blend = safe_te_blend_candidates[
    int(np.argmin([candidate["mse"] for candidate in safe_te_blend_candidates]))
]

best_safe_te_weights = best_safe_te_blend["weights"]

safe_te_blend_oof_pred = np.clip(
    safe_te_oof_matrix @ best_safe_te_weights,
    0,
    100,
)

safe_te_blend_test_pred = np.clip(
    safe_te_test_matrix @ best_safe_te_weights,
    0,
    100,
)

safe_te_blend_oof_mse = mean_squared_error(
    y_safe_te_blend,
    safe_te_blend_oof_pred,
)

pure_te_oof_mse_for_blend = mean_squared_error(
    y_safe_te_blend,
    safe_te_oof_matrix[:, te_component_idx],
)

safe_te_blend_gain_vs_te = pure_te_oof_mse_for_blend - safe_te_blend_oof_mse

safe_te_blend_weight_table = (
    pd.DataFrame({
        "component": safe_te_component_names,
        "display_name": safe_te_component_display_names,
        "weight": best_safe_te_weights,
        "component_oof_mse": component_mses,
    })
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)

print("Best candidate:", best_safe_te_blend["label"])
print("Safe TE blend OOF MSE:", safe_te_blend_oof_mse)
print("Pure TE LightGBM OOF MSE:", pure_te_oof_mse_for_blend)
print("Gain vs pure TE LightGBM:", safe_te_blend_gain_vs_te)

print("\nBest blend weights:")
display(safe_te_blend_weight_table)

print("\nTop blend candidates:")
safe_te_blend_screen.head(20)

Best candidate: random_dirichlet_te_focused_75k
Safe TE blend OOF MSE: 78.07151037235208
Pure TE LightGBM OOF MSE: 82.90761370632671
Gain vs pure TE LightGBM: 4.836103333974634

Best blend weights:


,component,display_name,weight,component_oof_mse
0,lgbm_te_v1,Target/statistical-encoded LightGBM,0.624592,82.907614
1,lgbm_long100k,LightGBM long 100k,0.207687,93.726214
2,lgbm_long80k,LightGBM long 80k,0.127595,94.163485
3,extratrees_safe,ExtraTrees,0.040027,110.749294
4,lgbm_12k,LightGBM selected 12k,0.000082,98.779406
5,rf500,Random Forest 500,0.000017,122.328024



Top blend candidates:


,label,mse,w_rf500,w_extratrees_safe,w_lgbm_12k,w_lgbm_long80k,w_lgbm_long100k,w_lgbm_te_v1
0,random_dirichlet_te_focused_75k,78.071510,0.000017,0.040027,0.000082,0.127595,0.207687,0.624592
1,random_dirichlet_top_focused_75k,78.085494,0.000026,0.034714,0.002192,0.130861,0.196293,0.635913
2,random_dirichlet_uniform_50k,78.129569,0.001443,0.029954,0.011069,0.100940,0.249669,0.606925
3,pair_lgbm_long100k__lgbm_te_v1,78.158831,0.000000,0.000000,0.000000,0.000000,0.356000,0.644000
4,pair_lgbm_long80k__lgbm_te_v1,78.206523,0.000000,0.000000,0.000000,0.352000,0.000000,0.648000
5,pair_lgbm_12k__lgbm_te_v1,79.778222,0.000000,0.000000,0.289000,0.000000,0.000000,0.711000
6,pair_extratrees_safe__lgbm_te_v1,81.633548,0.000000,0.173000,0.000000,0.000000,0.000000,0.827000
7,pair_rf500__lgbm_te_v1,82.103802,0.124000,0.000000,0.000000,0.000000,0.000000,0.876000
8,pure_lgbm_te_v1,82.907614,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
9,pair_extratrees_safe__lgbm_long100k,88.966368,0.000000,0.319000,0.000000,0.000000,0.681000,0.000000


THIS MODEL IS IMPORTANT! to be used with accounting solver models later (28G), which together becomes a final submission

In [293]:
safe_te_blend_oof_df = pd.DataFrame({
    "row_index": np.arange(len(y_safe_te_blend)),
    TARGET_COL: y_safe_te_blend,
    "pred_clipped": safe_te_blend_oof_pred,
})

safe_te_blend_test_df = pd.DataFrame({
    ID_COL: test_id_reference,
    TARGET_COL: safe_te_blend_test_pred,
})

safe_te_blend_submission = safe_te_blend_test_df[
    [ID_COL, TARGET_COL]
].copy()

safe_te_blend_oof_df.to_csv(
    SAFE_TE_BLEND_OOF_PATH,
    index=False,
)

safe_te_blend_test_df.to_csv(
    SAFE_TE_BLEND_TEST_PATH,
    index=False,
)

safe_te_blend_submission.to_csv(
    SAFE_TE_BLEND_SUBMISSION_PATH,
    index=False,
)

safe_te_blend_screen.to_csv(
    SAFE_TE_BLEND_SCREEN_PATH,
    index=False,
)

safe_te_blend_weight_table.to_csv(
    SAFE_TE_BLEND_WEIGHTS_PATH,
    index=False,
)

SAFE_TE_BLEND_PASS = True
SAFE_TE_BLEND_SUBMISSION_WORTHY = bool(safe_te_blend_gain_vs_te >= 1.0)

print("SAFE_TE_BLEND_PASS:", SAFE_TE_BLEND_PASS)
print("SAFE_TE_BLEND_SUBMISSION_WORTHY:", SAFE_TE_BLEND_SUBMISSION_WORTHY)

print("\nSaved files:")
print("-", SAFE_TE_BLEND_OOF_PATH)
print("-", SAFE_TE_BLEND_TEST_PATH)
print("-", SAFE_TE_BLEND_SUBMISSION_PATH)
print("-", SAFE_TE_BLEND_SCREEN_PATH)
print("-", SAFE_TE_BLEND_WEIGHTS_PATH)

print("\nSubmission prediction summary:")
pd.Series(safe_te_blend_test_pred).describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

SAFE_TE_BLEND_PASS: True
SAFE_TE_BLEND_SUBMISSION_WORTHY: True

Saved files:
- model_results/oof_blend_te_verified_noresid_weighted.csv
- model_results/testpred_blend_te_verified_noresid_weighted.csv
- submission_blend_te_verified_noresid_weighted.csv
- model_results/blend_te_verified_noresid_weight_screen.csv
- model_results/blend_te_verified_noresid_best_weights.csv

Submission prediction summary:


count    48307.000000
mean        54.338382
std         24.551656
min          0.201202
1%           7.931219
5%          16.555257
25%         34.791862
50%         52.371320
75%         74.537515
95%         95.197915
99%         99.362563
max         99.999194
dtype: float64

In [294]:
if "model_scoreboard" in globals():
    model_scoreboard = pd.concat(
        [
            model_scoreboard,
            pd.DataFrame([
                {
                    "model_family": "Blend",
                    "model": SAFE_TE_BLEND_NAME,
                    "feature_space": "OOF prediction blend",
                    "metric_source": "5-fold OOF",
                    "train_mse": np.nan,
                    "val_mse": safe_te_blend_oof_mse,
                    "notes": "; ".join(
                        [
                            f"w_{row.component}={row.weight:.3f}"
                            for row in safe_te_blend_weight_table.itertuples(index=False)
                            if row.weight > 0.0005
                        ]
                    ),
                }
            ])
        ],
        axis=0,
        ignore_index=True,
    )

    model_scoreboard = (
        model_scoreboard
        .drop_duplicates(
            subset=["model_family", "model", "feature_space", "val_mse", "notes"],
            keep="last",
        )
        .sort_values("val_mse")
        .reset_index(drop=True)
    )

    display(model_scoreboard.head(20))

else:
    print("model_scoreboard not found; skipping scoreboard update.")

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Blend,blend_te_verified_noresid_weighted,OOF prediction blend,5-fold OOF,NaN,78.071510,w_lgbm_te_v1=0.625; w_lgbm_long100k=0.208; w_l...
1,Target/statistical-encoded boosting,lgbm_te_base_5fold_oof_v1,base + target/statistical encoding,5-fold OOF,NaN,82.907608,LightGBM with leakage-safe target/statistical ...
2,Blend,stack_ridge_saved_predictions,saved prediction stack,5-fold OOF,NaN,87.976479,ridge stack over saved predictions
3,Blend,long_lightgbm_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,88.966368,w_extratrees=0.319; w_lgbm_100k_lr02=0.681
4,Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
5,Boosting,lgbm_t03_base_5fold_oof_long100k_lr02,base,5-fold OOF,NaN,93.726219,long selected LightGBM artifact
6,Boosting,lgbm_t03_base_5fold_oof_long80k_lr03,base,5-fold OOF,NaN,94.163485,long selected LightGBM artifact
7,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
8,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...
9,Boosting,lightgbm_targeted_holdout_refinement,base,fixed holdout,27.643127,104.496354,lgbm_target_l95_child60_l2_5; best_iteration=8000


In [295]:
submission_tracker_path = RESULTS_DIR / "submission_tracker.csv"

safe_te_public_checkpoint = pd.DataFrame([
    {
        "file": SAFE_TE_BLEND_SUBMISSION_PATH.name,
        "model_family": "Verified non-residual TE blend",
        "public_mse": 69.689,
        "notes": (
            "Convex blend of RF, ExtraTrees, base LightGBM, long LightGBM, "
            "and target/statistical-encoded LightGBM; residual-stack artifacts excluded"
        ),
    }
])

if submission_tracker_path.exists():
    submission_tracker = pd.read_csv(submission_tracker_path)

    if "file" not in submission_tracker.columns and "submission_file" in submission_tracker.columns:
        submission_tracker = submission_tracker.rename(columns={"submission_file": "file"})

    submission_tracker = pd.concat(
        [submission_tracker, safe_te_public_checkpoint],
        ignore_index=True,
    )
else:
    submission_tracker = safe_te_public_checkpoint.copy()

submission_tracker = (
    submission_tracker
    .drop_duplicates(subset=["file"], keep="last")
    .reset_index(drop=True)
)

submission_tracker.to_csv(
    submission_tracker_path,
    index=False,
)

submission_tracker.tail(10)

,file,model_family,public_mse,notes
0,submission_blend_auto_et_lgbm_long_oof_weighte...,Blend,80.662,ExtraTrees + long 100k LightGBM blend
1,submission_lgbm_te_base_5fold_oof_v1_foldavg.csv,Target/statistical-encoded boosting,70.997,LightGBM on base features plus leakage-safe ta...
2,submission_blend_te_verified_noresid_weighted.csv,Verified non-residual TE blend,69.689,"Convex blend of RF, ExtraTrees, base LightGBM,..."


In [296]:
safe_te_blend_diagnostic = pd.DataFrame([
    {
        "check": "Safe TE blend preflight",
        "value": SAFE_TE_BLEND_PREFLIGHT_PASS,
        "expected": True,
        "pass": bool(SAFE_TE_BLEND_PREFLIGHT_PASS),
    },
    {
        "check": "Component count",
        "value": len(safe_te_component_names),
        "expected": 6,
        "pass": bool(len(safe_te_component_names) == 6),
    },
    {
        "check": "Random Forest included",
        "value": "rf500" in safe_te_component_names,
        "expected": True,
        "pass": bool("rf500" in safe_te_component_names),
    },
    {
        "check": "ExtraTrees included",
        "value": "extratrees_safe" in safe_te_component_names,
        "expected": True,
        "pass": bool("extratrees_safe" in safe_te_component_names),
    },
    {
        "check": "Pure TE OOF MSE",
        "value": pure_te_oof_mse_for_blend,
        "expected": "about 82.9076",
        "pass": bool(abs(pure_te_oof_mse_for_blend - 82.9076) < 0.10),
    },
    {
        "check": "Safe TE blend OOF MSE",
        "value": safe_te_blend_oof_mse,
        "expected": "about 78.0715",
        "pass": bool(safe_te_blend_oof_mse < 78.25),
    },
    {
        "check": "Gain vs pure TE",
        "value": safe_te_blend_gain_vs_te,
        "expected": "about 4.836",
        "pass": bool(safe_te_blend_gain_vs_te > 4.5),
    },
    {
        "check": "Test prediction rows",
        "value": len(safe_te_blend_test_pred),
        "expected": len(test_ids),
        "pass": bool(len(safe_te_blend_test_pred) == len(test_ids)),
    },
])

safe_te_blend_diagnostic

,check,value,expected,pass
0,Safe TE blend preflight,True,True,True
1,Component count,6,6,True
2,Random Forest included,True,True,True
3,ExtraTrees included,True,True,True
4,Pure TE OOF MSE,82.907614,about 82.9076,True
5,Safe TE blend OOF MSE,78.07151,about 78.0715,True
6,Gain vs pure TE,4.836103,about 4.836,True
7,Test prediction rows,48307,48307,True


## Data modelling, Part 7: Accounting / solver models

The previous section produced a strong, leakage-safe convex ensemble prior. This section uses that prior inside a subgroup accounting solver.

The key observation is that many rows are related by subgroup count identities within the same `SCHOOL × ASSESSMENT_NAME` group:

$$
\text{All Students} = \text{Female} + \text{Male}
$$

and

$$
\text{All Students} = \text{Economically Disadvantaged} + \text{Not Economically Disadvantaged}.
$$

Because the target is a percentage and each row contains `N_STUDENTS`, predictions can be converted into estimated proficient counts. The solver adjusts those count estimates so that the subgroup accounting identities hold while remaining close to the machine-learning prior.

The final report model uses the Part 6 convex ensemble as the prior. A target-encoded LightGBM-only prior is also included as a simple sensitivity check.

In [297]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

try:
    from scipy.optimize import lsq_linear
    HAVE_ACCOUNTING_LSQ_LINEAR = True
except Exception:
    HAVE_ACCOUNTING_LSQ_LINEAR = False

TARGET_COL = globals().get("TARGET_COL", "PERCENT_PROFICIENT")
ID_COL = globals().get("ID_COL", "ASSESSMENT_ID")
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ACCOUNTING_N_SPLITS = 5

required_accounting_objects = [
    "raw_train_te",
    "raw_test_te",
    "y_train",
]

missing_accounting_objects = [
    name for name in required_accounting_objects
    if name not in globals()
]

if missing_accounting_objects:
    raise ValueError(f"Missing required accounting objects: {missing_accounting_objects}")

required_accounting_columns = [
    ID_COL,
    "SCHOOL",
    "ASSESSMENT_NAME",
    "SUBGROUP_NAME",
    "N_STUDENTS",
]

missing_train_accounting_cols = [
    col for col in required_accounting_columns
    if col not in raw_train_te.columns
]

missing_test_accounting_cols = [
    col for col in required_accounting_columns
    if col not in raw_test_te.columns
]

if missing_train_accounting_cols:
    raise ValueError(f"raw_train_te is missing columns: {missing_train_accounting_cols}")

if missing_test_accounting_cols:
    raise ValueError(f"raw_test_te is missing columns: {missing_test_accounting_cols}")

y_accounting = np.asarray(y_train, dtype=np.float64).reshape(-1)

if len(raw_train_te) != len(y_accounting):
    raise ValueError("raw_train_te and y_train have different row counts.")

n_train_accounting = len(raw_train_te)
n_test_accounting = len(raw_test_te)

train_id_reference = pd.Series(raw_train_te[ID_COL]).astype(str).to_numpy()
test_id_reference = pd.Series(raw_test_te[ID_COL]).astype(str).to_numpy()
test_ids_output = raw_test_te[ID_COL].to_numpy()


def clean_accounting_string(values):
    return pd.Series(values).astype("string").fillna("<NA>").astype(str)


def accounting_numeric_array(values):
    return (
        pd.to_numeric(values, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float64)
    )


train_accounting_df = pd.DataFrame({
    "row_index": np.arange(n_train_accounting),
    ID_COL: raw_train_te[ID_COL].to_numpy(),
    "SCHOOL": clean_accounting_string(raw_train_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_accounting_string(raw_train_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_accounting_string(raw_train_te["SUBGROUP_NAME"]),
    "N_STUDENTS": accounting_numeric_array(raw_train_te["N_STUDENTS"]),
    TARGET_COL: y_accounting,
})

test_accounting_df = pd.DataFrame({
    "row_index": np.arange(n_test_accounting),
    ID_COL: raw_test_te[ID_COL].to_numpy(),
    "SCHOOL": clean_accounting_string(raw_test_te["SCHOOL"]),
    "ASSESSMENT_NAME": clean_accounting_string(raw_test_te["ASSESSMENT_NAME"]),
    "SUBGROUP_NAME": clean_accounting_string(raw_test_te["SUBGROUP_NAME"]),
    "N_STUDENTS": accounting_numeric_array(raw_test_te["N_STUDENTS"]),
})

for frame in [train_accounting_df, test_accounting_df]:
    frame["N_STUDENTS"] = np.where(
        np.isfinite(frame["N_STUDENTS"]) & (frame["N_STUDENTS"] > 0),
        frame["N_STUDENTS"],
        np.nan,
    )

    frame["group_key"] = (
        frame["SCHOOL"].astype(str)
        + "||"
        + frame["ASSESSMENT_NAME"].astype(str)
    )

train_accounting_df["prof_count"] = np.rint(
    np.clip(train_accounting_df[TARGET_COL].to_numpy(dtype=np.float64), 0, 100)
    / 100.0
    * train_accounting_df["N_STUDENTS"].to_numpy(dtype=np.float64)
)

train_accounting_df["prof_count"] = np.clip(
    train_accounting_df["prof_count"],
    0,
    train_accounting_df["N_STUDENTS"],
)

candidate_accounting_identities = [
    ("All Students", "Female", "Male"),
    ("All Students", "Economically Disadvantaged", "Not Economically Disadvantaged"),
]

subgroups_seen = set(train_accounting_df["SUBGROUP_NAME"]).union(
    set(test_accounting_df["SUBGROUP_NAME"])
)

accounting_identities = [
    identity for identity in candidate_accounting_identities
    if all(subgroup in subgroups_seen for subgroup in identity)
]

if not accounting_identities:
    raise ValueError("No usable subgroup accounting identities were found.")

subgroup_count_summary = pd.concat(
    [
        train_accounting_df["SUBGROUP_NAME"].value_counts().rename("train_rows"),
        test_accounting_df["SUBGROUP_NAME"].value_counts().rename("test_rows"),
    ],
    axis=1,
).fillna(0).astype(int)

print("Have scipy.optimize.lsq_linear:", HAVE_ACCOUNTING_LSQ_LINEAR)
print("Training rows:", n_train_accounting)
print("Test rows:", n_test_accounting)

print("\nAccounting identities used:")
for all_s, a_s, b_s in accounting_identities:
    print(f"{all_s} = {a_s} + {b_s}")

subgroup_count_summary

Have scipy.optimize.lsq_linear: True
Training rows: 144921
Test rows: 48307

Accounting identities used:
All Students = Female + Male
All Students = Economically Disadvantaged + Not Economically Disadvantaged


,train_rows,test_rows
SUBGROUP_NAME,,
All Students,36711,12428
Male,29363,9589
Female,29110,9777
Economically Disadvantaged,25137,8487
Not Economically Disadvantaged,24600,8026


#### 7A. Accounting priors

The solver needs a prior prediction for every row. Two simple priors are checked:

- target/statistical-encoded LightGBM alone,
- the verified convex ensemble from Part 6.

The final report model uses the convex ensemble prior because it is strong, transparent, and already validated in the previous section.

In [298]:
def first_available_prediction_column(df, candidate_cols, path, label):
    """
    Return the first prediction column present in a saved prediction file.
    """
    for col in candidate_cols:
        if col in df.columns:
            return col

    numeric_cols = [
        col for col in df.columns
        if col not in ["row_index", ID_COL, "fold"]
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    if TARGET_COL in numeric_cols and len(numeric_cols) > 1:
        numeric_cols = [col for col in numeric_cols if col != TARGET_COL]

    if numeric_cols:
        return numeric_cols[0]

    raise ValueError(
        f"No usable prediction column found for {label} in {path}. "
        f"Observed columns: {list(df.columns)}"
    )


def load_accounting_oof_prior(path, candidate_cols, label):
    """
    Load and align one OOF prior prediction vector.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    pred_col = first_available_prediction_column(df, candidate_cols, path, label)

    aligned = df.copy()

    if "row_index" in aligned.columns:
        aligned = aligned.sort_values("row_index").reset_index(drop=True)

        if len(aligned) != n_train_accounting:
            raise ValueError(f"OOF row count mismatch for {label}.")

        expected_row_index = np.arange(n_train_accounting)

        if not np.array_equal(aligned["row_index"].to_numpy(), expected_row_index):
            raise ValueError(f"OOF row_index is not consecutive for {label}.")

    elif ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in OOF file for {label}.")

        train_order = pd.DataFrame({
            ID_COL: train_id_reference,
            "_train_order": np.arange(n_train_accounting),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = train_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"OOF predictions could not be aligned for {label}.")

        aligned = (
            aligned
            .sort_values("_train_order")
            .drop(columns=["_train_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

        if len(aligned) != n_train_accounting:
            raise ValueError(f"OOF row count mismatch for {label}.")

    if TARGET_COL in aligned.columns and pred_col != TARGET_COL:
        saved_target = pd.to_numeric(aligned[TARGET_COL], errors="coerce").to_numpy(dtype=np.float64)

        if np.isfinite(saved_target).all() and not np.allclose(saved_target, y_accounting):
            raise ValueError(f"OOF target alignment failed for {label}.")

    pred = pd.to_numeric(aligned[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite OOF prior predictions found for {label}.")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


def load_accounting_test_prior(path, candidate_cols, label):
    """
    Load and align one test prior prediction vector.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    pred_col = first_available_prediction_column(df, candidate_cols, path, label)

    aligned = df.copy()

    if ID_COL in aligned.columns:
        if aligned[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in test file for {label}.")

        test_order = pd.DataFrame({
            ID_COL: test_id_reference,
            "_test_order": np.arange(n_test_accounting),
        })

        aligned[ID_COL] = aligned[ID_COL].astype(str)

        aligned = test_order.merge(
            aligned,
            on=ID_COL,
            how="left",
        )

        if aligned[pred_col].isna().any():
            raise ValueError(f"Test predictions could not be aligned for {label}.")

        aligned = (
            aligned
            .sort_values("_test_order")
            .drop(columns=["_test_order"])
            .reset_index(drop=True)
        )

    else:
        aligned = aligned.reset_index(drop=True)

        if len(aligned) != n_test_accounting:
            raise ValueError(f"Test row count mismatch for {label}.")

    pred = pd.to_numeric(aligned[pred_col], errors="coerce").to_numpy(dtype=np.float64)

    if not np.isfinite(pred).all():
        raise ValueError(f"Non-finite test prior predictions found for {label}.")

    return np.clip(pred, 0, 100).astype(np.float32), pred_col


accounting_prior_specs = [
    {
        "name": "te_lgbm_prior",
        "display_name": "Target/statistical-encoded LightGBM prior",
        "oof_path": RESULTS_DIR / "oof_lgbm_te_base_5fold_oof_v1.csv",
        "test_path": RESULTS_DIR / "testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv",
        "oof_candidate_cols": ["pred_clipped"],
        "test_candidate_cols": [TARGET_COL, "PERCENT_PROFICIENT"],
    },
    {
        "name": "convex_te_blend_prior",
        "display_name": "Verified convex ensemble prior",
        "oof_path": RESULTS_DIR / "oof_blend_te_verified_noresid_weighted.csv",
        "test_path": RESULTS_DIR / "testpred_blend_te_verified_noresid_weighted.csv",
        "oof_candidate_cols": ["pred_clipped"],
        "test_candidate_cols": [TARGET_COL, "PERCENT_PROFICIENT"],
    },
]

accounting_prior_data = {}
accounting_prior_rows = []

for spec in accounting_prior_specs:
    oof_pred, oof_col = load_accounting_oof_prior(
        path=spec["oof_path"],
        candidate_cols=spec["oof_candidate_cols"],
        label=spec["name"],
    )

    test_pred, test_col = load_accounting_test_prior(
        path=spec["test_path"],
        candidate_cols=spec["test_candidate_cols"],
        label=spec["name"],
    )

    prior_oof_mse = mean_squared_error(y_accounting, oof_pred)

    accounting_prior_data[spec["name"]] = {
        "display_name": spec["display_name"],
        "oof_pred": oof_pred,
        "test_pred": test_pred,
        "oof_path": spec["oof_path"],
        "test_path": spec["test_path"],
        "oof_column": oof_col,
        "test_column": test_col,
        "base_oof_mse": prior_oof_mse,
    }

    accounting_prior_rows.append({
        "prior": spec["name"],
        "display_name": spec["display_name"],
        "base_oof_mse": prior_oof_mse,
        "oof_file": str(spec["oof_path"]),
        "oof_column": oof_col,
        "test_file": str(spec["test_path"]),
        "test_column": test_col,
        "test_pred_mean": test_pred.mean(),
        "test_pred_std": test_pred.std(),
    })

accounting_prior_summary = (
    pd.DataFrame(accounting_prior_rows)
    .sort_values("base_oof_mse")
    .reset_index(drop=True)
)

ACCOUNTING_PRIORS_READY = bool(
    set(accounting_prior_data) == {"te_lgbm_prior", "convex_te_blend_prior"}
    and all(len(data["oof_pred"]) == n_train_accounting for data in accounting_prior_data.values())
    and all(len(data["test_pred"]) == n_test_accounting for data in accounting_prior_data.values())
)

print("ACCOUNTING_PRIORS_READY:", ACCOUNTING_PRIORS_READY)

if not ACCOUNTING_PRIORS_READY:
    raise ValueError("Accounting prior loading failed.")

accounting_prior_summary

ACCOUNTING_PRIORS_READY: True


,prior,display_name,base_oof_mse,oof_file,oof_column,test_file,test_column,test_pred_mean,test_pred_std
0,convex_te_blend_prior,Verified convex ensemble prior,78.071510,model_results/oof_blend_te_verified_noresid_we...,pred_clipped,model_results/testpred_blend_te_verified_nores...,PERCENT_PROFICIENT,54.338383,24.551403
1,te_lgbm_prior,Target/statistical-encoded LightGBM prior,82.907614,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,pred_clipped,model_results/testpred_lgbm_te_base_5fold_oof_...,PERCENT_PROFICIENT,54.500694,24.778822


#### 7B. Constrained accounting solver

For each `SCHOOL × ASSESSMENT_NAME` group, the solver works with proficient counts rather than percentages.

For known training rows, the proficient count is computed as:

$$
k_i = \operatorname{round}(N_i y_i / 100).
$$

For unknown validation or test rows, the prior count is:

$$
m_i = N_i q_i / 100.
$$

The solver keeps the unknown counts close to the prior counts while penalizing violations of the subgroup accounting identities.

In [299]:
def accounting_n_tolerance(n_students):
    """
    Tolerance for checking whether subgroup N_STUDENTS totals are compatible.
    """
    if not np.isfinite(n_students):
        return 1.0

    return max(1.0, 0.02 * float(n_students))


def build_accounting_known_lookup(df, row_indices):
    """
    Build known subgroup proficient-count lookup from selected training rows.
    """
    subset = df.iloc[row_indices]

    lookup = {}

    for group_key, group_df in subset.groupby("group_key", sort=False):
        group_lookup = {}

        for subgroup, subgroup_df in group_df.groupby("SUBGROUP_NAME", sort=False):
            n_values = subgroup_df["N_STUDENTS"].to_numpy(dtype=np.float64)
            k_values = subgroup_df["prof_count"].to_numpy(dtype=np.float64)

            good = (
                np.isfinite(n_values)
                & (n_values > 0)
                & np.isfinite(k_values)
            )

            if not good.any():
                continue

            n_mean = float(np.mean(n_values[good]))
            k_mean = float(np.mean(k_values[good]))

            group_lookup[str(subgroup)] = {
                "n": n_mean,
                "k": float(np.clip(k_mean, 0, n_mean)),
            }

        if group_lookup:
            lookup[str(group_key)] = group_lookup

    return lookup


def solve_one_accounting_group(
    query_group_df,
    known_group,
    prior_pct,
    identities,
    equation_weight,
    prior_weight,
):
    """
    Solve one SCHOOL x ASSESSMENT_NAME accounting system.
    """
    n_query = len(query_group_df)

    pred_pct = np.full(n_query, np.nan, dtype=np.float32)
    touched = np.zeros(n_query, dtype=bool)
    eq_count_used = 0

    subgroup_to_var = {}
    var_to_local = []
    var_n = []
    var_prior_count = []

    for local_idx, row in query_group_df.iterrows():
        subgroup = str(row["SUBGROUP_NAME"])
        n_students = float(row["N_STUDENTS"])
        prior_value = float(prior_pct[local_idx])

        if not np.isfinite(n_students) or n_students <= 0:
            continue

        if not np.isfinite(prior_value):
            continue

        if subgroup in subgroup_to_var:
            continue

        var_idx = len(var_to_local)

        subgroup_to_var[subgroup] = var_idx
        var_to_local.append(local_idx)
        var_n.append(n_students)
        var_prior_count.append(float(np.clip(prior_value, 0, 100) / 100.0 * n_students))

    n_vars = len(var_to_local)

    if n_vars == 0:
        return pred_pct, touched, eq_count_used

    def subgroup_present(subgroup):
        return (
            subgroup in subgroup_to_var
            or (
                known_group is not None
                and subgroup in known_group
            )
        )

    def subgroup_n(subgroup):
        if subgroup in subgroup_to_var:
            return var_n[subgroup_to_var[subgroup]]

        return known_group[subgroup]["n"]

    def subgroup_known_count(subgroup):
        return known_group[subgroup]["k"]

    a_rows = []
    b_values = []

    sqrt_prior = float(np.sqrt(prior_weight))
    sqrt_equation = float(np.sqrt(equation_weight))

    for var_idx in range(n_vars):
        row = np.zeros(n_vars, dtype=np.float64)
        row[var_idx] = sqrt_prior

        a_rows.append(row)
        b_values.append(sqrt_prior * var_prior_count[var_idx])

    touched_vars = set()

    for all_s, a_s, b_s in identities:
        if not (
            subgroup_present(all_s)
            and subgroup_present(a_s)
            and subgroup_present(b_s)
        ):
            continue

        n_all = subgroup_n(all_s)
        n_a = subgroup_n(a_s)
        n_b = subgroup_n(b_s)

        if not (
            np.isfinite(n_all)
            and np.isfinite(n_a)
            and np.isfinite(n_b)
        ):
            continue

        if abs(n_all - (n_a + n_b)) > accounting_n_tolerance(n_all):
            continue

        signs = {
            all_s: 1.0,
            a_s: -1.0,
            b_s: -1.0,
        }

        row = np.zeros(n_vars, dtype=np.float64)
        known_sum = 0.0
        vars_in_equation = []

        for subgroup, sign in signs.items():
            if subgroup in subgroup_to_var:
                var_idx = subgroup_to_var[subgroup]
                row[var_idx] += sign
                vars_in_equation.append(var_idx)
            else:
                known_sum += sign * subgroup_known_count(subgroup)

        if not vars_in_equation:
            continue

        a_rows.append(sqrt_equation * row)
        b_values.append(sqrt_equation * (-known_sum))

        eq_count_used += 1
        touched_vars.update(vars_in_equation)

    if eq_count_used == 0 or not touched_vars:
        return pred_pct, touched, eq_count_used

    A = np.vstack(a_rows)
    b = np.asarray(b_values, dtype=np.float64)

    lower = np.zeros(n_vars, dtype=np.float64)
    upper = np.asarray(var_n, dtype=np.float64)

    try:
        if HAVE_ACCOUNTING_LSQ_LINEAR:
            solution = lsq_linear(
                A,
                b,
                bounds=(lower, upper),
                method="trf",
                lsmr_tol="auto",
                max_iter=100,
            )

            solved_counts = solution.x

        else:
            solved_counts, *_ = np.linalg.lstsq(A, b, rcond=None)
            solved_counts = np.clip(solved_counts, lower, upper)

    except Exception:
        solved_counts, *_ = np.linalg.lstsq(A, b, rcond=None)
        solved_counts = np.clip(solved_counts, lower, upper)

    for var_idx in touched_vars:
        local_idx = var_to_local[var_idx]
        n_students = var_n[var_idx]

        if np.isfinite(n_students) and n_students > 0:
            pred_value = 100.0 * float(solved_counts[var_idx]) / n_students
            pred_pct[local_idx] = np.float32(np.clip(pred_value, 0, 100))
            touched[local_idx] = True

    return pred_pct, touched, eq_count_used


def solve_many_accounting_groups(
    query_df,
    known_lookup,
    prior_pct,
    identities,
    equation_weight,
    prior_weight,
):
    """
    Solve all accounting groups for one validation or test frame.
    """
    query_df = query_df.reset_index(drop=True).copy()
    query_df["local_pos"] = np.arange(len(query_df))

    prior_pct = np.asarray(prior_pct, dtype=np.float32).reshape(-1)

    pred = np.full(len(query_df), np.nan, dtype=np.float32)
    touched = np.zeros(len(query_df), dtype=bool)
    eq_counts = np.zeros(len(query_df), dtype=np.float32)

    for group_key, group_df in query_df.groupby("group_key", sort=False):
        local_positions = group_df["local_pos"].to_numpy(dtype=np.int64)

        known_group = known_lookup.get(str(group_key), {})

        group_pred, group_touched, group_eq_count = solve_one_accounting_group(
            query_group_df=group_df.reset_index(drop=True),
            known_group=known_group,
            prior_pct=prior_pct[local_positions],
            identities=identities,
            equation_weight=equation_weight,
            prior_weight=prior_weight,
        )

        pred[local_positions] = group_pred
        touched[local_positions] = group_touched
        eq_counts[local_positions] = group_eq_count

    return pred, touched, eq_counts

#### 7C. Accounting solver with simple priors

The solver is screened using a small grid of equation weights and solver shrinkage values.

The final submitted report model is the convex ensemble prior with the accounting correction. The target-encoded LightGBM-only prior is retained only as a simple sensitivity check.

In [300]:
def run_accounting_solver_for_prior(
    prior_name,
    prior_display_name,
    base_oof,
    base_test,
    equation_weight_grid,
    lambda_grid,
    prior_weight=1.0,
):
    """
    Run the accounting solver for one prior model and save OOF/test artifacts.
    """
    print("\n" + "=" * 100)
    print("Accounting solver prior:", prior_display_name)
    print("=" * 100)

    base_oof = np.asarray(base_oof, dtype=np.float32).reshape(-1)
    base_test = np.asarray(base_test, dtype=np.float32).reshape(-1)

    if len(base_oof) != n_train_accounting:
        raise ValueError(f"OOF prior length mismatch for {prior_name}.")

    if len(base_test) != n_test_accounting:
        raise ValueError(f"Test prior length mismatch for {prior_name}.")

    base_oof_mse = float(mean_squared_error(y_accounting, base_oof))

    print("Base OOF MSE:", base_oof_mse)

    folds = list(
        KFold(
            n_splits=ACCOUNTING_N_SPLITS,
            shuffle=True,
            random_state=RANDOM_STATE,
        ).split(np.arange(n_train_accounting))
    )

    screen_rows = []
    solver_oof_by_equation_weight = {}
    touched_oof_by_equation_weight = {}

    for equation_weight in equation_weight_grid:
        equation_weight = float(equation_weight)

        solver_oof = np.full(n_train_accounting, np.nan, dtype=np.float32)
        touched_oof = np.zeros(n_train_accounting, dtype=bool)

        print(f"\nEquation weight = {equation_weight:g}")

        for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
            known_lookup_fold = build_accounting_known_lookup(
                train_accounting_df,
                train_idx,
            )

            query_fold = train_accounting_df.iloc[valid_idx].reset_index(drop=True)
            prior_fold = base_oof[valid_idx]

            fold_solver_pred, fold_touched, fold_eq_counts = solve_many_accounting_groups(
                query_df=query_fold,
                known_lookup=known_lookup_fold,
                prior_pct=prior_fold,
                identities=accounting_identities,
                equation_weight=equation_weight,
                prior_weight=prior_weight,
            )

            solver_oof[valid_idx] = fold_solver_pred
            touched_oof[valid_idx] = fold_touched

            print(
                f"  fold {fold_id}: covered {int(fold_touched.sum())} / {len(valid_idx)}"
            )

        solver_oof_by_equation_weight[equation_weight] = solver_oof
        touched_oof_by_equation_weight[equation_weight] = touched_oof

        covered_oof = np.isfinite(solver_oof) & touched_oof

        print(f"  total OOF covered: {int(covered_oof.sum())} / {n_train_accounting}")

        if int(covered_oof.sum()) == 0:
            continue

        for lambda_solver in lambda_grid:
            blended_oof = base_oof.copy()

            blended_oof[covered_oof] = (
                base_oof[covered_oof]
                + float(lambda_solver) * (
                    solver_oof[covered_oof] - base_oof[covered_oof]
                )
            )

            blended_oof = np.clip(blended_oof, 0, 100).astype(np.float32)

            oof_mse = float(mean_squared_error(y_accounting, blended_oof))

            screen_rows.append({
                "prior_name": prior_name,
                "prior_display_name": prior_display_name,
                "equation_weight": equation_weight,
                "prior_weight": float(prior_weight),
                "lambda_solver": float(lambda_solver),
                "covered_rows": int(covered_oof.sum()),
                "coverage_rate": float(covered_oof.mean()),
                "base_oof_mse": base_oof_mse,
                "oof_mse": oof_mse,
                "gain_vs_base": base_oof_mse - oof_mse,
            })

    if not screen_rows:
        raise ValueError(f"No accounting solver rows were covered for {prior_name}.")

    screen = (
        pd.DataFrame(screen_rows)
        .sort_values("oof_mse")
        .reset_index(drop=True)
    )

    best = screen.iloc[0].copy()

    best_equation_weight = float(best["equation_weight"])
    best_lambda_solver = float(best["lambda_solver"])

    best_solver_oof = solver_oof_by_equation_weight[best_equation_weight]
    best_touched_oof = touched_oof_by_equation_weight[best_equation_weight]
    best_covered_oof = np.isfinite(best_solver_oof) & best_touched_oof

    best_oof = base_oof.copy()

    best_oof[best_covered_oof] = (
        base_oof[best_covered_oof]
        + best_lambda_solver * (
            best_solver_oof[best_covered_oof] - base_oof[best_covered_oof]
        )
    )

    best_oof = np.clip(best_oof, 0, 100).astype(np.float32)
    best_oof_mse = float(mean_squared_error(y_accounting, best_oof))

    known_lookup_full = build_accounting_known_lookup(
        train_accounting_df,
        np.arange(n_train_accounting),
    )

    solver_test, touched_test, eq_counts_test = solve_many_accounting_groups(
        query_df=test_accounting_df.reset_index(drop=True),
        known_lookup=known_lookup_full,
        prior_pct=base_test,
        identities=accounting_identities,
        equation_weight=best_equation_weight,
        prior_weight=prior_weight,
    )

    covered_test = np.isfinite(solver_test) & touched_test

    best_test = base_test.copy()

    best_test[covered_test] = (
        base_test[covered_test]
        + best_lambda_solver * (
            solver_test[covered_test] - base_test[covered_test]
        )
    )

    best_test = np.clip(best_test, 0, 100).astype(np.float32)

    screen_path = RESULTS_DIR / f"accounting_solver_{prior_name}_screen.csv"
    oof_path = RESULTS_DIR / f"oof_accounting_solver_{prior_name}.csv"
    testpred_path = RESULTS_DIR / f"testpred_accounting_solver_{prior_name}.csv"

    base_submission_path = Path(f"submission_base_{prior_name}.csv")
    accounting_submission_path = Path(f"submission_accounting_solver_{prior_name}.csv")

    screen.to_csv(screen_path, index=False)

    pd.DataFrame({
        "row_index": np.arange(n_train_accounting),
        TARGET_COL: y_accounting,
        "prior_name": prior_name,
        "base_pred": base_oof,
        "solver_pred": best_solver_oof,
        "solver_covered": best_covered_oof.astype(int),
        "equation_weight": best_equation_weight,
        "prior_weight": prior_weight,
        "lambda_solver": best_lambda_solver,
        "pred_clipped": best_oof,
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: test_ids_output,
        "prior_name": prior_name,
        "base_pred": base_test,
        "solver_pred": solver_test,
        "solver_covered": covered_test.astype(int),
        "equation_weight": best_equation_weight,
        "prior_weight": prior_weight,
        "lambda_solver": best_lambda_solver,
        TARGET_COL: best_test,
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_COL: test_ids_output,
        TARGET_COL: np.clip(base_test, 0, 100),
    }).to_csv(base_submission_path, index=False)

    final_submission = pd.DataFrame({
        ID_COL: test_ids_output,
        TARGET_COL: best_test,
    })

    final_submission.to_csv(accounting_submission_path, index=False)

    if final_submission.shape != (n_test_accounting, 2):
        raise ValueError(f"Submission shape is wrong for {prior_name}: {final_submission.shape}")

    if list(final_submission.columns) != [ID_COL, TARGET_COL]:
        raise ValueError(f"Submission columns are wrong for {prior_name}: {list(final_submission.columns)}")

    if final_submission[ID_COL].isna().any():
        raise ValueError(f"Missing IDs in submission for {prior_name}.")

    if final_submission[TARGET_COL].isna().any():
        raise ValueError(f"Missing predictions in submission for {prior_name}.")

    if not np.isfinite(final_submission[TARGET_COL]).all():
        raise ValueError(f"Non-finite predictions in submission for {prior_name}.")

    if not final_submission[TARGET_COL].between(0, 100).all():
        raise ValueError(f"Predictions outside [0, 100] in submission for {prior_name}.")

    summary = {
        "prior_name": prior_name,
        "prior_display_name": prior_display_name,
        "base_oof_mse": base_oof_mse,
        "best_accounting_oof_mse": best_oof_mse,
        "gain_vs_base": base_oof_mse - best_oof_mse,
        "equation_weight": best_equation_weight,
        "prior_weight": float(prior_weight),
        "lambda_solver": best_lambda_solver,
        "oof_covered_rows": int(best_covered_oof.sum()),
        "oof_coverage_rate": float(best_covered_oof.mean()),
        "test_covered_rows": int(covered_test.sum()),
        "test_coverage_rate": float(covered_test.mean()),
        "base_submission_path": str(base_submission_path),
        "accounting_submission_path": str(accounting_submission_path),
        "screen_path": str(screen_path),
        "oof_path": str(oof_path),
        "testpred_path": str(testpred_path),
    }

    print("\nBest accounting config:")
    print(pd.Series(summary).to_string())

    print("\nTop 10 screen rows:")
    display(screen.head(10))

    print("\nSaved:")
    print("-", base_submission_path)
    print("-", accounting_submission_path)
    print("-", screen_path)
    print("-", oof_path)
    print("-", testpred_path)

    gc.collect()

    return summary, screen

In [301]:
ACCOUNTING_EQUATION_WEIGHT_GRID = [
    100.0,
    1000.0,
    10000.0,
    100000.0,
]

ACCOUNTING_LAMBDA_GRID = np.unique(
    np.concatenate([
        np.linspace(-0.50, 1.50, 501),
        np.array([
            0.0,
            0.25,
            0.50,
            0.75,
            0.916,
            0.924,
            0.932,
            0.992,
            1.0,
        ]),
    ])
)

accounting_run_order = [
    "te_lgbm_prior",
    "convex_te_blend_prior",
]

accounting_summary_rows = []
accounting_screen_frames = []

for prior_name in accounting_run_order:
    prior_data = accounting_prior_data[prior_name]

    summary, screen = run_accounting_solver_for_prior(
        prior_name=prior_name,
        prior_display_name=prior_data["display_name"],
        base_oof=prior_data["oof_pred"],
        base_test=prior_data["test_pred"],
        equation_weight_grid=ACCOUNTING_EQUATION_WEIGHT_GRID,
        lambda_grid=ACCOUNTING_LAMBDA_GRID,
        prior_weight=1.0,
    )

    accounting_summary_rows.append(summary)
    accounting_screen_frames.append(screen)

accounting_solver_summary = (
    pd.DataFrame(accounting_summary_rows)
    .sort_values("best_accounting_oof_mse")
    .reset_index(drop=True)
)

accounting_solver_full_screen = (
    pd.concat(accounting_screen_frames, ignore_index=True)
    .sort_values("oof_mse")
    .reset_index(drop=True)
)

ACCOUNTING_SOLVER_SUMMARY_PATH = RESULTS_DIR / "accounting_solver_prior_comparison_summary.csv"
ACCOUNTING_SOLVER_FULL_SCREEN_PATH = RESULTS_DIR / "accounting_solver_prior_comparison_full_screen.csv"

accounting_solver_summary.to_csv(
    ACCOUNTING_SOLVER_SUMMARY_PATH,
    index=False,
)

accounting_solver_full_screen.to_csv(
    ACCOUNTING_SOLVER_FULL_SCREEN_PATH,
    index=False,
)

FINAL_ACCOUNTING_PRIOR_NAME = "convex_te_blend_prior"

final_accounting_row = (
    accounting_solver_summary
    .loc[accounting_solver_summary["prior_name"] == FINAL_ACCOUNTING_PRIOR_NAME]
    .iloc[0]
)

FINAL_ACCOUNTING_SUBMISSION_PATH = Path(final_accounting_row["accounting_submission_path"])
FINAL_ACCOUNTING_OOF_PATH = Path(final_accounting_row["oof_path"])
FINAL_ACCOUNTING_TESTPRED_PATH = Path(final_accounting_row["testpred_path"])

print("\nAccounting solver prior comparison:")
display(accounting_solver_summary)

print("\nFinal report accounting model:")
print(final_accounting_row.to_string())

print("\nSaved comparison files:")
print("-", ACCOUNTING_SOLVER_SUMMARY_PATH)
print("-", ACCOUNTING_SOLVER_FULL_SCREEN_PATH)


Accounting solver prior: Target/statistical-encoded LightGBM prior
Base OOF MSE: 82.90761366243048

Equation weight = 100
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation weight = 1000
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation weight = 10000
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation weight = 100000
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Best accounting c

,prior_name,prior_display_name,equation_weight,prior_weight,lambda_solver,covered_rows,coverage_rate,base_oof_mse,oof_mse,gain_vs_base
0,te_lgbm_prior,Target/statistical-encoded LightGBM prior,100000.0,1.0,0.916,81593,0.563017,82.907614,58.908045,23.999568
1,te_lgbm_prior,Target/statistical-encoded LightGBM prior,100000.0,1.0,0.916,81593,0.563017,82.907614,58.908045,23.999568
2,te_lgbm_prior,Target/statistical-encoded LightGBM prior,100000.0,1.0,0.912,81593,0.563017,82.907614,58.908163,23.999450
3,te_lgbm_prior,Target/statistical-encoded LightGBM prior,10000.0,1.0,0.916,81593,0.563017,82.907614,58.908255,23.999359
4,te_lgbm_prior,Target/statistical-encoded LightGBM prior,10000.0,1.0,0.916,81593,0.563017,82.907614,58.908255,23.999359
5,te_lgbm_prior,Target/statistical-encoded LightGBM prior,10000.0,1.0,0.912,81593,0.563017,82.907614,58.908392,23.999222
6,te_lgbm_prior,Target/statistical-encoded LightGBM prior,100000.0,1.0,0.920,81593,0.563017,82.907614,58.908845,23.998768
7,te_lgbm_prior,Target/statistical-encoded LightGBM prior,10000.0,1.0,0.920,81593,0.563017,82.907614,58.909037,23.998577
8,te_lgbm_prior,Target/statistical-encoded LightGBM prior,100000.0,1.0,0.908,81593,0.563017,82.907614,58.909200,23.998414
9,te_lgbm_prior,Target/statistical-encoded LightGBM prior,10000.0,1.0,0.908,81593,0.563017,82.907614,58.909446,23.998167



Saved:
- submission_base_te_lgbm_prior.csv
- submission_accounting_solver_te_lgbm_prior.csv
- model_results/accounting_solver_te_lgbm_prior_screen.csv
- model_results/oof_accounting_solver_te_lgbm_prior.csv
- model_results/testpred_accounting_solver_te_lgbm_prior.csv

Accounting solver prior: Verified convex ensemble prior
Base OOF MSE: 78.07151040664739

Equation weight = 100
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation weight = 1000
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covered: 81593 / 144921

Equation weight = 10000
  fold 1: covered 16333 / 28985
  fold 2: covered 16245 / 28984
  fold 3: covered 16362 / 28984
  fold 4: covered 16367 / 28984
  fold 5: covered 16286 / 28984
  total OOF covere

,prior_name,prior_display_name,equation_weight,prior_weight,lambda_solver,covered_rows,coverage_rate,base_oof_mse,oof_mse,gain_vs_base
0,convex_te_blend_prior,Verified convex ensemble prior,100000.0,1.0,0.932,81593,0.563017,78.07151,54.292981,23.778529
1,convex_te_blend_prior,Verified convex ensemble prior,100000.0,1.0,0.932,81593,0.563017,78.07151,54.292981,23.778529
2,convex_te_blend_prior,Verified convex ensemble prior,10000.0,1.0,0.932,81593,0.563017,78.07151,54.293135,23.778376
3,convex_te_blend_prior,Verified convex ensemble prior,10000.0,1.0,0.932,81593,0.563017,78.07151,54.293135,23.778376
4,convex_te_blend_prior,Verified convex ensemble prior,100000.0,1.0,0.928,81593,0.563017,78.07151,54.293156,23.778355
5,convex_te_blend_prior,Verified convex ensemble prior,10000.0,1.0,0.928,81593,0.563017,78.07151,54.293327,23.778183
6,convex_te_blend_prior,Verified convex ensemble prior,100000.0,1.0,0.936,81593,0.563017,78.07151,54.293685,23.777826
7,convex_te_blend_prior,Verified convex ensemble prior,10000.0,1.0,0.936,81593,0.563017,78.07151,54.293820,23.777691
8,convex_te_blend_prior,Verified convex ensemble prior,100000.0,1.0,0.924,81593,0.563017,78.07151,54.294208,23.777302
9,convex_te_blend_prior,Verified convex ensemble prior,100000.0,1.0,0.924,81593,0.563017,78.07151,54.294208,23.777302



Saved:
- submission_base_convex_te_blend_prior.csv
- submission_accounting_solver_convex_te_blend_prior.csv
- model_results/accounting_solver_convex_te_blend_prior_screen.csv
- model_results/oof_accounting_solver_convex_te_blend_prior.csv
- model_results/testpred_accounting_solver_convex_te_blend_prior.csv

Accounting solver prior comparison:


,prior_name,prior_display_name,base_oof_mse,best_accounting_oof_mse,gain_vs_base,equation_weight,prior_weight,lambda_solver,oof_covered_rows,oof_coverage_rate,test_covered_rows,test_coverage_rate,base_submission_path,accounting_submission_path,screen_path,oof_path,testpred_path
0,convex_te_blend_prior,Verified convex ensemble prior,78.071510,54.292981,23.778529,100000.0,1.0,0.932,81593,0.563017,45105,0.933716,submission_base_convex_te_blend_prior.csv,submission_accounting_solver_convex_te_blend_p...,model_results/accounting_solver_convex_te_blen...,model_results/oof_accounting_solver_convex_te_...,model_results/testpred_accounting_solver_conve...
1,te_lgbm_prior,Target/statistical-encoded LightGBM prior,82.907614,58.908045,23.999568,100000.0,1.0,0.916,81593,0.563017,45105,0.933716,submission_base_te_lgbm_prior.csv,submission_accounting_solver_te_lgbm_prior.csv,model_results/accounting_solver_te_lgbm_prior_...,model_results/oof_accounting_solver_te_lgbm_pr...,model_results/testpred_accounting_solver_te_lg...



Final report accounting model:
prior_name                                                convex_te_blend_prior
prior_display_name                               Verified convex ensemble prior
base_oof_mse                                                           78.07151
best_accounting_oof_mse                                               54.292981
gain_vs_base                                                          23.778529
equation_weight                                                        100000.0
prior_weight                                                                1.0
lambda_solver                                                             0.932
oof_covered_rows                                                          81593
oof_coverage_rate                                                      0.563017
test_covered_rows                                                         45105
test_coverage_rate                                                     0.933716
base_sub

In [302]:
final_accounting_oof = pd.read_csv(FINAL_ACCOUNTING_OOF_PATH)
final_accounting_testpred = pd.read_csv(FINAL_ACCOUNTING_TESTPRED_PATH)
final_accounting_submission = pd.read_csv(FINAL_ACCOUNTING_SUBMISSION_PATH)

if "row_index" in final_accounting_oof.columns:
    final_accounting_oof = (
        final_accounting_oof
        .sort_values("row_index")
        .reset_index(drop=True)
    )

final_accounting_oof_pred = final_accounting_oof["pred_clipped"].to_numpy(dtype=float)

final_accounting_oof_mse = mean_squared_error(
    y_accounting,
    final_accounting_oof_pred,
)

final_accounting_base_oof_mse = float(final_accounting_row["base_oof_mse"])
final_accounting_gain = final_accounting_base_oof_mse - final_accounting_oof_mse

ACCOUNTING_SOLVER_FINAL_PASS = bool(
    FINAL_ACCOUNTING_SUBMISSION_PATH.exists()
    and final_accounting_submission.shape == (n_test_accounting, 2)
    and list(final_accounting_submission.columns) == [ID_COL, TARGET_COL]
    and final_accounting_submission[ID_COL].notna().all()
    and final_accounting_submission[TARGET_COL].notna().all()
    and np.isfinite(final_accounting_submission[TARGET_COL]).all()
    and final_accounting_submission[TARGET_COL].between(0, 100).all()
    and final_accounting_oof_mse < final_accounting_base_oof_mse
)

print("ACCOUNTING_SOLVER_FINAL_PASS:", ACCOUNTING_SOLVER_FINAL_PASS)
print("Final accounting prior:", FINAL_ACCOUNTING_PRIOR_NAME)
print("Final accounting submission:", FINAL_ACCOUNTING_SUBMISSION_PATH)
print("Base prior OOF MSE:", final_accounting_base_oof_mse)
print("Accounting solver OOF MSE:", final_accounting_oof_mse)
print("Gain vs prior:", final_accounting_gain)

print("\nFinal submission prediction summary:")
display(
    final_accounting_submission[TARGET_COL].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

ACCOUNTING_SOLVER_FINAL_PASS: True
Final accounting prior: convex_te_blend_prior
Final accounting submission: submission_accounting_solver_convex_te_blend_prior.csv
Base prior OOF MSE: 78.07151040664739
Accounting solver OOF MSE: 54.292981157635786
Gain vs prior: 23.778529249011605

Final submission prediction summary:


count    48307.000000
mean        54.181317
std         25.796468
min          0.013682
1%           3.926658
5%          13.766418
25%         33.667019
50%         52.848274
75%         75.253703
95%         96.861732
99%         99.942367
max         99.999950
Name: PERCENT_PROFICIENT, dtype: float64

In [303]:
if "model_scoreboard" in globals():
    model_scoreboard = pd.concat(
        [
            model_scoreboard,
            pd.DataFrame([
                {
                    "model_family": "Accounting solver",
                    "model": "accounting_solver_convex_te_blend_prior",
                    "feature_space": "convex ensemble prior plus subgroup accounting constraints",
                    "metric_source": "5-fold OOF",
                    "train_mse": np.nan,
                    "val_mse": final_accounting_oof_mse,
                    "notes": (
                        "Final report model; equation_weight="
                        f"{final_accounting_row['equation_weight']:.0f}; "
                        "lambda_solver="
                        f"{final_accounting_row['lambda_solver']:.3f}"
                    ),
                }
            ])
        ],
        axis=0,
        ignore_index=True,
    )

    model_scoreboard = (
        model_scoreboard
        .drop_duplicates(
            subset=["model_family", "model", "feature_space", "val_mse", "notes"],
            keep="last",
        )
        .sort_values("val_mse")
        .reset_index(drop=True)
    )

    display(model_scoreboard.head(20))

else:
    print("model_scoreboard not found; skipping scoreboard update.")

,model_family,model,feature_space,metric_source,train_mse,val_mse,notes
0,Accounting solver,accounting_solver_convex_te_blend_prior,convex ensemble prior plus subgroup accounting...,5-fold OOF,NaN,54.292981,Final report model; equation_weight=100000; la...
1,Blend,blend_te_verified_noresid_weighted,OOF prediction blend,5-fold OOF,NaN,78.071510,w_lgbm_te_v1=0.625; w_lgbm_long100k=0.208; w_l...
2,Target/statistical-encoded boosting,lgbm_te_base_5fold_oof_v1,base + target/statistical encoding,5-fold OOF,NaN,82.907608,LightGBM with leakage-safe target/statistical ...
3,Blend,stack_ridge_saved_predictions,saved prediction stack,5-fold OOF,NaN,87.976479,ridge stack over saved predictions
4,Blend,long_lightgbm_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,88.966368,w_extratrees=0.319; w_lgbm_100k_lr02=0.681
5,Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
6,Boosting,lgbm_t03_base_5fold_oof_long100k_lr02,base,5-fold OOF,NaN,93.726219,long selected LightGBM artifact
7,Boosting,lgbm_t03_base_5fold_oof_long80k_lr03,base,5-fold OOF,NaN,94.163485,long selected LightGBM artifact
8,Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,lgbm_t03_base_5fold_oof; mean_best_iteration=1...
9,Boosting,lightgbm_oof_base,base,5-fold OOF,NaN,103.251549,lgbm_target_l95_child60_l2_5; mean_best_iterat...


In [304]:
submission_tracker_path = RESULTS_DIR / "submission_tracker.csv"

final_accounting_public_checkpoint = pd.DataFrame([
    {
        "file": FINAL_ACCOUNTING_SUBMISSION_PATH.name,
        "model_family": "Accounting solver",
        "public_mse": 31.361,
        "notes": (
            "Constrained accounting solver using the verified convex ensemble prior. "
            "This corresponds to the clean version of the successful convex-prior accounting submission."
        ),
    }
])

if submission_tracker_path.exists():
    submission_tracker = pd.read_csv(submission_tracker_path)

    if "file" not in submission_tracker.columns and "submission_file" in submission_tracker.columns:
        submission_tracker = submission_tracker.rename(columns={"submission_file": "file"})

    submission_tracker = pd.concat(
        [submission_tracker, final_accounting_public_checkpoint],
        ignore_index=True,
    )

else:
    submission_tracker = final_accounting_public_checkpoint.copy()

submission_tracker = (
    submission_tracker
    .drop_duplicates(subset=["file"], keep="last")
    .reset_index(drop=True)
)

submission_tracker.to_csv(
    submission_tracker_path,
    index=False,
)

submission_tracker.tail(10)

,file,model_family,public_mse,notes
0,submission_blend_auto_et_lgbm_long_oof_weighte...,Blend,80.662,ExtraTrees + long 100k LightGBM blend
1,submission_lgbm_te_base_5fold_oof_v1_foldavg.csv,Target/statistical-encoded boosting,70.997,LightGBM on base features plus leakage-safe ta...
2,submission_blend_te_verified_noresid_weighted.csv,Verified non-residual TE blend,69.689,"Convex blend of RF, ExtraTrees, base LightGBM,..."
3,submission_accounting_solver_convex_te_blend_p...,Accounting solver,31.361,Constrained accounting solver using the verifi...


In [305]:
accounting_solver_diagnostic = pd.DataFrame([
    {
        "check": "Accounting priors loaded",
        "value": ACCOUNTING_PRIORS_READY,
        "expected": True,
        "pass": bool(ACCOUNTING_PRIORS_READY),
    },
    {
        "check": "Final prior name",
        "value": FINAL_ACCOUNTING_PRIOR_NAME,
        "expected": "convex_te_blend_prior",
        "pass": bool(FINAL_ACCOUNTING_PRIOR_NAME == "convex_te_blend_prior"),
    },
    {
        "check": "Convex prior base OOF MSE",
        "value": final_accounting_base_oof_mse,
        "expected": "about 78.0715",
        "pass": bool(abs(final_accounting_base_oof_mse - 78.0715) < 0.10),
    },
    {
        "check": "Final accounting OOF MSE",
        "value": final_accounting_oof_mse,
        "expected": "about 54.2930",
        "pass": bool(final_accounting_oof_mse < 54.40),
    },
    {
        "check": "Equation weight",
        "value": float(final_accounting_row["equation_weight"]),
        "expected": 100000.0,
        "pass": bool(float(final_accounting_row["equation_weight"]) == 100000.0),
    },
    {
        "check": "Solver shrinkage",
        "value": float(final_accounting_row["lambda_solver"]),
        "expected": 0.932,
        "pass": bool(abs(float(final_accounting_row["lambda_solver"]) - 0.932) < 1e-9),
    },
    {
        "check": "OOF covered rows",
        "value": int(final_accounting_row["oof_covered_rows"]),
        "expected": 81593,
        "pass": bool(int(final_accounting_row["oof_covered_rows"]) == 81593),
    },
    {
        "check": "Test covered rows",
        "value": int(final_accounting_row["test_covered_rows"]),
        "expected": 45105,
        "pass": bool(int(final_accounting_row["test_covered_rows"]) == 45105),
    },
    {
        "check": "Final submission rows",
        "value": len(final_accounting_submission),
        "expected": n_test_accounting,
        "pass": bool(len(final_accounting_submission) == n_test_accounting),
    },
    {
        "check": "Final submission valid",
        "value": ACCOUNTING_SOLVER_FINAL_PASS,
        "expected": True,
        "pass": bool(ACCOUNTING_SOLVER_FINAL_PASS),
    },
])

accounting_solver_diagnostic

,check,value,expected,pass
0,Accounting priors loaded,True,True,True
1,Final prior name,convex_te_blend_prior,convex_te_blend_prior,True
2,Convex prior base OOF MSE,78.07151,about 78.0715,True
3,Final accounting OOF MSE,54.292981,about 54.2930,True
4,Equation weight,100000.0,100000.0,True
5,Solver shrinkage,0.932,0.932,True
6,OOF covered rows,81593,81593,True
7,Test covered rows,45105,45105,True
8,Final submission rows,48307,48307,True
9,Final submission valid,True,True,True


#### 7D. Final accounting model selection

The final report model is the constrained accounting solver using the verified convex ensemble prior.

This model is selected because it combines two interpretable pieces:

- a convex machine-learning prior from the strongest verified non-residual models,
- a constrained accounting correction based on subgroup count identities.

The target/statistical-encoded LightGBM prior is retained only as a sensitivity check. The final submitted model uses the convex ensemble prior.

In [310]:
FINAL_REPORT_MODEL_NAME = "accounting_solver_convex_te_blend_prior"

FINAL_REPORT_SUBMISSION_SOURCE = Path("submission_accounting_solver_convex_te_blend_prior.csv")
FINAL_REPORT_SUBMISSION_PATH = Path("submission_final_accounting_solver_convex_te_blend_prior.csv")

if not FINAL_REPORT_SUBMISSION_SOURCE.exists():
    raise FileNotFoundError(
        f"Expected final accounting submission not found: {FINAL_REPORT_SUBMISSION_SOURCE}"
    )

final_report_submission = pd.read_csv(FINAL_REPORT_SUBMISSION_SOURCE)

if final_report_submission.shape != (n_test_accounting, 2):
    raise ValueError(
        f"Final submission has wrong shape: {final_report_submission.shape}"
    )

if list(final_report_submission.columns) != [ID_COL, TARGET_COL]:
    raise ValueError(
        f"Final submission has wrong columns: {list(final_report_submission.columns)}"
    )

if final_report_submission[ID_COL].isna().any():
    raise ValueError("Final submission contains missing assessment IDs.")

if final_report_submission[TARGET_COL].isna().any():
    raise ValueError("Final submission contains missing predictions.")

if not np.isfinite(final_report_submission[TARGET_COL]).all():
    raise ValueError("Final submission contains non-finite predictions.")

if not final_report_submission[TARGET_COL].between(0, 100).all():
    raise ValueError("Final submission contains predictions outside [0, 100].")

final_report_submission.to_csv(
    FINAL_REPORT_SUBMISSION_PATH,
    index=False,
)

FINAL_REPORT_SUBMISSION_READY = True

print("FINAL_REPORT_SUBMISSION_READY:", FINAL_REPORT_SUBMISSION_READY)
print("Final report model:", FINAL_REPORT_MODEL_NAME)
print("Source submission:", FINAL_REPORT_SUBMISSION_SOURCE)
print("Final named submission copy:", FINAL_REPORT_SUBMISSION_PATH)

final_report_submission.head()

FINAL_REPORT_SUBMISSION_READY: True
Final report model: accounting_solver_convex_te_blend_prior
Source submission: submission_accounting_solver_convex_te_blend_prior.csv
Final named submission copy: submission_final_accounting_solver_convex_te_blend_prior.csv


,ASSESSMENT_ID,PERCENT_PROFICIENT
0,8af5e0382a81,78.981340
1,e1591bf8db41,39.006866
2,547ec44dcea6,35.748108
3,0e200399fc40,59.727604
4,c2c40438dac7,82.227130


In [311]:
final_model_summary = pd.DataFrame([
    {
        "stage": "Part 6 prior",
        "model": "Verified convex ensemble prior",
        "oof_mse": final_accounting_base_oof_mse,
        "public_mse": 69.689,
        "notes": (
            "Convex blend of target/statistical-encoded LightGBM, long LightGBM, "
            "ExtraTrees, selected LightGBM, and Random Forest."
        ),
    },
    {
        "stage": "Part 7 final model",
        "model": "Constrained accounting solver with convex ensemble prior",
        "oof_mse": final_accounting_oof_mse,
        "public_mse": 31.361,
        "notes": (
            "Final report model. Applies subgroup accounting identities with "
            f"equation_weight={float(final_accounting_row['equation_weight']):.0f} "
            f"and lambda_solver={float(final_accounting_row['lambda_solver']):.3f}."
        ),
    },
])

final_model_summary

,stage,model,oof_mse,public_mse,notes
0,Part 6 prior,Verified convex ensemble prior,78.071510,69.689,Convex blend of target/statistical-encoded Lig...
1,Part 7 final model,Constrained accounting solver with convex ense...,54.292981,31.361,Final report model. Applies subgroup accountin...


# Evaluation and final submission

The final selected submission is the constrained accounting solver with the verified convex ensemble prior.

The final submission file is:

`submission_final_accounting_solver_convex_te_blend_prior.csv`

This file contains only the required competition columns: `ASSESSMENT_ID` and `PERCENT_PROFICIENT`.

In [315]:
if "submission_tracker" in globals():
    submission_tracker = pd.read_csv(submission_tracker_path)

    if "file" not in submission_tracker.columns and "submission_file" in submission_tracker.columns:
        submission_tracker = submission_tracker.rename(columns={"submission_file": "file"})

else:
    submission_tracker_path = RESULTS_DIR / "submission_tracker.csv"

    if submission_tracker_path.exists():
        submission_tracker = pd.read_csv(submission_tracker_path)

        if "file" not in submission_tracker.columns and "submission_file" in submission_tracker.columns:
            submission_tracker = submission_tracker.rename(columns={"submission_file": "file"})

    else:
        submission_tracker = pd.DataFrame(columns=["file", "model_family", "public_mse", "notes"])

final_submission_tracker_row = pd.DataFrame([
    {
        "file": FINAL_REPORT_SUBMISSION_PATH.name,
        "model_family": "Final report model",
        "public_mse": 31.361,
        "notes": (
            "Constrained accounting solver using the verified convex ensemble prior. "
            "Clean final notebook submission copy."
        ),
    }
])

submission_tracker = pd.concat(
    [submission_tracker, final_submission_tracker_row],
    ignore_index=True,
)

submission_tracker = (
    submission_tracker
    .drop_duplicates(subset=["file"], keep="last")
    .reset_index(drop=True)
)

submission_tracker.to_csv(
    submission_tracker_path,
    index=False,
)

submission_tracker.sort_values("public_mse").head(15)

,file,model_family,public_mse,notes
3,submission_accounting_solver_convex_te_blend_p...,Accounting solver,31.361,Constrained accounting solver using the verifi...
4,submission_final_accounting_solver_convex_te_b...,Final report model,31.361,Constrained accounting solver using the verifi...
2,submission_blend_te_verified_noresid_weighted.csv,Verified non-residual TE blend,69.689,"Convex blend of RF, ExtraTrees, base LightGBM,..."
1,submission_lgbm_te_base_5fold_oof_v1_foldavg.csv,Target/statistical-encoded boosting,70.997,LightGBM on base features plus leakage-safe ta...
0,submission_blend_auto_et_lgbm_long_oof_weighte...,Blend,80.662,ExtraTrees + long 100k LightGBM blend


# Evaluation and final model review

The final selected submission is the constrained accounting solver with the verified convex ensemble prior.

The final submission file is:

`submission_final_accounting_solver_convex_te_blend_prior.csv`

This file contains only the required competition columns: `ASSESSMENT_ID` and `PERCENT_PROFICIENT`.

The table below reviews the main models developed in the notebook. It reports the available internal validation or OOF score, public checkpoint score when available, and the role each model played in the final workflow.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Recovery cell:
# This rebuilds the final review table from the original model_scoreboard.
# It does not retrain models and does not delete model artifacts.

if "RESULTS_DIR" not in globals():
    RESULTS_DIR = Path("model_results")

FINAL_SCOREBOARD_PATH = RESULTS_DIR / "final_model_review_scoreboard.csv"
RESTORED_SCOREBOARD_PATH = RESULTS_DIR / "final_model_review_scoreboard_restored_full.csv"

if "model_scoreboard" not in globals():
    raise ValueError(
        "model_scoreboard is not available in memory. "
        "Do not use the collapsed final_model_review_scoreboard as the source. "
        "Restarting/rerunning the notebook up to the final scoreboard section will recreate model_scoreboard."
    )

scoreboard_required_columns = [
    "section",
    "model_family",
    "model",
    "feature_space",
    "metric_source",
    "train_mse",
    "validation_or_oof_mse",
    "public_mse",
    "submission_file",
    "selected_final_model",
    "notes",
]

restored_scoreboard = model_scoreboard.copy()

if "rank_by_internal_score" in restored_scoreboard.columns:
    restored_scoreboard = restored_scoreboard.drop(columns=["rank_by_internal_score"])

if "val_mse" in restored_scoreboard.columns:
    restored_scoreboard = restored_scoreboard.rename(
        columns={"val_mse": "validation_or_oof_mse"}
    )

if "valid_mse" in restored_scoreboard.columns:
    restored_scoreboard = restored_scoreboard.rename(
        columns={"valid_mse": "validation_or_oof_mse"}
    )

for col in scoreboard_required_columns:
    if col not in restored_scoreboard.columns:
        if col in ["train_mse", "validation_or_oof_mse", "public_mse"]:
            restored_scoreboard[col] = np.nan
        elif col == "selected_final_model":
            restored_scoreboard[col] = False
        else:
            restored_scoreboard[col] = ""

restored_scoreboard = restored_scoreboard[scoreboard_required_columns].copy()

canonical_late_stage_models = [
    "lgbm_te_base_5fold_oof_v1",
    "blend_te_verified_noresid_weighted",
    "accounting_solver_te_lgbm_prior",
    "accounting_solver_convex_te_blend_prior",
]

# Remove only old versions of the late-stage canonical rows.
# This does NOT remove polynomial/interactions variants.
restored_scoreboard = restored_scoreboard[
    ~restored_scoreboard["model"].astype(str).isin(canonical_late_stage_models)
].copy()


def infer_model_section(row):
    section = str(row.get("section", "")).strip()

    if section:
        return section

    family = str(row.get("model_family", "")).strip()
    model = str(row.get("model", "")).strip()

    if family == "Linear":
        if model == "linear_regression":
            return "Data modelling, Part 1"

        return "Data modelling, Part 2"

    if family == "Step functions":
        return "Data modelling, Part 3"

    if family in ["Tree", "Bagging"]:
        return "Data modelling, Part 4"

    if family == "Boosting":
        return "Data modelling, Part 5"

    if family in ["Target/statistical-encoded boosting", "Blend"]:
        return "Data modelling, Part 6"

    if family == "Accounting solver":
        return "Data modelling, Part 7"

    return section


restored_scoreboard["section"] = restored_scoreboard.apply(
    infer_model_section,
    axis=1,
)

te_accounting_mse = np.nan

if "accounting_solver_summary" in globals():
    if "te_lgbm_prior" in set(accounting_solver_summary["prior_name"]):
        te_accounting_mse = (
            accounting_solver_summary
            .loc[
                accounting_solver_summary["prior_name"] == "te_lgbm_prior",
                "best_accounting_oof_mse",
            ]
            .iloc[0]
        )

canonical_rows = pd.DataFrame([
    {
        "section": "Data modelling, Part 6",
        "model_family": "Target/statistical-encoded boosting",
        "model": "lgbm_te_base_5fold_oof_v1",
        "feature_space": "Base features plus leakage-safe target/statistical encodings",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": globals().get("te_lgbm_oof_mse_clipped", np.nan),
        "public_mse": 70.997,
        "submission_file": "submission_lgbm_te_base_5fold_oof_v1_foldavg.csv",
        "selected_final_model": False,
        "notes": "Strongest individual LightGBM model; used inside the convex ensemble prior.",
    },
    {
        "section": "Data modelling, Part 6",
        "model_family": "Blend",
        "model": "blend_te_verified_noresid_weighted",
        "feature_space": "Convex blend of verified non-residual OOF predictions",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": globals().get("safe_te_blend_oof_mse", np.nan),
        "public_mse": 69.689,
        "submission_file": "submission_blend_te_verified_noresid_weighted.csv",
        "selected_final_model": False,
        "notes": (
            "Convex ensemble prior using TE LightGBM, long LightGBM models, "
            "ExtraTrees, selected LightGBM, and Random Forest."
        ),
    },
    {
        "section": "Data modelling, Part 7",
        "model_family": "Accounting solver",
        "model": "accounting_solver_te_lgbm_prior",
        "feature_space": "TE LightGBM prior plus subgroup accounting constraints",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": te_accounting_mse,
        "public_mse": np.nan,
        "submission_file": "submission_accounting_solver_te_lgbm_prior.csv",
        "selected_final_model": False,
        "notes": "Sensitivity check showing that the accounting solver improves even a single TE LightGBM prior.",
    },
    {
        "section": "Data modelling, Part 7",
        "model_family": "Accounting solver",
        "model": "accounting_solver_convex_te_blend_prior",
        "feature_space": "Convex ensemble prior plus subgroup accounting constraints",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": globals().get("final_accounting_oof_mse", np.nan),
        "public_mse": 31.361,
        "submission_file": "submission_final_accounting_solver_convex_te_blend_prior.csv",
        "selected_final_model": True,
        "notes": (
            "Final report model. Uses the verified convex ensemble prior and applies "
            "the constrained subgroup accounting solver."
        ),
    },
])

restored_scoreboard = pd.concat(
    [restored_scoreboard, canonical_rows],
    ignore_index=True,
)

restored_scoreboard["validation_or_oof_mse"] = pd.to_numeric(
    restored_scoreboard["validation_or_oof_mse"],
    errors="coerce",
)

restored_scoreboard["public_mse"] = pd.to_numeric(
    restored_scoreboard["public_mse"],
    errors="coerce",
)

restored_scoreboard["selected_final_model"] = (
    restored_scoreboard["selected_final_model"]
    .fillna(False)
    .astype(bool)
)

# Remove only exact repeated rows from rerunning this cell.
# This keeps different feature spaces for the same model name.
restored_scoreboard = (
    restored_scoreboard
    .drop_duplicates(
        subset=[
            "section",
            "model_family",
            "model",
            "feature_space",
            "metric_source",
            "validation_or_oof_mse",
            "notes",
        ],
        keep="last",
    )
    .sort_values(
        ["validation_or_oof_mse", "public_mse"],
        na_position="last",
    )
    .reset_index(drop=True)
)

restored_scoreboard.insert(
    0,
    "rank_by_internal_score",
    np.arange(1, len(restored_scoreboard) + 1),
)

final_model_review_scoreboard = restored_scoreboard.copy()

final_model_review_scoreboard.to_csv(
    RESTORED_SCOREBOARD_PATH,
    index=False,
)

final_model_review_scoreboard.to_csv(
    FINAL_SCOREBOARD_PATH,
    index=False,
)

linear_feature_space_check = (
    final_model_review_scoreboard
    .loc[
        final_model_review_scoreboard["model_family"] == "Linear",
        ["model", "feature_space", "validation_or_oof_mse", "section"]
    ]
    .sort_values(["model", "feature_space"])
    .reset_index(drop=True)
)

late_stage_duplicate_counts = (
    final_model_review_scoreboard
    .loc[
        final_model_review_scoreboard["model"].isin(canonical_late_stage_models),
        "model",
    ]
    .value_counts()
    .reindex(canonical_late_stage_models, fill_value=0)
)

final_scoreboard_recovery_check = pd.DataFrame([
    {
        "check": "Restored scoreboard saved",
        "value": RESTORED_SCOREBOARD_PATH.exists(),
        "expected": True,
        "pass": bool(RESTORED_SCOREBOARD_PATH.exists()),
    },
    {
        "check": "Final scoreboard overwritten with restored full version",
        "value": FINAL_SCOREBOARD_PATH.exists(),
        "expected": True,
        "pass": bool(FINAL_SCOREBOARD_PATH.exists()),
    },
    {
        "check": "Linear feature-space rows retained",
        "value": len(linear_feature_space_check),
        "expected": "more than collapsed version",
        "pass": bool(len(linear_feature_space_check) > 15),
    },
    {
        "check": "Polynomial/interactions variants retained",
        "value": bool(
            final_model_review_scoreboard["feature_space"]
            .astype(str)
            .str.contains("poly|interaction", case=False, regex=True)
            .any()
        ),
        "expected": True,
        "pass": bool(
            final_model_review_scoreboard["feature_space"]
            .astype(str)
            .str.contains("poly|interaction", case=False, regex=True)
            .any()
        ),
    },
    {
        "check": "No duplicated late-stage canonical models",
        "value": bool((late_stage_duplicate_counts == 1).all()),
        "expected": True,
        "pass": bool((late_stage_duplicate_counts == 1).all()),
    },
    {
        "check": "Exactly one selected final model",
        "value": int(final_model_review_scoreboard["selected_final_model"].sum()),
        "expected": 1,
        "pass": bool(int(final_model_review_scoreboard["selected_final_model"].sum()) == 1),
    },
])



In [321]:
from pathlib import Path

import numpy as np
import pandas as pd

# Final review ordering cell:
# Builds the final model review table in notebook workflow order:
# newest model at the top, oldest model at the bottom.

if "RESULTS_DIR" not in globals():
    RESULTS_DIR = Path("model_results")

FINAL_SCOREBOARD_PATH = RESULTS_DIR / "final_model_review_scoreboard.csv"
FINAL_WORKFLOW_SCOREBOARD_PATH = RESULTS_DIR / "final_model_review_scoreboard_workflow_order.csv"

scoreboard_required_columns = [
    "section",
    "model_family",
    "model",
    "feature_space",
    "metric_source",
    "train_mse",
    "validation_or_oof_mse",
    "public_mse",
    "submission_file",
    "selected_final_model",
    "notes",
]

if "model_scoreboard" in globals():
    workflow_scoreboard = model_scoreboard.copy()
else:
    raise ValueError(
        "model_scoreboard is not available. "
        "Do not build the final workflow table from the damaged/collapsed scoreboard. "
        "Rerun the notebook up to the final review section so model_scoreboard exists."
    )

if "rank_by_internal_score" in workflow_scoreboard.columns:
    workflow_scoreboard = workflow_scoreboard.drop(columns=["rank_by_internal_score"])

if "val_mse" in workflow_scoreboard.columns:
    workflow_scoreboard = workflow_scoreboard.rename(
        columns={"val_mse": "validation_or_oof_mse"}
    )

if "valid_mse" in workflow_scoreboard.columns:
    workflow_scoreboard = workflow_scoreboard.rename(
        columns={"valid_mse": "validation_or_oof_mse"}
    )

for col in scoreboard_required_columns:
    if col not in workflow_scoreboard.columns:
        if col in ["train_mse", "validation_or_oof_mse", "public_mse"]:
            workflow_scoreboard[col] = np.nan
        elif col == "selected_final_model":
            workflow_scoreboard[col] = False
        else:
            workflow_scoreboard[col] = ""

workflow_scoreboard = workflow_scoreboard[scoreboard_required_columns].copy()

canonical_late_stage_models = [
    "lgbm_te_base_5fold_oof_v1",
    "blend_te_verified_noresid_weighted",
    "accounting_solver_te_lgbm_prior",
    "accounting_solver_convex_te_blend_prior",
]

# Remove only older duplicate versions of the late-stage canonical rows.
# This does not remove polynomial, interaction, or feature-space variants.
workflow_scoreboard = workflow_scoreboard[
    ~workflow_scoreboard["model"].astype(str).isin(canonical_late_stage_models)
].copy()

te_accounting_mse = np.nan

if "accounting_solver_summary" in globals():
    if "te_lgbm_prior" in set(accounting_solver_summary["prior_name"]):
        te_accounting_mse = (
            accounting_solver_summary
            .loc[
                accounting_solver_summary["prior_name"] == "te_lgbm_prior",
                "best_accounting_oof_mse",
            ]
            .iloc[0]
        )

canonical_rows = pd.DataFrame([
    {
        "section": "Data modelling, Part 6",
        "model_family": "Target/statistical-encoded boosting",
        "model": "lgbm_te_base_5fold_oof_v1",
        "feature_space": "Base features plus leakage-safe target/statistical encodings",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": globals().get("te_lgbm_oof_mse_clipped", np.nan),
        "public_mse": 70.997,
        "submission_file": "submission_lgbm_te_base_5fold_oof_v1_foldavg.csv",
        "selected_final_model": False,
        "notes": "Strongest individual LightGBM model; used inside the convex ensemble prior.",
    },
    {
        "section": "Data modelling, Part 6",
        "model_family": "Blend",
        "model": "blend_te_verified_noresid_weighted",
        "feature_space": "Convex blend of verified non-residual OOF predictions",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": globals().get("safe_te_blend_oof_mse", np.nan),
        "public_mse": 69.689,
        "submission_file": "submission_blend_te_verified_noresid_weighted.csv",
        "selected_final_model": False,
        "notes": (
            "Convex ensemble prior using TE LightGBM, long LightGBM models, "
            "ExtraTrees, selected LightGBM, and Random Forest."
        ),
    },
    {
        "section": "Data modelling, Part 7",
        "model_family": "Accounting solver",
        "model": "accounting_solver_te_lgbm_prior",
        "feature_space": "TE LightGBM prior plus subgroup accounting constraints",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": te_accounting_mse,
        "public_mse": np.nan,
        "submission_file": "submission_accounting_solver_te_lgbm_prior.csv",
        "selected_final_model": False,
        "notes": "Sensitivity check showing that the accounting solver improves even a single TE LightGBM prior.",
    },
    {
        "section": "Data modelling, Part 7",
        "model_family": "Accounting solver",
        "model": "accounting_solver_convex_te_blend_prior",
        "feature_space": "Convex ensemble prior plus subgroup accounting constraints",
        "metric_source": "5-fold OOF",
        "train_mse": np.nan,
        "validation_or_oof_mse": globals().get("final_accounting_oof_mse", np.nan),
        "public_mse": 31.361,
        "submission_file": "submission_final_accounting_solver_convex_te_blend_prior.csv",
        "selected_final_model": True,
        "notes": (
            "Final report model. Uses the verified convex ensemble prior and applies "
            "the constrained subgroup accounting solver."
        ),
    },
])

workflow_scoreboard = pd.concat(
    [workflow_scoreboard, canonical_rows],
    ignore_index=True,
)

workflow_scoreboard["validation_or_oof_mse"] = pd.to_numeric(
    workflow_scoreboard["validation_or_oof_mse"],
    errors="coerce",
)

workflow_scoreboard["public_mse"] = pd.to_numeric(
    workflow_scoreboard["public_mse"],
    errors="coerce",
)

workflow_scoreboard["selected_final_model"] = (
    workflow_scoreboard["selected_final_model"]
    .fillna(False)
    .astype(bool)
)


def infer_workflow_section(row):
    section = str(row["section"]).strip()
    family = str(row["model_family"]).strip()
    model = str(row["model"]).strip()
    feature_space = str(row["feature_space"]).strip()

    if model in [
        "accounting_solver_convex_te_blend_prior",
        "accounting_solver_te_lgbm_prior",
    ]:
        return "Data modelling, Part 7"

    if model in [
        "blend_te_verified_noresid_weighted",
        "lgbm_te_base_5fold_oof_v1",
    ]:
        return "Data modelling, Part 6"

    if model in [
        "stack_ridge_saved_predictions",
        "long_lightgbm_extratrees_blend",
        "rf_extratrees_lightgbm_blend",
        "lgbm_t03_base_5fold_oof_long100k_lr02",
        "lgbm_t03_base_5fold_oof_long80k_lr03",
        "lightgbm_selected_12k_oof_base",
        "lightgbm_oof_base",
        "lightgbm_targeted_holdout_refinement",
        "lightgbm_broad_holdout_screen",
    ]:
        return "Data modelling, Part 5"

    if family in ["Tree", "Bagging"]:
        return "Data modelling, Part 4"

    if family == "Step functions":
        return "Data modelling, Part 3"

    if family == "Boosting":
        return "Data modelling, Part 5"

    if family == "Accounting solver":
        return "Data modelling, Part 7"

    if family in ["Target/statistical-encoded boosting", "Blend"]:
        if model == "blend_te_verified_noresid_weighted":
            return "Data modelling, Part 6"
        return section if section else "Data modelling, Part 5"

    if family == "Linear":
        if model == "linear_regression":
            return "Data modelling, Part 1"

        return "Data modelling, Part 2"

    return section


workflow_scoreboard["section"] = workflow_scoreboard.apply(
    infer_workflow_section,
    axis=1,
)


def section_number(section):
    section = str(section)

    for part in range(1, 8):
        if f"Part {part}" in section:
            return part

    return 0


def feature_order(feature_space):
    feature_space = str(feature_space)

    if "base_plus_poly_interactions_plus_steps" in feature_space:
        return 90

    if "base_plus_poly_interactions" in feature_space:
        return 80

    if "base_plus_poly" in feature_space:
        return 70

    if "base_plus_interactions" in feature_space:
        return 60

    if "base_plus_top5_interactions" in feature_space:
        return 50

    if "top10_degree2_polynomial" in feature_space:
        return 40

    if feature_space == "base":
        return 10

    return 20


specific_model_order = {
    # Part 7
    "accounting_solver_convex_te_blend_prior": 790,
    "accounting_solver_te_lgbm_prior": 780,

    # Part 6
    "blend_te_verified_noresid_weighted": 690,
    "lgbm_te_base_5fold_oof_v1": 680,

    # Part 5
    "stack_ridge_saved_predictions": 590,
    "long_lightgbm_extratrees_blend": 580,
    "rf_extratrees_lightgbm_blend": 570,
    "lgbm_t03_base_5fold_oof_long100k_lr02": 560,
    "lgbm_t03_base_5fold_oof_long80k_lr03": 550,
    "lightgbm_selected_12k_oof_base": 540,
    "lightgbm_oof_base": 530,
    "lightgbm_targeted_holdout_refinement": 520,
    "lightgbm_broad_holdout_screen": 510,

    # Part 4
    "rf_extratrees_blend": 490,
    "extratrees_oof_base": 480,
    "extratrees_holdout_screen": 470,
    "random_forest_oof_500": 460,
    "random_forest_refined": 450,
    "random_forest_screen": 440,
    "regression_tree_holdout_screen": 430,
    "regression_tree_cv": 420,

    # Part 3
    "lasso_with_steps": 390,
    "ridge_with_steps": 388,
    "focused_expanded_linear_regression_with_steps": 380,
    "expanded_linear_regression_with_steps": 370,
    "linear_regression_with_steps": 360,
}

linear_method_order = {
    "hybrid_stepwise": 250,
    "forward_stepwise": 240,
    "backward_stepwise": 230,
    "elastic_net": 220,
    "lasso": 210,
    "ridge": 200,
    "linear_regression": 100,
}


def workflow_sort_value(row):
    section_part = section_number(row["section"])
    model = str(row["model"])
    family = str(row["model_family"])
    feature_space = str(row["feature_space"])

    if model in specific_model_order:
        return section_part * 1000 + specific_model_order[model]

    if family == "Linear":
        return (
            section_part * 1000
            + linear_method_order.get(model, 100)
            + feature_order(feature_space)
        )

    return section_part * 1000 + feature_order(feature_space)


workflow_scoreboard["workflow_sort_value"] = workflow_scoreboard.apply(
    workflow_sort_value,
    axis=1,
)

# Remove only exact rerun duplicates. This preserves different feature spaces
# for the same model name.
workflow_scoreboard = (
    workflow_scoreboard
    .drop_duplicates(
        subset=[
            "section",
            "model_family",
            "model",
            "feature_space",
            "metric_source",
            "validation_or_oof_mse",
            "notes",
        ],
        keep="last",
    )
    .sort_values(
        ["workflow_sort_value", "validation_or_oof_mse"],
        ascending=[False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

workflow_scoreboard.insert(
    0,
    "review_order_newest_to_oldest",
    np.arange(1, len(workflow_scoreboard) + 1),
)

workflow_scoreboard = workflow_scoreboard.drop(columns=["workflow_sort_value"])

final_model_review_scoreboard = workflow_scoreboard.copy()

final_model_review_scoreboard.to_csv(
    FINAL_WORKFLOW_SCOREBOARD_PATH,
    index=False,
)

final_model_review_scoreboard.to_csv(
    FINAL_SCOREBOARD_PATH,
    index=False,
)

late_stage_models = [
    "accounting_solver_convex_te_blend_prior",
    "accounting_solver_te_lgbm_prior",
    "blend_te_verified_noresid_weighted",
    "lgbm_te_base_5fold_oof_v1",
]

late_stage_duplicate_counts = (
    final_model_review_scoreboard
    .loc[
        final_model_review_scoreboard["model"].isin(late_stage_models),
        "model",
    ]
    .value_counts()
    .reindex(late_stage_models, fill_value=0)
)

final_workflow_order_check = pd.DataFrame([
    {
        "check": "Workflow scoreboard saved",
        "value": FINAL_WORKFLOW_SCOREBOARD_PATH.exists(),
        "expected": True,
        "pass": bool(FINAL_WORKFLOW_SCOREBOARD_PATH.exists()),
    },
    {
        "check": "Newest model is first row",
        "value": final_model_review_scoreboard.iloc[0]["model"],
        "expected": "accounting_solver_convex_te_blend_prior",
        "pass": bool(final_model_review_scoreboard.iloc[0]["model"] == "accounting_solver_convex_te_blend_prior"),
    },
    {
        "check": "Exactly one selected final model",
        "value": int(final_model_review_scoreboard["selected_final_model"].sum()),
        "expected": 1,
        "pass": bool(int(final_model_review_scoreboard["selected_final_model"].sum()) == 1),
    },
    {
        "check": "No duplicated late-stage models",
        "value": bool((late_stage_duplicate_counts == 1).all()),
        "expected": True,
        "pass": bool((late_stage_duplicate_counts == 1).all()),
    },
    {
        "check": "Polynomial/interactions variants retained",
        "value": bool(
            final_model_review_scoreboard["feature_space"]
            .astype(str)
            .str.contains("poly|interaction", case=False, regex=True)
            .any()
        ),
        "expected": True,
        "pass": bool(
            final_model_review_scoreboard["feature_space"]
            .astype(str)
            .str.contains("poly|interaction", case=False, regex=True)
            .any()
        ),
    },
])

print("Saved workflow-ordered final model review scoreboard:")
print(FINAL_WORKFLOW_SCOREBOARD_PATH)
print(FINAL_SCOREBOARD_PATH)

print("\nWorkflow order check:")
display(final_workflow_order_check)

print("\nFinal model review scoreboard, newest to oldest:")
display(final_model_review_scoreboard)

Saved workflow-ordered final model review scoreboard:
model_results/final_model_review_scoreboard_workflow_order.csv
model_results/final_model_review_scoreboard.csv

Workflow order check:


,check,value,expected,pass
0,Workflow scoreboard saved,True,True,True
1,Newest model is first row,accounting_solver_convex_te_blend_prior,accounting_solver_convex_te_blend_prior,True
2,Exactly one selected final model,1,1,True
3,No duplicated late-stage models,True,True,True
4,Polynomial/interactions variants retained,True,True,True



Final model review scoreboard, newest to oldest:


,review_order_newest_to_oldest,section,model_family,model,feature_space,metric_source,train_mse,validation_or_oof_mse,public_mse,submission_file,selected_final_model,notes
0,1,"Data modelling, Part 7",Accounting solver,accounting_solver_convex_te_blend_prior,Convex ensemble prior plus subgroup accounting...,5-fold OOF,NaN,54.292981,31.361,submission_final_accounting_solver_convex_te_b...,True,Final report model. Uses the verified convex e...
1,2,"Data modelling, Part 7",Accounting solver,accounting_solver_te_lgbm_prior,TE LightGBM prior plus subgroup accounting con...,5-fold OOF,NaN,58.908045,NaN,submission_accounting_solver_te_lgbm_prior.csv,False,Sensitivity check showing that the accounting ...
2,3,"Data modelling, Part 6",Blend,blend_te_verified_noresid_weighted,Convex blend of verified non-residual OOF pred...,5-fold OOF,NaN,78.071510,69.689,submission_blend_te_verified_noresid_weighted.csv,False,"Convex ensemble prior using TE LightGBM, long ..."
3,4,"Data modelling, Part 6",Target/statistical-encoded boosting,lgbm_te_base_5fold_oof_v1,Base features plus leakage-safe target/statist...,5-fold OOF,NaN,82.907608,70.997,submission_lgbm_te_base_5fold_oof_v1_foldavg.csv,False,Strongest individual LightGBM model; used insi...
4,5,"Data modelling, Part 5",Blend,stack_ridge_saved_predictions,saved prediction stack,5-fold OOF,NaN,87.976479,NaN,,False,ridge stack over saved predictions
5,6,"Data modelling, Part 5",Blend,long_lightgbm_extratrees_blend,OOF prediction blend,5-fold OOF,NaN,88.966368,NaN,,False,w_extratrees=0.319; w_lgbm_100k_lr02=0.681
6,7,"Data modelling, Part 5",Blend,rf_extratrees_lightgbm_blend,OOF prediction blend,5-fold OOF,NaN,93.499458,NaN,,False,w_rf=0.000; w_extratrees=0.356; w_lgbm=0.644; ...
7,8,"Data modelling, Part 5",Boosting,lgbm_t03_base_5fold_oof_long100k_lr02,base,5-fold OOF,NaN,93.726219,NaN,,False,long selected LightGBM artifact
8,9,"Data modelling, Part 5",Boosting,lgbm_t03_base_5fold_oof_long80k_lr03,base,5-fold OOF,NaN,94.163485,NaN,,False,long selected LightGBM artifact
9,10,"Data modelling, Part 5",Boosting,lightgbm_selected_12k_oof_base,base,5-fold OOF,NaN,98.779404,NaN,,False,lgbm_t03_base_5fold_oof; mean_best_iteration=1...


# END OF PROJECT